<a href="https://colab.research.google.com/github/mrfriman666/mrfriman666/blob/main/nissan_logger_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 📦 Ячейка 1/5: Окружение V6 (~10 мин)
import os

print("=" * 60)
print("📥 Системные пакеты + Java 17")
print("=" * 60)
!apt-get update -qq
!apt-get install -y -qq curl git unzip xz-utils zip libglu1-mesa openjdk-17-jdk-headless ninja-build cmake > /dev/null

!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java 2>&1 | tail -2

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ['PATH']
!java -version 2>&1 | head -1

print("\n📥 Flutter SDK")
!git clone https://github.com/flutter/flutter.git -b stable --depth 1 /content/flutter 2>/dev/null

os.environ['PATH'] = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE'] = '/content/.pub-cache'
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

!flutter config --no-analytics --no-cli-animations 2>/dev/null
!flutter --disable-telemetry 2>/dev/null

print("\n📥 Android SDK + NDK 28")
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
!mkdir -p /content/android-sdk/cmdline-tools
!unzip -q /tmp/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
!mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest 2>/dev/null || true

os.environ['ANDROID_HOME'] = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/content/android-sdk'
os.environ['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']

!yes | sdkmanager --licenses > /dev/null 2>&1
!sdkmanager "platform-tools" "platforms;android-36" "build-tools;36.0.0" "ndk;28.2.13676358" > /dev/null 2>&1

!flutter config --android-sdk /content/android-sdk 2>/dev/null
!flutter precache --android 2>/dev/null

print("\n✅ Готово!")
!flutter --version

📥 Системные пакеты + Java 17
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/java to provide /usr/bin/java (java) in manual mode
openjdk version "17.0.19" 2026-04-21

📥 Flutter SDK
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  227M  100  227M    0     0   288M      0 --:--:-- --:--:-- --:--:--  288M
Analytics reporting disabled.
Setting "cli-animations" value to "false".

You may need to restart any open editors for them to read new settings.

📥 Android SDK + NDK 28
Setting "android-sdk" value to "/content/android-sdk".

You may need to restart any open editors for them to read new settings.
[1/12] Material Fonts                                              458ms
[2/1

In [ ]:
# @title 🏗️ Ячейка 2/5: Проект V6 + Модели
import os

os.chdir('/content')
!rm -rf /content/nissan_logger_v6
!flutter create --org com.nissanlogger --project-name nissan_logger_v6 nissan_logger_v6
os.chdir('/content/nissan_logger_v6')

for folder in ['models', 'services', 'screens', 'widgets']:
    os.makedirs(f'lib/{folder}', exist_ok=True)

# ============ pubspec.yaml ============
with open('pubspec.yaml', 'w') as f:
    f.write('''name: nissan_logger_v6
description: Nissan X-Trail T30 QR20DE Tuning Logger V6
version: 6.0.0+1
publish_to: none

environment:
  sdk: ">=3.0.0 <4.0.0"
  flutter: ">=3.10.0"

dependencies:
  flutter:
    sdk: flutter
  cupertino_icons: ^1.0.6
  fl_chart: 0.68.0
  flutter_bluetooth_serial: 0.4.0
  permission_handler: 11.3.1
  path_provider: 2.1.4
  path: ^1.9.0
  csv: 6.0.0
  shared_preferences: 2.3.2
  file_picker: 8.1.2
  share_plus: 10.0.2
  intl: ^0.19.0
  uuid: 4.5.0
  vibration: 2.0.0
  math_expressions: 2.5.0

dev_dependencies:
  flutter_test:
    sdk: flutter
  flutter_lints: ^4.0.0

flutter:
  uses-material-design: true
''')
print("✅ pubspec.yaml")

# ============ AndroidManifest.xml ============
with open('android/app/src/main/AndroidManifest.xml', 'w') as f:
    f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" />
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" />
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN"
        android:usesPermissionFlags="neverForLocation" />
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT" />
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" />
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" />
    <uses-permission android:name="android.permission.WRITE_EXTERNAL_STORAGE"
        android:maxSdkVersion="28" />
    <uses-permission android:name="android.permission.READ_EXTERNAL_STORAGE" />
    <uses-permission android:name="android.permission.VIBRATE" />
    <uses-permission android:name="android.permission.WAKE_LOCK" />
    <uses-permission android:name="android.permission.INTERNET" />
    <application
        android:label="Nissan Logger V6"
        android:name="${applicationName}"
        android:icon="@mipmap/ic_launcher"
        android:usesCleartextTraffic="true"
        android:requestLegacyExternalStorage="true"
        android:allowBackup="true">
        <activity
            android:name=".MainActivity"
            android:exported="true"
            android:launchMode="singleTop"
            android:theme="@style/LaunchTheme"
            android:configChanges="orientation|keyboardHidden|keyboard|screenSize|smallestScreenSize|locale|layoutDirection|fontScale|screenLayout|density|uiMode"
            android:hardwareAccelerated="true"
            android:windowSoftInputMode="adjustResize">
            <meta-data android:name="io.flutter.embedding.android.NormalTheme"
                android:resource="@style/NormalTheme" />
            <intent-filter>
                <action android:name="android.intent.action.MAIN"/>
                <category android:name="android.intent.category.LAUNCHER"/>
            </intent-filter>
        </activity>
        <meta-data android:name="flutterEmbedding" android:value="2" />
    </application>
</manifest>
''')
print("✅ AndroidManifest.xml")

# ============ MainActivity.kt ============
main_dir = 'android/app/src/main/kotlin/com/nissanlogger/logger_v6'
os.makedirs(main_dir, exist_ok=True)
!rm -rf android/app/src/main/java

with open(f'{main_dir}/MainActivity.kt', 'w') as f:
    f.write('''package com.nissanlogger.logger_v6
import android.os.Bundle
import android.view.WindowManager
import io.flutter.embedding.android.FlutterActivity
class MainActivity : FlutterActivity() {
    override fun onCreate(savedInstanceState: Bundle?) {
        super.onCreate(savedInstanceState)
        window.addFlags(WindowManager.LayoutParams.FLAG_KEEP_SCREEN_ON)
    }
}
''')
print("✅ MainActivity.kt")

# ============ Gradle files ============
with open('android/gradle/wrapper/gradle-wrapper.properties', 'w') as f:
    f.write('''distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\\://services.gradle.org/distributions/gradle-8.10-all.zip
''')

with open('android/settings.gradle.kts', 'w') as f:
    f.write('''pluginManagement {
    val flutterSdkPath = run {
        val properties = java.util.Properties()
        file("local.properties").inputStream().use { properties.load(it) }
        val flutterSdkPath = properties.getProperty("flutter.sdk")
        require(flutterSdkPath != null) { "flutter.sdk not set in local.properties" }
        flutterSdkPath
    }
    includeBuild("$flutterSdkPath/packages/flutter_tools/gradle")
    repositories { google(); mavenCentral(); gradlePluginPortal() }
}
plugins {
    id("dev.flutter.flutter-plugin-loader") version "1.0.0"
    id("com.android.application") version "8.6.0" apply false
    id("org.jetbrains.kotlin.android") version "1.9.24" apply false
}
include(":app")
''')

with open('android/build.gradle.kts', 'w') as f:
    f.write('''allprojects {
    repositories { google(); mavenCentral() }
    configurations.all {
        resolutionStrategy {
            force("androidx.core:core:1.13.1")
            force("androidx.core:core-ktx:1.13.1")
            force("androidx.appcompat:appcompat:1.7.0")
            force("androidx.annotation:annotation:1.8.2")
        }
    }
}
val newBuildDir: Directory = rootProject.layout.buildDirectory.dir("../../build").get()
rootProject.layout.buildDirectory.value(newBuildDir)
subprojects {
    val newSubprojectBuildDir: Directory = newBuildDir.dir(project.name)
    project.layout.buildDirectory.value(newSubprojectBuildDir)
}
subprojects { project.evaluationDependsOn(":app") }
tasks.register<Delete>("clean") { delete(rootProject.layout.buildDirectory) }
''')

with open('android/app/build.gradle.kts', 'w') as f:
    f.write('''plugins {
    id("com.android.application")
    id("kotlin-android")
    id("dev.flutter.flutter-gradle-plugin")
}
android {
    namespace = "com.nissanlogger.logger_v6"
    compileSdk = 36
    ndkVersion = "28.2.13676358"
    compileOptions {
        sourceCompatibility = JavaVersion.VERSION_17
        targetCompatibility = JavaVersion.VERSION_17
    }
    kotlinOptions { jvmTarget = JavaVersion.VERSION_17.toString() }
    defaultConfig {
        applicationId = "com.nissanlogger.logger_v6"
        minSdk = 21
        targetSdk = 34
        versionCode = 6
        versionName = "6.0.0"
        multiDexEnabled = true
    }
    buildTypes {
        release {
            signingConfig = signingConfigs.getByName("debug")
            isMinifyEnabled = false
            isShrinkResources = false
        }
    }
}
dependencies {
    implementation("androidx.core:core:1.13.1")
    implementation("androidx.core:core-ktx:1.13.1")
    implementation("androidx.appcompat:appcompat:1.7.0")
    implementation("androidx.multidex:multidex:2.0.1")
}
flutter { source = "../.." }
''')

with open('android/gradle.properties', 'w') as f:
    f.write('''org.gradle.jvmargs=-Xmx4G -XX:+UseParallelGC -XX:MaxMetaspaceSize=2G
android.useAndroidX=true
android.enableJetifier=true
android.nonTransitiveRClass=false
kotlin.code.style=official
org.gradle.parallel=true
org.gradle.caching=false
org.gradle.configuration-cache=false
kotlin.jvm.target.validation.mode=warning
android.suppressUnsupportedCompileSdk=36
''')
print("✅ Gradle конфиги")

# ================================================================
# МОДЕЛИ
# ================================================================

# ============ lib/constants.dart ============
with open('lib/constants.dart', 'w') as f:
    f.write('''class AppConstants {
  static const String appVersion = '6.0.0';
  static const String appName = 'Nissan Logger V6';

  // Двигатель
  static const double engineDisplacement = 2.0;

  // Пороги алертов
  static const double knockRetardWarning = 1.0;
  static const double knockRetardDanger  = 3.0;
  static const double afrLeanWarning     = 15.5;
  static const double afrLeanDanger      = 16.5;
  static const double afrRichWarning     = 11.5;
  static const int    coolantTempWarning = 100;
  static const int    coolantTempDanger  = 110;
  static const double fuelTrimWarning    = 12.0;
  static const double fuelTrimDanger     = 20.0;

  // Автолог
  static const int autoLogRpmThreshold   = 1500;
  static const int autoLogSpeedThreshold = 5;
  static const int autoLogIdleTimeoutSec = 30;

  // Опрос
  static const int defaultPollingInterval = 50;

  // Топливо
  static const double gasolineDensity   = 745.0;  // г/л
  static const double stoichiometricAFR = 14.7;

  // MAF Hitachi QR20DE — таблица V → g/s
  // Измерена/верифицирована по даташиту Hitachi AFH70M-37
  static const List<List<double>> mafVoltageTable = [
    [0.50,  0.00],
    [0.70,  0.80],
    [0.80,  1.40],
    [0.90,  2.00],
    [0.98,  2.60],  // ХХ прогретый
    [1.00,  2.80],
    [1.10,  3.80],
    [1.20,  5.20],
    [1.30,  6.90],
    [1.40,  9.00],
    [1.50, 11.50],
    [1.60, 14.40],
    [1.70, 17.80],
    [1.80, 21.70],
    [1.90, 26.20],
    [2.00, 31.30],
    [2.10, 37.10],
    [2.20, 43.60],
    [2.30, 51.00],
    [2.40, 59.20],
    [2.50, 68.30],
    [2.60, 78.40],
    [2.70, 89.50],
    [2.80,101.60],
    [2.90,114.80],
    [3.00,129.00],
    [3.10,144.30],
    [3.20,160.70],
    [3.30,178.20],
    [3.40,196.80],
    [3.50,216.50],
    [4.00,320.00],
    [4.50,440.00],
    [5.00,570.00],
  ];

  /// Конвертирует вольтаж MAF → g/s через интерполяцию таблицы
  static double mafVoltToGps(double voltage) {
    final t = mafVoltageTable;
    if (voltage <= t.first[0]) return 0.0;
    if (voltage >= t.last[0])  return t.last[1];
    for (int i = 0; i < t.length - 1; i++) {
      if (voltage >= t[i][0] && voltage <= t[i + 1][0]) {
        final ratio = (voltage - t[i][0]) / (t[i + 1][0] - t[i][0]);
        return t[i][1] + ratio * (t[i + 1][1] - t[i][1]);
      }
    }
    return 0.0;
  }
}
''')
print("✅ constants.dart (с таблицей MAF Hitachi QR20DE)")

# ============ lib/models/obd_data.dart ============
with open('lib/models/obd_data.dart', 'w') as f:
    f.write('''import '../constants.dart';

class OBDData {
  final DateTime timestamp;
  final int    rpm;
  final int    speed;
  final double engineLoad;
  final int    coolantTemp;
  final int    intakeTemp;
  final double mafVoltage;   // Сырой вольтаж с MAF сенсора (V)
  final double mafGps;       // Конвертированный расход воздуха (g/s)
  final double throttlePos;
  final double ignitionTiming;
  final double actualIgnition;
  final double shortFuelTrim;
  final double longFuelTrim;
  final double o2Voltage;
  final double afr;
  final double vtcTargetAngle;
  final double vtcActualAngle;
  final double knockRetard;
  final int    knockCount;
  final double injectorDuty;
  final double injectorPulseWidth;
  final double requestedTorque;
  final double actualTorque;
  final double oilTemp;
  final double afrTarget;
  final double lambda;
  final double manifoldPressure;
  final double acceleratorPedal;
  final double throttleActual;
  final double batteryVoltage;
  final double engineDisplacement;
  final double tripFuelL;

  const OBDData({
    required this.timestamp,
    this.rpm = 0,
    this.speed = 0,
    this.engineLoad = 0,
    this.coolantTemp = 0,
    this.intakeTemp = 0,
    this.mafVoltage = 0,
    this.mafGps = 0,
    this.throttlePos = 0,
    this.ignitionTiming = 0,
    this.actualIgnition = 0,
    this.shortFuelTrim = 0,
    this.longFuelTrim = 0,
    this.o2Voltage = 0,
    this.afr = 14.7,
    this.vtcTargetAngle = 0,
    this.vtcActualAngle = 0,
    this.knockRetard = 0,
    this.knockCount = 0,
    this.injectorDuty = 0,
    this.injectorPulseWidth = 0,
    this.requestedTorque = 0,
    this.actualTorque = 0,
    this.oilTemp = 0,
    this.afrTarget = 14.7,
    this.lambda = 1.0,
    this.manifoldPressure = 0,
    this.acceleratorPedal = 0,
    this.throttleActual = 0,
    this.batteryVoltage = 0,
    this.engineDisplacement = 2.0,
    this.tripFuelL = 0,
  });

  // ── Расчётные величины ──────────────────────────────────────

  /// Лошадиные силы по формуле BSFC (приближение для NA двигателя)
  /// HP = MAF(g/s) * 3600 / (AFR * BSFC)
  /// BSFC ≈ 250 g/kWh для прогретого NA → пересчёт в л.с.
  double get calculatedHP {
    if (mafGps <= 0 || rpm <= 0 || afr <= 0) return 0;
    // Мощность через MAF и BSFC
    // P(kW) = MAF_fuel(g/s) * 3600 / BSFC(g/kWh)
    // MAF_fuel = mafGps / afr
    final mafFuelGps = mafGps / afr;
    final powerKW = mafFuelGps * 3600.0 / 250.0;
    return (powerKW * 1.3596).clamp(0, 500); // kW → л.с.
  }

  double get calculatedTorqueNm {
    if (calculatedHP <= 0 || rpm <= 0) return 0;
    // T(Нм) = P(Вт) / ω(рад/с) = P(кВт)*1000 / (rpm*2π/60)
    final powerW = calculatedHP / 1.3596 * 1000;
    final omega  = rpm * 2 * 3.14159 / 60;
    return (powerW / omega).clamp(0, 400);
  }

  double get volumetricEfficiency {
    if (rpm <= 0 || mafGps <= 0) return 0;
    // VE = (MAF_actual / MAF_theoretical) * 100
    // MAF_theoretical = rpm * displacement(L) * airDensity(g/L) / 120
    const airDensity = 1.184; // г/л при 20°C, 101.3 кПа
    final theoretical = rpm * engineDisplacement * airDensity / 120.0;
    if (theoretical <= 0) return 0;
    return (mafGps / theoretical * 100).clamp(0, 150);
  }

  double get fuelMassFlowGps => mafGps > 0 && afr > 0 ? mafGps / afr : 0;

  double get fuelFlowLph =>
      fuelMassFlowGps * 3600.0 / AppConstants.gasolineDensity;

  double get fuelL100km {
    if (speed < 5) return 0;
    return fuelFlowLph / speed * 100.0;
  }

  double get totalFuelTrim => shortFuelTrim + longFuelTrim;
  double get vtcError => (vtcTargetAngle - vtcActualAngle).abs();

  String get engineMode {
    if (rpm < 100)                        return 'STOP';
    if (rpm < 900 && throttlePos < 5)     return 'IDLE';
    if (throttlePos > 80)                 return 'WOT';
    if (throttlePos < 10 && speed > 0)    return 'COAST';
    return 'CRUISE';
  }

  // ── CSV ─────────────────────────────────────────────────────

  List<dynamic> toCsvRow() => [
    timestamp.millisecondsSinceEpoch,
    rpm, speed,
    engineLoad.toStringAsFixed(2),
    coolantTemp, intakeTemp,
    mafVoltage.toStringAsFixed(4),
    mafGps.toStringAsFixed(3),
    throttlePos.toStringAsFixed(2),
    ignitionTiming.toStringAsFixed(2),
    shortFuelTrim.toStringAsFixed(2),
    longFuelTrim.toStringAsFixed(2),
    o2Voltage.toStringAsFixed(4),
    afr.toStringAsFixed(3),
    vtcTargetAngle.toStringAsFixed(2),
    vtcActualAngle.toStringAsFixed(2),
    knockRetard.toStringAsFixed(2),
    knockCount,
    actualIgnition.toStringAsFixed(2),
    injectorDuty.toStringAsFixed(2),
    injectorPulseWidth.toStringAsFixed(2),
    requestedTorque.toStringAsFixed(2),
    actualTorque.toStringAsFixed(2),
    oilTemp.toStringAsFixed(1),
    afrTarget.toStringAsFixed(3),
    lambda.toStringAsFixed(4),
    manifoldPressure.toStringAsFixed(2),
    acceleratorPedal.toStringAsFixed(2),
    throttleActual.toStringAsFixed(2),
    calculatedHP.toStringAsFixed(2),
    calculatedTorqueNm.toStringAsFixed(2),
    batteryVoltage.toStringAsFixed(2),
    volumetricEfficiency.toStringAsFixed(1),
    fuelFlowLph.toStringAsFixed(3),
    fuelL100km.toStringAsFixed(2),
    tripFuelL.toStringAsFixed(3),
  ];

  static List<String> csvHeaders() => [
    'Timestamp_ms', 'RPM', 'Speed_kmh', 'EngineLoad_pct',
    'CoolantTemp_C', 'IntakeTemp_C',
    'MAF_V', 'MAF_gps',
    'ThrottlePos_pct', 'IgnitionTiming_deg',
    'STFT_pct', 'LTFT_pct', 'O2Voltage_V', 'AFR',
    'VTC_Target_deg', 'VTC_Actual_deg',
    'KnockRetard_deg', 'KnockCount',
    'ActualIgnition_deg', 'InjectorDuty_pct', 'InjectorPW_ms',
    'RequestedTorque_Nm', 'ActualTorque_Nm', 'OilTemp_C',
    'AFR_Target', 'Lambda', 'ManifoldPressure_kPa',
    'AcceleratorPedal_pct', 'ThrottleActual_pct',
    'EstimatedHP', 'EstimatedTorque_Nm',
    'BatteryVoltage_V', 'VE_pct',
    'FuelFlow_Lph', 'FuelConsumption_L100km', 'TripFuel_L',
  ];

  /// Парсит строку CSV обратно в OBDData (для анализатора)
  static OBDData? fromCsvRow(List<dynamic> row) {
    try {
      double _d(int i, [double def = 0]) =>
          double.tryParse(row[i].toString()) ?? def;
      int _i(int i, [int def = 0]) =>
          int.tryParse(row[i].toString()) ?? def;

      return OBDData(
        timestamp: DateTime.fromMillisecondsSinceEpoch(_i(0)),
        rpm: _i(1), speed: _i(2),
        engineLoad: _d(3),
        coolantTemp: _i(4), intakeTemp: _i(5),
        mafVoltage: _d(6),
        mafGps: _d(7),
        throttlePos: _d(8),
        ignitionTiming: _d(9),
        shortFuelTrim: _d(10), longFuelTrim: _d(11),
        o2Voltage: _d(12),
        afr: _d(13, 14.7),
        vtcTargetAngle: _d(14), vtcActualAngle: _d(15),
        knockRetard: _d(16),
        knockCount: _i(17),
        actualIgnition: _d(18),
        injectorDuty: _d(19),
        injectorPulseWidth: _d(20),
        requestedTorque: _d(21), actualTorque: _d(22),
        oilTemp: _d(23),
        afrTarget: _d(24, 14.7),
        lambda: _d(25, 1.0),
        manifoldPressure: _d(26),
        acceleratorPedal: _d(27),
        throttleActual: _d(28),
        batteryVoltage: _d(31),
        tripFuelL: _d(35),
      );
    } catch (_) {
      return null;
    }
  }
}
''')
print("✅ obd_data.dart (mafVoltage + mafGps, правильная формула HP)")

# ============ lib/models/tuning_map.dart ============
with open('lib/models/tuning_map.dart', 'w') as f:
    f.write('''import 'dart:convert';

class TuningMap {
  final String name;
  final String address;
  final int    rows;
  final int    cols;
  final List<double> rpmAxis;
  final List<double> loadAxis;
  List<List<double>> data;
  final String units;
  final double minValue;
  final double maxValue;

  TuningMap({
    required this.name,
    required this.address,
    required this.rows,
    required this.cols,
    required this.rpmAxis,
    required this.loadAxis,
    required this.data,
    required this.units,
    this.minValue = -100,
    this.maxValue =  400,
  });

  // ── Доступ к ячейкам ────────────────────────────────────────

  double getValue(double rpm, double load) {
    final ri = _closest(rpmAxis,  rpm);
    final li = _closest(loadAxis, load);
    return data[ri][li];
  }

  void setValue(double rpm, double load, double value) {
    final ri = _closest(rpmAxis,  rpm);
    final li = _closest(loadAxis, load);
    data[ri][li] = value;
  }

  int _closest(List<double> axis, double v) {
    int idx = 0;
    double best = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  // ── Утилиты ─────────────────────────────────────────────────

  TuningMap copy() => TuningMap(
    name: name, address: address, rows: rows, cols: cols,
    rpmAxis:  List<double>.from(rpmAxis),
    loadAxis: List<double>.from(loadAxis),
    data: data.map((r) => List<double>.from(r)).toList(),
    units: units, minValue: minValue, maxValue: maxValue,
  );

  double get avgValue {
    double sum = 0; int n = 0;
    for (final row in data) { for (final v in row) { sum += v; n++; } }
    return n > 0 ? sum / n : 0;
  }

  double get minDataValue {
    double m = double.infinity;
    for (final row in data) { for (final v in row) { if (v < m) m = v; } }
    return m == double.infinity ? 0 : m;
  }

  double get maxDataValue {
    double m = -double.infinity;
    for (final row in data) { for (final v in row) { if (v > m) m = v; } }
    return m == -double.infinity ? 0 : m;
  }

  // ── Сериализация ─────────────────────────────────────────────

  Map<String, dynamic> toJson() => {
    'name': name, 'address': address, 'rows': rows, 'cols': cols,
    'rpmAxis': rpmAxis, 'loadAxis': loadAxis, 'data': data,
    'units': units, 'minValue': minValue, 'maxValue': maxValue,
  };

  factory TuningMap.fromJson(Map<String, dynamic> j) {
    final rawData = j['data'] as List;
    final data = rawData.map<List<double>>(
      (row) => (row as List).map<double>((v) => (v as num).toDouble()).toList()
    ).toList();
    return TuningMap(
      name:     j['name']    as String,
      address:  j['address'] as String,
      rows:     j['rows']    as int,
      cols:     j['cols']    as int,
      rpmAxis:  (j['rpmAxis']  as List).map<double>((v) => (v as num).toDouble()).toList(),
      loadAxis: (j['loadAxis'] as List).map<double>((v) => (v as num).toDouble()).toList(),
      data:     data,
      units:    j['units']    as String,
      minValue: (j['minValue'] as num).toDouble(),
      maxValue: (j['maxValue'] as num).toDouble(),
    );
  }

  String toJsonString() => jsonEncode(toJson());
  factory TuningMap.fromJsonString(String s) =>
      TuningMap.fromJson(jsonDecode(s) as Map<String, dynamic>);
}
''')
print("✅ tuning_map.dart")

# ============ lib/models/analysis_result.dart ============
with open('lib/models/analysis_result.dart', 'w') as f:
    f.write('''class MapCell {
  final int    rpmIndex;
  final int    loadIndex;
  final double rpm;
  final double load;
  final double currentValue;
  final double suggestedValue;
  final double confidence;
  final int    sampleCount;
  final String reason;

  const MapCell({
    required this.rpmIndex,
    required this.loadIndex,
    required this.rpm,
    required this.load,
    required this.currentValue,
    required this.suggestedValue,
    required this.confidence,
    required this.sampleCount,
    required this.reason,
  });

  double get delta        => suggestedValue - currentValue;
  double get deltaPercent =>
      currentValue != 0 ? (delta / currentValue.abs()) * 100 : 0;
}

class AnalysisResult {
  final String       mapName;
  final DateTime     analyzedAt;
  final int          totalSamples;
  final List<MapCell> changes;
  final String       summary;
  final String       patternName;

  const AnalysisResult({
    required this.mapName,
    required this.analyzedAt,
    required this.totalSamples,
    required this.changes,
    required this.summary,
    this.patternName = '',
  });
}
''')
print("✅ analysis_result.dart")

# ============ lib/models/alert.dart ============
with open('lib/models/alert.dart', 'w') as f:
    f.write('''enum AlertLevel { info, warning, danger }

class AlertSnapshot {
  final int    rpm;
  final int    speed;
  final double engineLoad;
  final int    coolantTemp;
  final int    intakeTemp;
  final double mafGps;
  final double throttlePos;
  final double afr;
  final double knockRetard;
  final double shortFuelTrim;
  final double longFuelTrim;
  final double ignitionTiming;
  final double vtcActualAngle;

  const AlertSnapshot({
    required this.rpm,
    required this.speed,
    required this.engineLoad,
    required this.coolantTemp,
    required this.intakeTemp,
    required this.mafGps,
    required this.throttlePos,
    required this.afr,
    required this.knockRetard,
    required this.shortFuelTrim,
    required this.longFuelTrim,
    required this.ignitionTiming,
    required this.vtcActualAngle,
  });
}

class Alert {
  final String        message;
  final AlertLevel    level;
  final DateTime      timestamp;
  final String?       category;
  final AlertSnapshot? snapshot;
  final String?       explanation;

  const Alert({
    required this.message,
    required this.level,
    required this.timestamp,
    this.category,
    this.snapshot,
    this.explanation,
  });
}
''')
print("✅ alert.dart")

# ============ lib/models/dtc_code.dart ============
with open('lib/models/dtc_code.dart', 'w') as f:
    f.write('''class DTCCode {
  final String  code;
  final String  description;
  final DTCType type;
  final bool    isPending;

  const DTCCode({
    required this.code,
    required this.description,
    required this.type,
    this.isPending = false,
  });
}

enum DTCType { powertrain, chassis, body, network }
''')
print("✅ dtc_code.dart")

# ============ lib/models/custom_pid.dart ============
with open('lib/models/custom_pid.dart', 'w') as f:
    f.write('''import 'dart:convert';

/// Статус пользовательского PID по отношению к дефолтному
enum PidStatus {
  defaultUnchanged, // дефолтный, не изменён
  defaultModified,  // дефолтный, изменён пользователем
  defaultDeleted,   // дефолтный, удалён пользователем
  userAdded,        // добавлен пользователем
}

class CustomPid {
  final String    id;          // уникальный ID
  final String    cmd;         // hex команда
  final String    answer;      // ожидаемый префикс ответа
  final String    name;        // короткое имя
  final String    desc;        // описание
  final String    unit;        // единица
  final int       bytesCount;  // байт в ответе
  final String    formula;     // формула (X = raw value)
  final double    minVal;
  final double    maxVal;
  final int       priority;    // 1-3
  final String    category;
  final PidStatus status;
  final String?   originalId;  // ID дефолтного PID если modified

  const CustomPid({
    required this.id,
    required this.cmd,
    required this.answer,
    required this.name,
    required this.desc,
    required this.unit,
    required this.bytesCount,
    required this.formula,
    this.minVal   = 0,
    this.maxVal   = 255,
    this.priority = 3,
    this.category = 'other',
    this.status   = PidStatus.userAdded,
    this.originalId,
  });

  Map<String, dynamic> toJson() => {
    'id': id, 'cmd': cmd, 'answer': answer,
    'name': name, 'desc': desc, 'unit': unit,
    'bytesCount': bytesCount, 'formula': formula,
    'minVal': minVal, 'maxVal': maxVal,
    'priority': priority, 'category': category,
    'status': status.name, 'originalId': originalId,
  };

  factory CustomPid.fromJson(Map<String, dynamic> j) => CustomPid(
    id:         j['id']         as String,
    cmd:        j['cmd']        as String,
    answer:     j['answer']     as String,
    name:       j['name']       as String,
    desc:       j['desc']       as String,
    unit:       j['unit']       as String,
    bytesCount: j['bytesCount'] as int,
    formula:    j['formula']    as String,
    minVal:     (j['minVal']    as num).toDouble(),
    maxVal:     (j['maxVal']    as num).toDouble(),
    priority:   j['priority']   as int,
    category:   j['category']   as String,
    status:     PidStatus.values.firstWhere(
                  (e) => e.name == (j['status'] as String),
                  orElse: () => PidStatus.userAdded),
    originalId: j['originalId'] as String?,
  );

  String toJsonString() => jsonEncode(toJson());
  factory CustomPid.fromJsonString(String s) =>
      CustomPid.fromJson(jsonDecode(s) as Map<String, dynamic>);

  CustomPid copyWith({
    String? name, String? desc, String? unit,
    String? formula, double? minVal, double? maxVal,
    int? priority, String? category, PidStatus? status,
  }) => CustomPid(
    id: id, cmd: cmd, answer: answer,
    name:       name       ?? this.name,
    desc:       desc       ?? this.desc,
    unit:       unit       ?? this.unit,
    bytesCount: bytesCount,
    formula:    formula    ?? this.formula,
    minVal:     minVal     ?? this.minVal,
    maxVal:     maxVal     ?? this.maxVal,
    priority:   priority   ?? this.priority,
    category:   category   ?? this.category,
    status:     status     ?? this.status,
    originalId: originalId,
  );
}
''')
print("✅ custom_pid.dart")

# ============ lib/models/vehicle_profile.dart ============
with open('lib/models/vehicle_profile.dart', 'w') as f:
    f.write('''import 'dart:convert';
import 'custom_pid.dart';

class AlertThresholds {
  final double knockWarning;
  final double knockDanger;
  final int    coolantWarning;
  final int    coolantDanger;
  final double fuelTrimWarning;
  final double fuelTrimDanger;
  final double afrLeanWarning;
  final double afrLeanDanger;

  const AlertThresholds({
    this.knockWarning    = 1.0,
    this.knockDanger     = 3.0,
    this.coolantWarning  = 100,
    this.coolantDanger   = 110,
    this.fuelTrimWarning = 12.0,
    this.fuelTrimDanger  = 20.0,
    this.afrLeanWarning  = 15.5,
    this.afrLeanDanger   = 16.5,
  });

  Map<String, dynamic> toJson() => {
    'knockWarning': knockWarning, 'knockDanger': knockDanger,
    'coolantWarning': coolantWarning, 'coolantDanger': coolantDanger,
    'fuelTrimWarning': fuelTrimWarning, 'fuelTrimDanger': fuelTrimDanger,
    'afrLeanWarning': afrLeanWarning, 'afrLeanDanger': afrLeanDanger,
  };

  factory AlertThresholds.fromJson(Map<String, dynamic> j) => AlertThresholds(
    knockWarning:    (j['knockWarning']    as num).toDouble(),
    knockDanger:     (j['knockDanger']     as num).toDouble(),
    coolantWarning:  j['coolantWarning']   as int,
    coolantDanger:   j['coolantDanger']    as int,
    fuelTrimWarning: (j['fuelTrimWarning'] as num).toDouble(),
    fuelTrimDanger:  (j['fuelTrimDanger']  as num).toDouble(),
    afrLeanWarning:  (j['afrLeanWarning']  as num).toDouble(),
    afrLeanDanger:   (j['afrLeanDanger']   as num).toDouble(),
  );
}

class VehicleProfile {
  final String id;
  final String name;
  final String make;
  final String model;
  final String year;
  final String engine;
  final double displacement;
  final String ecuFirmware;
  final DateTime createdAt;

  // Калибровки
  final double mafMultiplier;      // доп. множитель поверх таблицы
  final double speedMultiplier;    // коррекция скорости
  final double fuelCorrection;     // коэффициент расхода

  // UI
  final List<String> dashboardLayout; // 12 ID параметров дашборда
  final int pollingInterval;          // мс

  // PID персонализация
  final List<CustomPid> customPids;    // добавленные/изменённые
  final List<String>    deletedPidIds; // ID удалённых дефолтных

  // Алерты
  final bool            alertsEnabled;
  final bool            soundEnabled;
  final bool            vibrationEnabled;
  final AlertThresholds thresholds;

  // BT
  final String? lastBtAddress;

  const VehicleProfile({
    required this.id,
    required this.name,
    required this.make,
    required this.model,
    required this.year,
    required this.engine,
    this.displacement  = 2.0,
    this.ecuFirmware   = '',
    required this.createdAt,
    this.mafMultiplier   = 1.0,
    this.speedMultiplier = 1.0,
    this.fuelCorrection  = 1.0,
    this.dashboardLayout = const [
      'timing','knock','vtc','load','throttle','maf_gps',
      'afr','ect','iat','batt','inj','fuel_lh',
    ],
    this.pollingInterval = 50,
    this.customPids      = const [],
    this.deletedPidIds   = const [],
    this.alertsEnabled   = true,
    this.soundEnabled    = true,
    this.vibrationEnabled = true,
    this.thresholds      = const AlertThresholds(),
    this.lastBtAddress,
  });

  Map<String, dynamic> toJson() => {
    'id': id, 'name': name, 'make': make, 'model': model,
    'year': year, 'engine': engine,
    'displacement': displacement, 'ecuFirmware': ecuFirmware,
    'createdAt': createdAt.toIso8601String(),
    'mafMultiplier': mafMultiplier,
    'speedMultiplier': speedMultiplier,
    'fuelCorrection': fuelCorrection,
    'dashboardLayout': dashboardLayout,
    'pollingInterval': pollingInterval,
    'customPids': customPids.map((p) => p.toJson()).toList(),
    'deletedPidIds': deletedPidIds,
    'alertsEnabled': alertsEnabled,
    'soundEnabled': soundEnabled,
    'vibrationEnabled': vibrationEnabled,
    'thresholds': thresholds.toJson(),
    'lastBtAddress': lastBtAddress,
  };

  factory VehicleProfile.fromJson(Map<String, dynamic> j) {
    final pids = (j['customPids'] as List? ?? [])
        .map((p) => CustomPid.fromJson(p as Map<String, dynamic>))
        .toList();
    final deleted = (j['deletedPidIds'] as List? ?? [])
        .map((e) => e as String)
        .toList();
    final layout = (j['dashboardLayout'] as List? ?? [])
        .map((e) => e as String)
        .toList();
    return VehicleProfile(
      id:           j['id']           as String,
      name:         j['name']         as String,
      make:         j['make']         as String,
      model:        j['model']        as String,
      year:         j['year']         as String,
      engine:       j['engine']       as String,
      displacement: (j['displacement'] as num).toDouble(),
      ecuFirmware:  j['ecuFirmware']  as String? ?? '',
      createdAt:    DateTime.parse(j['createdAt'] as String),
      mafMultiplier:    (j['mafMultiplier']    as num? ?? 1.0).toDouble(),
      speedMultiplier:  (j['speedMultiplier']  as num? ?? 1.0).toDouble(),
      fuelCorrection:   (j['fuelCorrection']   as num? ?? 1.0).toDouble(),
      dashboardLayout:  layout.isNotEmpty ? layout : const [
        'timing','knock','vtc','load','throttle','maf_gps',
        'afr','ect','iat','batt','inj','fuel_lh',
      ],
      pollingInterval:  j['pollingInterval']  as int? ?? 50,
      customPids:       pids,
      deletedPidIds:    deleted,
      alertsEnabled:    j['alertsEnabled']    as bool? ?? true,
      soundEnabled:     j['soundEnabled']     as bool? ?? true,
      vibrationEnabled: j['vibrationEnabled'] as bool? ?? true,
      thresholds: j['thresholds'] != null
          ? AlertThresholds.fromJson(j['thresholds'] as Map<String, dynamic>)
          : const AlertThresholds(),
      lastBtAddress: j['lastBtAddress'] as String?,
    );
  }

  String toJsonString() => jsonEncode(toJson());
  factory VehicleProfile.fromJsonString(String s) =>
      VehicleProfile.fromJson(jsonDecode(s) as Map<String, dynamic>);

  VehicleProfile copyWith({
    String? name, String? make, String? model, String? year,
    String? engine, double? displacement, String? ecuFirmware,
    double? mafMultiplier, double? speedMultiplier, double? fuelCorrection,
    List<String>? dashboardLayout, int? pollingInterval,
    List<CustomPid>? customPids, List<String>? deletedPidIds,
    bool? alertsEnabled, bool? soundEnabled, bool? vibrationEnabled,
    AlertThresholds? thresholds, String? lastBtAddress,
  }) => VehicleProfile(
    id: id, createdAt: createdAt,
    name:             name             ?? this.name,
    make:             make             ?? this.make,
    model:            model            ?? this.model,
    year:             year             ?? this.year,
    engine:           engine           ?? this.engine,
    displacement:     displacement     ?? this.displacement,
    ecuFirmware:      ecuFirmware      ?? this.ecuFirmware,
    mafMultiplier:    mafMultiplier    ?? this.mafMultiplier,
    speedMultiplier:  speedMultiplier  ?? this.speedMultiplier,
    fuelCorrection:   fuelCorrection   ?? this.fuelCorrection,
    dashboardLayout:  dashboardLayout  ?? this.dashboardLayout,
    pollingInterval:  pollingInterval  ?? this.pollingInterval,
    customPids:       customPids       ?? this.customPids,
    deletedPidIds:    deletedPidIds    ?? this.deletedPidIds,
    alertsEnabled:    alertsEnabled    ?? this.alertsEnabled,
    soundEnabled:     soundEnabled     ?? this.soundEnabled,
    vibrationEnabled: vibrationEnabled ?? this.vibrationEnabled,
    thresholds:       thresholds       ?? this.thresholds,
    lastBtAddress:    lastBtAddress    ?? this.lastBtAddress,
  );
}
''')
print("✅ vehicle_profile.dart (полный профиль с PID + калибровки + алерты)")

# ============ lib/models/custom_map_def.dart ============
with open('lib/models/custom_map_def.dart', 'w') as f:
    f.write('''import 'dart:convert';

class CustomMapDef {
  final String  id;
  final String  name;
  final int     address;
  final int     rows;
  final int     cols;
  final bool    isUInt16;
  final String  formula;
  final String  units;
  final List<double>? xAxis;
  final List<double>? yAxis;
  final DateTime createdAt;

  const CustomMapDef({
    required this.id,
    required this.name,
    required this.address,
    required this.rows,
    required this.cols,
    this.isUInt16 = true,
    required this.formula,
    this.units = '',
    this.xAxis,
    this.yAxis,
    required this.createdAt,
  });

  int get bytesPerCell => isUInt16 ? 2 : 1;
  int get totalBytes   => rows * cols * bytesPerCell;
  String get addressHex =>
      '0x' + address.toRadixString(16).toUpperCase().padLeft(4, '0');

  Map<String, dynamic> toJson() => {
    'id': id, 'name': name, 'address': address,
    'rows': rows, 'cols': cols, 'isUInt16': isUInt16,
    'formula': formula, 'units': units,
    'xAxis': xAxis, 'yAxis': yAxis,
    'createdAt': createdAt.toIso8601String(),
  };

  factory CustomMapDef.fromJson(Map<String, dynamic> j) => CustomMapDef(
    id:        j['id']        as String,
    name:      j['name']      as String,
    address:   j['address']   as int,
    rows:      j['rows']      as int,
    cols:      j['cols']      as int,
    isUInt16:  j['isUInt16']  as bool? ?? true,
    formula:   j['formula']   as String,
    units:     j['units']     as String? ?? '',
    xAxis: j['xAxis'] != null
        ? (j['xAxis'] as List).map<double>((e) => (e as num).toDouble()).toList()
        : null,
    yAxis: j['yAxis'] != null
        ? (j['yAxis'] as List).map<double>((e) => (e as num).toDouble()).toList()
        : null,
    createdAt: DateTime.parse(j['createdAt'] as String),
  );

  String toJsonString() => jsonEncode(toJson());
  factory CustomMapDef.fromJsonString(String s) =>
      CustomMapDef.fromJson(jsonDecode(s) as Map<String, dynamic>);
}

class EcuMapReadResult {
  final CustomMapDef  def;
  final List<List<double>> data;
  final DateTime      readAt;
  final String        rawHex;

  const EcuMapReadResult({
    required this.def,
    required this.data,
    required this.readAt,
    this.rawHex = '',
  });
}
''')
print("✅ custom_map_def.dart")

print()
print("=" * 60)
print("✅ Ячейка 2/5 готова! Все модели созданы.")
print("=" * 60)
!echo "Файлы моделей:" && ls lib/models/
!echo "Прочие файлы:" && ls lib/*.dart

Creating project nissan_logger_v6...
Resolving dependencies in `nissan_logger_v6`...
Got dependencies in `nissan_logger_v6`.
Wrote 131 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd nissan_logger_v6
  $ flutter run

Your application code is in nissan_logger_v6/lib/main.dart.

✅ pubspec.yaml
✅ AndroidManifest.xml
✅ MainActivity.kt
✅ Gradle конфиги
✅ constants.dart (с таблицей MAF Hitachi QR20DE)
✅ obd_data.dart (mafVoltage + mafGps, правильная формула HP)
✅ tuning_map.dart
✅ analysis_result.dart
✅ alert.dart
✅ dtc_code.dart
✅ custom_pid.dart
✅ vehicle_profile.dart (полный профиль с PID + калибровки + алерты)
✅ custom_map_def.dart

✅ Ячейка 2/5 готова! Все модели созданы.
Файлы моделей:
alert.dart	      custom_map_def.dart  dtc_code.dart  tunin

In [ ]:
# @title 🔧 Ячейка 3/5: Все сервисы V6
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# settings_service.dart — единое хранилище настроек
# ================================================================
with open('lib/services/settings_service.dart', 'w') as f:
    f.write('''import 'package:shared_preferences/shared_preferences.dart';

/// Глобальное хранилище настроек приложения.
/// Профильные настройки хранятся в VehicleProfile.
class SettingsService {
  static SharedPreferences? _p;
  static Future<void> init() async { _p ??= await SharedPreferences.getInstance(); }

  // ── Активный профиль ────────────────────────────────────────
  static const _kActiveProfile = 'active_profile_id';
  static String? get activeProfileId => _p?.getString(_kActiveProfile);
  static Future<void> setActiveProfileId(String? id) async {
    await init();
    if (id == null) await _p!.remove(_kActiveProfile);
    else await _p!.setString(_kActiveProfile, id);
  }

  // ── BT ──────────────────────────────────────────────────────
  static const _kLastBt = 'last_bt_device';
  static String? get lastBtDevice => _p?.getString(_kLastBt);
  static Future<void> setLastBtDevice(String? v) async {
    await init();
    if (v == null) await _p!.remove(_kLastBt);
    else await _p!.setString(_kLastBt, v);
  }

  static const _kAutoConnect = 'auto_connect';
  static bool get autoConnect => _p?.getBool(_kAutoConnect) ?? false;
  static Future<void> setAutoConnect(bool v) async {
    await init(); await _p!.setBool(_kAutoConnect, v);
  }

  // ── Опрос ───────────────────────────────────────────────────
  static const _kPolling = 'polling_interval';
  static int get pollingInterval => _p?.getInt(_kPolling) ?? 50;
  static Future<void> setPollingInterval(int v) async {
    await init(); await _p!.setInt(_kPolling, v);
  }

  // ── Автолог ─────────────────────────────────────────────────
  static const _kAutoLog = 'auto_log';
  static bool get autoLog => _p?.getBool(_kAutoLog) ?? false;
  static Future<void> setAutoLog(bool v) async {
    await init(); await _p!.setBool(_kAutoLog, v);
  }

  // ── Кеш PID ─────────────────────────────────────────────────
  static const _kCachedPids  = 'cached_pids';
  static const _kCachedEcuId = 'cached_ecu_id';
  static List<String> get cachedPidList  => _p?.getStringList(_kCachedPids)  ?? [];
  static String?      get cachedEcuId    => _p?.getString(_kCachedEcuId);
  static Future<void> setCachedPidList(List<String> v) async {
    await init(); await _p!.setStringList(_kCachedPids, v);
  }
  static Future<void> setCachedEcuId(String? v) async {
    await init();
    if (v == null) await _p!.remove(_kCachedEcuId);
    else await _p!.setString(_kCachedEcuId, v);
  }
  static Future<void> clearPidCache() async {
    await init();
    await _p!.remove(_kCachedPids);
    await _p!.remove(_kCachedEcuId);
  }

  // ── Поездка (расход) ────────────────────────────────────────
  static const _kTripFuel = 'trip_fuel_l';
  static double get tripFuelL => _p?.getDouble(_kTripFuel) ?? 0.0;
  static Future<void> setTripFuelL(double v) async {
    await init(); await _p!.setDouble(_kTripFuel, v);
  }
  static Future<void> resetTripFuel() async {
    await init(); await _p!.setDouble(_kTripFuel, 0.0);
  }

  // ── Профили (JSON-строки) ───────────────────────────────────
  static const _kProfiles = 'vehicle_profiles';
  static List<String> get profilesJson => _p?.getStringList(_kProfiles) ?? [];
  static Future<void> setProfilesJson(List<String> v) async {
    await init(); await _p!.setStringList(_kProfiles, v);
  }

  // ── Команда чтения памяти ───────────────────────────────────
  static const _kMemCmd = 'mem_read_command';
  static String? get memReadCommand => _p?.getString(_kMemCmd);
  static Future<void> setMemReadCommand(String? v) async {
    await init();
    if (v == null) await _p!.remove(_kMemCmd);
    else await _p!.setString(_kMemCmd, v);
  }
}
''')
print("✅ settings_service.dart")

# ================================================================
# profile_service.dart — CRUD профилей + активный профиль
# ================================================================
with open('lib/services/profile_service.dart', 'w') as f:
    f.write('''import 'package:uuid/uuid.dart';
import '../models/vehicle_profile.dart';
import '../models/custom_pid.dart';
import 'settings_service.dart';

/// Управление профилями автомобилей.
/// Каждый профиль хранит все настройки: калибровки, PID, дашборд, алерты.
class ProfileService {
  static const _uuid = Uuid();

  // ── Чтение ──────────────────────────────────────────────────

  List<VehicleProfile> getAll() {
    return SettingsService.profilesJson
        .map((s) => VehicleProfile.fromJsonString(s))
        .toList();
  }

  VehicleProfile? getActive() {
    final id = SettingsService.activeProfileId;
    if (id == null) return null;
    try {
      return getAll().firstWhere((p) => p.id == id);
    } catch (_) {
      return null;
    }
  }

  VehicleProfile getActiveOrDefault() {
    return getActive() ?? _defaultProfile();
  }

  // ── Запись ──────────────────────────────────────────────────

  Future<VehicleProfile> create({
    required String name,
    required String make,
    required String model,
    required String year,
    required String engine,
    double displacement = 2.0,
    String ecuFirmware  = '',
  }) async {
    final profile = VehicleProfile(
      id:           _uuid.v4(),
      name:         name,
      make:         make,
      model:        model,
      year:         year,
      engine:       engine,
      displacement: displacement,
      ecuFirmware:  ecuFirmware,
      createdAt:    DateTime.now(),
    );
    final all = getAll()..add(profile);
    await _save(all);
    if (all.length == 1) {
      await SettingsService.setActiveProfileId(profile.id);
    }
    return profile;
  }

  Future<void> update(VehicleProfile profile) async {
    final all = getAll();
    final idx = all.indexWhere((p) => p.id == profile.id);
    if (idx >= 0) all[idx] = profile;
    else all.add(profile);
    await _save(all);
  }

  Future<void> delete(String id) async {
    final all = getAll()..removeWhere((p) => p.id == id);
    await _save(all);
    if (SettingsService.activeProfileId == id) {
      await SettingsService.setActiveProfileId(
          all.isNotEmpty ? all.first.id : null);
    }
  }

  Future<void> setActive(String id) async {
    await SettingsService.setActiveProfileId(id);
  }

  // ── PID персонализация ───────────────────────────────────────

  /// Добавить/обновить кастомный PID в активном профиле
  Future<void> saveCustomPid(CustomPid pid) async {
    final profile = getActiveOrDefault();
    final pids = List<CustomPid>.from(profile.customPids);
    final idx = pids.indexWhere((p) => p.id == pid.id);
    if (idx >= 0) pids[idx] = pid;
    else pids.add(pid);
    await update(profile.copyWith(customPids: pids));
  }

  /// Удалить кастомный PID (если дефолтный — добавляем в deletedPidIds)
  Future<void> deletePid(String pidId, {bool isDefault = false}) async {
    final profile = getActiveOrDefault();
    final pids    = List<CustomPid>.from(profile.customPids)
        ..removeWhere((p) => p.id == pidId);
    final deleted = List<String>.from(profile.deletedPidIds);
    if (isDefault && !deleted.contains(pidId)) deleted.add(pidId);
    await update(profile.copyWith(customPids: pids, deletedPidIds: deleted));
  }

  /// Восстановить удалённый дефолтный PID
  Future<void> restoreDefaultPid(String pidId) async {
    final profile = getActiveOrDefault();
    final deleted = List<String>.from(profile.deletedPidIds)
        ..remove(pidId);
    await update(profile.copyWith(deletedPidIds: deleted));
  }

  // ── Вспомогательное ─────────────────────────────────────────

  Future<void> _save(List<VehicleProfile> all) async {
    await SettingsService.setProfilesJson(
        all.map((p) => p.toJsonString()).toList());
  }

  VehicleProfile _defaultProfile() => VehicleProfile(
    id:        'default',
    name:      'Nissan X-Trail T30',
    make:      'Nissan',
    model:     'X-Trail T30',
    year:      '2004',
    engine:    'QR20DE',
    createdAt: DateTime(2024),
  );
}
''')
print("✅ profile_service.dart")

# ================================================================
# nissan_pid_library.dart — полная библиотека PID
# ================================================================
with open('lib/services/nissan_pid_library.dart', 'w') as f:
    f.write(r'''import '../models/custom_pid.dart';

class NissanPidDef {
  final String id;
  final String cmd;
  final String answer;
  final String name;
  final String desc;
  final String unit;
  final int    bytesCount;
  final double Function(List<int>) formula;
  final double minVal;
  final double maxVal;
  final int    priority;
  final String category;

  const NissanPidDef({
    required this.id,
    required this.cmd,
    required this.answer,
    required this.name,
    required this.desc,
    required this.unit,
    required this.bytesCount,
    required this.formula,
    this.minVal    = 0,
    this.maxVal    = 255,
    this.priority  = 3,
    this.category  = 'other',
  });

  /// Конвертировать в CustomPid (для сохранения в профиле)
  CustomPid toCustomPid({
    String? formulaStr,
    CustomPid Function(NissanPidDef)? modifier,
  }) =>
      CustomPid(
        id:         id,
        cmd:        cmd,
        answer:     answer,
        name:       name,
        desc:       desc,
        unit:       unit,
        bytesCount: bytesCount,
        formula:    formulaStr ?? 'X',
        minVal:     minVal,
        maxVal:     maxVal,
        priority:   priority,
        category:   category,
        status:     CustomPid.defaultUnchanged == null
                    ? PidStatus.defaultUnchanged
                    : PidStatus.defaultUnchanged,
        originalId: id,
      );
}

// ignore: avoid_classes_with_only_static_members
class NissanPidLibrary {
  static final List<NissanPidDef> all = [
    // ── Приоритет 1: быстрые (каждый цикл) ─────────────────────
    NissanPidDef(id:'RPM',    cmd:'2212010401', answer:'621201',
      name:'RPM',    desc:'Обороты',      unit:'RPM',    bytesCount:2,
      priority:1, category:'engine',    minVal:0, maxVal:8000,
      formula: (b) => (b[0]*256+b[1])*12.5),

    NissanPidDef(id:'TIMING', cmd:'22110A0401', answer:'62110A',
      name:'TIMING', desc:'УОЗ факт',     unit:'°BTDC',  bytesCount:1,
      priority:1, category:'ignition',  minVal:-20, maxVal:60,
      formula: (b) => (110-b[0]).toDouble()),

    NissanPidDef(id:'KNOCK',  cmd:'22112D0401', answer:'62112D',
      name:'KNOCK',  desc:'Корр.УОЗ',    unit:'°',      bytesCount:1,
      priority:1, category:'ignition',  minVal:-30, maxVal:30,
      formula: (b) { int v=b[0]; if(v>=128) v-=256; return v.toDouble(); }),

    NissanPidDef(id:'TPS',    cmd:'22111E0401', answer:'62111E',
      name:'TPS',    desc:'Дроссель',     unit:'%',      bytesCount:1,
      priority:1, category:'throttle',  minVal:0, maxVal:100,
      formula: (b) => b[0]*0.35),

    /// MAF возвращает ВОЛЬТАЖ (не g/s).
    /// Конвертация V→g/s выполняется в OBDService через таблицу Hitachi.
    NissanPidDef(id:'MAF_V',  cmd:'2212040401', answer:'621204',
      name:'MAF_V',  desc:'MAF напряжение', unit:'V',   bytesCount:2,
      priority:1, category:'air',       minVal:0, maxVal:5,
      formula: (b) => (b[0]*256+b[1])*0.005),

    NissanPidDef(id:'ECT',    cmd:'2211010401', answer:'621101',
      name:'ECT',    desc:'Темп. ОЖ',     unit:'°C',     bytesCount:1,
      priority:1, category:'temp',      minVal:-30, maxVal:130,
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'LOAD',   cmd:'2211170401', answer:'621117',
      name:'LOAD',   desc:'Нагрузка',     unit:'%',      bytesCount:1,
      priority:1, category:'engine',    minVal:0, maxVal:100,
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'SPEED',  cmd:'2211020401', answer:'621102',
      name:'SPEED',  desc:'Скорость',     unit:'км/ч',   bytesCount:1,
      priority:1, category:'engine',    minVal:0, maxVal:200,
      formula: (b) => b[0]*2.0),

    NissanPidDef(id:'VTC_ACT',cmd:'2211350401', answer:'621135',
      name:'VTC_ACT',desc:'VTC факт B1',  unit:'°CA',    bytesCount:1,
      priority:1, category:'vtc',       minVal:-10, maxVal:50,
      formula: (b) => b[0]*0.5-64),

    NissanPidDef(id:'STFT',   cmd:'2211230401', answer:'621123',
      name:'STFT',   desc:'STFT B1',      unit:'%',      bytesCount:1,
      priority:1, category:'fuel',      minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'LTFT',   cmd:'2211250401', answer:'621125',
      name:'LTFT',   desc:'LTFT B1',      unit:'%',      bytesCount:1,
      priority:1, category:'fuel',      minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'INJ_B1', cmd:'2212060401', answer:'621206',
      name:'INJ_B1', desc:'Впрыск B1',    unit:'ms',     bytesCount:2,
      priority:1, category:'fuel',      minVal:0, maxVal:30,
      formula: (b) => (b[0]*256+b[1])*0.01),

    NissanPidDef(id:'O2_B1S1',cmd:'2211180401', answer:'621118',
      name:'O2_B1S1',desc:'O2 B1S1',     unit:'V',      bytesCount:1,
      priority:1, category:'fuel',      minVal:0, maxVal:1,
      formula: (b) => b[0]*0.01),

    // ── Приоритет 2: средние (каждый 3-й цикл) ──────────────────
    NissanPidDef(id:'PEDAL',  cmd:'22117C0401', answer:'62117C',
      name:'PEDAL',  desc:'Педаль газа',  unit:'%',      bytesCount:1,
      priority:2, category:'throttle',
      formula: (b) => b[0]*0.5),

    NissanPidDef(id:'BATT',   cmd:'2211030401', answer:'621103',
      name:'BATT',   desc:'Напряжение',   unit:'V',      bytesCount:1,
      priority:2, category:'electric',  minVal:8, maxVal:16,
      formula: (b) => b[0]*0.08),

    NissanPidDef(id:'IAT',    cmd:'2211060401', answer:'621106',
      name:'IAT',    desc:'Темп. впуска', unit:'°C',     bytesCount:1,
      priority:2, category:'temp',      minVal:-30, maxVal:100,
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'MAP_V',  cmd:'22112A0401', answer:'62112A',
      name:'MAP_V',  desc:'MAP датчик',   unit:'V',      bytesCount:1,
      priority:2, category:'air',
      formula: (b) => b[0]*0.02),

    NissanPidDef(id:'IACV',   cmd:'22110B0401', answer:'62110B',
      name:'IACV',   desc:'Клапан ХХ',   unit:'%',      bytesCount:1,
      priority:2, category:'idle',
      formula: (b) => b[0]*0.5),

    NissanPidDef(id:'VTC_SOL',cmd:'2211380401', answer:'621138',
      name:'VTC_SOL',desc:'VTC Sol B1',   unit:'%',      bytesCount:1,
      priority:2, category:'vtc',
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'STFT_B2',cmd:'2211240401', answer:'621124',
      name:'STFT_B2',desc:'STFT B2',      unit:'%',      bytesCount:1,
      priority:2, category:'fuel',      minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'LTFT_B2',cmd:'2211260401', answer:'621126',
      name:'LTFT_B2',desc:'LTFT B2',      unit:'%',      bytesCount:1,
      priority:2, category:'fuel',      minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'INJ_BASE',cmd:'2212080401',answer:'621208',
      name:'INJ_BASE',desc:'Впрыск баз.',  unit:'ms',    bytesCount:2,
      priority:2, category:'fuel',
      formula: (b) => (b[0]*256+b[1])/2048.0),

    NissanPidDef(id:'VTC_DUTY',cmd:'22122D0401',answer:'62122D',
      name:'VTC_DUTY',desc:'VTC Duty B1',  unit:'%',     bytesCount:2,
      priority:2, category:'vtc',
      formula: (b) => (b[0]*256+b[1])*3200.0/32768.0),

    NissanPidDef(id:'POWER',  cmd:'2212570401', answer:'621257',
      name:'POWER',  desc:'Мощность запр.', unit:'kW',  bytesCount:2,
      priority:2, category:'engine',
      formula: (b) => (b[0]*256+b[1])*0.03125),

    NissanPidDef(id:'TORQUE', cmd:'2212280401', answer:'621228',
      name:'TORQUE', desc:'Момент',        unit:'Nm',    bytesCount:2,
      priority:2, category:'engine',
      formula: (b) {
        int v=b[0]*256+b[1];
        if(v>=32768) v-=65536;
        return v/4.0;
      }),

    // ── Приоритет 3: медленные (каждый 10-й цикл) ───────────────
    NissanPidDef(id:'OIL_T',  cmd:'22111F0401', answer:'62111F',
      name:'OIL_T',  desc:'Темп. масла',  unit:'°C',    bytesCount:1,
      priority:3, category:'temp',
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'FUEL_T', cmd:'2211040401', answer:'621104',
      name:'FUEL_T', desc:'Темп. топлива', unit:'°C',   bytesCount:1,
      priority:3, category:'temp',
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'O2_B2S1',cmd:'2211190401', answer:'621119',
      name:'O2_B2S1',desc:'O2 B2S1',      unit:'V',     bytesCount:1,
      priority:3, category:'fuel',
      formula: (b) => b[0]*0.01),

    NissanPidDef(id:'O2_B1S2',cmd:'22111A0401', answer:'62111A',
      name:'O2_B1S2',desc:'O2 B1S2',      unit:'V',     bytesCount:1,
      priority:3, category:'fuel',
      formula: (b) => b[0]*0.01),

    NissanPidDef(id:'AF_B1S1',cmd:'2212250401', answer:'621225',
      name:'AF_B1S1',desc:'A/F B1S1',     unit:'V',     bytesCount:2,
      priority:3, category:'fuel',
      formula: (b) => (b[0]*256+b[1])*0.005),

    NissanPidDef(id:'BARO',   cmd:'2211290401', answer:'621129',
      name:'BARO',   desc:'Атм. давление', unit:'V',    bytesCount:1,
      priority:3, category:'air',
      formula: (b) => b[0]*0.02),

    NissanPidDef(id:'ALT_SPD',cmd:'2211900401', answer:'621190',
      name:'ALT_SPD',desc:'Об. генератора', unit:'RPM', bytesCount:1,
      priority:3, category:'electric',
      formula: (b) => b[0]*0.75),

    NissanPidDef(id:'BAT_SOC',cmd:'2211510401', answer:'621151',
      name:'BAT_SOC',desc:'Заряд АКБ',    unit:'%',     bytesCount:1,
      priority:3, category:'electric',
      formula: (b) => b[0].toDouble()),

    NissanPidDef(id:'FAN',    cmd:'2211470401', answer:'621147',
      name:'FAN',    desc:'Вентилятор',   unit:'%',     bytesCount:1,
      priority:3, category:'other',
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'IDLE_BS',cmd:'22110D0401', answer:'62110D',
      name:'IDLE_BS',desc:'Базовые ХХ',   unit:'RPM',   bytesCount:1,
      priority:3, category:'idle',       maxVal:3200,
      formula: (b) => b[0]*12.5),

    NissanPidDef(id:'CAT_T',  cmd:'2213020401', answer:'621302',
      name:'CAT_T',  desc:'Темп. катал.', unit:'°C',    bytesCount:2,
      priority:3, category:'temp',      minVal:0, maxVal:1000,
      formula: (b) => (b[0]*256+b[1])*0.1-40),

    NissanPidDef(id:'AMBIENT',cmd:'22130A0401', answer:'62130A',
      name:'AMBIENT',desc:'Темп. воздуха',unit:'°C',    bytesCount:2,
      priority:3, category:'temp',      minVal:-40, maxVal:60,
      formula: (b) => (b[0]*256+b[1])*0.1-40),

    NissanPidDef(id:'MISFIRE1',cmd:'2212220401',answer:'621222',
      name:'MISFIRE1',desc:'Пропуски Ц1', unit:'cnt',   bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),

    NissanPidDef(id:'MISFIRE2',cmd:'2212230401',answer:'621223',
      name:'MISFIRE2',desc:'Пропуски Ц2', unit:'cnt',   bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),
  ];

  static List<NissanPidDef> byPriority(int p) =>
      all.where((x) => x.priority == p).toList();

  static NissanPidDef? byId(String id) {
    try { return all.firstWhere((p) => p.id == id); }
    catch (_) { return null; }
  }

  static List<NissanPidDef> byCategory(String c) =>
      all.where((x) => x.category == c).toList();

  static List<String> get categories =>
      all.map((p) => p.category).toSet().toList()..sort();
}
''')
print("✅ nissan_pid_library.dart")

# ================================================================
# formula_evaluator.dart
# ================================================================
with open('lib/services/formula_evaluator.dart', 'w') as f:
    f.write(r'''import 'package:math_expressions/math_expressions.dart';

/// Вычисляет формулу с переменной X (raw byte value).
class FormulaEvaluator {
  final String formulaStr;
  Expression?  _expr;

  FormulaEvaluator(this.formulaStr) { _parse(); }

  void _parse() {
    try {
      final parser = Parser();
      final cleaned = formulaStr
          .trim()
          .replaceAll('X', 'x')
          .replaceAll(' ', '');
      _expr = parser.parse(cleaned);
    } catch (_) {
      _expr = null;
    }
  }

  bool get isValid => _expr != null;

  double evaluate(num rawValue) {
    if (_expr == null) return rawValue.toDouble();
    try {
      final cm = ContextModel();
      cm.bindVariable(Variable('x'), Number(rawValue.toDouble()));
      final result = _expr!.evaluate(EvaluationType.REAL, cm);
      if (result is num) return result.toDouble();
      return rawValue.toDouble();
    } catch (_) {
      return rawValue.toDouble();
    }
  }

  static bool isFormulaValid(String formula) {
    final ev = FormulaEvaluator(formula);
    if (!ev.isValid) return false;
    final t = ev.evaluate(100);
    return !t.isNaN && !t.isInfinite;
  }

  /// Предустановленные формулы Nissan
  static const Map<String, String> presets = {
    'Torque (X-32768)/10.24': '(X-32768)/10.24',
    'Force X-32768':          'X-32768',
    'VTC (X-128)/2':          '(X-128)/2',
    'Spark (X-16384)/128':    '(X-16384)/128',
    'VE X/256':               'X/256',
    'Lambda X/128':           'X/128',
    'RPM X*10':               'X*10',
    'Raw X':                  'X',
    'Temp X-50':              'X-50',
    'Voltage X*0.02':         'X*0.02',
  };
}
''')
print("✅ formula_evaluator.dart")

# ================================================================
# obd_service.dart — с правильным MAF (V→g/s), паузой polling
# ================================================================
with open('lib/services/obd_service.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import '../models/obd_data.dart';
import '../models/vehicle_profile.dart';
import '../constants.dart';
import 'nissan_pid_library.dart';
import 'settings_service.dart';

class OBDService {
  // ── BT ──────────────────────────────────────────────────────
  BluetoothConnection? _connection;
  StreamSubscription?  _inputSub;
  final StringBuffer   _rxBuf = StringBuffer();

  // ── Команды ─────────────────────────────────────────────────
  bool       _cmdInProgress = false;
  Completer<String>? _cmdCompleter;

  // ── Polling ─────────────────────────────────────────────────
  bool _isPolling    = false;
  bool _pollPaused   = false;
  int  _pollCounter  = 0;
  int  _mediumIdx    = 0;
  int  _slowIdx      = 0;
  double _pollFps    = 0.0;
  int    _lastPollMs = 0;

  // ── ECU ─────────────────────────────────────────────────────
  bool   _ecuResponds  = false;
  bool   _initialized  = false;
  String _protocolInfo = '';
  String _ecuId        = '';

  // ── PID ─────────────────────────────────────────────────────
  List<NissanPidDef> _activePids  = [];
  List<NissanPidDef> _fastPids    = [];
  List<NissanPidDef> _medPids     = [];
  List<NissanPidDef> _slowPids    = [];
  final Map<String, double>   _values  = {};
  final Map<String, List<int>>_rawData = {};

  // ── Профиль ──────────────────────────────────────────────────
  VehicleProfile? _profile;

  // ── Расход ──────────────────────────────────────────────────
  double    _tripFuelL  = 0;
  DateTime? _lastFuelTs;

  // ── Reconnect ────────────────────────────────────────────────
  bool   _autoReconnect   = true;
  int    _reconnectTries  = 0;
  String? _lastAddress;
  Timer?  _reconnectTimer;
  static const _maxReconnect = 3;

  // ── Streams ──────────────────────────────────────────────────
  final _dataCtrl = StreamController<OBDData>.broadcast();
  final _logCtrl  = StreamController<String>.broadcast();

  Stream<OBDData> get dataStream => _dataCtrl.stream;
  Stream<String>  get logStream  => _logCtrl.stream;

  // ── Геттеры ──────────────────────────────────────────────────
  bool   get isConnected   => _connection?.isConnected ?? false;
  bool   get isInitialized => _initialized;
  bool   get ecuResponds   => _ecuResponds;
  String get protocolInfo  => _protocolInfo;
  String get ecuId         => _ecuId;
  int    get pollFps       => _pollFps.toInt();
  int    get lastPollMs    => _lastPollMs;
  double get tripFuelL     => _tripFuelL;
  List<NissanPidDef>        get activePids  => _activePids;
  Map<String, double>        get pidValues   => Map.unmodifiable(_values);
  Map<String, List<int>>     get rawPidData  => Map.unmodifiable(_rawData);

  // ── Профиль ──────────────────────────────────────────────────
  void applyProfile(VehicleProfile profile) {
    _profile = profile;
    _tripFuelL = SettingsService.tripFuelL;
  }

  // ── Топливо ──────────────────────────────────────────────────
  Future<void> resetTripFuel() async {
    _tripFuelL = 0;
    _lastFuelTs = null;
    await SettingsService.resetTripFuel();
  }

  Future<void> loadTripFuel() async {
    _tripFuelL = SettingsService.tripFuelL;
  }

  // ── Лог ──────────────────────────────────────────────────────
  void _log(String msg) {
    _logCtrl.add(msg);
  }

  // ── BT API ───────────────────────────────────────────────────
  Future<List<BluetoothDevice>> getBondedDevices() async {
    try { return await FlutterBluetoothSerial.instance.getBondedDevices(); }
    catch (_) { return []; }
  }

  Future<BluetoothState> getBluetoothState() async =>
      FlutterBluetoothSerial.instance.state;

  Future<bool?> requestEnable() async =>
      FlutterBluetoothSerial.instance.requestEnable();

  // ── Подключение ──────────────────────────────────────────────
  Future<bool> connect(String address) async {
    try {
      _log('=== BT $address ===');
      _initialized  = false;
      _ecuResponds  = false;
      _lastAddress  = address;
      _reconnectTries = 0;

      _connection = await BluetoothConnection.toAddress(address);
      _log('BT OK');

      _inputSub = _connection!.input!.listen(
        _onData,
        onDone:  _onDisconnected,
        onError: (e) => _log('BT Error: $e'),
      );

      await Future.delayed(const Duration(milliseconds: 1500));
      _rxBuf.clear();
      _cmdInProgress = false;
      _cmdCompleter  = null;

      // Будим ELM
      _connection!.output.add(Uint8List.fromList([13, 13, 13]));
      await _connection!.output.allSent;
      await Future.delayed(const Duration(milliseconds: 500));
      _rxBuf.clear();

      final r = await sendCommand('ATZ', timeout: 5000);
      _log('ATZ: [$r]');
      await Future.delayed(const Duration(milliseconds: 1500));

      if (r.toUpperCase().contains('ELM')) {
        _initialized = true;
        await SettingsService.setLastBtDevice(address);
        _log('ELM OK');
        return true;
      }
      return false;
    } catch (e) {
      _log('ERR: $e');
      return false;
    }
  }

  // ── Инициализация ЭБУ ────────────────────────────────────────
  Future<bool> initECU({bool useCache = true}) async {
    if (!isConnected) return false;
    _log('=== INIT ECU ===');
    _ecuResponds = false;
    _rawData.clear();
    _activePids.clear();
    _fastPids.clear();
    _medPids.clear();
    _slowPids.clear();

    // Инициализация ELM
    await sendCommand('ATZ',     timeout: 4000);
    await Future.delayed(const Duration(milliseconds: 1000));
    for (final cmd in ['ATE0','ATL0','ATS0','ATH0','ATAL','ATSW00',
                        'ATST19','ATAT2','ATIB10','ATSP5','ATSH8110FC']) {
      await sendCommand(cmd, timeout: 1500);
    }
    await sendCommand('ATFI', timeout: 3000);
    await Future.delayed(const Duration(milliseconds: 200));

    // Проверка шины
    final r = await sendCommand('2211000401', timeout: 8000);
    _log('BUS: [$r]');
    if (!r.replaceAll(' ','').toUpperCase().contains('6211')) {
      _log('BUS FAIL');
      return false;
    }
    _log('ECU RESPONDS!');
    _ecuResponds = true;

    // ECU ID
    final idR = await sendCommand('1A81', timeout: 3000);
    final idC = idR.replaceAll(' ','').toUpperCase();
    if (idC.contains('5A')) {
      final idx = idC.indexOf('5A');
      _ecuId = _hexToAscii(idC.substring(idx + 2));
      _log('ECU ID: $_ecuId');
    }

    // Кеш
    if (useCache) {
      final ce = SettingsService.cachedEcuId;
      final cp = SettingsService.cachedPidList;
      if (ce == _ecuId && cp.isNotEmpty) {
        _log('CACHE ${cp.length} pids');
        for (final name in cp) {
          final pid = NissanPidLibrary.byId(name);
          if (pid != null) _activePids.add(pid);
        }
        _buildPidLists();
        _protocolInfo = 'Nissan $_ecuId (кеш)';
        Future.delayed(const Duration(milliseconds: 300), startPolling);
        return true;
      }
    }

    // Сканирование
    _log('SCAN ${NissanPidLibrary.all.length} pids');
    for (final pid in NissanPidLibrary.all) {
      final resp = await sendCommand(pid.cmd, timeout: 600);
      final clean = resp.replaceAll(' ','').toUpperCase();
      if (clean.contains(pid.answer)) {
        final bytes = _extractBytes(resp, pid.answer);
        if (bytes.length >= pid.bytesCount) _activePids.add(pid);
      }
      await Future.delayed(const Duration(milliseconds: 20));
    }
    _buildPidLists();
    _log('Found ${_activePids.length} pids');

    if (_activePids.isEmpty) return false;

    await SettingsService.setCachedEcuId(_ecuId);
    await SettingsService.setCachedPidList(_activePids.map((p) => p.id).toList());

    _protocolInfo = 'Nissan $_ecuId (${_activePids.length} pid)';
    Future.delayed(const Duration(milliseconds: 300), startPolling);
    return true;
  }

  void _buildPidLists() {
    _fastPids = _activePids.where((p) => p.priority == 1).toList();
    _medPids  = _activePids.where((p) => p.priority == 2).toList();
    _slowPids = _activePids.where((p) => p.priority == 3).toList();
  }

  // ── Polling ─────────────────────────────────────────────────
  void startPolling() {
    if (_isPolling || !_ecuResponds || _activePids.isEmpty) return;
    _isPolling = true;
    _log('POLL START');
    _pollLoop();
  }

  void stopPolling() { _isPolling = false; }

  Future<void> _pollLoop() async {
    final fpsTimer = Stopwatch()..start();
    int fpsCount = 0;

    while (_isPolling && isConnected && _ecuResponds) {
      // Ждём если polling на паузе (sendCommand из UI)
      while (_pollPaused && _isPolling) {
        await Future.delayed(const Duration(milliseconds: 10));
      }
      if (!_isPolling) break;

      _pollCounter++;
      final sw = Stopwatch()..start();

      // Fast PIDs — каждый цикл
      for (final pid in _fastPids) {
        if (!_isPolling || _pollPaused) break;
        await _pollPid(pid);
      }

      // Medium PIDs — каждый 3-й цикл
      if (_pollCounter % 3 == 0 && _medPids.isNotEmpty && !_pollPaused) {
        for (int i = 0; i < 2 && !_pollPaused; i++) {
          await _pollPid(_medPids[_mediumIdx % _medPids.length]);
          _mediumIdx++;
        }
      }

      // Slow PIDs — каждый 10-й цикл
      if (_pollCounter % 10 == 0 && _slowPids.isNotEmpty && !_pollPaused) {
        await _pollPid(_slowPids[_slowIdx % _slowPids.length]);
        _slowIdx++;
      }

      sw.stop();
      _lastPollMs = sw.elapsedMilliseconds;
      fpsCount++;
      if (fpsTimer.elapsedMilliseconds >= 1000) {
        _pollFps = fpsCount * 1000.0 / fpsTimer.elapsedMilliseconds;
        fpsCount = 0;
        fpsTimer.reset();
      }

      _publish();

      final interval = _profile?.pollingInterval
          ?? SettingsService.pollingInterval;
      if (interval > 0) {
        await Future.delayed(Duration(milliseconds: interval));
      }
    }
  }

  Future<void> _pollPid(NissanPidDef pid) async {
    try {
      final r = await sendCommand(pid.cmd, timeout: 300, pausePolling: false);
      final bytes = _extractBytes(r, pid.answer);
      if (bytes.length >= pid.bytesCount) {
        _values[pid.id]  = pid.formula(bytes);
        _rawData[pid.cmd] = bytes;
      }
    } catch (_) {}
  }

  // ── Publish data ─────────────────────────────────────────────
  void _publish() {
    final mafVolt = _v('MAF_V');

    // Конвертация V→g/s через таблицу Hitachi QR20DE
    final mafGpsRaw  = AppConstants.mafVoltToGps(mafVolt);
    final mafMult    = _profile?.mafMultiplier    ?? 1.0;
    final mafGps     = mafGpsRaw * mafMult;

    final speedRaw   = _v('SPEED');
    final speedMult  = _profile?.speedMultiplier  ?? 1.0;
    final speed      = (speedRaw * speedMult).toInt().clamp(0, 300);

    final disp       = _profile?.displacement ?? 2.0;

    // AFR из O2 + STFT
    final o2  = _v('O2_B1S1');
    final stft = _v('STFT');
    final afr = _calcAfr(o2, stft);

    final data = OBDData(
      timestamp:       DateTime.now(),
      rpm:             _v('RPM').toInt().clamp(0, 9999),
      speed:           speed,
      engineLoad:      _v('LOAD').clamp(0, 100),
      coolantTemp:     _v('ECT').toInt().clamp(-40, 200),
      intakeTemp:      _v('IAT').toInt().clamp(-40, 100),
      mafVoltage:      mafVolt,
      mafGps:          mafGps,
      throttlePos:     _v('TPS').clamp(0, 100),
      ignitionTiming:  _v('TIMING'),
      actualIgnition:  _v('TIMING'),
      vtcActualAngle:  _v('VTC_ACT'),
      knockRetard:     _v('KNOCK').abs(),
      shortFuelTrim:   _v('STFT').clamp(-100, 100),
      longFuelTrim:    _v('LTFT').clamp(-100, 100),
      o2Voltage:       o2,
      afr:             afr,
      injectorPulseWidth: _v('INJ_B1'),
      injectorDuty:    (_v('INJ_B1') / 20.0 * 100).clamp(0, 100),
      manifoldPressure: _v('MAP_V') * 40,
      acceleratorPedal: _v('PEDAL'),
      throttleActual:   _v('TPS'),
      batteryVoltage:   _v('BATT'),
      engineDisplacement: disp,
      tripFuelL:        _tripFuelL,
    );

    // Обновляем счётчик расхода
    _updateTripFuel(data.fuelFlowLph);

    _dataCtrl.add(data);
  }

  double _v(String id) => _values[id] ?? 0;

  double _calcAfr(double o2, double stft) {
    double lambda;
    if      (o2 > 0.85) lambda = 0.87;
    else if (o2 > 0.75) lambda = 0.92;
    else if (o2 > 0.60) lambda = 0.97;
    else if (o2 > 0.45) lambda = 1.00;
    else if (o2 > 0.30) lambda = 1.03;
    else if (o2 > 0.15) lambda = 1.05;
    else                lambda = 1.10;
    lambda *= (1 + stft / 100.0 * 0.3);
    return (lambda * 14.7).clamp(10.0, 20.0);
  }

  void _updateTripFuel(double fuelLph) {
    final now = DateTime.now();
    if (_lastFuelTs != null && fuelLph > 0) {
      final dt = now.difference(_lastFuelTs!).inMilliseconds / 1000.0;
      _tripFuelL += (fuelLph / 3600.0) * dt;
      if (_pollCounter % 100 == 0) {
        SettingsService.setTripFuelL(_tripFuelL);
      }
    }
    _lastFuelTs = now;
  }

  // ── sendCommand ──────────────────────────────────────────────
  Future<String> sendCommand(
    String cmd, {
    int     timeout       = 1000,
    bool    pausePolling  = true,
  }) async {
    if (!isConnected) return '';

    if (pausePolling && _isPolling) {
      _pollPaused = true;
      // Ждём завершения текущей команды polling (макс 300мс)
      int wait = 0;
      while (_cmdInProgress && wait < 30) {
        await Future.delayed(const Duration(milliseconds: 10));
        wait++;
      }
    }

    // Ждём освобождения шины
    int guard = 0;
    while (_cmdInProgress && guard < 50) {
      await Future.delayed(const Duration(milliseconds: 5));
      guard++;
    }
    if (_cmdInProgress) {
      _cmdInProgress = false;
      _cmdCompleter  = null;
    }

    _rxBuf.clear();
    _cmdInProgress = true;
    _cmdCompleter  = Completer<String>();

    try {
      final bytes = [...cmd.codeUnits, 13];
      _connection!.output.add(Uint8List.fromList(bytes));
      await _connection!.output.allSent;

      String resp = '';
      try {
        resp = await _cmdCompleter!.future
            .timeout(Duration(milliseconds: timeout));
      } catch (_) {
        resp = _rxBuf.toString();
      }

      _cmdInProgress = false;
      _cmdCompleter  = null;

      if (pausePolling && _isPolling) {
        await Future.delayed(const Duration(milliseconds: 30));
        _pollPaused = false;
      }

      return _clean(resp);
    } catch (e) {
      _cmdInProgress = false;
      _cmdCompleter  = null;
      if (pausePolling) _pollPaused = false;
      return '';
    }
  }

  String _clean(String r) =>
      r.replaceAll('>','').replaceAll('\r',' ').replaceAll('\n',' ')
       .replaceAll('  ',' ').trim();

  // ── BT callbacks ─────────────────────────────────────────────
  void _onData(Uint8List data) {
    final s = String.fromCharCodes(data);
    _rxBuf.write(s);
    if (s.contains('>') && _cmdCompleter != null && !_cmdCompleter!.isCompleted) {
      _cmdCompleter!.complete(_rxBuf.toString());
    }
  }

  void _onDisconnected() {
    _log('BT DISCONNECTED');
    stopPolling();
    _initialized   = false;
    _ecuResponds   = false;
    _cmdInProgress = false;
    _cmdCompleter  = null;
    _connection    = null;

    if (_autoReconnect && _lastAddress != null &&
        _reconnectTries < _maxReconnect) {
      _scheduleReconnect();
    }
  }

  void _scheduleReconnect() {
    _reconnectTries++;
    _log('Reconnect attempt $_reconnectTries');
    _reconnectTimer?.cancel();
    _reconnectTimer = Timer(
      Duration(seconds: 3 * _reconnectTries),
      () async {
        if (_lastAddress == null) return;
        final ok = await connect(_lastAddress!);
        if (ok) await initECU(useCache: true);
        else if (_reconnectTries < _maxReconnect) _scheduleReconnect();
      },
    );
  }

  // ── Отключение ───────────────────────────────────────────────
  Future<void> disconnect() async {
    _autoReconnect = false;
    _reconnectTimer?.cancel();
    stopPolling();
    _initialized   = false;
    _ecuResponds   = false;
    _cmdInProgress = false;
    _cmdCompleter  = null;
    await _inputSub?.cancel();
    _inputSub = null;
    await _connection?.close();
    _connection = null;
    _autoReconnect = true;
    await SettingsService.setTripFuelL(_tripFuelL);
  }

  void dispose() {
    disconnect();
    _dataCtrl.close();
    _logCtrl.close();
  }

  // ── Утилиты ──────────────────────────────────────────────────
  List<int> _extractBytes(String resp, String prefix) {
    final s = resp.replaceAll(' ','').replaceAll('BUSINIT:OK','')
                  .toUpperCase();
    final idx = s.indexOf(prefix);
    if (idx < 0) return [];
    final hex = s.substring(idx + prefix.length);
    final result = <int>[];
    for (int i = 0; i + 1 < hex.length; i += 2) {
      final h = hex.substring(i, i + 2);
      if (!RegExp(r'^[0-9A-F]+$').hasMatch(h)) break;
      try { result.add(int.parse(h, radix: 16)); } catch (_) { break; }
    }
    return result;
  }

  String _hexToAscii(String hex) {
    final sb = StringBuffer();
    for (int i = 0; i + 1 < hex.length; i += 2) {
      try {
        final b = int.parse(hex.substring(i, i + 2), radix: 16);
        if (b >= 0x20 && b <= 0x7E) sb.writeCharCode(b);
      } catch (_) { break; }
    }
    return sb.toString();
  }
}
''')
print("✅ obd_service.dart (MAF V→g/s через таблицу Hitachi, паузы polling)")

# ================================================================
# logger_service.dart — быстрый IOSink с try/catch
# ================================================================
with open('lib/services/logger_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:async';
import 'package:path_provider/path_provider.dart';
import 'package:intl/intl.dart';
import '../models/obd_data.dart';
import '../constants.dart';
import 'settings_service.dart';

class LoggerService {
  final List<String> _buf  = [];
  bool      _isLogging     = false;
  bool      _isAutoLogging = false;
  String?   _currentPath;
  DateTime? _lastActivity;
  int       _totalRecords  = 0;
  IOSink?   _sink;
  Timer?    _flushTimer;

  bool    get isLogging     => _isLogging;
  bool    get isAutoLogging => _isAutoLogging;
  int     get recordCount   => _totalRecords;
  String? get currentPath   => _currentPath;

  Future<void> startLogging({bool auto = false}) async {
    _buf.clear();
    _totalRecords = 0;
    _isLogging    = true;
    _isAutoLogging = auto;
    _lastActivity  = DateTime.now();

    final ts   = DateFormat('yyyyMMdd_HHmmss').format(DateTime.now());
    final pref = auto ? 'auto_' : '';
    final name = 'nissan_${pref}log_$ts.csv';
    final dir  = await getApplicationDocumentsDirectory();
    _currentPath = '${dir.path}/$name';

    try {
      _sink = File(_currentPath!).openWrite(mode: FileMode.writeOnly);
      _sink!.writeln(OBDData.csvHeaders().join(','));
    } catch (e) {
      _isLogging = false;
      return;
    }

    // Автосброс каждые 2 сек
    _flushTimer = Timer.periodic(
      const Duration(seconds: 2),
      (_) { if (_isLogging) _flush(); },
    );
  }

  void addData(OBDData data) {
    if (_isLogging) {
      final row  = data.toCsvRow();
      final line = row.join(',');
      _buf.add(line);
      _totalRecords++;
      if (_buf.length >= 200) _flush();
    }
    if (SettingsService.autoLog) _handleAutoLog(data);
  }

  void _handleAutoLog(OBDData data) {
    final moving = data.rpm   > AppConstants.autoLogRpmThreshold ||
                   data.speed > AppConstants.autoLogSpeedThreshold;
    if (moving) {
      _lastActivity = DateTime.now();
      if (!_isLogging) startLogging(auto: true);
    } else if (_isLogging && _isAutoLogging && _lastActivity != null) {
      final idle = DateTime.now().difference(_lastActivity!).inSeconds;
      if (idle >= AppConstants.autoLogIdleTimeoutSec) stopLogging();
    }
  }

  void _flush() {
    if (_sink == null || _buf.isEmpty) return;
    try {
      _sink!.writeln(_buf.join('\n'));
      _buf.clear();
    } catch (_) {}
  }

  Future<String?> stopLogging() async {
    if (!_isLogging) return null;
    _isLogging    = false;
    _isAutoLogging = false;
    _flushTimer?.cancel();
    _flushTimer = null;
    _flush();
    try {
      await _sink?.flush();
      await _sink?.close();
    } catch (_) {}
    _sink = null;
    return _currentPath;
  }

  Future<List<FileSystemEntity>> getSavedLogs() async {
    final dir   = await getApplicationDocumentsDirectory();
    final files = dir
        .listSync()
        .where((f) => f.path.endsWith('.csv'))
        .toList()
      ..sort((a, b) => b.path.compareTo(a.path));
    return files;
  }

  Future<void> deleteLog(String path) async {
    final f = File(path);
    if (await f.exists()) await f.delete();
  }
}
''')
print("✅ logger_service.dart (IOSink + try/catch + буфер 200)")

# ================================================================
# dtc_service.dart + dtc_database.dart
# ================================================================
with open('lib/services/dtc_database.dart', 'w') as f:
    f.write(r'''class DTCDatabase {
  static const Map<String, String> codes = {
    // P01xx — MAF/MAP/IAT
    'P0100': 'MAF — нет сигнала',
    'P0101': 'MAF — сигнал вне диапазона',
    'P0102': 'MAF — низкий сигнал',
    'P0103': 'MAF — высокий сигнал',
    'P0110': 'IAT — неисправность цепи',
    'P0112': 'IAT — низкий сигнал',
    'P0113': 'IAT — высокий сигнал',
    // P011x — ECT
    'P0115': 'ECT — неисправность цепи',
    'P0117': 'ECT — низкий сигнал',
    'P0118': 'ECT — высокий сигнал',
    // P012x — TPS
    'P0120': 'Датчик дросселя A — неисправность',
    'P0121': 'Датчик дросселя A — вне диапазона',
    'P0122': 'Датчик дросселя A — низкий сигнал',
    'P0123': 'Датчик дросселя A — высокий сигнал',
    // P013x — O2
    'P0130': 'O2 B1S1 — неисправность',
    'P0131': 'O2 B1S1 — низкий сигнал',
    'P0132': 'O2 B1S1 — высокий сигнал',
    'P0133': 'O2 B1S1 — медленный отклик',
    'P0134': 'O2 B1S1 — нет активности',
    'P0135': 'O2 B1S1 — нагреватель',
    'P0136': 'O2 B1S2 — неисправность',
    'P0141': 'O2 B1S2 — нагреватель',
    // P017x — Смесь
    'P0171': 'Смесь бедная B1',
    'P0172': 'Смесь богатая B1',
    // P020x — Форсунки
    'P0201': 'Форсунка цилиндр 1',
    'P0202': 'Форсунка цилиндр 2',
    'P0203': 'Форсунка цилиндр 3',
    'P0204': 'Форсунка цилиндр 4',
    // P030x — Пропуски
    'P0300': 'Множественные пропуски зажигания',
    'P0301': 'Пропуски — цилиндр 1',
    'P0302': 'Пропуски — цилиндр 2',
    'P0303': 'Пропуски — цилиндр 3',
    'P0304': 'Пропуски — цилиндр 4',
    // P032x — Детонация
    'P0325': 'Датчик детонации — неисправность',
    'P0326': 'Датчик детонации — сигнал вне диапазона',
    // P033x — Датчики вала
    'P0335': 'ДПКВ — нет сигнала',
    'P0336': 'ДПКВ — сигнал вне диапазона',
    'P0340': 'ДРВК — нет сигнала',
    'P0341': 'ДРВК — сигнал вне диапазона',
    // P042x — Катализатор
    'P0420': 'Катализатор B1 — ниже нормы',
    // P044x — EVAP
    'P0440': 'EVAP — общая неисправность',
    'P0441': 'EVAP — неверный поток',
    'P0442': 'EVAP — небольшая утечка',
    'P0446': 'EVAP — клапан вентиляции',
    // P050x
    'P0500': 'Датчик скорости VSS',
    'P0505': 'Регулятор холостого хода IACV',
    // P060x
    'P0605': 'ECU ROM — неисправность',
    // P070x
    'P0700': 'АКПП — общая неисправность',
    // P11xx — Nissan специфика
    'P1111': 'VTC — клапан OCV B1',
    'P1128': 'Дроссель ETCS — блокировка',
    'P1130': 'A/F датчик — вне диапазона',
    'P1145': 'VTC — неисправность B2',
    'P1345': 'VVT — датчик положения',
    'P1614': 'NATS — иммобилайзер',
    'P1652': 'IACV — неисправность',
    'P1656': 'OCV — VVT соленоид B1',
    // U — CAN/сеть
    'U1000': 'CAN — нет связи',
    'U1001': 'CAN — сбой связи',
    'U1002': 'CAN — тайм-аут',
    'U1010': 'CAN — неисправность BCM',
  };

  static String getDescription(String code) =>
      codes[code] ?? 'Неизвестная ошибка ($code)';
}
''')

with open('lib/services/dtc_service.dart', 'w') as f:
    f.write(r'''import 'obd_service.dart';
import '../models/dtc_code.dart';
import 'dtc_database.dart';

/// DTC Service для Nissan EFIv0[KL] (QR20DE, ECU 1EQ010).
/// Команды подтверждены K-line сниффером:
///   Чтение:  A330 → E3 [N] [B1 B2 STATUS] ...
///   Стирание: 14A1 → 54E1
class DTCService {
  final OBDService _obd;
  DTCService(this._obd);

  Future<List<DTCCode>> readStoredDTC() async {
    final resp  = await _obd.sendCommand('A330', timeout: 3000);
    final clean = resp.replaceAll(' ', '').toUpperCase();

    final idx = clean.indexOf('E3');
    if (idx < 0) return [];
    if (idx + 4 > clean.length) return [];

    final count = int.tryParse(clean.substring(idx + 2, idx + 4), radix: 16) ?? 0;
    if (count == 0 || count > 20) return [];

    final result = <DTCCode>[];
    final seen   = <String>{};
    final start  = idx + 4;

    for (int i = 0; i < count; i++) {
      final rs = start + i * 6;
      if (rs + 6 > clean.length) break;

      final b1     = int.tryParse(clean.substring(rs,     rs + 2), radix: 16) ?? 0;
      final b2     = int.tryParse(clean.substring(rs + 2, rs + 4), radix: 16) ?? 0;
      final status = int.tryParse(clean.substring(rs + 4, rs + 6), radix: 16) ?? 0;

      if (b1 == 0 && b2 == 0) continue;

      final code = _decode(b1, b2);
      if (seen.contains(code)) continue;
      seen.add(code);

      result.add(DTCCode(
        code:        code,
        description: DTCDatabase.getDescription(code),
        type:        _type(code),
        isPending:   status == 0x80 || status == 0x40,
      ));
    }
    return result;
  }

  Future<List<DTCCode>> readPendingDTC() async => [];

  Future<bool> clearDTC() async {
    final r = await _obd.sendCommand('14A1', timeout: 3000);
    return r.replaceAll(' ', '').toUpperCase().contains('54');
  }

  /// Декодирует 2 байта Nissan → строку DTC (P0121, U1001 и т.д.)
  String _decode(int b1, int b2) {
    const pfx = ['P', 'C', 'B', 'U'];
    final p  = pfx[(b1 >> 6) & 0x3];
    final d1 = ((b1 >> 4) & 0x3).toString();
    final d2 = (b1 & 0x0F).toRadixString(16).toUpperCase();
    final d34 = b2.toRadixString(16).padLeft(2, '0').toUpperCase();
    return '$p$d1$d2$d34';
  }

  DTCType _type(String c) {
    if (c.startsWith('P')) return DTCType.powertrain;
    if (c.startsWith('C')) return DTCType.chassis;
    if (c.startsWith('B')) return DTCType.body;
    return DTCType.network;
  }
}
''')
print("✅ dtc_service.dart + dtc_database.dart (P0120/P0121/U1001 добавлены)")

# ================================================================
# alert_service.dart
# ================================================================
with open('lib/services/alert_service.dart', 'w') as f:
    f.write(r'''import 'package:flutter/services.dart';
import 'package:vibration/vibration.dart';
import '../models/obd_data.dart';
import '../models/alert.dart';
import '../models/vehicle_profile.dart';

class AlertService {
  DateTime?    _lastAlertTime;
  final List<Alert> _recent = [];
  final List<Alert> _all    = [];

  List<Alert> get recentAlerts => List.unmodifiable(_recent);
  List<Alert> get allAlerts    => List.unmodifiable(_all);

  VehicleProfile? _profile;
  void applyProfile(VehicleProfile p) { _profile = p; }

  List<Alert> checkData(OBDData data) {
    final p = _profile;
    if (p != null && !p.alertsEnabled) return [];

    final thr = p?.thresholds ?? const AlertThresholds();
    final alerts = <Alert>[];

    final snap = AlertSnapshot(
      rpm: data.rpm, speed: data.speed, engineLoad: data.engineLoad,
      coolantTemp: data.coolantTemp, intakeTemp: data.intakeTemp,
      mafGps: data.mafGps, throttlePos: data.throttlePos,
      afr: data.afr, knockRetard: data.knockRetard,
      shortFuelTrim: data.shortFuelTrim, longFuelTrim: data.longFuelTrim,
      ignitionTiming: data.actualIgnition, vtcActualAngle: data.vtcActualAngle,
    );

    // Детонация
    if (data.knockRetard >= thr.knockDanger) {
      alerts.add(Alert(
        message: 'СИЛЬНАЯ ДЕТОНАЦИЯ ${data.knockRetard.toStringAsFixed(1)}°',
        level: AlertLevel.danger, timestamp: DateTime.now(),
        category: 'knock', snapshot: snap,
        explanation: 'ЭБУ снижает УОЗ на ${data.knockRetard.toStringAsFixed(1)}°.\n'
            'Причины: плохой бензин, перегрев (ОЖ ${data.coolantTemp}°C), '
            'бедная смесь (AFR ${data.afr.toStringAsFixed(2)}), '
            'нагрузка ${data.engineLoad.toStringAsFixed(0)}%',
      ));
    } else if (data.knockRetard >= thr.knockWarning) {
      alerts.add(Alert(
        message: 'Детонация ${data.knockRetard.toStringAsFixed(1)}°',
        level: AlertLevel.warning, timestamp: DateTime.now(),
        category: 'knock', snapshot: snap,
        explanation: 'Лёгкая детонация. RPM: ${data.rpm}, '
            'нагрузка: ${data.engineLoad.toStringAsFixed(0)}%',
      ));
    }

    // Перегрев
    if (data.coolantTemp >= thr.coolantDanger) {
      alerts.add(Alert(
        message: 'ПЕРЕГРЕВ! ОЖ = ${data.coolantTemp}°C',
        level: AlertLevel.danger, timestamp: DateTime.now(),
        category: 'temp', snapshot: snap,
        explanation: 'КРИТИЧЕСКАЯ температура!\n'
            'Термостат, помпа, вентилятор, утечка ОЖ.\n'
            'ЗАГЛУШИ ДВИГАТЕЛЬ!',
      ));
    } else if (data.coolantTemp >= thr.coolantWarning) {
      alerts.add(Alert(
        message: 'ОЖ высокая: ${data.coolantTemp}°C',
        level: AlertLevel.warning, timestamp: DateTime.now(),
        category: 'temp', snapshot: snap,
        explanation: 'Норма 85-95°C. Проверь вентилятор, уровень ОЖ.',
      ));
    }

    // Смесь бедная при нагрузке
    if (data.engineLoad > 70 && data.afr > thr.afrLeanDanger) {
      alerts.add(Alert(
        message: 'ОЧЕНЬ БЕДНАЯ! AFR=${data.afr.toStringAsFixed(2)}',
        level: AlertLevel.danger, timestamp: DateTime.now(),
        category: 'afr', snapshot: snap,
        explanation: 'Бедная смесь при нагрузке ${data.engineLoad.toStringAsFixed(0)}%.\n'
            'Подсос воздуха, забиты форсунки, слабый насос.\n'
            'STFT ${data.shortFuelTrim.toStringAsFixed(1)}%, '
            'LTFT ${data.longFuelTrim.toStringAsFixed(1)}%',
      ));
    }

    // Топливные коррекции
    final trim = data.totalFuelTrim.abs();
    if (trim >= thr.fuelTrimDanger) {
      final dir = data.totalFuelTrim > 0 ? 'бедно' : 'богато';
      alerts.add(Alert(
        message: 'Коррекции ${data.totalFuelTrim.toStringAsFixed(1)}%',
        level: AlertLevel.danger, timestamp: DateTime.now(),
        category: 'fuel', snapshot: snap,
        explanation: 'ЭБУ корректирует ($dir).\n'
            'STFT: ${data.shortFuelTrim.toStringAsFixed(1)}%, '
            'LTFT: ${data.longFuelTrim.toStringAsFixed(1)}%',
      ));
    }

    if (alerts.isNotEmpty) {
      _triggerFeedback(alerts, p);
      _recent.insertAll(0, alerts);
      _all.addAll(alerts);
      if (_recent.length > 50) _recent.removeRange(50, _recent.length);
      if (_all.length    > 500) _all.removeRange(0, _all.length - 500);
    }
    return alerts;
  }

  void _triggerFeedback(List<Alert> alerts, VehicleProfile? p) {
    if (_lastAlertTime != null &&
        DateTime.now().difference(_lastAlertTime!).inSeconds < 3) return;
    _lastAlertTime = DateTime.now();

    final danger = alerts.any((a) => a.level == AlertLevel.danger);
    if (p?.vibrationEnabled ?? true) _vibrate(danger);
    if (p?.soundEnabled    ?? true)  _sound(danger);
  }

  Future<void> _vibrate(bool danger) async {
    try {
      final has = await Vibration.hasVibrator() ?? false;
      if (!has) { await HapticFeedback.heavyImpact(); return; }
      if (danger) Vibration.vibrate(pattern: [0, 500, 200, 500, 200, 500]);
      else        Vibration.vibrate(duration: 300);
    } catch (_) {
      await HapticFeedback.heavyImpact();
    }
  }

  Future<void> _sound(bool danger) async {
    try {
      await SystemSound.play(
          danger ? SystemSoundType.alert : SystemSoundType.click);
    } catch (_) {}
  }

  void clearAlerts() { _recent.clear(); _all.clear(); }
  void dispose() {}
}
''')
print("✅ alert_service.dart (использует пороги из профиля)")

# ================================================================
# map_storage_service.dart
# ================================================================
with open('lib/services/map_storage_service.dart', 'w') as f:
    f.write(r'''import 'dart:convert';
import 'package:shared_preferences/shared_preferences.dart';
import '../models/tuning_map.dart';

class MapStorageService {
  static const _prefix  = 'map_edit_';
  static const _listKey = 'map_edit_list';
  static SharedPreferences? _p;

  static Future<void> _init() async {
    _p ??= await SharedPreferences.getInstance();
  }

  static Future<void> saveMap(TuningMap map) async {
    await _init();
    final key  = _prefix + map.address;
    final json = jsonEncode({
      'name': map.name, 'address': map.address,
      'rows': map.rows, 'cols': map.cols,
      'data': map.data,
      'updatedAt': DateTime.now().toIso8601String(),
    });
    await _p!.setString(key, json);
    final list = _p!.getStringList(_listKey) ?? [];
    if (!list.contains(map.address)) {
      list.add(map.address);
      await _p!.setStringList(_listKey, list);
    }
  }

  static Future<List<List<double>>?> loadMapData(String address) async {
    await _init();
    final s = _p!.getString(_prefix + address);
    if (s == null) return null;
    try {
      final j   = jsonDecode(s) as Map<String, dynamic>;
      final raw = j['data'] as List;
      return raw.map<List<double>>(
        (row) => (row as List).map<double>((v) => (v as num).toDouble()).toList(),
      ).toList();
    } catch (_) { return null; }
  }

  static Future<bool>      hasEdits(String address) async {
    await _init();
    return _p!.containsKey(_prefix + address);
  }

  static Future<DateTime?> getUpdatedAt(String address) async {
    await _init();
    final s = _p!.getString(_prefix + address);
    if (s == null) return null;
    try {
      final j = jsonDecode(s) as Map<String, dynamic>;
      return DateTime.tryParse(j['updatedAt'] as String? ?? '');
    } catch (_) { return null; }
  }

  static Future<void> resetMap(String address) async {
    await _init();
    await _p!.remove(_prefix + address);
    final list = _p!.getStringList(_listKey) ?? [];
    list.remove(address);
    await _p!.setStringList(_listKey, list);
  }

  static Future<void> resetAll() async {
    await _init();
    final list = _p!.getStringList(_listKey) ?? [];
    for (final a in list) await _p!.remove(_prefix + a);
    await _p!.remove(_listKey);
  }
}
''')
print("✅ map_storage_service.dart")

# ================================================================
# map_history_service.dart
# ================================================================
with open('lib/services/map_history_service.dart', 'w') as f:
    f.write(r'''import 'dart:convert';
import 'package:shared_preferences/shared_preferences.dart';
import 'package:intl/intl.dart';
import '../models/tuning_map.dart';

class MapRevision {
  final String   address;
  final String   name;
  final int      version;
  final DateTime savedAt;
  final List<List<double>> data;
  final String   comment;

  const MapRevision({
    required this.address,
    required this.name,
    required this.version,
    required this.savedAt,
    required this.data,
    this.comment = '',
  });

  String get label =>
      'v$version • ${DateFormat("dd.MM HH:mm").format(savedAt)}'
      '${comment.isNotEmpty ? " — $comment" : ""}';

  Map<String, dynamic> toJson() => {
    'address': address, 'name': name, 'version': version,
    'savedAt': savedAt.toIso8601String(),
    'data': data, 'comment': comment,
  };

  factory MapRevision.fromJson(Map<String, dynamic> j) {
    final raw = j['data'] as List;
    return MapRevision(
      address: j['address'] as String,
      name:    j['name']    as String,
      version: j['version'] as int,
      savedAt: DateTime.parse(j['savedAt'] as String),
      data: raw.map<List<double>>(
        (r) => (r as List).map<double>((v) => (v as num).toDouble()).toList(),
      ).toList(),
      comment: j['comment'] as String? ?? '',
    );
  }
}

class MapHistoryService {
  static const _prefix    = 'map_hist_';
  static const _maxRevs   = 20;
  static SharedPreferences? _p;

  static Future<void> _init() async {
    _p ??= await SharedPreferences.getInstance();
  }

  static Future<List<MapRevision>> getHistory(String address) async {
    await _init();
    final s = _p!.getString(_prefix + address);
    if (s == null) return [];
    try {
      final list = jsonDecode(s) as List;
      return list
          .map((j) => MapRevision.fromJson(j as Map<String, dynamic>))
          .toList();
    } catch (_) { return []; }
  }

  static Future<int> saveRevision(TuningMap map, {String comment = ''}) async {
    await _init();
    final hist    = await getHistory(map.address);
    final version = hist.isEmpty ? 1 : hist.last.version + 1;
    final rev     = MapRevision(
      address: map.address, name: map.name,
      version: version,     savedAt: DateTime.now(),
      data:    map.data.map((r) => List<double>.from(r)).toList(),
      comment: comment.isEmpty ? 'Правка $version' : comment,
    );
    hist.add(rev);
    if (hist.length > _maxRevs) hist.removeRange(0, hist.length - _maxRevs);
    await _p!.setString(
      _prefix + map.address,
      jsonEncode(hist.map((r) => r.toJson()).toList()),
    );
    return version;
  }

  static Future<void> clearHistory(String address) async {
    await _init();
    await _p!.remove(_prefix + address);
  }
}
''')
print("✅ map_history_service.dart")

# ================================================================
# rom_map_reader.dart — чтение + запись .bin прошивки
# ================================================================
with open('lib/services/rom_map_reader.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:typed_data';
import 'package:file_picker/file_picker.dart';
import 'package:path_provider/path_provider.dart';
import '../models/tuning_map.dart';
import 'formula_evaluator.dart';
import 'map_storage_service.dart';

class RomMapDef {
  final String       name;
  final String       addressHex;
  final int          address;
  final int          rows;
  final int          cols;
  final bool         isU16;
  final String       formula;
  final String       units;
  final double       minVal;
  final double       maxVal;
  final List<double> rpmAxis;
  final List<double> loadAxis;
  final bool         transpose; // VE: в ROM хранится [load][rpm]

  const RomMapDef({
    required this.name,
    required this.addressHex,
    required this.address,
    required this.rows,
    required this.cols,
    required this.isU16,
    required this.formula,
    required this.units,
    required this.rpmAxis,
    required this.loadAxis,
    this.minVal    = -100,
    this.maxVal    = 400,
    this.transpose = false,
  });

  int get bytesPerCell => isU16 ? 2 : 1;
  int get totalBytes   => rows * cols * bytesPerCell;
}

class RomMapReader {
  Uint8List? _data;
  String?    _fileName;
  String?    _filePath;

  bool    get isLoaded  => _data != null;
  String? get fileName  => _fileName;
  String? get filePath  => _filePath;
  int     get length    => _data?.length ?? 0;

  // ── Стандартные карты Nissan 1EQ010 ──────────────────────────
  static final List<RomMapDef> standardMaps = [
    RomMapDef(
      name:'Spark Advance WOT', addressHex:'0x06EBC', address:0x6EBC,
      rows:16, cols:16, isU16:true, formula:'(X-16384)/128', units:'deg',
      minVal:-5, maxVal:45,
      rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
      loadAxis: [6,13,19,25,31,38,44,50,56,63,69,75,81,88,94,100],
    ),
    RomMapDef(
      name:'Engine Torque', addressHex:'0x07C3C', address:0x7C3C,
      rows:16, cols:16, isU16:true, formula:'(X-32768)/10.24', units:'Nm',
      minVal:-100, maxVal:200,
      rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
      loadAxis: [6,13,19,25,31,38,44,50,56,63,69,75,81,88,94,100],
    ),
    RomMapDef(
      name:'Fresh Air Rate / VE', addressHex:'0x0A754', address:0xA754,
      rows:16, cols:15, isU16:true, formula:'X/256', units:'%',
      minVal:15, maxVal:250, transpose:true,
      rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
      loadAxis: [10,20,30,40,50,60,70,75,80,85,90,92,94,96,100],
    ),
    RomMapDef(
      name:'VTC Intake', addressHex:'0x06BF1', address:0x6BF1,
      rows:8, cols:8, isU16:false, formula:'(X-128)/2', units:'deg',
      minVal:0, maxVal:40,
      rpmAxis:  [800,1600,2400,3200,4000,4800,5600,6400],
      loadAxis: [0,15,30,45,60,75,90,100],
    ),
    RomMapDef(
      name:'Powertrain Force', addressHex:'0x0A2BC', address:0xA2BC,
      rows:16, cols:16, isU16:true, formula:'X-32768', units:'N',
      minVal:-5000, maxVal:5000,
      rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
      loadAxis: [6,13,19,25,31,38,44,50,56,63,69,75,81,88,94,100],
    ),
    RomMapDef(
      name:'Enrichment Lambda', addressHex:'0x06521', address:0x6521,
      rows:8, cols:8, isU16:false, formula:'X/128', units:'Lambda',
      minVal:0.5, maxVal:1.5,
      rpmAxis:  [800,1600,2400,3200,4000,4800,5600,6400],
      loadAxis: [0,15,30,45,60,75,90,100],
    ),
  ];

  // ── Загрузка файла ───────────────────────────────────────────
  Future<bool> pickAndLoad() async {
    try {
      final result = await FilePicker.platform.pickFiles(
        type: FileType.custom,
        allowedExtensions: ['bin', 'rom', 'hex', 'dat'],
      );
      if (result == null || result.files.single.path == null) return false;
      _filePath = result.files.single.path!;
      _fileName = result.files.single.name;
      _data     = await File(_filePath!).readAsBytes();
      return true;
    } catch (_) { return false; }
  }

  // ── Чтение ──────────────────────────────────────────────────
  int _u16BE(int addr) {
    if (_data == null || addr + 2 > _data!.length) return 0;
    return (_data![addr] << 8) | _data![addr + 1];
  }

  int _u8(int addr) {
    if (_data == null || addr >= _data!.length) return 0;
    return _data![addr];
  }

  TuningMap? readMap(RomMapDef def) {
    if (_data == null) return null;
    if (def.address + def.totalBytes > _data!.length) return null;

    final ev = FormulaEvaluator(def.formula);
    if (!ev.isValid) return null;

    var data = <List<double>>[];
    for (int r = 0; r < def.rows; r++) {
      final row = <double>[];
      for (int c = 0; c < def.cols; c++) {
        final off = (r * def.cols + c) * def.bytesPerCell;
        final raw = def.isU16 ? _u16BE(def.address + off) : _u8(def.address + off);
        row.add(ev.evaluate(raw));
      }
      data.add(row);
    }

    // VE транспонирование: ROM хранит [load][rpm], нам нужен [rpm][load]
    if (def.transpose && data.isNotEmpty) {
      final transposed = <List<double>>[];
      for (int c = 0; c < data[0].length; c++) {
        transposed.add([for (int r = 0; r < data.length; r++) data[r][c]]);
      }
      data = transposed;
    }

    return TuningMap(
      name:     def.name,
      address:  def.addressHex,
      rows:     def.transpose ? def.cols : def.rows,
      cols:     def.transpose ? def.rows : def.cols,
      rpmAxis:  def.rpmAxis,
      loadAxis: def.loadAxis,
      data:     data,
      units:    def.units,
      minValue: def.minVal,
      maxValue: def.maxVal,
    );
  }

  List<TuningMap> readAllMaps() {
    if (_data == null) return [];
    return standardMaps
        .map(readMap)
        .where((m) => m != null)
        .cast<TuningMap>()
        .toList();
  }

  // ── Запись карты обратно в .bin ──────────────────────────────
  /// Кодирует значение обратно в raw байт(ы) по обратной формуле.
  /// Поддерживаемые формулы: линейные вида (X+b)/a или X*a+b.
  List<int> _encodeValue(double value, RomMapDef def) {
    // Обратное преобразование формулы
    double raw;
    final f = def.formula.replaceAll(' ', '');
    if (f == '(X-16384)/128')   raw = value * 128 + 16384;
    else if (f == '(X-32768)/10.24') raw = value * 10.24 + 32768;
    else if (f == 'X/256')       raw = value * 256;
    else if (f == '(X-128)/2')   raw = value * 2 + 128;
    else if (f == 'X-32768')     raw = value + 32768;
    else if (f == 'X/128')       raw = value * 128;
    else                          raw = value; // raw = X

    final rawInt = raw.round().clamp(0, def.isU16 ? 65535 : 255);
    if (def.isU16) {
      return [(rawInt >> 8) & 0xFF, rawInt & 0xFF];
    } else {
      return [rawInt & 0xFF];
    }
  }

  /// Записывает изменённую карту в буфер _data
  bool writeMapToBuffer(TuningMap map, RomMapDef def) {
    if (_data == null) return false;

    var data = map.data;

    // Обратное транспонирование для VE
    if (def.transpose && data.isNotEmpty) {
      final orig = <List<double>>[];
      for (int c = 0; c < data[0].length; c++) {
        orig.add([for (int r = 0; r < data.length; r++) data[r][c]]);
      }
      data = orig;
    }

    for (int r = 0; r < def.rows; r++) {
      for (int c = 0; c < def.cols; c++) {
        if (r >= data.length || c >= data[r].length) continue;
        final off  = def.address + (r * def.cols + c) * def.bytesPerCell;
        final bytes = _encodeValue(data[r][c], def);
        for (int i = 0; i < bytes.length; i++) {
          if (off + i < _data!.length) _data![off + i] = bytes[i];
        }
      }
    }
    return true;
  }

  /// Сохраняет изменённый .bin файл с префиксом NLP_MOD
  Future<String?> saveModifiedRom({int modIndex = 1}) async {
    if (_data == null || _filePath == null) return null;
    try {
      final dir      = await getApplicationDocumentsDirectory();
      final origName = _fileName ?? 'rom.bin';
      final modName  = 'NLP_MOD${modIndex}_$origName';
      final outPath  = '${dir.path}/$modName';
      await File(outPath).writeAsBytes(_data!);
      return outPath;
    } catch (_) { return null; }
  }

  /// Сохраняет все карты как дефолтные в MapStorageService
  Future<int> saveAllAsDefaults() async {
    final maps = readAllMaps();
    for (final m in maps) await MapStorageService.saveMap(m);
    return maps.length;
  }

  // ── Сравнение двух ROM ───────────────────────────────────────
  static List<RomDiff> compareRoms(
      RomMapReader rom1, RomMapReader rom2) {
    final diffs = <RomDiff>[];
    for (final def in standardMaps) {
      final m1 = rom1.readMap(def);
      final m2 = rom2.readMap(def);
      if (m1 == null || m2 == null) continue;

      final cellDiffs = <RomCellDiff>[];
      for (int r = 0; r < m1.rows && r < m2.rows; r++) {
        for (int c = 0; c < m1.cols && c < m2.cols; c++) {
          final v1 = m1.data[r][c];
          final v2 = m2.data[r][c];
          if ((v1 - v2).abs() > 0.001) {
            cellDiffs.add(RomCellDiff(
              rpmIdx: r, loadIdx: c,
              rpm:  m1.rpmAxis[r],
              load: m1.loadAxis[c],
              valueA: v1, valueB: v2,
            ));
          }
        }
      }
      diffs.add(RomDiff(
        mapName:   def.name,
        mapDef:    def,
        mapA:      m1,
        mapB:      m2,
        cellDiffs: cellDiffs,
      ));
    }
    return diffs;
  }
}

class RomCellDiff {
  final int    rpmIdx;
  final int    loadIdx;
  final double rpm;
  final double load;
  final double valueA;
  final double valueB;

  const RomCellDiff({
    required this.rpmIdx,
    required this.loadIdx,
    required this.rpm,
    required this.load,
    required this.valueA,
    required this.valueB,
  });

  double get delta        => valueB - valueA;
  double get deltaPercent =>
      valueA != 0 ? (delta / valueA.abs()) * 100 : 0;
}

class RomDiff {
  final String        mapName;
  final RomMapDef     mapDef;
  final TuningMap     mapA;
  final TuningMap     mapB;
  final List<RomCellDiff> cellDiffs;

  const RomDiff({
    required this.mapName,
    required this.mapDef,
    required this.mapA,
    required this.mapB,
    required this.cellDiffs,
  });

  bool get hasDiffs => cellDiffs.isNotEmpty;
}
''')
print("✅ rom_map_reader.dart (чтение + запись NLP_MOD + сравнение ROM)")

# ================================================================
# analyzer_service.dart — полный с паттернами
# ================================================================
with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:math';
import 'package:csv/csv.dart';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

// ── Паттерны настройки ───────────────────────────────────────
enum TuningPatternType { maxPower, economy, stability }

class TuningPattern {
  final TuningPatternType type;
  final String  name;
  final String  description;
  final double  knockTolerance;
  final double  afrLean;
  final double  afrRich;
  final double  timingAggression; // 0..1
  final double  fuelTrimThreshold;
  final double  vtcAggression;    // 0..1

  const TuningPattern({
    required this.type,
    required this.name,
    required this.description,
    this.knockTolerance    = 1.0,
    this.afrLean           = 15.0,
    this.afrRich           = 12.5,
    this.timingAggression  = 0.5,
    this.fuelTrimThreshold = 3.0,
    this.vtcAggression     = 0.5,
  });

  static const maxPower = TuningPattern(
    type: TuningPatternType.maxPower,
    name: 'Максимальная мощность',
    description: 'Агрессивный УОЗ, богатая WOT смесь, оптимальный VTC',
    knockTolerance:    0.3,
    afrLean:           14.0,
    afrRich:           11.5,
    timingAggression:  0.9,
    fuelTrimThreshold: 2.0,
    vtcAggression:     0.9,
  );

  static const economy = TuningPattern(
    type: TuningPatternType.economy,
    name: 'Минимальный расход',
    description: 'Бедная смесь на круизе, умеренный УОЗ',
    knockTolerance:    0.5,
    afrLean:           15.5,
    afrRich:           13.5,
    timingAggression:  0.3,
    fuelTrimThreshold: 5.0,
    vtcAggression:     0.3,
  );

  static const stability = TuningPattern(
    type: TuningPatternType.stability,
    name: 'Стабильная работа',
    description: 'Консервативные настройки, минимум детонации',
    knockTolerance:    2.0,
    afrLean:           14.7,
    afrRich:           13.0,
    timingAggression:  0.1,
    fuelTrimThreshold: 3.0,
    vtcAggression:     0.2,
  );

  static const List<TuningPattern> all = [maxPower, economy, stability];
}

// ── Сервис анализа ───────────────────────────────────────────
class AnalyzerService {
  static const _minSamples   = 2;
  static const _minConf      = 0.4;

  // ── Spark Advance ────────────────────────────────────────────
  Future<AnalysisResult> analyzeSparkMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log
        .where((d) => d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0)
        .toList();
    if (valid.length < 10) {
      return _empty('Spark Advance', log.length, 'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts  = entry.key.split(',');
      final ri     = int.parse(parts[0]);
      final li     = int.parse(parts[1]);
      final cur    = map.data[ri][li];

      final avgTiming = _avg(samples.map((d) => d.actualIgnition));
      final avgKnock  = _avg(samples.map((d) => d.knockRetard));
      final avgAFR    = _avg(samples.map((d) => d.afr));

      double sug  = cur;
      double conf = 0;
      String why  = '';

      if (avgKnock > pattern.knockTolerance * 2) {
        sug  = cur - min(avgKnock, 3.0);
        why  = 'Детонация ${avgKnock.toStringAsFixed(1)}°';
        conf = 0.9;
      } else if (avgKnock > pattern.knockTolerance) {
        sug  = cur - (1.0 * (1.0 - pattern.timingAggression));
        why  = 'Knock>${pattern.knockTolerance.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgTiming != 0 && (avgTiming - cur).abs() > 2) {
        sug  = avgTiming + (pattern.timingAggression * 2 - 1);
        why  = 'ЭБУ: ${avgTiming.toStringAsFixed(1)}° (${pattern.name})';
        conf = 0.6;
      } else if (avgKnock < pattern.knockTolerance * 0.2 &&
                 avgAFR   > pattern.afrRich &&
                 avgAFR   < pattern.afrLean) {
        sug  = cur + pattern.timingAggression * 2;
        why  = '+${(pattern.timingAggression * 2).toStringAsFixed(1)}° (стабильно)';
        conf = 0.5;
      }

      sug = sug.clamp(-5.0, 45.0);
      if ((sug - cur).abs() >= 0.5 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName:     'Spark Advance',
      analyzedAt:  DateTime.now(),
      totalSamples: log.length,
      changes:     changes,
      patternName: pattern.name,
      summary:     _summary('Spark', changes, valid.length, grouped.length),
    );
  }

  // ── Fuel / VE ────────────────────────────────────────────────
  Future<AnalysisResult> analyzeFuelMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log
        .where((d) => d.rpm > 400 && d.rpm < 7500 &&
                       d.engineLoad > 0 && d.longFuelTrim.abs() < 30)
        .toList();
    if (valid.length < 10) {
      return _empty('Fuel Map / VE', log.length, 'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri    = int.parse(parts[0]);
      final li    = int.parse(parts[1]);
      final cur   = map.data[ri][li];

      final avgSTFT = _avg(samples.map((d) => d.shortFuelTrim));
      final avgLTFT = _avg(samples.map((d) => d.longFuelTrim));
      final total   = avgSTFT + avgLTFT;
      final avgAFR  = _avg(samples.map((d) => d.afr));
      final avgLoad = _avg(samples.map((d) => d.engineLoad));

      double sug  = cur;
      double conf = 0;
      String why  = '';

      if (total > pattern.fuelTrimThreshold) {
        sug  = cur * (1 + total / 100.0);
        why  = 'Trim +${total.toStringAsFixed(1)}% (бедно)';
        conf = min(0.9, total.abs() / 10);
      } else if (total < -pattern.fuelTrimThreshold) {
        sug  = cur * (1 + total / 100.0);
        why  = 'Trim ${total.toStringAsFixed(1)}% (богато)';
        conf = min(0.9, total.abs() / 10);
      } else if (avgAFR > pattern.afrLean && avgLoad > 50) {
        sug  = cur * 1.04;
        why  = 'AFR ${avgAFR.toStringAsFixed(1)} бедно (нагрузка)';
        conf = 0.6;
      } else if (avgAFR < pattern.afrRich && avgLoad > 50) {
        sug  = cur * 0.96;
        why  = 'AFR ${avgAFR.toStringAsFixed(1)} богато (нагрузка)';
        conf = 0.6;
      }

      // Ограничение: не более 15% за раз
      final maxChange = cur * 0.15;
      final delta     = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + delta;

      if ((sug - cur).abs() / (cur.abs() + 0.001) >= 0.01 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName:      'Fuel Map / VE',
      analyzedAt:   DateTime.now(),
      totalSamples: log.length,
      changes:      changes,
      patternName:  pattern.name,
      summary:      _summary('Fuel', changes, valid.length, grouped.length),
    );
  }

  // ── VTC ──────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeVTCMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log
        .where((d) => d.rpm > 800 && d.rpm < 7000 && d.engineLoad > 5)
        .toList();
    if (valid.length < 10) {
      return _empty('VTC Map', log.length, 'Мало данных');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts  = entry.key.split(',');
      final ri     = int.parse(parts[0]);
      final li     = int.parse(parts[1]);
      final cur    = map.data[ri][li];

      final avgAct   = _avg(samples.map((d) => d.vtcActualAngle));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));

      double sug  = cur;
      double conf = 0;
      String why  = '';

      if ((avgAct - cur).abs() > 3) {
        sug  = avgAct + (pattern.vtcAggression * 2 - 1);
        why  = 'ЭБУ: ${avgAct.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgKnock > pattern.knockTolerance && cur > 15) {
        sug  = max(0, cur - 5);
        why  = 'Детонация — снизить VTC';
        conf = 0.75;
      }

      sug = sug.clamp(0.0, 45.0);
      if ((sug - cur).abs() >= 2 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName:      'VTC Map',
      analyzedAt:   DateTime.now(),
      totalSamples: log.length,
      changes:      changes,
      patternName:  pattern.name,
      summary:      _summary('VTC', changes, valid.length, grouped.length),
    );
  }

  // ── Torque ───────────────────────────────────────────────────
  Future<AnalysisResult> analyzeTorqueMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log
        .where((d) => d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0)
        .toList();
    if (valid.length < 10) {
      return _empty('Engine Torque', log.length, 'Мало данных');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri    = int.parse(parts[0]);
      final li    = int.parse(parts[1]);
      final cur   = map.data[ri][li];

      final avgCalc  = _avg(samples.map((d) => d.calculatedTorqueNm));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));

      double sug  = cur;
      double conf = 0;
      String why  = '';

      if (avgCalc > 0 && (avgCalc - cur).abs() > cur.abs() * 0.15) {
        sug  = avgCalc;
        why  = 'Расч. момент ${avgCalc.toStringAsFixed(1)} Нм';
        conf = 0.5;
      }
      if (avgKnock > pattern.knockTolerance * 2 && cur > 50) {
        sug  = cur * 0.9;
        why  = 'Детонация ${avgKnock.toStringAsFixed(1)}° — снизить';
        conf = 0.7;
      }

      if ((sug - cur).abs() >= 5 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName:      'Engine Torque',
      analyzedAt:   DateTime.now(),
      totalSamples: log.length,
      changes:      changes,
      patternName:  pattern.name,
      summary:      _summary('Torque', changes, valid.length, grouped.length),
    );
  }

  // ── Загрузка CSV ─────────────────────────────────────────────
  Future<List<OBDData>> loadLogFromCSV(String path) async {
    final content = await File(path).readAsString();
    final rows    = const CsvToListConverter().convert(content);
    if (rows.length < 2) return [];

    final result = <OBDData>[];
    for (int i = 1; i < rows.length; i++) {
      final d = OBDData.fromCsvRow(rows[i]);
      if (d != null) result.add(d);
    }
    return result;
  }

  // ── Сплиттер / объединитель логов ────────────────────────────
  /// Объединяет несколько логов с усреднением по ячейкам карты.
  /// Дубли по timestamp (±200ms) удаляются.
  Future<List<OBDData>> mergeLogs(List<List<OBDData>> logs) async {
    // 1. Собираем все записи
    final all = <OBDData>[];
    for (final log in logs) all.addAll(log);

    // 2. Сортируем по времени
    all.sort((a, b) => a.timestamp.compareTo(b.timestamp));

    // 3. Убираем дубли (±200ms)
    final deduped = <OBDData>[];
    for (final d in all) {
      if (deduped.isEmpty) { deduped.add(d); continue; }
      final last = deduped.last;
      final diff = d.timestamp.difference(last.timestamp).inMilliseconds.abs();
      if (diff > 200) deduped.add(d);
    }
    return deduped;
  }

  // ── Вспомогательное ─────────────────────────────────────────
  Map<String, List<OBDData>> _group(List<OBDData> data, TuningMap map) {
    final result = <String, List<OBDData>>{};
    for (final d in data) {
      final ri  = _closest(map.rpmAxis,  d.rpm.toDouble());
      final li  = _closest(map.loadAxis, d.engineLoad);
      final key = '$ri,$li';
      result.putIfAbsent(key, () => []).add(d);
    }
    return result;
  }

  int _closest(List<double> axis, double v) {
    int idx = 0;
    double best = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  double _avg(Iterable<num> vals) {
    if (vals.isEmpty) return 0;
    return vals.reduce((a, b) => a + b) / vals.length;
  }

  AnalysisResult _empty(String name, int total, String msg) =>
      AnalysisResult(
        mapName: name, analyzedAt: DateTime.now(),
        totalSamples: total, changes: const [], summary: msg,
      );

  String _summary(String type, List<MapCell> ch, int total, int cells) {
    if (ch.isEmpty) {
      return '$type оптимальна\n'
             'Данных: $total | Клеток: $cells';
    }
    final inc = ch.where((c) => c.delta > 0).length;
    final dec = ch.where((c) => c.delta < 0).length;
    final avg = ch.map((c) => c.delta.abs()).reduce((a, b) => a + b) / ch.length;
    final maxD = ch.map((c) => c.delta.abs()).reduce((a, b) => a > b ? a : b);
    final trend = inc > dec * 2 ? '↑ увеличение'
                : dec > inc * 2 ? '↓ уменьшение'
                : '↕ смешанное';
    return 'Данных: $total | Клеток: $cells\n'
           'Правок: ${ch.length} (+$inc -$dec) $trend\n'
           'Среднее: ${avg.toStringAsFixed(2)} | Макс: ${maxD.toStringAsFixed(2)}';
  }
}
''')
print("✅ analyzer_service.dart (Spark/Fuel/VTC/Torque + паттерны + mergeLogs)")

# ================================================================
# tuning_service.dart — дефолтные карты
# ================================================================
with open('lib/services/tuning_service.dart', 'w') as f:
    f.write(r'''import '../models/tuning_map.dart';
import 'map_storage_service.dart';

/// Предоставляет карты с учётом сохранённых правок.
class TuningService {

  Future<TuningMap> getSparkAdvanceMap()  async => _apply(_defaultSpark());
  Future<TuningMap> getFuelMap()          async => _apply(_defaultFuel());
  Future<TuningMap> getVTCMap()           async => _apply(_defaultVTC());
  Future<TuningMap> getEngineTorqueMap()  async => _apply(_defaultTorque());

  Future<TuningMap> _apply(TuningMap def) async {
    final saved = await MapStorageService.loadMapData(def.address);
    if (saved == null) return def;
    if (saved.length != def.rows) return def;
    if (saved.isNotEmpty && saved[0].length != def.cols) return def;
    return TuningMap(
      name: def.name, address: def.address,
      rows: def.rows, cols: def.cols,
      rpmAxis: def.rpmAxis, loadAxis: def.loadAxis,
      data: saved, units: def.units,
      minValue: def.minValue, maxValue: def.maxValue,
    );
  }

  TuningMap _defaultSpark() => TuningMap(
    name:'Spark Advance WOT', address:'0x06EBC',
    rows:16, cols:16,
    rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
    loadAxis: [6,13,19,25,31,38,44,50,56,63,69,75,81,88,94,100],
    units:'deg', minValue:-5, maxValue:45,
    data: [
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,21.25,25.3],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.96,16.64,23.04,30.0],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.96,16.64,23.04,30.0],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.95,8.96,16.64,23.04,30.0],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.25,30.08,30.0],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16.64,21.13,26.13,35.59,35.5],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.09,18.56,26.37,28.43,30.08,35.5],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.96,14.21,24.32,30.08,32.77,35.59,35.5],
    ],
  );

  TuningMap _defaultFuel() => TuningMap(
    name:'Fresh Air Rate / VE', address:'0x0A754',
    rows:16, cols:15,
    rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
    loadAxis: [10,20,30,40,50,60,70,75,80,85,90,92,94,96,100],
    units:'%', minValue:15, maxValue:250,
    data: [
      [64.00,61.91,59.82,51.45,51.45,47.27,41.00,34.73,32.64,29.29,25.11,21.77,18.00,14.24,6.71],
      [91.26,74.35,68.18,59.82,59.82,55.63,49.36,43.09,41.00,37.66,33.47,30.13,26.37,22.60,15.08],
      [107.12,86.50,75.95,68.87,68.18,64.00,57.73,51.45,49.36,46.02,41.84,39.49,34.73,30.96,23.44],
      [121.54,103.94,90.59,81.58,77.93,73.04,66.23,59.82,57.73,54.38,49.86,46.86,43.09,39.33,31.80],
      [129.99,117.54,104.00,93.43,89.17,82.53,75.81,67.86,65.75,62.74,58.56,55.21,51.45,47.69,40.16],
      [135.66,128.89,114.98,107.49,101.17,92.79,86.56,78.45,75.35,71.80,66.97,63.53,59.82,56.05,52.71],
      [140.20,134.15,122.14,115.11,108.75,100.35,93.13,88.07,83.96,78.95,74.55,69.98,65.65,61.07,61.07],
      [143.44,147.12,151.11,143.73,135.47,125.71,117.25,109.65,102.46,97.72,91.96,86.91,80.84,76.80,72.43],
      [147.62,151.80,160.11,160.88,156.85,145.47,135.90,124.60,120.53,119.43,114.84,107.54,101.61,95.66,84.73],
      [151.80,155.98,166.95,171.76,176.71,176.71,172.63,158.13,154.66,143.61,131.48,124.60,117.60,112.65,97.95],
      [155.98,160.17,174.85,183.95,188.13,186.97,182.62,175.94,165.69,161.12,149.06,140.58,131.89,123.63,114.47],
      [160.17,164.35,181.21,190.75,197.15,199.95,198.86,187.59,181.50,179.13,167.96,157.77,146.63,138.34,125.50],
      [164.35,168.53,185.25,198.47,201.98,201.98,195.30,195.60,185.31,186.81,180.72,170.89,163.82,151.98,138.27],
      [168.53,172.71,189.43,201.98,206.16,206.16,201.98,197.83,193.62,184.41,181.20,170.00,164.48,156.16,146.26],
      [172.71,176.89,193.62,206.16,210.34,206.16,201.98,193.62,197.80,189.43,185.25,175.83,165.99,149.62,149.62],
      [214.52,218.70,235.43,247.97,252.15,247.97,247.97,235.43,239.61,231.25,227.07,214.52,206.16,189.43,189.43],
    ],
  );

  TuningMap _defaultVTC() => TuningMap(
    name:'VTC Intake', address:'0x06BF1',
    rows:8, cols:8,
    rpmAxis:  [800,1600,2400,3200,4000,4800,5600,6400],
    loadAxis: [0,15,30,45,60,75,90,100],
    units:'deg', minValue:0, maxValue:40,
    data: [
      [0.0,15.0,20.0,25.0,30.0,10.0,0.0,35.0],
      [0.0,20.0,30.0,35.0,35.0,20.0,10.0,25.0],
      [0.0,25.0,35.0,35.0,35.0,20.0,10.0,25.0],
      [0.0,10.0,15.0,20.0,20.0,20.0,20.0,20.0],
      [0.0,10.0,20.0,20.0,20.0,20.0,20.0,20.0],
      [0.0,10.0,15.0,15.0,15.0,15.0,15.0,15.0],
      [0.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0],
      [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    ],
  );

  TuningMap _defaultTorque() => TuningMap(
    name:'Engine Torque', address:'0x07C3C',
    rows:16, cols:16,
    rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
    loadAxis: [6,13,19,25,31,38,44,50,56,63,69,75,81,88,94,100],
    units:'Nm', minValue:-100, maxValue:200,
    data: [
      [-30.76,-8.11,14.45,37.11,57.62,81.25,101.17,115.92,115.92,115.92,115.92,115.92,115.92,115.92,115.92,115.92],
      [-30.76,-8.11,14.45,37.11,57.62,81.25,101.17,115.92,115.92,115.92,115.92,115.92,115.92,115.92,115.92,115.92],
      [-31.93,-11.23,11.72,36.82,61.13,83.30,104.39,124.38,142.38,144.63,144.63,144.63,144.63,144.63,144.63,144.63],
      [-32.81,-11.23,11.43,36.62,62.21,85.16,108.69,129.49,150.39,150.39,150.39,150.39,150.39,150.39,150.39,150.39],
      [-33.89,-11.62,12.50,38.57,64.55,87.99,109.67,133.01,154.20,154.20,154.20,154.20,154.20,154.20,154.20,154.20],
      [-35.84,-13.57,9.18,35.25,61.72,86.43,109.77,134.11,154.20,166.60,166.60,166.60,166.60,166.60,166.60,166.60],
      [-37.50,-15.23,9.08,36.23,61.72,86.52,111.43,138.19,158.01,167.58,167.58,167.58,167.58,167.58,167.58,167.58],
      [-39.75,-15.14,10.06,36.52,59.86,84.28,106.25,131.93,165.72,165.72,165.72,165.72,165.72,165.72,165.72,165.72],
      [-42.87,-15.62,10.16,34.57,58.69,84.67,113.87,135.55,161.86,168.55,168.55,168.55,168.55,168.55,168.55,168.55],
      [-46.78,-18.75,8.69,34.77,61.52,87.40,110.64,135.42,152.05,178.12,178.12,178.12,178.12,178.12,178.12,178.12],
      [-50.10,-20.90,7.42,32.42,59.47,85.64,106.98,131.35,155.18,176.17,176.17,176.17,176.17,176.17,176.17,176.17],
      [-55.18,-23.63,7.03,30.86,55.76,81.64,106.25,129.10,147.56,170.21,175.29,175.29,175.29,175.29,175.29,175.29],
      [-60.55,-29.30,1.95,28.22,52.54,77.15,100.29,119.14,134.99,151.95,171.39,171.39,171.39,171.39,171.39,171.39],
      [-64.16,-31.84,0.59,25.98,50.59,75.88,97.27,116.89,133.40,156.84,158.01,158.01,158.01,158.01,158.01,158.01],
      [-66.70,-33.89,-0.98,23.83,48.14,72.75,95.61,114.26,132.13,152.25,152.25,152.25,152.25,152.25,152.25,152.25],
      [-64.26,-32.52,-0.88,23.83,47.14,72.75,95.61,114.26,132.13,152.25,152.25,152.25,152.25,152.25,152.25,152.25],
    ],
  );
}
''')
print("✅ tuning_service.dart")

# ================================================================
# ecu_map_reader.dart
# ================================================================
with open('lib/services/ecu_map_reader.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import '../models/custom_map_def.dart';
import 'obd_service.dart';
import 'settings_service.dart';
import 'formula_evaluator.dart';

class MemReadCommand {
  final String id;
  final String description;
  final String Function(int address, int length) builder;
  final String responsePrefix;

  const MemReadCommand({
    required this.id,
    required this.description,
    required this.builder,
    required this.responsePrefix,
  });
}

class EcuMapReaderService {
  final OBDService _obd;
  EcuMapReaderService(this._obd);

  static final List<MemReadCommand> knownCommands = [
    MemReadCommand(
      id: '23_kwp', description: 'KWP2000 (23)',
      responsePrefix: '63',
      builder: (addr, len) {
        final a = addr.toRadixString(16).toUpperCase().padLeft(6, '0');
        final l = len.toRadixString(16).toUpperCase().padLeft(2, '0');
        return '23$a$l';
      },
    ),
    MemReadCommand(
      id: '21_consult', description: 'Consult II (21)',
      responsePrefix: '61',
      builder: (addr, len) {
        final a = addr.toRadixString(16).toUpperCase().padLeft(4, '0');
        return '21$a';
      },
    ),
    MemReadCommand(
      id: 'd2_nissan', description: 'Nissan D2',
      responsePrefix: '92',
      builder: (addr, len) {
        final a = addr.toRadixString(16).toUpperCase().padLeft(6, '0');
        final l = len.toRadixString(16).toUpperCase().padLeft(2, '0');
        return 'D2$a$l';
      },
    ),
  ];

  MemReadCommand? getSavedCommand() {
    final id = SettingsService.memReadCommand;
    if (id == null) return null;
    try { return knownCommands.firstWhere((c) => c.id == id); }
    catch (_) { return null; }
  }

  Future<MemReadCommand?> autoDetectCommand({
    int testAddress = 0x7C3C,
    int testLength  = 4,
    void Function(String)? onProgress,
  }) async {
    if (!_obd.isConnected || !_obd.ecuResponds) {
      onProgress?.call('ЭБУ не подключён'); return null;
    }
    for (final cmd in knownCommands) {
      final testCmd = cmd.builder(testAddress, testLength);
      onProgress?.call('→ ${cmd.description}: $testCmd');
      final resp  = await _obd.sendCommand(testCmd, timeout: 3000);
      final clean = resp.replaceAll(' ', '').toUpperCase();
      onProgress?.call('  $resp');
      if (clean.startsWith(cmd.responsePrefix) && !clean.startsWith('7F')) {
        onProgress?.call('✅ ${cmd.id}');
        await SettingsService.setMemReadCommand(cmd.id);
        return cmd;
      }
      await Future.delayed(const Duration(milliseconds: 200));
    }
    onProgress?.call('❌ Не найдено');
    return null;
  }

  Future<String> testRawCommand(String rawHex) async {
    if (!_obd.isConnected) return 'Нет подключения';
    return _obd.sendCommand(rawHex, timeout: 5000);
  }

  Future<EcuMapReadResult?> readMap({
    required CustomMapDef def,
    MemReadCommand? command,
    void Function(int done, int total)? onProgress,
  }) async {
    final cmd = command ?? getSavedCommand();
    if (cmd == null || !_obd.isConnected || !_obd.ecuResponds) return null;

    final allBytes  = <int>[];
    int   bytesRead = 0;
    const chunkSize = 32;

    while (bytesRead < def.totalBytes) {
      final chunk = (def.totalBytes - bytesRead).clamp(0, chunkSize);
      final addr  = def.address + bytesRead;
      final resp  = await _obd.sendCommand(cmd.builder(addr, chunk), timeout: 5000);
      final clean = resp.replaceAll(' ', '').toUpperCase();
      final pi    = clean.indexOf(cmd.responsePrefix);
      if (pi < 0) return null;

      String hex = clean.substring(pi + cmd.responsePrefix.length);
      if (hex.length > chunk * 2 + 6) hex = hex.substring(6);

      for (int i = 0; i + 1 < hex.length && allBytes.length < bytesRead + chunk; i += 2) {
        try { allBytes.add(int.parse(hex.substring(i, i + 2), radix: 16)); }
        catch (_) { break; }
      }
      bytesRead = allBytes.length;
      onProgress?.call(bytesRead, def.totalBytes);
      await Future.delayed(const Duration(milliseconds: 100));
    }

    final ev   = FormulaEvaluator(def.formula);
    final data = <List<double>>[];
    int idx = 0;

    for (int r = 0; r < def.rows; r++) {
      final row = <double>[];
      for (int c = 0; c < def.cols; c++) {
        int raw = 0;
        if (def.isUInt16) {
          if (idx + 1 >= allBytes.length) break;
          raw = (allBytes[idx] << 8) | allBytes[idx + 1];
          idx += 2;
        } else {
          if (idx >= allBytes.length) break;
          raw = allBytes[idx];
          idx += 1;
        }
        row.add(ev.evaluate(raw));
      }
      if (row.isNotEmpty) data.add(row);
    }

    return EcuMapReadResult(def: def, data: data, readAt: DateTime.now());
  }
}
''')
print("✅ ecu_map_reader.dart")

# ================================================================
# nissan_unlock.dart — правильные битовые операции для Dart 64-bit
# ================================================================
with open('lib/services/nissan_unlock.dart', 'w') as f:
    f.write(r'''/// Nissan 1EQ010 Seed/Key Unlock
/// Реверс-инжиниринг алгоритма из ROM ЭБУ (Ghidra, SH7054).
/// ВАЖНО: Dart использует 64-битные int.
/// Все промежуточные значения маскируются до 32/16 бит вручную.
class NissanUnlock {
  static const int _ramInit1 = 0x7374;  // RAM[0xFFFF8416]
  static const int _ramInit2 = 0xBC6A;  // RAM[0xFFFF8418]

  /// Вычисляет KEY из 4 байт SEED.
  /// [seedBytes] — ровно 4 байта из ответа ЭБУ (27 01).
  static List<int> calcKey(List<int> seedBytes) {
    assert(seedBytes.length == 4, 'SEED must be 4 bytes');

    // Разбиваем на two 16-bit halves
    int high = ((seedBytes[0] << 8) | seedBytes[1]) & 0xFFFF;
    int low  = ((seedBytes[2] << 8) | seedBytes[3]) & 0xFFFF;

    // RAM переменные (эмуляция ОЗУ ЭБУ)
    int ram1 = _ramInit1 & 0xFFFF;
    int ram2 = _ramInit2 & 0xFFFF;

    // ── Round 1 (FUN_00015dcc) ──────────────────────────────
    int r6 = (low + ram1) & 0xFFFF;
    // r5 = r6 << 2 (32-bit)
    int r5 = (r6 << 2) & 0xFFFFFFFF;
    // r1 = r5 >> 16 (Dart: уже в 64-бит, маскируем до 16)
    int r1 = (r5 >> 16) & 0xFFFF;
    // r1 = r1 + r5 (32-bit)
    r1 = (r1 + r5) & 0xFFFFFFFF;
    // r1 = r1 + r6 (32-bit, r6 расширяется до 32)
    r1 = (r1 + r6) & 0xFFFFFFFF;
    // constant + r1 (маска 0xFFFF из ROM @ 0x15E20)
    r5 = (0xFFFF + r1) & 0xFFFFFFFF;
    // XOR с high
    r5 = (r5 ^ high) & 0xFFFF;
    high = r5;

    // ── Round 2: SWAP ───────────────────────────────────────
    final tmp = high;
    high = low;
    low  = tmp;

    // ── Round 3 (FUN_00015e24) ──────────────────────────────
    r5 = (low + ram2) & 0xFFFFFFFF;
    r6 = (r5 & 0xFFFF);
    // r6 = r6 << 1
    r6 = (r6 << 1) & 0xFFFFFFFF;
    r1 = (r6 >> 16) & 0xFFFF;
    r1 = (r1 + r6) & 0xFFFFFFFF;
    r1 = (r1 + r5) & 0xFFFFFFFF;
    r5 = (0xFFFF + r1) & 0xFFFFFFFF;
    r6 = (r5 & 0xFFFF);
    // r6 = r6 << 4
    r6 = (r6 << 4) & 0xFFFFFFFF;
    r1 = (r6 >> 16) & 0xFFFF;
    r1 = (r1 + r6) & 0xFFFFFFFF;
    r5 = (r5 ^ r1) & 0xFFFFFFFF;
    r5 = (r5 ^ high) & 0xFFFF;
    high = r5;

    // ── Сборка KEY ──────────────────────────────────────────
    final keyU32 = ((high & 0xFFFF) << 16) | (low & 0xFFFF);
    return [
      (keyU32 >> 24) & 0xFF,
      (keyU32 >> 16) & 0xFF,
      (keyU32 >>  8) & 0xFF,
       keyU32        & 0xFF,
    ];
  }

  static String keyToHex(List<int> key) =>
      key.map((b) => b.toRadixString(16).padLeft(2, '0').toUpperCase()).join('');

  static String seedToHex(List<int> seed) => keyToHex(seed);
}
''')
print("✅ nissan_unlock.dart (правильные 64-bit маски)")

# ================================================================
# performance_service.dart
# ================================================================
with open('lib/services/performance_service.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import '../models/obd_data.dart';

class PerformanceService {
  bool _isRunning = false;
  bool _isWaiting = false;
  DateTime? _startTime;
  DateTime? _lastTime;
  double _distance = 0;

  double _time0to60  = 0;
  double _time0to100 = 0;
  double _time400m   = 0;
  double _maxSpeed   = 0;
  int    _maxRPM     = 0;
  double _maxHP      = 0;

  final _ctrl = StreamController<void>.broadcast();
  Stream<void> get runStream => _ctrl.stream;

  bool   get isRunning  => _isRunning;
  bool   get isWaiting  => _isWaiting;
  double get time0to60  => _time0to60;
  double get time0to100 => _time0to100;
  double get time400m   => _time400m;
  double get maxSpeed   => _maxSpeed;
  int    get maxRPM     => _maxRPM;
  double get maxHP      => _maxHP;

  void startWaiting() {
    _isWaiting = true;
    _reset();
  }

  void stop() {
    _isRunning = false;
    _isWaiting = false;
  }

  void _reset() {
    _startTime = null;
    _lastTime  = null;
    _distance  = 0;
    _time0to60 = _time0to100 = _time400m = 0;
    _maxSpeed  = 0;
    _maxRPM    = 0;
    _maxHP     = 0;
  }

  void processData(OBDData data) {
    if (!_isWaiting && !_isRunning) return;

    // Старт: скорость >= 1 и газ > 30%
    if (_isWaiting && data.speed >= 1 && data.throttlePos > 30) {
      _isRunning = true;
      _isWaiting = false;
      _startTime = DateTime.now();
      _lastTime  = _startTime;
    }

    if (!_isRunning) return;

    // Обновляем максимумы
    if (data.speed       > _maxSpeed) _maxSpeed = data.speed.toDouble();
    if (data.rpm         > _maxRPM)   _maxRPM   = data.rpm;
    if (data.calculatedHP > _maxHP)   _maxHP    = data.calculatedHP;

    final now     = DateTime.now();
    final elapsed = now.difference(_startTime!).inMilliseconds / 1000.0;

    // Дистанция
    if (_lastTime != null) {
      final dt = now.difference(_lastTime!).inMilliseconds / 1000.0;
      _distance += (data.speed / 3.6) * dt;
    }
    _lastTime = now;

    // Контрольные точки
    if (_time0to60  == 0 && data.speed >= 60)  _time0to60  = elapsed;
    if (_time0to100 == 0 && data.speed >= 100) _time0to100 = elapsed;
    if (_time400m   == 0 && _distance  >= 400) _time400m   = elapsed;

    _ctrl.add(null);

    if (_time400m > 0 && data.speed >= 200) stop();
  }

  void dispose() { _ctrl.close(); }
}
''')
print("✅ performance_service.dart")

# ================================================================
# export_service.dart
# ================================================================
with open('lib/services/export_service.dart', 'w') as f:
    f.write(r'''import 'dart:convert';
import 'dart:io';
import 'package:path_provider/path_provider.dart';
import 'package:intl/intl.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

class ExportService {
  String _ts() => DateFormat('yyyyMMdd_HHmmss').format(DateTime.now());

  Future<String> exportToWinOLS(TuningMap map) async {
    final dir  = await getApplicationDocumentsDirectory();
    final path = '${dir.path}/${_safeName(map.name)}_${_ts()}.ols';
    final sb   = StringBuffer();
    sb.writeln('# WinOLS export — Nissan Logger V6');
    sb.writeln('MAP_NAME=${map.name}');
    sb.writeln('MAP_ADDRESS=${map.address}');
    sb.writeln('X_AXIS=${map.loadAxis.join(",")}');
    sb.writeln('Y_AXIS=${map.rpmAxis.join(",")}');
    for (int i = 0; i < map.data.length; i++) {
      sb.writeln('ROW_$i=${map.data[i].map((v) => v.toStringAsFixed(2)).join(",")}');
    }
    await File(path).writeAsString(sb.toString());
    return path;
  }

  Future<String> exportToEcuEdit(TuningMap map) async {
    final dir  = await getApplicationDocumentsDirectory();
    final path = '${dir.path}/${_safeName(map.name)}_${_ts()}.txt';
    final sb   = StringBuffer();
    sb.writeln('[MAP]');
    sb.writeln('Name=${map.name}');
    sb.writeln('Address=${map.address}');
    sb.writeln('Units=${map.units}');
    sb.writeln('[DATA]');
    for (int i = 0; i < map.data.length; i++) {
      sb.write(map.rpmAxis[i].toStringAsFixed(0).padLeft(6));
      for (final v in map.data[i]) {
        sb.write(v.toStringAsFixed(2).padLeft(9));
      }
      sb.writeln();
    }
    await File(path).writeAsString(sb.toString());
    return path;
  }

  Future<String> exportToJson(
      AnalysisResult result, TuningMap orig, TuningMap upd) async {
    final dir  = await getApplicationDocumentsDirectory();
    final path = '${dir.path}/tuning_${_ts()}.json';
    final data = {
      'meta': {
        'app':      'Nissan Logger V6',
        'exported': DateTime.now().toIso8601String(),
        'pattern':  result.patternName,
      },
      'original': orig.toJson(),
      'updated':  upd.toJson(),
      'changes':  result.changes.map((c) => {
        'rpm': c.rpm, 'load': c.load,
        'from': c.currentValue, 'to': c.suggestedValue,
        'delta': c.delta, 'reason': c.reason,
      }).toList(),
    };
    await File(path).writeAsString(
        const JsonEncoder.withIndent('  ').convert(data));
    return path;
  }

  Future<String> exportHexPatch(TuningMap map) async {
    final dir     = await getApplicationDocumentsDirectory();
    final path    = '${dir.path}/${_safeName(map.name)}_${_ts()}.hex';
    final base    = int.tryParse(
        map.address.replaceAll('0x','').replaceAll('0X',''), radix: 16) ?? 0;
    final sb      = StringBuffer();
    sb.writeln('; Hex patch — ${map.name}');
    sb.writeln('; Base: ${map.address}');
    for (int i = 0; i < map.data.length; i++) {
      for (int j = 0; j < map.data[i].length; j++) {
        final addr = base + (i * map.cols + j) * 2;
        sb.writeln(
          '${addr.toRadixString(16).padLeft(8,"0").toUpperCase()}: '
          '${map.data[i][j].toStringAsFixed(2)}',
        );
      }
    }
    await File(path).writeAsString(sb.toString());
    return path;
  }

  String _safeName(String n) => n.replaceAll(RegExp(r'[^a-zA-Z0-9_]'), '_');
}
''')
print("✅ export_service.dart")

print()
print("=" * 60)
print("✅ Ячейка 3/5 готова! Все сервисы созданы.")
print("=" * 60)
!ls lib/services/

✅ settings_service.dart
✅ profile_service.dart
✅ nissan_pid_library.dart
✅ formula_evaluator.dart
✅ obd_service.dart (MAF V→g/s через таблицу Hitachi, паузы polling)
✅ logger_service.dart (IOSink + try/catch + буфер 200)
✅ dtc_service.dart + dtc_database.dart (P0120/P0121/U1001 добавлены)
✅ alert_service.dart (использует пороги из профиля)
✅ map_storage_service.dart
✅ map_history_service.dart
✅ rom_map_reader.dart (чтение + запись NLP_MOD + сравнение ROM)
✅ analyzer_service.dart (Spark/Fuel/VTC/Torque + паттерны + mergeLogs)
✅ tuning_service.dart
✅ ecu_map_reader.dart
✅ nissan_unlock.dart (правильные 64-bit маски)
✅ performance_service.dart
✅ export_service.dart

✅ Ячейка 3/5 готова! Все сервисы созданы.
alert_service.dart     formula_evaluator.dart	 obd_service.dart
analyzer_service.dart  logger_service.dart	 performance_service.dart
dtc_database.dart      map_history_service.dart  profile_service.dart
dtc_service.dart       map_storage_service.dart  rom_map_reader.dart
ecu_map_reader

In [ ]:
# @title 🎨 Ячейка 4/5 ЧАСТЬ A: main + виджеты + основные экраны
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# main.dart
# ================================================================
with open('lib/main.dart', 'w') as f:
    f.write('''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'screens/home_screen.dart';
import 'services/settings_service.dart';

void main() async {
  WidgetsFlutterBinding.ensureInitialized();
  FlutterError.onError = (d) => FlutterError.presentError(d);
  await SettingsService.init();
  await SystemChrome.setPreferredOrientations([
    DeviceOrientation.portraitUp,
    DeviceOrientation.landscapeLeft,
    DeviceOrientation.landscapeRight,
  ]);
  runApp(const NissanLoggerApp());
}

class NissanLoggerApp extends StatelessWidget {
  const NissanLoggerApp({super.key});
  @override
  Widget build(BuildContext context) {
    return MaterialApp(
      title: 'Nissan Logger V6',
      debugShowCheckedModeBanner: false,
      theme: ThemeData(
        brightness: Brightness.dark,
        primarySwatch: Colors.blue,
        scaffoldBackgroundColor: const Color(0xFF1A1A2E),
        cardColor: const Color(0xFF16213E),
        colorScheme: const ColorScheme.dark(
          primary: Color(0xFFE94560),
          secondary: Color(0xFF0F3460),
          surface: Color(0xFF16213E),
        ),
        elevatedButtonTheme: ElevatedButtonThemeData(
          style: ElevatedButton.styleFrom(foregroundColor: Colors.white),
        ),
        textButtonTheme: TextButtonThemeData(
          style: TextButton.styleFrom(foregroundColor: Colors.white),
        ),
      ),
      home: const HomeScreen(),
    );
  }
}
''')
print("✅ main.dart")

# ================================================================
# widgets/fps_indicator.dart
# ================================================================
with open('lib/widgets/fps_indicator.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'package:flutter/material.dart';
import '../services/obd_service.dart';

class FpsIndicator extends StatefulWidget {
  final OBDService obdService;
  const FpsIndicator({super.key, required this.obdService});
  @override
  State<FpsIndicator> createState() => _FpsIndicatorState();
}

class _FpsIndicatorState extends State<FpsIndicator> {
  Timer? _t;
  @override
  void initState() {
    super.initState();
    _t = Timer.periodic(const Duration(seconds: 1), (_) {
      if (mounted) setState(() {});
    });
  }
  @override
  void dispose() { _t?.cancel(); super.dispose(); }

  @override
  Widget build(BuildContext context) {
    final conn = widget.obdService.isConnected;
    final ecu  = widget.obdService.ecuResponds;
    final fps  = widget.obdService.pollFps;

    Color color;
    IconData icon;
    String text;

    if (!conn) {
      color = Colors.red;
      icon  = Icons.bluetooth_disabled;
      text  = 'OFFLINE';
    } else if (!ecu) {
      color = Colors.orange;
      icon  = Icons.bluetooth_connected;
      text  = 'BT ONLY';
    } else {
      color = fps >= 5 ? Colors.green : (fps >= 3 ? Colors.orange : Colors.red);
      icon  = Icons.bluetooth_connected;
      text  = '$fps Hz';
    }

    return Padding(
      padding: const EdgeInsets.only(right: 8),
      child: Row(mainAxisSize: MainAxisSize.min, children: [
        Icon(icon, color: color, size: 18),
        const SizedBox(width: 4),
        Text(text, style: TextStyle(
          color: color, fontSize: 12, fontWeight: FontWeight.bold)),
      ]),
    );
  }
}
''')
print("✅ fps_indicator.dart")

# ================================================================
# widgets/map_table_view.dart — с градиентом + история + save/reset
# ================================================================
with open('lib/widgets/map_table_view.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';
import '../services/map_storage_service.dart';
import '../services/map_history_service.dart';

class MapTableView extends StatefulWidget {
  final TuningMap originalMap;
  final TuningMap? updatedMap;
  final List<MapCell> changes;
  final bool isFullscreen;
  final VoidCallback? onFullscreenTap;
  final VoidCallback? onSaved;

  const MapTableView({
    super.key,
    required this.originalMap,
    this.updatedMap,
    required this.changes,
    this.isFullscreen = false,
    this.onFullscreenTap,
    this.onSaved,
  });

  @override
  State<MapTableView> createState() => _MapTableViewState();
}

class _MapTableViewState extends State<MapTableView> {
  int? _selRow, _selCol;
  MapCell? _selChange;
  double _cellSize = 52;
  Map<String, MapCell> _changesMap = {};
  bool _hasSavedEdits = false;
  DateTime? _savedAt;

  @override
  void initState() {
    super.initState();
    _rebuild();
    _cellSize = widget.isFullscreen ? 62 : 52;
    _checkSaved();
  }

  @override
  void didUpdateWidget(MapTableView old) {
    super.didUpdateWidget(old);
    _rebuild();
    _checkSaved();
  }

  void _rebuild() {
    _changesMap = {};
    for (final c in widget.changes) {
      _changesMap['${c.rpmIndex}_${c.loadIndex}'] = c;
    }
  }

  Future<void> _checkSaved() async {
    final has = await MapStorageService.hasEdits(widget.originalMap.address);
    final at  = await MapStorageService.getUpdatedAt(widget.originalMap.address);
    if (mounted) setState(() { _hasSavedEdits = has; _savedAt = at; });
  }

  Future<void> _save() async {
    if (widget.updatedMap == null) return;
    final ok = await _confirm('Сохранить правки?',
        'Правок: ${widget.changes.length}');
    if (ok != true) return;
    await MapHistoryService.saveRevision(widget.updatedMap!);
    await MapStorageService.saveMap(widget.updatedMap!);
    _snack('Сохранено', Colors.green);
    await _checkSaved();
    widget.onSaved?.call();
  }

  Future<void> _reset({bool all = false}) async {
    final ok = await _confirm(
      all ? 'Сбросить ВСЕ карты?' : 'Сбросить эту карту?', '');
    if (ok != true) return;
    if (all) await MapStorageService.resetAll();
    else     await MapStorageService.resetMap(widget.originalMap.address);
    _snack('Сброшено', Colors.green);
    await _checkSaved();
    widget.onSaved?.call();
  }

  Future<bool?> _confirm(String title, String body) => showDialog<bool>(
    context: context,
    builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: Text(title),
      content: body.isNotEmpty
          ? Text(body, style: const TextStyle(color: Colors.white70))
          : null,
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false),
          child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.green),
          child: const Text('ДА')),
      ],
    ),
  );

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(
      SnackBar(content: Text(m), backgroundColor: c));
  }

  // Градиент: зелёный→жёлтый→красный
  Color _cellColor(int r, int l) {
    final change = _changesMap['${r}_$l'];
    if (change == null) {
      final val   = widget.originalMap.data[r][l];
      final range = widget.originalMap.maxValue - widget.originalMap.minValue;
      final norm  = range > 0
          ? ((val - widget.originalMap.minValue) / range).clamp(0.0, 1.0)
          : 0.5;
      if (norm < 0.5) {
        return Color.lerp(
          const Color(0xFF1B5E20), const Color(0xFF827717), norm * 2)!
            .withOpacity(0.85);
      } else {
        return Color.lerp(
          const Color(0xFF827717), const Color(0xFFB71C1C), (norm - 0.5) * 2)!
            .withOpacity(0.85);
      }
    }
    final pct = change.deltaPercent.abs();
    if (pct < 3)  return change.delta > 0 ? Colors.yellow.shade800 : Colors.lightBlue.shade800;
    if (pct < 8)  return change.delta > 0 ? Colors.orange.shade800 : Colors.blue.shade800;
    return change.delta > 0 ? Colors.red.shade800 : Colors.blueAccent.shade700;
  }

  @override
  Widget build(BuildContext context) {
    final map = widget.originalMap;
    return Column(children: [
      // Заголовок
      Container(
        padding: const EdgeInsets.all(6),
        color: const Color(0xFF16213E),
        child: Column(children: [
          Row(children: [
            const Icon(Icons.grid_on, color: Colors.cyan, size: 16),
            const SizedBox(width: 4),
            Expanded(child: Column(
              crossAxisAlignment: CrossAxisAlignment.start,
              children: [
                Text(map.name, style: const TextStyle(
                  color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12),
                  overflow: TextOverflow.ellipsis),
                if (_hasSavedEdits && _savedAt != null)
                  Text('С правками • ${DateFormat("dd.MM HH:mm").format(_savedAt!)}',
                    style: const TextStyle(color: Colors.orange, fontSize: 9)),
              ],
            )),
            Text('${map.rows}x${map.cols}',
              style: const TextStyle(color: Colors.white54, fontSize: 10)),
            if (!widget.isFullscreen && widget.onFullscreenTap != null) ...[
              const SizedBox(width: 6),
              GestureDetector(
                onTap: widget.onFullscreenTap,
                child: Container(
                  padding: const EdgeInsets.all(4),
                  decoration: BoxDecoration(
                    color: Colors.cyan.withOpacity(0.3),
                    borderRadius: BorderRadius.circular(3)),
                  child: const Icon(Icons.fullscreen, color: Colors.cyan, size: 18)),
              ),
            ],
            if (widget.isFullscreen) ...[
              const SizedBox(width: 6),
              GestureDetector(
                onTap: () => setState(() => _cellSize = (_cellSize - 8).clamp(32, 90)),
                child: const Icon(Icons.remove_circle_outline, color: Colors.white70, size: 20)),
              Text('${_cellSize.toInt()}', style: const TextStyle(color: Colors.white54, fontSize: 10)),
              GestureDetector(
                onTap: () => setState(() => _cellSize = (_cellSize + 8).clamp(32, 90)),
                child: const Icon(Icons.add_circle_outline, color: Colors.white70, size: 20)),
            ],
          ]),
          // Кнопки save/reset
          if (widget.changes.isNotEmpty || _hasSavedEdits)
            Padding(
              padding: const EdgeInsets.only(top: 6),
              child: Row(children: [
                if (widget.changes.isNotEmpty && widget.updatedMap != null)
                  Expanded(child: ElevatedButton.icon(
                    onPressed: _save,
                    icon: const Icon(Icons.save, size: 14),
                    label: const Text('Сохранить', style: TextStyle(fontSize: 11)),
                    style: ElevatedButton.styleFrom(
                      backgroundColor: Colors.green, padding: const EdgeInsets.symmetric(vertical: 8)),
                  )),
                if (widget.changes.isNotEmpty && _hasSavedEdits) const SizedBox(width: 6),
                if (_hasSavedEdits)
                  Expanded(child: PopupMenuButton<String>(
                    onSelected: (v) => v == 'one' ? _reset() : _reset(all: true),
                    itemBuilder: (_) => const [
                      PopupMenuItem(value: 'one', child: Text('Сбросить эту')),
                      PopupMenuItem(value: 'all', child: Text('Сбросить ВСЕ')),
                    ],
                    child: Container(
                      padding: const EdgeInsets.symmetric(vertical: 8, horizontal: 12),
                      decoration: BoxDecoration(
                        color: Colors.orange, borderRadius: BorderRadius.circular(4)),
                      child: const Row(
                        mainAxisAlignment: MainAxisAlignment.center,
                        children: [
                          Icon(Icons.restore, color: Colors.white, size: 16),
                          SizedBox(width: 4),
                          Text('Сброс', style: TextStyle(
                            color: Colors.white, fontSize: 11, fontWeight: FontWeight.bold)),
                        ]),
                    ),
                  )),
              ]),
            ),
        ]),
      ),
      // Выбранная ячейка
      if (_selChange != null)
        Container(
          padding: const EdgeInsets.all(6),
          margin: const EdgeInsets.all(4),
          decoration: BoxDecoration(
            color: Colors.cyan.withOpacity(0.15),
            borderRadius: BorderRadius.circular(6),
            border: Border.all(color: Colors.cyan.withOpacity(0.5))),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            Row(children: [
              Text('RPM ${_selChange!.rpm.toInt()} | Load ${_selChange!.load.toInt()}%',
                style: const TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 12)),
              const Spacer(),
              GestureDetector(
                onTap: () => setState(() { _selChange = null; _selRow = null; _selCol = null; }),
                child: const Icon(Icons.close, size: 14, color: Colors.white54)),
            ]),
            Text(
              '${_selChange!.currentValue.toStringAsFixed(2)} → ${_selChange!.suggestedValue.toStringAsFixed(2)}',
              style: TextStyle(
                color: _selChange!.delta > 0 ? Colors.green : Colors.orange, fontSize: 13)),
            Text(_selChange!.reason, style: const TextStyle(color: Colors.white70, fontSize: 10)),
          ]),
        ),
      // Таблица
      Expanded(
        child: InteractiveViewer(
          constrained: false,
          minScale: 0.5,
          maxScale: 3.0,
          boundaryMargin: const EdgeInsets.all(20),
          child: _buildTable(map),
        ),
      ),
    ]);
  }

  Widget _buildTable(TuningMap map) {
    const headerH = 28.0;
    return Padding(
      padding: const EdgeInsets.all(4),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        // Заголовок колонок
        Row(children: [
          Container(
            width: _cellSize, height: headerH,
            decoration: BoxDecoration(
              color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: const Center(child: Text('RPM/%',
              style: TextStyle(fontSize: 9, color: Colors.white70))),
          ),
          ...List.generate(map.cols, (j) => Container(
            width: _cellSize, height: headerH,
            decoration: BoxDecoration(
              color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: Center(child: Text(map.loadAxis[j].toStringAsFixed(0),
              style: const TextStyle(fontSize: 10, color: Colors.white70))),
          )),
        ]),
        // Строки
        ...List.generate(map.rows, (i) => Row(children: [
          Container(
            width: _cellSize, height: _cellSize,
            decoration: BoxDecoration(
              color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: Center(child: Text(map.rpmAxis[i].toStringAsFixed(0),
              style: const TextStyle(fontSize: 10, color: Colors.white70))),
          ),
          ...List.generate(map.cols, (j) => _cell(i, j, map)),
        ])),
      ]),
    );
  }

  Widget _cell(int i, int j, TuningMap map) {
    final change = _changesMap['${i}_$j'];
    final hasC   = change != null;
    final isSel  = _selRow == i && _selCol == j;
    final orig   = map.data[i][j];
    final upd    = widget.updatedMap?.data[i][j] ?? orig;

    return GestureDetector(
      onTap: () => setState(() { _selRow = i; _selCol = j; _selChange = change; }),
      child: Container(
        width: _cellSize, height: _cellSize,
        decoration: BoxDecoration(
          color: _cellColor(i, j),
          border: Border.all(
            color: isSel ? Colors.cyan : hasC ? Colors.white54 : Colors.white12,
            width: isSel ? 2 : (hasC ? 1 : 0.5)),
        ),
        child: hasC
          ? Column(mainAxisAlignment: MainAxisAlignment.center, children: [
              Text(orig.toStringAsFixed(1),
                style: TextStyle(fontSize: 9, color: Colors.white.withOpacity(0.7),
                  decoration: TextDecoration.lineThrough)),
              Text(upd.toStringAsFixed(1),
                style: TextStyle(fontSize: 11, fontWeight: FontWeight.bold,
                  color: change!.delta > 0 ? Colors.greenAccent : Colors.yellowAccent)),
            ])
          : Center(child: Text(orig.toStringAsFixed(1),
              style: const TextStyle(fontSize: 10, color: Colors.white70))),
      ),
    );
  }
}

/// Полноэкранный просмотр карты
class MapFullscreenView extends StatelessWidget {
  final TuningMap originalMap;
  final TuningMap? updatedMap;
  final List<MapCell> changes;
  final VoidCallback? onSaved;

  const MapFullscreenView({
    super.key,
    required this.originalMap,
    this.updatedMap,
    required this.changes,
    this.onSaved,
  });

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      backgroundColor: const Color(0xFF1A1A2E),
      appBar: AppBar(
        title: Text(originalMap.name),
        backgroundColor: const Color(0xFF16213E),
        leading: IconButton(
          icon: const Icon(Icons.close),
          onPressed: () => Navigator.pop(context)),
      ),
      body: MapTableView(
        originalMap: originalMap,
        updatedMap:  updatedMap,
        changes:     changes,
        isFullscreen: true,
        onSaved:     onSaved,
      ),
    );
  }
}
''')
print("✅ map_table_view.dart (градиент + история + save/reset)")

# ================================================================
# screens/home_screen.dart — с StreamSubscription (фикс утечки)
# ================================================================
with open('lib/screens/home_screen.dart', 'w') as f:
    f.write('''import 'dart:async';
import 'package:flutter/material.dart';
import '../services/obd_service.dart';
import '../services/logger_service.dart';
import '../services/alert_service.dart';
import '../services/profile_service.dart';
import '../services/performance_service.dart';
import '../services/settings_service.dart';
import 'dashboard_screen.dart';
import 'graph_screen.dart';
import 'log_graph_screen.dart';
import 'logging_screen.dart';
import 'events_screen.dart';
import 'dtc_screen.dart';
import 'analyzer_screen.dart';
import 'ecu_read_screen.dart';
import 'service_screen.dart';
import 'performance_screen.dart';
import 'export_screen.dart';
import 'custom_pid_screen.dart';
import 'profile_screen.dart';
import 'rom_compare_screen.dart';
import 'terminal_screen.dart';
import 'settings_screen.dart';

class _NavItem {
  final IconData icon;
  final String label;
  final Widget screen;
  const _NavItem(this.icon, this.label, this.screen);
}

class HomeScreen extends StatefulWidget {
  const HomeScreen({super.key});
  @override
  State<HomeScreen> createState() => _HomeScreenState();
}

class _HomeScreenState extends State<HomeScreen> {
  int _currentIndex = 0;

  final OBDService         _obd     = OBDService();
  final LoggerService      _logger  = LoggerService();
  final AlertService       _alert   = AlertService();
  final ProfileService     _profile = ProfileService();
  final PerformanceService _perf    = PerformanceService();

  late final List<_NavItem> _items;
  StreamSubscription? _dataSub; // ← FIX: сохраняем подписку

  @override
  void initState() {
    super.initState();
    _obd.loadTripFuel();

    // Применяем профиль
    final activeProfile = _profile.getActiveOrDefault();
    _obd.applyProfile(activeProfile);
    _alert.applyProfile(activeProfile);

    // Подписка на данные с сохранением для dispose
    _dataSub = _obd.dataStream.listen((data) {
      _logger.addData(data);
      _alert.checkData(data);
      _perf.processData(data);
    });

    _tryAutoConnect();

    _items = [
      _NavItem(Icons.speed, 'Приборы',
        DashboardScreen(obdService: _obd, alertService: _alert, profileService: _profile)),
      _NavItem(Icons.show_chart, 'Графики',
        GraphScreen(obdService: _obd)),
      _NavItem(Icons.timeline, 'ЛогГраф',
        const LogGraphScreen()),
      _NavItem(Icons.fiber_manual_record, 'Лог',
        LoggingScreen(obdService: _obd, loggerService: _logger)),
      _NavItem(Icons.notifications_active, 'События',
        EventsScreen(alertService: _alert)),
      _NavItem(Icons.warning, 'DTC',
        DTCScreen(obdService: _obd)),
      _NavItem(Icons.analytics, 'Анализ',
        AnalyzerScreen(obdService: _obd)),
      _NavItem(Icons.memory, 'ЭБУ',
        EcuReadScreen(obdService: _obd)),
      _NavItem(Icons.build, 'Сервис',
        ServiceScreen(obdService: _obd)),
      _NavItem(Icons.timer, 'Замер',
        PerformanceScreen(obdService: _obd, performanceService: _perf)),
      _NavItem(Icons.upload_file, 'Экспорт',
        const ExportScreen()),
      _NavItem(Icons.code, 'PID',
        CustomPIDScreen(obdService: _obd, profileService: _profile)),
      _NavItem(Icons.directions_car, 'Авто',
        ProfileScreen(profileService: _profile)),
      _NavItem(Icons.compare, 'ROM Diff',
        const RomCompareScreen()),
      _NavItem(Icons.terminal, 'Терминал',
        TerminalScreen(obdService: _obd)),
      _NavItem(Icons.settings, 'Настройки',
        SettingsScreen(obdService: _obd, alertService: _alert, profileService: _profile)),
    ];
  }

  Future<void> _tryAutoConnect() async {
    await Future.delayed(const Duration(seconds: 2));
    if (!SettingsService.autoConnect) return;
    final addr = SettingsService.lastBtDevice;
    if (addr != null && !_obd.isConnected) {
      try {
        final ok = await _obd.connect(addr);
        if (ok) await _obd.initECU(useCache: true);
      } catch (_) {}
    }
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      body: _items[_currentIndex].screen,
      bottomNavigationBar: Container(
        height: 72,
        decoration: const BoxDecoration(
          color: Color(0xFF16213E),
          border: Border(top: BorderSide(color: Color(0xFF0F3460), width: 0.5))),
        child: SingleChildScrollView(
          scrollDirection: Axis.horizontal,
          child: Row(
            children: List.generate(_items.length, (i) {
              final item = _items[i];
              final sel  = i == _currentIndex;
              return InkWell(
                onTap: () => setState(() => _currentIndex = i),
                child: Container(
                  width: 72,
                  padding: const EdgeInsets.symmetric(vertical: 8),
                  decoration: sel
                    ? const BoxDecoration(border: Border(
                        top: BorderSide(color: Color(0xFFE94560), width: 3)))
                    : null,
                  child: Column(
                    mainAxisAlignment: MainAxisAlignment.center,
                    children: [
                      Icon(item.icon,
                        color: sel ? const Color(0xFFE94560) : Colors.white54, size: 20),
                      const SizedBox(height: 3),
                      Text(item.label, style: TextStyle(
                        color: sel ? const Color(0xFFE94560) : Colors.white54,
                        fontSize: 9,
                        fontWeight: sel ? FontWeight.bold : FontWeight.normal),
                      ),
                    ],
                  ),
                ),
              );
            }),
          ),
        ),
      ),
    );
  }

  @override
  void dispose() {
    _dataSub?.cancel(); // ← FIX: отменяем подписку
    _obd.dispose();
    _alert.dispose();
    _perf.dispose();
    super.dispose();
  }
}
''')
print("✅ home_screen.dart (StreamSubscription fix)")

# ================================================================
# screens/dashboard_screen.dart — настраиваемый + STFT/LTFT/режим
# ================================================================
with open('lib/screens/dashboard_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import '../models/obd_data.dart';
import '../models/alert.dart';
import '../models/vehicle_profile.dart';
import '../services/obd_service.dart';
import '../services/alert_service.dart';
import '../services/profile_service.dart';
import '../widgets/fps_indicator.dart';

class _P {
  final String id, label, unit;
  final double Function(OBDData) get;
  final int digits;
  final Color color;
  const _P(this.id, this.label, this.unit, this.get, this.digits, this.color);
}

final List<_P> _allParams = [
  _P('rpm',      'RPM',       '',      (d) => d.rpm.toDouble(),         0, Colors.blue),
  _P('speed',    'Скорость',  'км/ч',  (d) => d.speed.toDouble(),       0, Colors.cyan),
  _P('timing',   'Зажигание', '°',     (d) => d.actualIgnition,         1, Colors.green),
  _P('knock',    'Knock',     '°',     (d) => d.knockRetard,            1, Colors.red),
  _P('vtc',      'VTC',       '°',     (d) => d.vtcActualAngle,         1, Colors.cyan),
  _P('load',     'Нагрузка',  '%',     (d) => d.engineLoad,             0, Colors.orange),
  _P('throttle', 'Дроссель',  '%',     (d) => d.throttlePos,            0, Colors.green),
  _P('maf_v',    'MAF V',     'V',     (d) => d.mafVoltage,             3, Colors.purple),
  _P('maf_gps',  'MAF',       'g/s',   (d) => d.mafGps,                 2, Colors.purple),
  _P('afr',      'AFR',       '',      (d) => d.afr,                    2, Colors.teal),
  _P('ect',      'ОЖ',        '°C',    (d) => d.coolantTemp.toDouble(),  0, Colors.red),
  _P('iat',      'Впуск',     '°C',    (d) => d.intakeTemp.toDouble(),   0, Colors.cyan),
  _P('batt',     'Батарея',   'V',     (d) => d.batteryVoltage,         2, Colors.yellow),
  _P('inj',      'Форсунки',  'ms',    (d) => d.injectorPulseWidth,     2, Colors.amber),
  _P('stft',     'STFT',      '%',     (d) => d.shortFuelTrim,          1, Colors.lime),
  _P('ltft',     'LTFT',      '%',     (d) => d.longFuelTrim,           1, Colors.teal),
  _P('o2',       'O2',        'V',     (d) => d.o2Voltage,              3, Colors.indigo),
  _P('hp',       'Мощность',  'л.с.',  (d) => d.calculatedHP,           1, Colors.yellow),
  _P('torque',   'Момент',    'Нм',    (d) => d.calculatedTorqueNm,     0, Colors.orange),
  _P('ve',       'VE',        '%',     (d) => d.volumetricEfficiency,    0, Colors.lightBlue),
  _P('fuel_lh',  'Расход',    'L/ч',   (d) => d.fuelFlowLph,            2, Colors.pink),
  _P('fuel_100', 'L/100км',   '',      (d) => d.fuelL100km,             1, Colors.pinkAccent),
  _P('pedal',    'Педаль',    '%',     (d) => d.acceleratorPedal,       0, Colors.green),
  _P('injduty',  'Впрыск%',  '%',     (d) => d.injectorDuty,           0, Colors.deepOrange),
  _P('trip',     'Поездка',  'L',     (d) => d.tripFuelL,              3, Colors.orange),
];

_P _param(String id) {
  try { return _allParams.firstWhere((p) => p.id == id); }
  catch (_) { return _allParams.first; }
}

class DashboardScreen extends StatefulWidget {
  final OBDService obdService;
  final AlertService alertService;
  final ProfileService profileService;
  const DashboardScreen({
    super.key,
    required this.obdService,
    required this.alertService,
    required this.profileService,
  });
  @override
  State<DashboardScreen> createState() => _DashboardScreenState();
}

class _DashboardScreenState extends State<DashboardScreen> {
  OBDData _data = OBDData(timestamp: DateTime.now());
  List<Alert> _alerts = [];
  List<String> _layout = [];

  @override
  void initState() {
    super.initState();
    final profile = widget.profileService.getActiveOrDefault();
    _layout = List<String>.from(profile.dashboardLayout);
    if (_layout.length < 6) {
      _layout = ['timing','knock','vtc','load','throttle','maf_gps',
                  'afr','ect','iat','batt','inj','fuel_lh'];
    }
    widget.obdService.dataStream.listen((data) {
      if (mounted) setState(() {
        _data = data;
        _alerts = widget.alertService.recentAlerts.take(3).toList();
      });
    });
  }

  Future<void> _saveLayout() async {
    final profile = widget.profileService.getActiveOrDefault();
    await widget.profileService.update(
      profile.copyWith(dashboardLayout: _layout));
  }

  void _pickParam(int index) {
    showModalBottomSheet(
      context: context,
      backgroundColor: const Color(0xFF16213E),
      builder: (c) => Container(
        height: 400,
        padding: const EdgeInsets.all(12),
        child: Column(children: [
          const Text('Выберите параметр',
            style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
          const SizedBox(height: 8),
          Expanded(child: GridView.builder(
            gridDelegate: const SliverGridDelegateWithFixedCrossAxisCount(
              crossAxisCount: 3, childAspectRatio: 2.5,
              crossAxisSpacing: 6, mainAxisSpacing: 6),
            itemCount: _allParams.length,
            itemBuilder: (_, i) {
              final p   = _allParams[i];
              final sel = _layout.contains(p.id);
              return GestureDetector(
                onTap: () {
                  setState(() => _layout[index] = p.id);
                  _saveLayout();
                  Navigator.pop(c);
                },
                child: Container(
                  decoration: BoxDecoration(
                    color: sel ? p.color.withOpacity(0.3) : const Color(0xFF0F3460),
                    borderRadius: BorderRadius.circular(6),
                    border: Border.all(color: p.color.withOpacity(0.5))),
                  child: Center(child: Column(
                    mainAxisSize: MainAxisSize.min,
                    children: [
                      Text(p.label, style: TextStyle(
                        color: p.color, fontSize: 11, fontWeight: FontWeight.bold)),
                      Text(p.unit, style: TextStyle(
                        color: p.color.withOpacity(0.6), fontSize: 9)),
                    ],
                  )),
                ),
              );
            },
          )),
        ]),
      ),
    );
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(
        title: const Text('Приборная панель'),
        backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)],
      ),
      body: SingleChildScrollView(
        padding: const EdgeInsets.all(8),
        child: Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
          // Алерты
          if (_alerts.isNotEmpty)
            Card(
              color: _alerts.first.level == AlertLevel.danger
                ? Colors.red.withOpacity(0.3)
                : Colors.orange.withOpacity(0.3),
              child: Padding(
                padding: const EdgeInsets.all(8),
                child: Column(
                  crossAxisAlignment: CrossAxisAlignment.start,
                  children: _alerts.map((a) => Text(a.message,
                    style: const TextStyle(fontWeight: FontWeight.bold,
                      fontSize: 11, color: Colors.white))).toList(),
                ),
              ),
            ),
          // RPM + Speed
          Row(children: [
            Expanded(child: _bigGauge('RPM', _data.rpm.toString(),
              _data.rpm > 6500 ? Colors.red : _data.rpm > 5500 ? Colors.orange : Colors.green)),
            const SizedBox(width: 6),
            Expanded(child: _bigGauge('KM/H', _data.speed.toString(), Colors.blue)),
          ]),
          const SizedBox(height: 6),
          // Настраиваемая сетка
          GridView.builder(
            shrinkWrap: true,
            physics: const NeverScrollableScrollPhysics(),
            gridDelegate: const SliverGridDelegateWithFixedCrossAxisCount(
              crossAxisCount: 3, childAspectRatio: 1.8,
              crossAxisSpacing: 4, mainAxisSpacing: 4),
            itemCount: _layout.length,
            itemBuilder: (_, i) {
              final p   = _param(_layout[i]);
              final val = p.get(_data);
              return GestureDetector(
                onLongPress: () => _pickParam(i),
                child: _paramCard(p.label, val.toStringAsFixed(p.digits), p.unit, p.color),
              );
            },
          ),
          const SizedBox(height: 6),
          // HP / Torque / VE
          Card(
            color: const Color(0xFF16213E),
            child: Padding(
              padding: const EdgeInsets.all(10),
              child: Row(children: [
                _bigStat('HP', _data.calculatedHP.toStringAsFixed(1), Colors.yellow),
                _bigStat('Нм', _data.calculatedTorqueNm.toStringAsFixed(0), Colors.orange),
                _bigStat('VE%', _data.volumetricEfficiency.toStringAsFixed(0), Colors.lightBlue),
              ]),
            ),
          ),
          const SizedBox(height: 6),
          // STFT / LTFT
          Card(
            color: const Color(0xFF16213E),
            child: Padding(
              padding: const EdgeInsets.all(10),
              child: Column(children: [
                const Text('ТОПЛИВНЫЕ КОРРЕКЦИИ',
                  style: TextStyle(color: Colors.white70, fontSize: 11)),
                const SizedBox(height: 6),
                Row(children: [
                  Expanded(child: _trim('STFT', _data.shortFuelTrim)),
                  Expanded(child: _trim('LTFT', _data.longFuelTrim)),
                ]),
              ]),
            ),
          ),
          const SizedBox(height: 6),
          // Режим
          Card(
            color: const Color(0xFF16213E),
            child: Padding(
              padding: const EdgeInsets.all(10),
              child: Column(children: [
                const Text('РЕЖИМ РАБОТЫ',
                  style: TextStyle(color: Colors.white70, fontSize: 11)),
                Text(_data.engineMode, style: const TextStyle(
                  fontSize: 16, color: Colors.cyan, fontWeight: FontWeight.bold)),
              ]),
            ),
          ),
          const SizedBox(height: 4),
          const Center(child: Text('Удерживайте ячейку для замены параметра',
            style: TextStyle(color: Colors.white30, fontSize: 10))),
        ]),
      ),
    );
  }

  Widget _bigGauge(String label, String value, Color color) => Card(
    color: const Color(0xFF16213E),
    child: Padding(
      padding: const EdgeInsets.all(12),
      child: Column(children: [
        Text(label, style: const TextStyle(color: Colors.white70, fontSize: 12)),
        const SizedBox(height: 4),
        FittedBox(child: Text(value, style: TextStyle(
          color: color, fontSize: 38, fontWeight: FontWeight.bold))),
      ]),
    ),
  );

  Widget _paramCard(String label, String value, String unit, Color color) => Card(
    color: const Color(0xFF16213E),
    shape: RoundedRectangleBorder(
      borderRadius: BorderRadius.circular(8),
      side: BorderSide(color: color.withOpacity(0.2))),
    child: Padding(
      padding: const EdgeInsets.all(4),
      child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
        Text(label, style: const TextStyle(color: Colors.white54, fontSize: 9)),
        FittedBox(child: Row(
          mainAxisSize: MainAxisSize.min,
          crossAxisAlignment: CrossAxisAlignment.baseline,
          textBaseline: TextBaseline.alphabetic,
          children: [
            Text(value, style: TextStyle(
              color: color, fontSize: 15, fontWeight: FontWeight.bold)),
            if (unit.isNotEmpty)
              Text(' $unit', style: TextStyle(
                color: color.withOpacity(0.6), fontSize: 9)),
          ],
        )),
      ]),
    ),
  );

  Widget _bigStat(String label, String value, Color color) => Expanded(
    child: Column(children: [
      Text(label, style: const TextStyle(color: Colors.white70, fontSize: 11)),
      Text(value, style: TextStyle(
        fontSize: 20, color: color, fontWeight: FontWeight.bold)),
    ]),
  );

  Widget _trim(String label, double value) {
    final c = value.abs() > 15 ? Colors.red
            : value.abs() > 10 ? Colors.orange
            : Colors.green;
    return Column(children: [
      Text(label, style: const TextStyle(color: Colors.white70, fontSize: 11)),
      Text('${value.toStringAsFixed(1)}%', style: TextStyle(
        color: c, fontSize: 18, fontWeight: FontWeight.bold)),
    ]);
  }
}
''')
print("✅ dashboard_screen.dart (настраиваемый + профиль + STFT/LTFT/режим)")

# ================================================================
# Заглушки для экранов которые будут в Части B
# (чтобы home_screen компилировался)
# ================================================================
stubs = {
  'graph_screen': 'Графики',
  'log_graph_screen': 'ЛогГраф',
  'logging_screen': 'Логирование',
  'events_screen': 'События',
  'dtc_screen': 'DTC',
  'analyzer_screen': 'Анализатор',
  'ecu_read_screen': 'ЭБУ Карты',
  'service_screen': 'Сервис',
  'performance_screen': 'Замер',
  'export_screen': 'Экспорт',
  'custom_pid_screen': 'PID',
  'profile_screen': 'Профили',
  'rom_compare_screen': 'ROM Diff',
  'terminal_screen': 'Терминал',
  'settings_screen': 'Настройки',
}

for fname, title in stubs.items():
    path = f'lib/screens/{fname}.dart'
    if os.path.exists(path):
        continue
    # Определяем конструктор по имени
    class_name = ''.join(w.capitalize() for w in fname.split('_'))

    # Параметры конструктора
    params = ''
    imports = ''
    if 'obd' in fname or fname in ['settings_screen', 'dtc_screen', 'service_screen',
                                      'ecu_read_screen', 'terminal_screen',
                                      'performance_screen']:
        imports += "import '../services/obd_service.dart';\n"
        params += 'final OBDService obdService; '
    if 'logger' in fname or fname == 'logging_screen':
        imports += "import '../services/logger_service.dart';\n"
        params += 'final LoggerService loggerService; '
    if 'alert' in fname or fname == 'events_screen':
        imports += "import '../services/alert_service.dart';\n"
        params += 'final AlertService alertService; '
    if 'profile' in fname or fname in ['settings_screen', 'custom_pid_screen',
                                        'profile_screen']:
        imports += "import '../services/profile_service.dart';\n"
        params += 'final ProfileService profileService; '
    if 'performance' in fname:
        imports += "import '../services/performance_service.dart';\n"
        params += 'final PerformanceService performanceService; '

    # Строим конструктор
    if params:
        req = ', '.join(f'required this.{p.split()[-1].rstrip(";")}'
                        for p in params.strip().split('; ') if p.strip())
        req = req.replace(';;', ';').replace('; ', ', ')
        # Упрощаем
        fields = [p.strip().rstrip(';') for p in params.split(';') if p.strip()]
        field_decl = '\n'.join(f'  final {f};' for f in fields if f)
        req_params = ', '.join(
            f'required this.{f.split()[-1]}'
            for f in fields if f)
        const_prefix = '' if params else 'const '

        with open(path, 'w') as f:
            f.write(f'''import 'package:flutter/material.dart';
{imports}
class {class_name} extends StatelessWidget {{
{field_decl}
  const {class_name}({{super.key, {req_params}}});
  @override
  Widget build(BuildContext context) {{
    return Scaffold(
      appBar: AppBar(title: const Text('{title}'),
        backgroundColor: const Color(0xFF16213E)),
      body: const Center(child: Text('STUB: {title}',
        style: TextStyle(color: Colors.white54, fontSize: 18))),
    );
  }}
}}
''')
    else:
        with open(path, 'w') as f:
            f.write(f'''import 'package:flutter/material.dart';

class {class_name} extends StatelessWidget {{
  const {class_name}({{super.key}});
  @override
  Widget build(BuildContext context) {{
    return Scaffold(
      appBar: AppBar(title: const Text('{title}'),
        backgroundColor: const Color(0xFF16213E)),
      body: const Center(child: Text('STUB: {title}',
        style: TextStyle(color: Colors.white54, fontSize: 18))),
    );
  }}
}}
''')
    print(f"  📝 Заглушка: {fname}.dart")

print()
print("=" * 60)
print("✅ Ячейка 4A готова!")
print("  main.dart, home_screen, dashboard_screen")
print("  fps_indicator, map_table_view")
print("  + заглушки для 15 экранов (будут заменены в части B)")
print("=" * 60)
!ls lib/screens/
# @title 🎨 Ячейка 4/5 ЧАСТЬ B: Все экраны V6
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# settings_screen.dart — ПОЛНЫЙ
# ================================================================
with open('lib/screens/settings_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import 'package:permission_handler/permission_handler.dart';
import '../services/obd_service.dart';
import '../services/alert_service.dart';
import '../services/profile_service.dart';
import '../services/settings_service.dart';
import '../models/vehicle_profile.dart';
import '../widgets/fps_indicator.dart';

class SettingsScreen extends StatefulWidget {
  final OBDService obdService;
  final AlertService alertService;
  final ProfileService profileService;
  const SettingsScreen({super.key,
    required this.obdService, required this.alertService,
    required this.profileService});
  @override
  State<SettingsScreen> createState() => _SettingsScreenState();
}

class _SettingsScreenState extends State<SettingsScreen> {
  List<BluetoothDevice> _devices = [];
  bool _scanning = false, _connecting = false, _initializing = false;
  String _btStatus = '...';

  late VehicleProfile _prof;

  @override
  void initState() {
    super.initState();
    _prof = widget.profileService.getActiveOrDefault();
    _checkBt();
  }

  Future<void> _checkBt() async {
    await Permission.bluetoothScan.request();
    await Permission.bluetoothConnect.request();
    await Permission.location.request();
    try {
      final s = await widget.obdService.getBluetoothState();
      setState(() => _btStatus = s == BluetoothState.STATE_ON ? 'Включён' : 'Выключен');
      if (s == BluetoothState.STATE_ON) _loadDevices();
    } catch (_) { setState(() => _btStatus = 'Ошибка'); }
  }

  Future<void> _loadDevices() async {
    setState(() => _scanning = true);
    try {
      final d = await widget.obdService.getBondedDevices();
      setState(() { _devices = d; _scanning = false; });
    } catch (_) { setState(() => _scanning = false); }
  }

  Future<void> _connect(BluetoothDevice d) async {
    setState(() => _connecting = true);
    _snack('Подключение...', Colors.blue);
    final ok = await widget.obdService.connect(d.address);
    setState(() => _connecting = false);
    if (ok) _snack('BT OK! Нажми ИНИЦИАЛИЗАЦИЯ', Colors.orange);
    else    _snack('Не удалось', Colors.red);
  }

  Future<void> _initECU({bool cache = true}) async {
    if (!widget.obdService.isConnected) {
      _snack('Сначала подключись', Colors.red); return;
    }
    setState(() => _initializing = true);
    _snack('Инициализация...', Colors.blue);
    final ok = await widget.obdService.initECU(useCache: cache);
    setState(() => _initializing = false);
    if (ok) _snack('ЭБУ OK! ${widget.obdService.activePids.length} PID', Colors.green);
    else    _snack('ЭБУ не отвечает', Colors.red);
  }

  Future<void> _disconnect() async {
    await widget.obdService.disconnect();
    setState(() {});
    _snack('Отключено', Colors.orange);
  }

  Future<void> _updateProfile(VehicleProfile Function(VehicleProfile) fn) async {
    _prof = fn(_prof);
    await widget.profileService.update(_prof);
    widget.obdService.applyProfile(_prof);
    widget.alertService.applyProfile(_prof);
    setState(() {});
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(
      SnackBar(content: Text(m), backgroundColor: c,
        duration: const Duration(seconds: 2)));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Настройки'),
        backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.refresh), onPressed: _loadDevices),
        ]),
      body: ListView(padding: const EdgeInsets.all(12), children: [
        // ── BT ──────────────────────────────────────────────
        _section('BLUETOOTH', [
          Row(children: [
            Icon(_btStatus == 'Включён' ? Icons.bluetooth : Icons.bluetooth_disabled,
              color: _btStatus == 'Включён' ? Colors.green : Colors.red),
            const SizedBox(width: 8),
            Text('Статус: $_btStatus'),
          ]),
          if (_btStatus != 'Включён')
            ElevatedButton.icon(
              onPressed: () async {
                await widget.obdService.requestEnable();
                await _checkBt();
              },
              icon: const Icon(Icons.bluetooth),
              label: const Text('Включить BT'),
              style: ElevatedButton.styleFrom(backgroundColor: Colors.blue)),
        ]),
        const SizedBox(height: 8),

        // ── ELM327 ──────────────────────────────────────────
        _section('ELM327 + ИНИЦИАЛИЗАЦИЯ', [
          if (widget.obdService.ecuResponds)
            Container(
              padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
              decoration: BoxDecoration(
                color: Colors.green.withOpacity(0.3),
                borderRadius: BorderRadius.circular(4)),
              child: const Text('ЭБУ ОК',
                style: TextStyle(color: Colors.green, fontSize: 10, fontWeight: FontWeight.bold))),
          const Padding(
            padding: EdgeInsets.symmetric(vertical: 4),
            child: Text('ВАЖНО: заведи двигатель!',
              style: TextStyle(color: Colors.orange, fontSize: 11, fontWeight: FontWeight.bold))),
          if (_scanning) const Center(child: CircularProgressIndicator())
          else if (_devices.isEmpty) const Text('Нет устройств')
          else ..._devices.map(_deviceTile),
          const SizedBox(height: 8),
          if (widget.obdService.isConnected && !widget.obdService.ecuResponds)
            SizedBox(
              width: double.infinity, height: 55,
              child: ElevatedButton.icon(
                onPressed: _initializing ? null : () => _initECU(),
                icon: _initializing
                  ? const SizedBox(width: 20, height: 20,
                      child: CircularProgressIndicator(color: Colors.white, strokeWidth: 2))
                  : const Icon(Icons.settings_input_component),
                label: Text(_initializing ? 'ИНИЦИАЛИЗАЦИЯ...' : 'ИНИЦИАЛИЗАЦИЯ ЭБУ',
                  style: const TextStyle(fontSize: 15, fontWeight: FontWeight.bold)),
                style: ElevatedButton.styleFrom(backgroundColor: Colors.deepOrange))),
          if (widget.obdService.ecuResponds)
            Padding(
              padding: const EdgeInsets.only(top: 6),
              child: Row(children: [
                Expanded(child: Text(
                  'Протокол: ${widget.obdService.protocolInfo}\n'
                  'PID: ${widget.obdService.activePids.length} | FPS: ${widget.obdService.pollFps}',
                  style: const TextStyle(color: Colors.green, fontSize: 11))),
                TextButton(
                  onPressed: () => _initECU(cache: false),
                  child: const Text('Пересканировать', style: TextStyle(fontSize: 10))),
              ])),
          if (widget.obdService.isConnected)
            Padding(
              padding: const EdgeInsets.only(top: 8),
              child: SizedBox(width: double.infinity,
                child: ElevatedButton.icon(
                  onPressed: _disconnect,
                  icon: const Icon(Icons.bluetooth_disabled),
                  label: const Text('ОТКЛЮЧИТЬ', style: TextStyle(fontWeight: FontWeight.bold)),
                  style: ElevatedButton.styleFrom(backgroundColor: Colors.red)))),
        ]),
        const SizedBox(height: 8),

        // ── Автоматизация ────────────────────────────────────
        _section('АВТОМАТИЗАЦИЯ', [
          SwitchListTile(
            contentPadding: EdgeInsets.zero, dense: true,
            title: const Text('Автоподключение'),
            value: SettingsService.autoConnect,
            onChanged: (v) async {
              await SettingsService.setAutoConnect(v);
              setState(() {});
            }),
          SwitchListTile(
            contentPadding: EdgeInsets.zero, dense: true,
            title: const Text('Автолог при движении'),
            subtitle: const Text('RPM>1500 или скорость>5', style: TextStyle(fontSize: 11)),
            value: SettingsService.autoLog,
            onChanged: (v) async {
              await SettingsService.setAutoLog(v);
              setState(() {});
            }),
        ]),
        const SizedBox(height: 8),

        // ── Интервал опроса ──────────────────────────────────
        _section('ИНТЕРВАЛ ОПРОСА', [
          Text('Пауза: ${_prof.pollingInterval} мс'),
          Slider(
            value: _prof.pollingInterval.toDouble(),
            min: 0, max: 500, divisions: 50,
            label: '${_prof.pollingInterval} мс',
            onChanged: (v) => _updateProfile(
              (p) => p.copyWith(pollingInterval: v.toInt()))),
        ]),
        const SizedBox(height: 8),

        // ── MAF множитель ────────────────────────────────────
        _section('MAF МНОЖИТЕЛЬ', [
          const Text('Поверх таблицы Hitachi QR20DE',
            style: TextStyle(color: Colors.white54, fontSize: 10)),
          Text('× ${_prof.mafMultiplier.toStringAsFixed(2)}',
            style: const TextStyle(fontSize: 16)),
          Slider(
            value: _prof.mafMultiplier, min: 0.5, max: 2.0, divisions: 30,
            onChanged: (v) => _updateProfile(
              (p) => p.copyWith(mafMultiplier: v))),
          const Text('На ХХ прогретого MAF g/s ≈ 2.0-3.5',
            style: TextStyle(color: Colors.white54, fontSize: 10)),
          Row(children: [
            for (final v in [0.8, 0.9, 1.0, 1.1, 1.2]) ...[
              _preset('${v}x', _prof.mafMultiplier, v,
                (val) => _updateProfile((p) => p.copyWith(mafMultiplier: val))),
              if (v != 1.2) const SizedBox(width: 3),
            ],
          ]),
        ]),
        const SizedBox(height: 8),

        // ── Скорость ─────────────────────────────────────────
        _section('КАЛИБРОВКА СКОРОСТИ', [
          Text('× ${_prof.speedMultiplier.toStringAsFixed(3)}'),
          Slider(
            value: _prof.speedMultiplier, min: 0.8, max: 1.3, divisions: 50,
            onChanged: (v) => _updateProfile(
              (p) => p.copyWith(speedMultiplier: v))),
          const Text('Приборка обычно завышает на 3-7%',
            style: TextStyle(color: Colors.white54, fontSize: 10)),
          Row(children: [
            for (final v in [1.0, 1.03, 1.05, 1.07, 1.10]) ...[
              _preset(v.toStringAsFixed(2), _prof.speedMultiplier, v,
                (val) => _updateProfile((p) => p.copyWith(speedMultiplier: val))),
              if (v != 1.10) const SizedBox(width: 3),
            ],
          ]),
        ]),
        const SizedBox(height: 8),

        // ── Коррекция расхода ────────────────────────────────
        _section('КАЛИБРОВКА РАСХОДА', [
          Text('× ${_prof.fuelCorrection.toStringAsFixed(1)}',
            style: const TextStyle(fontSize: 16)),
          Slider(
            value: _prof.fuelCorrection, min: 0.5, max: 3.0, divisions: 25,
            onChanged: (v) => _updateProfile(
              (p) => p.copyWith(fuelCorrection: v))),
          const Text('На ХХ прогретого должно быть 0.8-1.2 L/ч',
            style: TextStyle(color: Colors.white54, fontSize: 10)),
          Row(children: [
            for (final v in [0.5, 1.0, 1.5, 2.0, 3.0]) ...[
              _preset('${v}x', _prof.fuelCorrection, v,
                (val) => _updateProfile((p) => p.copyWith(fuelCorrection: val))),
              if (v != 3.0) const SizedBox(width: 3),
            ],
          ]),
        ]),
        const SizedBox(height: 8),

        // ── Топливо поездки ──────────────────────────────────
        _section('РАСХОД ЗА ПОЕЗДКУ', [
          Text('Накоплено: ${widget.obdService.tripFuelL.toStringAsFixed(3)} L',
            style: const TextStyle(color: Colors.pink, fontSize: 14)),
          const SizedBox(height: 8),
          SizedBox(width: double.infinity,
            child: ElevatedButton.icon(
              onPressed: () async {
                await widget.obdService.resetTripFuel();
                setState(() {});
                _snack('Сброшено', Colors.green);
              },
              icon: const Icon(Icons.restart_alt),
              label: const Text('СБРОСИТЬ СЧЁТЧИК'),
              style: ElevatedButton.styleFrom(backgroundColor: Colors.orange))),
        ]),
        const SizedBox(height: 8),

        // ── Уведомления ──────────────────────────────────────
        _section('УВЕДОМЛЕНИЯ', [
          SwitchListTile(
            contentPadding: EdgeInsets.zero, dense: true,
            title: const Text('Алерты'), value: _prof.alertsEnabled,
            onChanged: (v) => _updateProfile((p) => p.copyWith(alertsEnabled: v))),
          SwitchListTile(
            contentPadding: EdgeInsets.zero, dense: true,
            title: const Text('Звук'), value: _prof.soundEnabled,
            onChanged: _prof.alertsEnabled
              ? (v) => _updateProfile((p) => p.copyWith(soundEnabled: v))
              : null),
          SwitchListTile(
            contentPadding: EdgeInsets.zero, dense: true,
            title: const Text('Вибрация'), value: _prof.vibrationEnabled,
            onChanged: _prof.alertsEnabled
              ? (v) => _updateProfile((p) => p.copyWith(vibrationEnabled: v))
              : null),
        ]),
        const SizedBox(height: 8),

        // ── Кеш ──────────────────────────────────────────────
        _section('КЕШ', [
          Text('ECU: ${SettingsService.cachedEcuId ?? "нет"}', style: const TextStyle(fontSize: 12)),
          Text('PID: ${SettingsService.cachedPidList.length}', style: const TextStyle(fontSize: 12)),
          const SizedBox(height: 6),
          OutlinedButton.icon(
            onPressed: () async {
              await SettingsService.clearPidCache();
              _snack('Кеш очищен', Colors.orange);
            },
            icon: const Icon(Icons.delete_outline),
            label: const Text('Очистить кеш PID')),
        ]),
        const SizedBox(height: 20),
      ]),
    );
  }

  Widget _section(String title, List<Widget> children) => Card(
    color: const Color(0xFF16213E),
    child: Padding(
      padding: const EdgeInsets.all(12),
      child: Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Text(title, style: const TextStyle(
            color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
          const SizedBox(height: 6),
          ...children,
        ]),
    ),
  );

  Widget _preset(String label, double current, double value,
      void Function(double) onTap) {
    final active = (current - value).abs() < 0.005;
    return Expanded(child: ElevatedButton(
      onPressed: () => onTap(value),
      style: ElevatedButton.styleFrom(
        backgroundColor: active ? Colors.orange : const Color(0xFF0F3460),
        padding: const EdgeInsets.symmetric(vertical: 6)),
      child: Text(label, style: const TextStyle(fontSize: 11, fontWeight: FontWeight.bold)),
    ));
  }

  Widget _deviceTile(BluetoothDevice d) {
    final isOBD = (d.name ?? '').toUpperCase().contains('OBD') ||
                  (d.name ?? '').toUpperCase().contains('ELM');
    return Card(
      color: isOBD ? const Color(0xFF0F3460) : const Color(0xFF1A1A2E),
      child: ListTile(
        dense: true,
        leading: Icon(Icons.bluetooth, color: isOBD ? Colors.orange : Colors.white70),
        title: Text(d.name ?? 'Unknown'),
        subtitle: Text(d.address, style: const TextStyle(fontSize: 10)),
        trailing: _connecting
          ? const SizedBox(width: 24, height: 24,
              child: CircularProgressIndicator(strokeWidth: 2))
          : ElevatedButton(
              onPressed: widget.obdService.isConnected ? null : () => _connect(d),
              style: ElevatedButton.styleFrom(
                backgroundColor: const Color(0xFFE94560),
                padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 6)),
              child: Text(widget.obdService.isConnected ? 'OK' : 'CONNECT',
                style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold))),
      ),
    );
  }
}
''')
print("✅ settings_screen.dart")

# ================================================================
# graph_screen.dart
# ================================================================
with open('lib/screens/graph_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:fl_chart/fl_chart.dart';
import '../models/obd_data.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';

class _GCfg {
  final String title, unit;
  final double Function(OBDData) get;
  final Color color;
  final double minY, maxY;
  const _GCfg(this.title, this.unit, this.get, this.color, this.minY, this.maxY);
}

class GraphScreen extends StatefulWidget {
  final OBDService obdService;
  const GraphScreen({super.key, required this.obdService});
  @override
  State<GraphScreen> createState() => _GraphScreenState();
}

class _GraphScreenState extends State<GraphScreen> {
  static const _max = 100;
  final List<OBDData> _h = [];
  String _profile = 'Зажигание';

  final _profiles = <String, List<_GCfg>>{
    'Зажигание': [
      _GCfg('RPM',       'об/мин', (d) => d.rpm.toDouble(),  Colors.blue,  0, 7000),
      _GCfg('Зажигание', '°',      (d) => d.actualIgnition,  Colors.green, -10, 45),
      _GCfg('Knock',     '°',      (d) => d.knockRetard,     Colors.red,   0, 15),
      _GCfg('Нагрузка',  '%',      (d) => d.engineLoad,      Colors.orange,0, 100),
    ],
    'Топливо': [
      _GCfg('AFR',  '', (d) => d.afr,           Colors.green,  10, 17),
      _GCfg('STFT', '%',(d) => d.shortFuelTrim, Colors.orange, -30, 30),
      _GCfg('LTFT', '%',(d) => d.longFuelTrim,  Colors.red,    -30, 30),
      _GCfg('O2',   'V',(d) => d.o2Voltage,     Colors.yellow, 0, 1),
    ],
    'Расход': [
      _GCfg('L/ч',      '', (d) => d.fuelFlowLph,       Colors.pink,      0, 40),
      _GCfg('L/100км',  '', (d) => d.fuelL100km,        Colors.pinkAccent, 0, 30),
      _GCfg('Скорость','км/ч',(d) => d.speed.toDouble(), Colors.cyan,     0, 200),
      _GCfg('MAF',  'g/s',(d) => d.mafGps,              Colors.purple,    0, 100),
    ],
    'Общий': [
      _GCfg('RPM',     'об/мин',(d) => d.rpm.toDouble(),       Colors.blue, 0, 7000),
      _GCfg('ОЖ',      '°C',   (d) => d.coolantTemp.toDouble(),Colors.red,  0, 130),
      _GCfg('Дроссель','%',     (d) => d.throttlePos,          Colors.green,0, 100),
      _GCfg('Скорость','км/ч',  (d) => d.speed.toDouble(),     Colors.cyan, 0, 200),
    ],
  };

  @override
  void initState() {
    super.initState();
    widget.obdService.dataStream.listen((d) {
      if (!mounted) return;
      setState(() {
        _h.add(d);
        if (_h.length > _max) _h.removeAt(0);
      });
    });
  }

  @override
  Widget build(BuildContext context) {
    final graphs = _profiles[_profile] ?? _profiles.values.first;
    return Scaffold(
      appBar: AppBar(title: const Text('Графики'),
        backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.clear),
            onPressed: () => setState(() => _h.clear())),
        ]),
      body: Column(children: [
        Padding(
          padding: const EdgeInsets.all(8),
          child: DropdownButtonFormField<String>(
            value: _profile, dropdownColor: const Color(0xFF16213E),
            decoration: InputDecoration(
              labelText: 'Профиль',
              border: OutlineInputBorder(borderRadius: BorderRadius.circular(8)),
              filled: true, fillColor: const Color(0xFF16213E)),
            items: _profiles.keys
              .map((k) => DropdownMenuItem(value: k, child: Text(k)))
              .toList(),
            onChanged: (v) { if (v != null) setState(() => _profile = v); }),
        ),
        Expanded(child: SingleChildScrollView(
          child: Column(children: graphs.map(_graph).toList()),
        )),
      ]),
    );
  }

  Widget _graph(_GCfg cfg) {
    if (_h.isEmpty) {
      return Card(color: const Color(0xFF16213E), margin: const EdgeInsets.all(6),
        child: Container(height: 130, alignment: Alignment.center,
          child: Text(cfg.title, style: TextStyle(color: cfg.color))));
    }
    final cur   = cfg.get(_h.last);
    final spots = <FlSpot>[];
    for (int i = 0; i < _h.length; i++) spots.add(FlSpot(i.toDouble(), cfg.get(_h[i])));

    return Card(
      color: const Color(0xFF16213E), margin: const EdgeInsets.all(6),
      child: Padding(padding: const EdgeInsets.all(10), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Row(mainAxisAlignment: MainAxisAlignment.spaceBetween, children: [
            Text(cfg.title, style: TextStyle(color: cfg.color, fontSize: 14, fontWeight: FontWeight.bold)),
            Text('${cur.toStringAsFixed(2)} ${cfg.unit}',
              style: TextStyle(color: cfg.color, fontSize: 18, fontWeight: FontWeight.bold)),
          ]),
          const SizedBox(height: 6),
          SizedBox(height: 120, child: LineChart(LineChartData(
            gridData: const FlGridData(show: false),
            titlesData: const FlTitlesData(show: false),
            borderData: FlBorderData(show: true,
              border: Border.all(color: Colors.white.withOpacity(0.1))),
            minX: 0, maxX: _max.toDouble(),
            minY: cfg.minY, maxY: cfg.maxY,
            lineBarsData: [
              LineChartBarData(spots: spots, isCurved: true, color: cfg.color,
                barWidth: 2, dotData: const FlDotData(show: false),
                belowBarData: BarAreaData(show: true, color: cfg.color.withOpacity(0.15))),
            ],
          ))),
        ],
      )),
    );
  }
}
''')
print("✅ graph_screen.dart")

# ================================================================
# logging_screen.dart
# ================================================================
with open('lib/screens/logging_screen.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:async';
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import 'package:share_plus/share_plus.dart';
import '../services/obd_service.dart';
import '../services/logger_service.dart';
import '../widgets/fps_indicator.dart';

class LoggingScreen extends StatefulWidget {
  final OBDService obdService;
  final LoggerService loggerService;
  const LoggingScreen({super.key, required this.obdService, required this.loggerService});
  @override
  State<LoggingScreen> createState() => _LoggingScreenState();
}

class _LoggingScreenState extends State<LoggingScreen> {
  List<FileSystemEntity> _logs = [];
  Timer? _t;

  @override
  void initState() { super.initState(); _load();
    _t = Timer.periodic(const Duration(seconds: 1), (_) { if (mounted) setState(() {}); });
  }
  @override
  void dispose() { _t?.cancel(); super.dispose(); }

  Future<void> _load() async {
    final l = await widget.loggerService.getSavedLogs();
    setState(() => _logs = l);
  }

  Future<void> _toggle() async {
    if (widget.loggerService.isLogging) {
      await widget.loggerService.stopLogging();
      await _load();
    } else {
      if (!widget.obdService.isConnected) return;
      await widget.loggerService.startLogging();
    }
    setState(() {});
  }

  String _size(int b) => b < 1024 ? '$b B'
      : b < 1048576 ? '${(b/1024).toStringAsFixed(1)} KB'
      : '${(b/1048576).toStringAsFixed(1)} MB';

  @override
  Widget build(BuildContext context) {
    final logging = widget.loggerService.isLogging;
    final auto    = widget.loggerService.isAutoLogging;
    return Scaffold(
      appBar: AppBar(title: const Text('Логирование'),
        backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.refresh), onPressed: _load)]),
      body: Column(children: [
        Card(
          color: logging
            ? (auto ? Colors.blue.withOpacity(0.3) : Colors.red.withOpacity(0.3))
            : const Color(0xFF16213E),
          margin: const EdgeInsets.all(12),
          child: Padding(padding: const EdgeInsets.all(16), child: Column(children: [
            Text(logging ? (auto ? 'АВТОЛОГ' : 'ЗАПИСЬ') : 'Не записывает',
              style: TextStyle(fontSize: 20, fontWeight: FontWeight.bold,
                color: logging ? (auto ? Colors.blue : Colors.red) : Colors.white)),
            const SizedBox(height: 4),
            Text('Записей: ${widget.loggerService.recordCount}',
              style: const TextStyle(color: Colors.white70)),
            const SizedBox(height: 16),
            SizedBox(width: double.infinity, height: 60,
              child: ElevatedButton.icon(
                onPressed: _toggle,
                icon: Icon(logging ? Icons.stop : Icons.fiber_manual_record, size: 32),
                label: Text(logging ? 'STOP' : 'START',
                  style: const TextStyle(fontSize: 20, fontWeight: FontWeight.bold)),
                style: ElevatedButton.styleFrom(
                  backgroundColor: logging ? Colors.red : Colors.green))),
          ])),
        ),
        Expanded(child: _logs.isEmpty
          ? const Center(child: Text('Нет логов', style: TextStyle(color: Colors.white54)))
          : ListView.builder(
              padding: const EdgeInsets.all(8), itemCount: _logs.length,
              itemBuilder: (_, i) {
                final f = _logs[i];
                final name = f.path.split('/').last;
                final stat = File(f.path).statSync();
                return Card(color: const Color(0xFF16213E),
                  child: ListTile(
                    leading: CircleAvatar(
                      backgroundColor: name.contains('auto')
                        ? Colors.blue : const Color(0xFF0F3460),
                      child: Icon(name.contains('auto')
                        ? Icons.auto_awesome : Icons.description,
                        color: Colors.white, size: 18)),
                    title: Text(name, style: const TextStyle(fontSize: 13)),
                    subtitle: Text(
                      '${DateFormat("dd.MM.yyyy HH:mm").format(stat.modified)} • ${_size(stat.size)}',
                      style: const TextStyle(fontSize: 11)),
                    trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                      IconButton(icon: const Icon(Icons.share, color: Colors.blue),
                        onPressed: () async {
                          try { await Share.shareXFiles([XFile(f.path)]); } catch (_) {}
                        }),
                      IconButton(icon: const Icon(Icons.delete, color: Colors.red),
                        onPressed: () async {
                          await widget.loggerService.deleteLog(f.path);
                          await _load();
                        }),
                    ])));
              })),
      ]),
    );
  }
}
''')
print("✅ logging_screen.dart")

# ================================================================
# events_screen.dart
# ================================================================
with open('lib/screens/events_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import '../models/alert.dart';
import '../services/alert_service.dart';

class EventsScreen extends StatefulWidget {
  final AlertService alertService;
  const EventsScreen({super.key, required this.alertService});
  @override
  State<EventsScreen> createState() => _EventsScreenState();
}

class _EventsScreenState extends State<EventsScreen> {
  Timer? _t;
  @override
  void initState() { super.initState();
    _t = Timer.periodic(const Duration(seconds: 1), (_) { if (mounted) setState(() {}); });
  }
  @override
  void dispose() { _t?.cancel(); super.dispose(); }

  Color _color(AlertLevel l) => switch (l) {
    AlertLevel.danger  => Colors.red,
    AlertLevel.warning => Colors.orange,
    AlertLevel.info    => Colors.blue,
  };

  void _detail(Alert a) {
    showDialog(context: context, builder: (c) => Dialog(
      backgroundColor: const Color(0xFF16213E),
      child: Container(
        constraints: const BoxConstraints(maxWidth: 500),
        padding: const EdgeInsets.all(16),
        child: SingleChildScrollView(child: Column(
          crossAxisAlignment: CrossAxisAlignment.start,
          mainAxisSize: MainAxisSize.min,
          children: [
            Text(a.message, style: TextStyle(color: _color(a.level),
              fontWeight: FontWeight.bold, fontSize: 15)),
            Text(DateFormat('dd.MM.yyyy HH:mm:ss').format(a.timestamp),
              style: const TextStyle(color: Colors.white54, fontSize: 11)),
            const Divider(color: Colors.white24),
            if (a.snapshot != null) ...[
              const Text('СТОП-КАДР:', style: TextStyle(color: Colors.cyan, fontSize: 12)),
              _row('RPM', a.snapshot!.rpm.toString()),
              _row('Скорость', '${a.snapshot!.speed} км/ч'),
              _row('Нагрузка', '${a.snapshot!.engineLoad.toStringAsFixed(1)}%'),
              _row('MAF', '${a.snapshot!.mafGps.toStringAsFixed(2)} g/s'),
              _row('ОЖ', '${a.snapshot!.coolantTemp}°C'),
              _row('AFR', a.snapshot!.afr.toStringAsFixed(2)),
              _row('УОЗ', '${a.snapshot!.ignitionTiming.toStringAsFixed(1)}°'),
              _row('Knock', '${a.snapshot!.knockRetard.toStringAsFixed(1)}°'),
              _row('STFT', '${a.snapshot!.shortFuelTrim.toStringAsFixed(1)}%'),
              _row('LTFT', '${a.snapshot!.longFuelTrim.toStringAsFixed(1)}%'),
            ],
            if (a.explanation != null) ...[
              const Divider(color: Colors.white24),
              const Text('ПРИЧИНА:', style: TextStyle(color: Colors.yellow, fontSize: 12)),
              const SizedBox(height: 4),
              Container(
                padding: const EdgeInsets.all(8),
                decoration: BoxDecoration(color: Colors.black26, borderRadius: BorderRadius.circular(4)),
                child: Text(a.explanation!, style: const TextStyle(color: Colors.white70, fontSize: 12))),
            ],
            const SizedBox(height: 12),
            Align(alignment: Alignment.centerRight,
              child: TextButton(onPressed: () => Navigator.pop(c),
                child: const Text('Закрыть'))),
          ],
        )),
      ),
    ));
  }

  Widget _row(String l, String v) => Padding(
    padding: const EdgeInsets.symmetric(vertical: 1),
    child: Row(children: [
      SizedBox(width: 90, child: Text(l, style: const TextStyle(color: Colors.white54, fontSize: 12))),
      Expanded(child: Text(v, style: const TextStyle(color: Colors.white, fontSize: 13, fontWeight: FontWeight.bold))),
    ]),
  );

  @override
  Widget build(BuildContext context) {
    final events = widget.alertService.allAlerts.reversed.toList();
    return Scaffold(
      appBar: AppBar(title: const Text('События'), backgroundColor: const Color(0xFF16213E),
        actions: [IconButton(icon: const Icon(Icons.delete_sweep),
          onPressed: () { widget.alertService.clearAlerts(); setState(() {}); })]),
      body: events.isEmpty
        ? const Center(child: Text('Пока событий нет', style: TextStyle(color: Colors.white54)))
        : ListView.builder(
            padding: const EdgeInsets.all(8), itemCount: events.length,
            itemBuilder: (_, i) {
              final e = events[i];
              return Card(color: const Color(0xFF16213E),
                child: ListTile(
                  leading: Icon(Icons.warning, color: _color(e.level)),
                  title: Text(e.message, style: TextStyle(color: _color(e.level),
                    fontWeight: FontWeight.bold, fontSize: 13)),
                  subtitle: Text(DateFormat('dd.MM HH:mm:ss').format(e.timestamp),
                    style: const TextStyle(color: Colors.white54, fontSize: 11)),
                  trailing: e.snapshot != null
                    ? const Icon(Icons.info_outline, color: Colors.cyan, size: 18) : null,
                  dense: true,
                  onTap: e.snapshot != null ? () => _detail(e) : null,
                ));
            }),
    );
  }
}
''')
print("✅ events_screen.dart")

# ================================================================
# dtc_screen.dart
# ================================================================
with open('lib/screens/dtc_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import '../services/obd_service.dart';
import '../services/dtc_service.dart';
import '../models/dtc_code.dart';
import '../widgets/fps_indicator.dart';

class DTCScreen extends StatefulWidget {
  final OBDService obdService;
  const DTCScreen({super.key, required this.obdService});
  @override
  State<DTCScreen> createState() => _DTCScreenState();
}

class _DTCScreenState extends State<DTCScreen> {
  late final DTCService _dtc;
  List<DTCCode> _stored = [];
  bool _loading = false, _scanned = false;

  @override
  void initState() { super.initState(); _dtc = DTCService(widget.obdService); }

  Future<void> _read() async {
    if (!widget.obdService.isConnected) return;
    setState(() => _loading = true);
    _stored = await _dtc.readStoredDTC();
    setState(() { _loading = false; _scanned = true; });
  }

  Future<void> _clear() async {
    final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Стереть ошибки?'),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false), child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.red),
          child: const Text('СТЕРЕТЬ')),
      ]));
    if (ok == true) {
      await _dtc.clearDTC();
      setState(() => _stored = []);
    }
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Ошибки DTC'), backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.refresh), onPressed: _loading ? null : _read),
          IconButton(icon: const Icon(Icons.delete_forever, color: Colors.red),
            onPressed: _loading || _stored.isEmpty ? null : _clear),
        ]),
      body: _loading
        ? const Center(child: CircularProgressIndicator())
        : Column(children: [
            Card(
              color: _stored.isEmpty && _scanned
                ? Colors.green.withOpacity(0.2)
                : _stored.isNotEmpty ? Colors.orange.withOpacity(0.2) : const Color(0xFF16213E),
              margin: const EdgeInsets.all(12),
              child: Padding(padding: const EdgeInsets.all(16), child: Row(children: [
                Icon(
                  _stored.isEmpty && _scanned ? Icons.check_circle
                    : _stored.isNotEmpty ? Icons.warning : Icons.info,
                  color: _stored.isEmpty && _scanned ? Colors.green
                    : _stored.isNotEmpty ? Colors.orange : Colors.blue,
                  size: 40),
                const SizedBox(width: 16),
                Expanded(child: Text(
                  _stored.isEmpty && _scanned ? 'Ошибок нет!'
                    : _stored.isNotEmpty ? 'Ошибок: ${_stored.length}'
                    : 'Нажмите СКАНИРОВАТЬ',
                  style: const TextStyle(fontSize: 18, fontWeight: FontWeight.bold))),
              ])),
            ),
            if (!_scanned)
              Padding(padding: const EdgeInsets.symmetric(horizontal: 16),
                child: SizedBox(width: double.infinity, height: 50,
                  child: ElevatedButton.icon(
                    onPressed: _read,
                    icon: const Icon(Icons.search),
                    label: const Text('СКАНИРОВАТЬ',
                      style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
                    style: ElevatedButton.styleFrom(backgroundColor: const Color(0xFFE94560))))),
            Expanded(child: ListView(padding: const EdgeInsets.all(8), children: [
              ..._stored.map((d) => Card(color: const Color(0xFF16213E),
                child: ListTile(
                  leading: CircleAvatar(
                    backgroundColor: d.isPending ? Colors.yellow : Colors.orange,
                    child: Text(d.code[0], style: const TextStyle(
                      color: Colors.black, fontWeight: FontWeight.bold))),
                  title: Text('${d.code}${d.isPending ? " (pending)" : ""}',
                    style: const TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
                  subtitle: Text(d.description,
                    style: const TextStyle(color: Colors.white70, fontSize: 12))))),
            ])),
          ]),
    );
  }
}
''')
print("✅ dtc_screen.dart")

# ================================================================
# analyzer_screen.dart — с скроллом в онлайн + паттерны
# ================================================================
with open('lib/screens/analyzer_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:file_picker/file_picker.dart';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';
import '../services/analyzer_service.dart';
import '../services/tuning_service.dart';
import '../services/export_service.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';
import '../widgets/map_table_view.dart';

class AnalyzerScreen extends StatefulWidget {
  final OBDService obdService;
  const AnalyzerScreen({super.key, required this.obdService});
  @override
  State<AnalyzerScreen> createState() => _AnalyzerScreenState();
}

class _AnalyzerScreenState extends State<AnalyzerScreen>
    with SingleTickerProviderStateMixin {
  final _analyzer = AnalyzerService();
  final _tuning   = TuningService();
  final _export   = ExportService();
  late final TabController _tab;

  String _map = 'Spark Advance';
  TuningPattern _pattern = TuningPattern.stability;
  TuningMap? _orig, _upd;

  // Из лога
  List<OBDData>? _log;
  String? _logName;
  AnalysisResult? _logResult;
  bool _busy = false;

  // Онлайн
  bool _recording = false;
  final List<OBDData> _buf = [];
  AnalysisResult? _onlineResult;
  StreamSubscription? _dataSub;
  Timer? _autoTimer;

  @override
  void initState() { super.initState(); _tab = TabController(length: 2, vsync: this); }
  @override
  void dispose() { _tab.dispose(); _dataSub?.cancel(); _autoTimer?.cancel(); super.dispose(); }

  // ── Общее ────────────────────────────────────────────────────
  Future<(AnalysisResult, TuningMap, TuningMap)?> _doAnalysis(List<OBDData> data) async {
    TuningMap m;
    AnalysisResult r;
    switch (_map) {
      case 'Fuel Map / VE':
        m = await _tuning.getFuelMap();
        r = await _analyzer.analyzeFuelMap(data, m, pattern: _pattern);
      case 'VTC':
        m = await _tuning.getVTCMap();
        r = await _analyzer.analyzeVTCMap(data, m, pattern: _pattern);
      case 'Torque':
        m = await _tuning.getEngineTorqueMap();
        r = await _analyzer.analyzeTorqueMap(data, m, pattern: _pattern);
      default:
        m = await _tuning.getSparkAdvanceMap();
        r = await _analyzer.analyzeSparkMap(data, m, pattern: _pattern);
    }
    final u = m.copy();
    for (final c in r.changes) u.data[c.rpmIndex][c.loadIndex] = c.suggestedValue;
    return (r, m, u);
  }

  void _openMap() {
    if (_orig == null) return;
    Navigator.push(context, MaterialPageRoute(builder: (_) => MapFullscreenView(
      originalMap: _orig!, updatedMap: _upd,
      changes: (_logResult?.changes ?? _onlineResult?.changes) ?? [])));
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(m), backgroundColor: c));
  }

  // ── Онлайн ───────────────────────────────────────────────────
  void _startRec() {
    if (!widget.obdService.isConnected || !widget.obdService.ecuResponds) {
      _snack('Нет подключения', Colors.red); return;
    }
    setState(() { _recording = true; _buf.clear(); _onlineResult = null; });
    _dataSub = widget.obdService.dataStream.listen((d) {
      if (_recording) { _buf.add(d); if (_buf.length > 5000) _buf.removeRange(0, 1000); }
    });
    _autoTimer = Timer.periodic(const Duration(seconds: 5), (_) async {
      if (_buf.length >= 20 && mounted) {
        final r = await _doAnalysis(_buf);
        if (r != null && mounted) setState(() {
          _onlineResult = r.$1; _orig = r.$2; _upd = r.$3;
        });
      }
      if (mounted) setState(() {});
    });
  }

  void _stopRec() {
    setState(() => _recording = false);
    _dataSub?.cancel(); _autoTimer?.cancel();
  }

  // ── Из лога ──────────────────────────────────────────────────
  Future<void> _loadLog() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv']);
      if (r == null) return;
      setState(() => _busy = true);
      _log = await _analyzer.loadLogFromCSV(r.files.single.path!);
      _logName = r.files.single.name;
      setState(() => _busy = false);
      _snack('Загружено: ${_log!.length}', Colors.green);
    } catch (_) { setState(() => _busy = false); }
  }

  Future<void> _analyzeLog() async {
    if (_log == null || _log!.isEmpty) return;
    setState(() => _busy = true);
    final r = await _doAnalysis(_log!);
    if (r != null) setState(() { _logResult = r.$1; _orig = r.$2; _upd = r.$3; _busy = false; });
    else setState(() => _busy = false);
  }

  // ── Сплиттер ─────────────────────────────────────────────────
  Future<void> _mergeAndAnalyze() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv'], allowMultiple: true);
      if (r == null || r.files.length < 2) {
        _snack('Выберите 2+ файла', Colors.orange); return;
      }
      setState(() => _busy = true);
      final logs = <List<OBDData>>[];
      for (final f in r.files) {
        if (f.path != null) logs.add(await _analyzer.loadLogFromCSV(f.path!));
      }
      _log = await _analyzer.mergeLogs(logs);
      _logName = '${r.files.length} логов → ${_log!.length} записей';
      setState(() => _busy = false);
      _snack('Объединено: ${_log!.length}', Colors.green);
    } catch (_) { setState(() => _busy = false); }
  }

  // ── UI ───────────────────────────────────────────────────────
  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Анализатор'), backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)],
        bottom: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.wifi), text: 'Онлайн'),
          Tab(icon: Icon(Icons.folder_open), text: 'Из лога'),
        ])),
      body: TabBarView(controller: _tab, children: [
        _onlineTab(), _logTab(),
      ]),
    );
  }

  Widget _mapSelector() {
    return Column(children: [
      Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
        child: Row(children: [
          const Text('Карта:', style: TextStyle(color: Colors.white70, fontSize: 12)),
          const SizedBox(width: 8),
          Expanded(child: DropdownButtonFormField<String>(
            value: _map, dropdownColor: const Color(0xFF16213E), isDense: true,
            items: const [
              DropdownMenuItem(value: 'Spark Advance', child: Text('Зажигание')),
              DropdownMenuItem(value: 'Fuel Map / VE', child: Text('Топливо/VE')),
              DropdownMenuItem(value: 'VTC', child: Text('VTC')),
              DropdownMenuItem(value: 'Torque', child: Text('Момент')),
            ],
            onChanged: (v) => setState(() => _map = v!))),
        ]))),
      const SizedBox(height: 4),
      Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          const Text('Паттерн:', style: TextStyle(color: Colors.orange, fontSize: 11, fontWeight: FontWeight.bold)),
          const SizedBox(height: 4),
          Wrap(spacing: 6, runSpacing: 6, children: TuningPattern.all.map((p) {
            final sel = _pattern.type == p.type;
            return GestureDetector(
              onTap: () => setState(() => _pattern = p),
              child: Container(
                padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 6),
                decoration: BoxDecoration(
                  color: sel ? Colors.orange.withOpacity(0.3) : const Color(0xFF0F3460),
                  borderRadius: BorderRadius.circular(6),
                  border: Border.all(color: sel ? Colors.orange : Colors.white24)),
                child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                  Text(p.name, style: TextStyle(
                    color: sel ? Colors.orange : Colors.white70,
                    fontSize: 11, fontWeight: sel ? FontWeight.bold : FontWeight.normal)),
                  Text(p.description, style: const TextStyle(color: Colors.white38, fontSize: 8)),
                ])));
          }).toList()),
        ]))),
    ]);
  }

  Widget _onlineTab() {
    final conn = widget.obdService.isConnected && widget.obdService.ecuResponds;
    return SingleChildScrollView(
      padding: const EdgeInsets.all(6),
      child: Column(children: [
        Card(color: _recording ? Colors.green.withOpacity(0.2) : const Color(0xFF16213E),
          child: Padding(padding: const EdgeInsets.all(8), child: Column(children: [
            Row(children: [
              Icon(conn ? Icons.check_circle : Icons.error,
                color: conn ? Colors.green : Colors.red, size: 18),
              const SizedBox(width: 6),
              Expanded(child: Text(conn ? 'ЭБУ готов' : 'Нет подключения',
                style: const TextStyle(fontSize: 12))),
              if (_recording)
                Container(
                  padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
                  decoration: BoxDecoration(color: Colors.red, borderRadius: BorderRadius.circular(4)),
                  child: const Text('REC', style: TextStyle(
                    color: Colors.white, fontWeight: FontWeight.bold, fontSize: 10))),
            ]),
            const SizedBox(height: 4),
            Row(children: [
              _stat('Записей', _buf.length.toString(), Colors.blue),
              const SizedBox(width: 4),
              _stat('Правок', (_onlineResult?.changes.length ?? 0).toString(), Colors.orange),
            ]),
          ]))),
        const SizedBox(height: 4),
        _mapSelector(),
        const SizedBox(height: 4),
        Row(children: [
          Expanded(child: ElevatedButton.icon(
            onPressed: _recording ? _stopRec : (conn ? _startRec : null),
            icon: Icon(_recording ? Icons.stop : Icons.play_arrow, size: 18),
            label: Text(_recording ? 'СТОП' : 'ЗАПИСЬ', style: const TextStyle(fontSize: 12)),
            style: ElevatedButton.styleFrom(
              backgroundColor: _recording ? Colors.red : Colors.green,
              minimumSize: const Size.fromHeight(40)))),
          const SizedBox(width: 4),
          Expanded(child: ElevatedButton.icon(
            onPressed: _buf.length >= 20 ? () async {
              final r = await _doAnalysis(_buf);
              if (r != null && mounted) setState(() {
                _onlineResult = r.$1; _orig = r.$2; _upd = r.$3;
              });
            } : null,
            icon: const Icon(Icons.refresh, size: 18),
            label: const Text('АНАЛИЗ', style: TextStyle(fontSize: 12)),
            style: ElevatedButton.styleFrom(minimumSize: const Size.fromHeight(40)))),
        ]),
        if (_onlineResult != null) ...[
          const SizedBox(height: 8),
          _resultCard(_onlineResult!),
        ],
        if (_onlineResult == null)
          const Padding(padding: EdgeInsets.all(20),
            child: Text('Нажми ЗАПИСЬ → покатайся → правки автоматически',
              style: TextStyle(color: Colors.white54, fontSize: 12),
              textAlign: TextAlign.center)),
      ]),
    );
  }

  Widget _logTab() {
    return SingleChildScrollView(
      padding: const EdgeInsets.all(6),
      child: Column(children: [
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
          child: Column(children: [
            Row(children: [
              Expanded(child: ElevatedButton.icon(
                onPressed: _busy ? null : _loadLog,
                icon: const Icon(Icons.folder_open, size: 18),
                label: const Text('CSV'),
                style: ElevatedButton.styleFrom(minimumSize: const Size.fromHeight(38)))),
              const SizedBox(width: 4),
              Expanded(child: ElevatedButton.icon(
                onPressed: _busy ? null : _mergeAndAnalyze,
                icon: const Icon(Icons.merge_type, size: 18),
                label: const Text('Объединить'),
                style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.purple,
                  minimumSize: const Size.fromHeight(38)))),
            ]),
            if (_logName != null) ...[
              const SizedBox(height: 4),
              Text(_logName!, style: const TextStyle(color: Colors.white70, fontSize: 11)),
              Text('Записей: ${_log?.length ?? 0}',
                style: const TextStyle(color: Colors.green, fontWeight: FontWeight.bold)),
            ],
          ]))),
        const SizedBox(height: 4),
        _mapSelector(),
        const SizedBox(height: 4),
        ElevatedButton.icon(
          onPressed: _busy || _log == null ? null : _analyzeLog,
          icon: const Icon(Icons.analytics, size: 18),
          label: const Text('АНАЛИЗИРОВАТЬ'),
          style: ElevatedButton.styleFrom(
            backgroundColor: Colors.green, minimumSize: const Size.fromHeight(40))),
        if (_logResult != null) ...[
          const SizedBox(height: 8),
          _resultCard(_logResult!),
        ],
      ]),
    );
  }

  Widget _stat(String l, String v, Color c) => Expanded(
    child: Container(
      padding: const EdgeInsets.all(3),
      decoration: BoxDecoration(color: const Color(0xFF0F3460), borderRadius: BorderRadius.circular(4)),
      child: Column(children: [
        Text(l, style: const TextStyle(color: Colors.white70, fontSize: 9)),
        Text(v, style: TextStyle(color: c, fontSize: 13, fontWeight: FontWeight.bold)),
      ])));

  Widget _resultCard(AnalysisResult r) {
    return Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(6),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Row(children: [
          const Icon(Icons.assessment, color: Colors.green, size: 18),
          const SizedBox(width: 4),
          Expanded(child: Text(r.mapName, style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold))),
          Text('${r.changes.length} правок', style: const TextStyle(color: Colors.orange, fontSize: 12)),
        ]),
        if (r.patternName.isNotEmpty)
          Text('Паттерн: ${r.patternName}', style: const TextStyle(color: Colors.white54, fontSize: 10)),
        Text(r.summary, style: const TextStyle(color: Colors.white70, fontSize: 10)),
        const SizedBox(height: 4),
        if (_orig != null) SizedBox(width: double.infinity,
          child: ElevatedButton.icon(onPressed: _openMap,
            icon: const Icon(Icons.grid_on, size: 20),
            label: const Text('ОТКРЫТЬ КАРТУ', style: TextStyle(fontSize: 13, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.cyan.shade700, minimumSize: const Size.fromHeight(42)))),
        const SizedBox(height: 6),
        SizedBox(height: 250,
          child: r.changes.isEmpty
            ? const Center(child: Text('Правок нет', style: TextStyle(color: Colors.green, fontSize: 14)))
            : ListView.builder(
                itemCount: r.changes.length,
                itemBuilder: (_, i) {
                  final c = r.changes[i];
                  final dc = c.delta > 0 ? Colors.green : Colors.orange;
                  return Card(color: const Color(0xFF0F3460), margin: const EdgeInsets.symmetric(vertical: 2),
                    child: ListTile(dense: true,
                      title: Text('RPM ${c.rpm.toInt()} | ${c.load.toInt()}%',
                        style: const TextStyle(fontSize: 11, fontWeight: FontWeight.bold)),
                      subtitle: Text(
                        '${c.currentValue.toStringAsFixed(1)} → ${c.suggestedValue.toStringAsFixed(1)} (${c.reason})',
                        style: TextStyle(color: dc, fontSize: 10)),
                      trailing: Text('${(c.confidence * 100).toInt()}%',
                        style: TextStyle(
                          color: c.confidence > 0.8 ? Colors.green : Colors.orange,
                          fontSize: 11, fontWeight: FontWeight.bold))));
                })),
        const SizedBox(height: 4),
        Wrap(spacing: 4, runSpacing: 4, children: [
          _expBtn('WinOLS', Colors.blue, () async {
            if (_upd != null) { final p = await _export.exportToWinOLS(_upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
          _expBtn('JSON', Colors.green, () async {
            if (_upd != null && _orig != null) {
              final p = await _export.exportToJson(r, _orig!, _upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
          _expBtn('HEX', Colors.orange, () async {
            if (_upd != null) { final p = await _export.exportHexPatch(_upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
        ]),
      ])));
  }

  Widget _expBtn(String l, Color c, VoidCallback onTap) => ElevatedButton.icon(
    onPressed: onTap, icon: const Icon(Icons.download, size: 12),
    label: Text(l, style: const TextStyle(fontSize: 10)),
    style: ElevatedButton.styleFrom(
      backgroundColor: c, padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4)));
}
''')
print("✅ analyzer_screen.dart (скролл обе вкладки + паттерны + сплиттер)")

# ================================================================
# log_graph_screen.dart
# ================================================================
with open('lib/screens/log_graph_screen.dart', 'w') as f:
    f.write(r'''import 'dart:math';
import 'package:flutter/material.dart';
import 'package:fl_chart/fl_chart.dart';
import 'package:file_picker/file_picker.dart';
import '../models/obd_data.dart';
import '../services/analyzer_service.dart';

class _PI {
  final String key, label, unit;
  final Color color;
  final double Function(OBDData) get;
  final int digits;
  const _PI(this.key, this.label, this.unit, this.color, this.get, this.digits);
}

const _params = <_PI>[
  _PI('RPM',     'RPM',       'об/мин', Colors.blue,       (d) => d.rpm.toDouble(),  0),
  _PI('Speed',   'Скорость',  'км/ч',   Colors.cyan,       (d) => d.speed.toDouble(),0),
  _PI('Timing',  'УОЗ',       '°',      Colors.lightGreen, (d) => d.actualIgnition,  1),
  _PI('Knock',   'Knock',     '°',      Colors.deepOrange, (d) => d.knockRetard,     1),
  _PI('Load',    'Нагрузка',  '%',      Colors.amber,      (d) => d.engineLoad,      1),
  _PI('MAF',     'MAF',       'g/s',    Colors.purple,     (d) => d.mafGps,          2),
  _PI('AFR',     'AFR',       '',       Colors.yellow,     (d) => d.afr,             2),
  _PI('ECT',     'ОЖ',        '°C',     Colors.red,        (d) => d.coolantTemp.toDouble(), 0),
  _PI('STFT',    'STFT',      '%',      Colors.lime,       (d) => d.shortFuelTrim,   1),
  _PI('LTFT',    'LTFT',      '%',      Colors.teal,       (d) => d.longFuelTrim,    1),
  _PI('VTC',     'VTC',       '°',      Colors.pink,       (d) => d.vtcActualAngle,  1),
  _PI('TPS',     'Дроссель',  '%',      Colors.green,      (d) => d.throttlePos,     1),
  _PI('Batt',    'Батарея',   'V',      Colors.lightBlueAccent, (d) => d.batteryVoltage, 2),
  _PI('HP',      'HP',        'л.с.',   Colors.yellowAccent, (d) => d.calculatedHP,   1),
  _PI('FuelLH',  'Расход',    'L/ч',    Colors.pink,       (d) => d.fuelFlowLph,     2),
];

_PI _p(String k) {
  try { return _params.firstWhere((p) => p.key == k); }
  catch (_) { return _params.first; }
}

class LogGraphScreen extends StatefulWidget {
  const LogGraphScreen({super.key});
  @override
  State<LogGraphScreen> createState() => _LogGraphScreenState();
}

class _LogGraphScreenState extends State<LogGraphScreen> {
  final _analyzer = AnalyzerService();
  List<OBDData>? _log;
  String? _name;
  bool _loading = false;
  Set<String> _sel = {'RPM', 'Timing', 'Knock'};
  double _rStart = 0, _rEnd = 1;
  int? _touchIdx;

  Future<void> _load() async {
    try {
      final r = await FilePicker.platform.pickFiles(type: FileType.custom, allowedExtensions: ['csv']);
      if (r == null) return;
      setState(() => _loading = true);
      _log = await _analyzer.loadLogFromCSV(r.files.single.path!);
      _name = r.files.single.name;
      _rStart = 0; _rEnd = 1; _touchIdx = null;
      setState(() => _loading = false);
    } catch (_) { setState(() => _loading = false); }
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Графики лога'), backgroundColor: const Color(0xFF16213E),
        actions: [
          if (_log != null) IconButton(icon: const Icon(Icons.zoom_out_map),
            onPressed: () => setState(() { _rStart = 0; _rEnd = 1; _touchIdx = null; })),
        ]),
      body: SingleChildScrollView(child: Column(children: [
        Padding(padding: const EdgeInsets.all(8), child: Column(children: [
          ElevatedButton.icon(onPressed: _loading ? null : _load,
            icon: const Icon(Icons.folder_open),
            label: const Text('Загрузить CSV'),
            style: ElevatedButton.styleFrom(minimumSize: const Size.fromHeight(45))),
          if (_name != null) ...[
            const SizedBox(height: 4),
            Text('$_name — ${_log?.length ?? 0} записей',
              style: const TextStyle(color: Colors.green)),
          ],
        ])),
        // Параметры
        if (_log != null)
          Padding(padding: const EdgeInsets.symmetric(horizontal: 8),
            child: Wrap(spacing: 4, runSpacing: 4, children: _params.map((p) {
              final sel = _sel.contains(p.key);
              return FilterChip(
                label: Text(p.label, style: TextStyle(fontSize: 11,
                  color: sel ? Colors.white : Colors.white70)),
                selected: sel,
                onSelected: (s) => setState(() {
                  if (s) _sel.add(p.key);
                  else if (_sel.length > 1) _sel.remove(p.key);
                }),
                selectedColor: p.color.withOpacity(0.5),
                backgroundColor: const Color(0xFF0F3460),
                materialTapTargetSize: MaterialTapTargetSize.shrinkWrap);
            }).toList())),
        if (_log != null && _log!.isNotEmpty) ...[
          // Статистика
          Padding(padding: const EdgeInsets.all(8), child: _buildStats()),
          // Совмещённый
          Padding(padding: const EdgeInsets.all(8), child: Card(color: const Color(0xFF16213E),
            child: Padding(padding: const EdgeInsets.all(10), child: Column(
              crossAxisAlignment: CrossAxisAlignment.start, children: [
                const Text('СОВМЕЩЁННЫЙ (0-100%)',
                  style: TextStyle(color: Colors.white70, fontSize: 11, fontWeight: FontWeight.bold)),
                const SizedBox(height: 8),
                SizedBox(height: 300, child: _buildCombined()),
              ])))),
          // Отдельные
          ..._sel.map((k) {
            final p = _p(k);
            return Padding(padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4),
              child: Card(color: const Color(0xFF16213E),
                child: Padding(padding: const EdgeInsets.all(10), child: Column(
                  crossAxisAlignment: CrossAxisAlignment.start, children: [
                    Text('${p.label} (${p.unit})', style: TextStyle(
                      color: p.color, fontSize: 13, fontWeight: FontWeight.bold)),
                    const SizedBox(height: 6),
                    SizedBox(height: 200, child: _buildSingle(p)),
                  ]))));
          }),
          // Zoom slider
          Padding(padding: const EdgeInsets.all(8), child: _buildSlider()),
          const SizedBox(height: 20),
        ],
      ])),
    );
  }

  Widget _buildStats() {
    if (_log == null) return const SizedBox();
    return Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
      child: Table(
        border: TableBorder.all(color: Colors.white12, width: 0.5),
        columnWidths: const {0: FlexColumnWidth(2), 1: FlexColumnWidth(1.5),
          2: FlexColumnWidth(1.5), 3: FlexColumnWidth(1.5)},
        children: [
          const TableRow(
            decoration: BoxDecoration(color: Color(0xFF0F3460)),
            children: [
              Padding(padding: EdgeInsets.all(4), child: Text('Параметр', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold))),
              Padding(padding: EdgeInsets.all(4), child: Text('Мин', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
              Padding(padding: EdgeInsets.all(4), child: Text('Сред', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
              Padding(padding: EdgeInsets.all(4), child: Text('Макс', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
            ]),
          ..._sel.map((k) {
            final p = _p(k);
            final vals = _log!.map((d) => p.get(d)).toList();
            return TableRow(children: [
              Padding(padding: const EdgeInsets.all(4), child: Text(p.label, style: const TextStyle(fontSize: 11))),
              Padding(padding: const EdgeInsets.all(4), child: Text(vals.reduce(min).toStringAsFixed(p.digits),
                style: const TextStyle(fontSize: 11, fontFamily: 'monospace'), textAlign: TextAlign.center)),
              Padding(padding: const EdgeInsets.all(4), child: Text(
                (vals.reduce((a, b) => a + b) / vals.length).toStringAsFixed(p.digits),
                style: const TextStyle(fontSize: 11, fontFamily: 'monospace'), textAlign: TextAlign.center)),
              Padding(padding: const EdgeInsets.all(4), child: Text(vals.reduce(max).toStringAsFixed(p.digits),
                style: const TextStyle(fontSize: 11, fontFamily: 'monospace'), textAlign: TextAlign.center)),
            ]);
          }),
        ])));
  }

  List<OBDData> get _range {
    if (_log == null) return [];
    final s = (_rStart * _log!.length).floor();
    final e = (_rEnd * _log!.length).ceil().clamp(s + 1, _log!.length);
    return _log!.sublist(s, e);
  }

  Widget _buildCombined() {
    final data = _range;
    if (data.isEmpty) return const SizedBox();
    int step = (data.length / 300).ceil().clamp(1, 100);
    final lines = <LineChartBarData>[];
    for (final k in _sel) {
      final p = _p(k);
      final vals = data.map((d) => p.get(d)).toList();
      final minV = vals.reduce(min);
      final maxV = vals.reduce(max);
      final r = (maxV - minV).abs() < 0.001 ? 1.0 : maxV - minV;
      final spots = <FlSpot>[];
      for (int i = 0; i < data.length; i += step) {
        spots.add(FlSpot(i.toDouble(), ((p.get(data[i]) - minV) / r * 100)));
      }
      lines.add(LineChartBarData(spots: spots, isCurved: false, color: p.color,
        barWidth: 1.5, dotData: const FlDotData(show: false)));
    }
    return LineChart(LineChartData(
      gridData: const FlGridData(show: false),
      titlesData: const FlTitlesData(show: false),
      borderData: FlBorderData(show: true, border: Border.all(color: Colors.white.withOpacity(0.1))),
      minX: 0, maxX: data.length.toDouble(), minY: 0, maxY: 100,
      lineBarsData: lines,
    ));
  }

  Widget _buildSingle(_PI p) {
    final data = _range;
    if (data.isEmpty) return const SizedBox();
    int step = (data.length / 300).ceil().clamp(1, 100);
    final spots = <FlSpot>[];
    double minY = double.infinity, maxY = -double.infinity;
    for (int i = 0; i < data.length; i += step) {
      final v = p.get(data[i]);
      spots.add(FlSpot(i.toDouble(), v));
      if (v < minY) minY = v;
      if (v > maxY) maxY = v;
    }
    if (minY == maxY) { minY -= 1; maxY += 1; }
    final margin = (maxY - minY) * 0.05;
    minY -= margin; maxY += margin;
    return LineChart(LineChartData(
      gridData: const FlGridData(show: false),
      titlesData: FlTitlesData(
        rightTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        topTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        bottomTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        leftTitles: AxisTitles(sideTitles: SideTitles(showTitles: true, reservedSize: 48,
          interval: (maxY - minY) / 4 == 0 ? 1 : (maxY - minY) / 4,
          getTitlesWidget: (v, _) => Text(v.toStringAsFixed(p.digits),
            style: const TextStyle(color: Colors.white54, fontSize: 9))))),
      borderData: FlBorderData(show: true, border: Border.all(color: Colors.white.withOpacity(0.1))),
      minX: 0, maxX: data.length.toDouble(), minY: minY, maxY: maxY,
      lineBarsData: [LineChartBarData(spots: spots, isCurved: false, color: p.color,
        barWidth: 1.5, dotData: const FlDotData(show: false),
        belowBarData: BarAreaData(show: true, color: p.color.withOpacity(0.15)))],
    ));
  }

  Widget _buildSlider() {
    if (_log == null || _log!.length < 10) return const SizedBox();
    return Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
      child: Column(children: [
        Text('ZOOM: ${(_rStart * _log!.length).toInt()} — ${(_rEnd * _log!.length).toInt()} / ${_log!.length}',
          style: const TextStyle(color: Colors.white70, fontSize: 11)),
        RangeSlider(values: RangeValues(_rStart, _rEnd), min: 0, max: 1, divisions: 100,
          activeColor: const Color(0xFFE94560), inactiveColor: Colors.white24,
          onChanged: (v) => setState(() {
            if (v.end - v.start >= 0.02) { _rStart = v.start; _rEnd = v.end; }
          })),
      ])));
  }
}
''')
print("✅ log_graph_screen.dart")

# ================================================================
# performance_screen.dart
# ================================================================
with open('lib/screens/performance_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import '../services/obd_service.dart';
import '../services/performance_service.dart';
import '../widgets/fps_indicator.dart';

class PerformanceScreen extends StatefulWidget {
  final OBDService obdService;
  final PerformanceService performanceService;
  const PerformanceScreen({super.key, required this.obdService, required this.performanceService});
  @override
  State<PerformanceScreen> createState() => _PerformanceScreenState();
}

class _PerformanceScreenState extends State<PerformanceScreen> {
  @override
  void initState() {
    super.initState();
    widget.performanceService.runStream.listen((_) { if (mounted) setState(() {}); });
  }

  void _toggle() {
    if (widget.performanceService.isRunning || widget.performanceService.isWaiting)
      widget.performanceService.stop();
    else {
      if (!widget.obdService.isConnected) return;
      widget.performanceService.startWaiting();
    }
    setState(() {});
  }

  String _fmt(double s) => s == 0 ? '--' : s.toStringAsFixed(2);

  @override
  Widget build(BuildContext context) {
    final p = widget.performanceService;
    return Scaffold(
      appBar: AppBar(title: const Text('Замер'), backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)]),
      body: Padding(padding: const EdgeInsets.all(16), child: Column(children: [
        Card(
          color: p.isRunning ? Colors.red.withOpacity(0.3)
              : p.isWaiting ? Colors.orange.withOpacity(0.3)
              : const Color(0xFF16213E),
          child: Padding(padding: const EdgeInsets.all(20), child: Column(children: [
            Icon(p.isRunning ? Icons.speed : p.isWaiting ? Icons.hourglass_bottom : Icons.timer,
              size: 60, color: p.isRunning ? Colors.red : p.isWaiting ? Colors.orange : Colors.blue),
            const SizedBox(height: 12),
            Text(p.isRunning ? 'ЗАМЕР!' : p.isWaiting ? 'ЖДУ СТАРТА...' : 'ГОТОВ',
              style: const TextStyle(fontSize: 20, fontWeight: FontWeight.bold)),
            const SizedBox(height: 16),
            SizedBox(width: double.infinity, height: 60,
              child: ElevatedButton.icon(onPressed: _toggle,
                icon: Icon(p.isRunning || p.isWaiting ? Icons.stop : Icons.play_arrow, size: 32),
                label: Text(p.isRunning || p.isWaiting ? 'СТОП' : 'НАЧАТЬ',
                  style: const TextStyle(fontSize: 18, fontWeight: FontWeight.bold)),
                style: ElevatedButton.styleFrom(
                  backgroundColor: p.isRunning || p.isWaiting ? Colors.red : Colors.green))),
          ]))),
        const SizedBox(height: 16),
        Expanded(child: GridView.count(crossAxisCount: 2, childAspectRatio: 1.5,
          crossAxisSpacing: 8, mainAxisSpacing: 8, children: [
            _c('0-60',      _fmt(p.time0to60),  'сек',  Colors.green),
            _c('0-100',     _fmt(p.time0to100), 'сек',  Colors.blue),
            _c('400м',      _fmt(p.time400m),   'сек',  Colors.orange),
            _c('Макс скор', p.maxSpeed.toStringAsFixed(0), 'км/ч', Colors.purple),
            _c('Макс RPM',  p.maxRPM.toString(),     'об', Colors.red),
            _c('Макс HP',   p.maxHP.toStringAsFixed(0), 'л.с.', Colors.yellow),
          ])),
      ])));
  }

  Widget _c(String l, String v, String u, Color c) => Card(color: const Color(0xFF16213E),
    child: Padding(padding: const EdgeInsets.all(12),
      child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
        Text(l, style: const TextStyle(color: Colors.white70, fontSize: 12)),
        const SizedBox(height: 8),
        FittedBox(child: Text(v, style: TextStyle(color: c, fontSize: 28, fontWeight: FontWeight.bold))),
        Text(u, style: TextStyle(color: c.withOpacity(0.7), fontSize: 11)),
      ])));
}
''')
print("✅ performance_screen.dart")

# ================================================================
# terminal_screen.dart
# ================================================================
with open('lib/screens/terminal_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';

class TerminalScreen extends StatefulWidget {
  final OBDService obdService;
  const TerminalScreen({super.key, required this.obdService});
  @override
  State<TerminalScreen> createState() => _TerminalScreenState();
}

class _TerminalScreenState extends State<TerminalScreen> {
  final List<String> _logs = [];
  final _cmd = TextEditingController();

  @override
  void initState() {
    super.initState();
    widget.obdService.logStream.listen((m) {
      if (mounted) setState(() { _logs.add(m); if (_logs.length > 500) _logs.removeAt(0); });
    });
  }

  Future<void> _send() async {
    final c = _cmd.text.trim().toUpperCase();
    if (c.isEmpty || !widget.obdService.isConnected) return;
    setState(() => _logs.add('>>> $c'));
    final r = await widget.obdService.sendCommand(c);
    setState(() => _logs.add('<<< $r'));
    _cmd.clear();
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Терминал'), backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.clear_all), onPressed: () => setState(() => _logs.clear())),
          IconButton(icon: const Icon(Icons.copy), onPressed: () =>
            Clipboard.setData(ClipboardData(text: _logs.join('\n')))),
        ]),
      body: Column(children: [
        Container(padding: const EdgeInsets.all(8), color: const Color(0xFF16213E),
          child: Wrap(spacing: 6, runSpacing: 6, children: [
            for (final e in [['ATZ','Reset'],['ATI','Info'],['ATRV','Volt'],
                              ['0100','Sup'],['010C','RPM'],['03','DTC'],
                              ['2701','Seed'],['1A81','ECU ID']])
              ElevatedButton(
                onPressed: () { _cmd.text = e[0]; _send(); },
                style: ElevatedButton.styleFrom(
                  backgroundColor: const Color(0xFF0F3460),
                  padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 4)),
                child: Column(mainAxisSize: MainAxisSize.min, children: [
                  Text(e[0], style: const TextStyle(fontFamily: 'monospace', fontSize: 11)),
                  Text(e[1], style: const TextStyle(fontSize: 9, color: Colors.white60)),
                ])),
          ])),
        Expanded(child: Container(color: Colors.black, padding: const EdgeInsets.all(8),
          child: SingleChildScrollView(
            child: SelectableText(_logs.join('\n'),
              style: const TextStyle(fontFamily: 'monospace', fontSize: 12, color: Colors.green))))),
        Container(color: const Color(0xFF16213E), padding: const EdgeInsets.all(8),
          child: Row(children: [
            Expanded(child: TextField(controller: _cmd,
              style: const TextStyle(fontFamily: 'monospace'),
              decoration: const InputDecoration(hintText: 'Команда...',
                border: OutlineInputBorder(),
                contentPadding: EdgeInsets.symmetric(horizontal: 12, vertical: 8)),
              onSubmitted: (_) => _send(),
              textCapitalization: TextCapitalization.characters)),
            const SizedBox(width: 8),
            IconButton(icon: const Icon(Icons.send, color: Colors.green), onPressed: _send),
          ])),
      ]),
    );
  }
}
''')
print("✅ terminal_screen.dart")

# ================================================================
# export_screen.dart
# ================================================================
with open('lib/screens/export_screen.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'package:flutter/material.dart';
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';
import '../services/tuning_service.dart';
import '../services/export_service.dart';

class ExportScreen extends StatefulWidget {
  const ExportScreen({super.key});
  @override
  State<ExportScreen> createState() => _ExportScreenState();
}

class _ExportScreenState extends State<ExportScreen> {
  final _t = TuningService();
  final _e = ExportService();
  List<FileSystemEntity> _files = [];

  @override
  void initState() { super.initState(); _load(); }

  Future<void> _load() async {
    try {
      final dir = await getApplicationDocumentsDirectory();
      final f = dir.listSync().where((f) =>
        f.path.endsWith('.ols') || f.path.endsWith('.hex') ||
        f.path.endsWith('.txt') || f.path.endsWith('.json')).toList();
      f.sort((a, b) => b.path.compareTo(a.path));
      setState(() => _files = f);
    } catch (_) {}
  }

  Future<void> _exp(String type, String fmt) async {
    try {
      final m = type == 'Spark' ? await _t.getSparkAdvanceMap()
              : type == 'Fuel'  ? await _t.getFuelMap()
              : type == 'VTC'   ? await _t.getVTCMap()
              : await _t.getEngineTorqueMap();
      String p;
      if (fmt == 'winols')      p = await _e.exportToWinOLS(m);
      else if (fmt == 'ecuedit') p = await _e.exportToEcuEdit(m);
      else                       p = await _e.exportHexPatch(m);
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(SnackBar(
        content: Text('→ ${p.split("/").last}'), backgroundColor: Colors.green));
      await _load();
    } catch (_) {}
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Экспорт карт'), backgroundColor: const Color(0xFF16213E),
        actions: [IconButton(icon: const Icon(Icons.refresh), onPressed: _load)]),
      body: Column(children: [
        Padding(padding: const EdgeInsets.all(12), child: Card(color: const Color(0xFF16213E),
          child: Padding(padding: const EdgeInsets.all(12), child: Column(
            crossAxisAlignment: CrossAxisAlignment.start, children: [
              const Text('ЭКСПОРТ КАРТ', style: TextStyle(
                color: Colors.white70, fontSize: 12, fontWeight: FontWeight.bold)),
              const SizedBox(height: 8),
              for (final e in [['Зажигание','Spark'],['Топливо','Fuel'],['VTC','VTC'],['Момент','Torque']])
                _row(e[0], e[1]),
            ])))),
        Expanded(child: _files.isEmpty
          ? const Center(child: Text('Нет файлов', style: TextStyle(color: Colors.white54)))
          : ListView.builder(padding: const EdgeInsets.all(8), itemCount: _files.length,
              itemBuilder: (_, i) {
                final f = _files[i]; final n = f.path.split('/').last;
                return Card(color: const Color(0xFF16213E), child: ListTile(
                  title: Text(n, style: const TextStyle(fontSize: 12)),
                  trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                    IconButton(icon: const Icon(Icons.share, color: Colors.blue),
                      onPressed: () async { try { await Share.shareXFiles([XFile(f.path)]); } catch (_) {} }),
                    IconButton(icon: const Icon(Icons.delete, color: Colors.red),
                      onPressed: () async { await File(f.path).delete(); await _load(); }),
                  ])));
              })),
      ]),
    );
  }

  Widget _row(String label, String type) => Padding(
    padding: const EdgeInsets.symmetric(vertical: 4),
    child: Row(children: [
      Expanded(child: Text(label)),
      IconButton(icon: const Icon(Icons.file_download, size: 20),
        color: Colors.blue, onPressed: () => _exp(type, 'winols')),
      IconButton(icon: const Icon(Icons.description, size: 20),
        color: Colors.purple, onPressed: () => _exp(type, 'ecuedit')),
      IconButton(icon: const Icon(Icons.memory, size: 20),
        color: Colors.orange, onPressed: () => _exp(type, 'hex')),
    ]));
}
''')
print("✅ export_screen.dart")

# ================================================================
# custom_pid_screen.dart — с CRUD
# ================================================================
with open('lib/screens/custom_pid_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'package:uuid/uuid.dart';
import '../models/custom_pid.dart';
import '../services/obd_service.dart';
import '../services/profile_service.dart';
import '../services/nissan_pid_library.dart';
import '../widgets/fps_indicator.dart';

class CustomPIDScreen extends StatefulWidget {
  final OBDService obdService;
  final ProfileService profileService;
  const CustomPIDScreen({super.key, required this.obdService, required this.profileService});
  @override
  State<CustomPIDScreen> createState() => _CustomPIDScreenState();
}

class _CustomPIDScreenState extends State<CustomPIDScreen> {
  Timer? _t;
  String _cat = 'all';

  @override
  void initState() { super.initState();
    _t = Timer.periodic(const Duration(milliseconds: 500), (_) { if (mounted) setState(() {}); });
  }
  @override
  void dispose() { _t?.cancel(); super.dispose(); }

  void _addPid() {
    final cmd    = TextEditingController();
    final answer = TextEditingController();
    final name   = TextEditingController();
    final desc   = TextEditingController();
    final unit   = TextEditingController();
    final formula = TextEditingController(text: 'X');
    final bytes  = TextEditingController(text: '1');

    showDialog(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Добавить PID'),
      content: SingleChildScrollView(child: Column(mainAxisSize: MainAxisSize.min, children: [
        TextField(controller: cmd, decoration: const InputDecoration(labelText: 'Команда (hex)', hintText: '2211FF0401')),
        TextField(controller: answer, decoration: const InputDecoration(labelText: 'Ответ (hex prefix)', hintText: '6211FF')),
        TextField(controller: name, decoration: const InputDecoration(labelText: 'Имя', hintText: 'MY_PID')),
        TextField(controller: desc, decoration: const InputDecoration(labelText: 'Описание')),
        TextField(controller: unit, decoration: const InputDecoration(labelText: 'Единица')),
        TextField(controller: formula, decoration: const InputDecoration(labelText: 'Формула (X = raw)', hintText: 'X * 0.01')),
        TextField(controller: bytes, decoration: const InputDecoration(labelText: 'Байт в ответе'), keyboardType: TextInputType.number),
      ])),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c), child: const Text('Отмена')),
        TextButton(
          onPressed: () async {
            if (cmd.text.isEmpty || name.text.isEmpty) return;
            final pid = CustomPid(
              id: const Uuid().v4(),
              cmd: cmd.text.trim().toUpperCase(),
              answer: answer.text.trim().toUpperCase(),
              name: name.text.trim(),
              desc: desc.text.trim(),
              unit: unit.text.trim(),
              bytesCount: int.tryParse(bytes.text) ?? 1,
              formula: formula.text.trim(),
              status: PidStatus.userAdded,
            );
            await widget.profileService.saveCustomPid(pid);
            Navigator.pop(c);
            setState(() {});
            if (mounted) ScaffoldMessenger.of(context).showSnackBar(
              SnackBar(content: Text('Добавлен: ${pid.name}'), backgroundColor: Colors.green));
          },
          style: TextButton.styleFrom(foregroundColor: Colors.green),
          child: const Text('ДОБАВИТЬ')),
      ]));
  }

  @override
  Widget build(BuildContext context) {
    final pids    = widget.obdService.activePids;
    final profile = widget.profileService.getActiveOrDefault();
    final custom  = profile.customPids;
    final deleted = profile.deletedPidIds;
    final filtered = _cat == 'all' ? pids : pids.where((p) => p.category == _cat).toList();
    final cats = ['all', ...NissanPidLibrary.categories];

    return Scaffold(
      appBar: AppBar(title: const Text('PID'), backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          IconButton(icon: const Icon(Icons.add), onPressed: _addPid),
          IconButton(icon: const Icon(Icons.copy), onPressed: () async {
            final sb = StringBuffer();
            sb.writeln('ECU: ${widget.obdService.ecuId}');
            for (final p in pids) {
              final v = widget.obdService.pidValues[p.id] ?? 0;
              sb.writeln('${p.cmd} [${p.desc}] = ${v.toStringAsFixed(2)} ${p.unit}');
            }
            await Clipboard.setData(ClipboardData(text: sb.toString()));
          }),
        ]),
      body: Column(children: [
        // ECU info
        if (widget.obdService.ecuId.isNotEmpty)
          Card(color: const Color(0xFF16213E), margin: const EdgeInsets.all(8),
            child: Padding(padding: const EdgeInsets.all(10),
              child: Text('ECU: ${widget.obdService.ecuId} | PID: ${pids.length} | Custom: ${custom.length}',
                style: const TextStyle(color: Colors.green)))),
        // Категории
        SizedBox(height: 40, child: ListView.builder(
          scrollDirection: Axis.horizontal,
          padding: const EdgeInsets.symmetric(horizontal: 8),
          itemCount: cats.length,
          itemBuilder: (_, i) => Padding(
            padding: const EdgeInsets.only(right: 4),
            child: FilterChip(
              label: Text(cats[i], style: const TextStyle(fontSize: 11)),
              selected: _cat == cats[i],
              onSelected: (_) => setState(() => _cat = cats[i]),
              backgroundColor: const Color(0xFF0F3460),
              selectedColor: const Color(0xFFE94560).withOpacity(0.5))))),
        // Custom PIDs
        if (custom.isNotEmpty) ...[
          const Padding(padding: EdgeInsets.symmetric(horizontal: 12, vertical: 4),
            child: Text('КАСТОМНЫЕ:', style: TextStyle(color: Colors.orange, fontSize: 11))),
          ...custom.map((p) => Card(color: Colors.orange.withOpacity(0.1),
            margin: const EdgeInsets.symmetric(horizontal: 8, vertical: 2),
            child: ListTile(dense: true,
              leading: const CircleAvatar(radius: 12, backgroundColor: Colors.orange,
                child: Icon(Icons.star, size: 14, color: Colors.white)),
              title: Text(p.name, style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)),
              subtitle: Text('${p.cmd} | ${p.formula}', style: const TextStyle(fontSize: 10, color: Colors.white54)),
              trailing: IconButton(icon: const Icon(Icons.delete, size: 18, color: Colors.red),
                onPressed: () async {
                  await widget.profileService.deletePid(p.id);
                  setState(() {});
                })))),
        ],
        // Активные PIDs
        Expanded(child: ListView.builder(
          padding: const EdgeInsets.all(8),
          itemCount: filtered.length,
          itemBuilder: (_, i) {
            final p = filtered[i];
            final v = widget.obdService.pidValues[p.id] ?? 0;
            final isDeleted = deleted.contains(p.id);
            return Card(
              color: isDeleted ? Colors.red.withOpacity(0.1) : const Color(0xFF16213E),
              child: ListTile(dense: true,
                leading: CircleAvatar(radius: 14,
                  backgroundColor: p.priority == 1 ? Colors.red
                    : p.priority == 2 ? Colors.orange : Colors.grey,
                  child: Text('P${p.priority}',
                    style: const TextStyle(fontSize: 9, color: Colors.white))),
                title: Text(p.desc, style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)),
                subtitle: Text(p.cmd, style: const TextStyle(
                  fontFamily: 'monospace', fontSize: 10, color: Colors.cyan)),
                trailing: Text('${v.toStringAsFixed(2)} ${p.unit}',
                  style: const TextStyle(color: Colors.yellow, fontWeight: FontWeight.bold, fontSize: 13))));
          })),
      ]),
    );
  }
}
''')
print("✅ custom_pid_screen.dart (CRUD)")

# ================================================================
# profile_screen.dart
# ================================================================
with open('lib/screens/profile_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:intl/intl.dart';
import '../services/profile_service.dart';
import '../models/vehicle_profile.dart';

class ProfileScreen extends StatefulWidget {
  final ProfileService profileService;
  const ProfileScreen({super.key, required this.profileService});
  @override
  State<ProfileScreen> createState() => _ProfileScreenState();
}

class _ProfileScreenState extends State<ProfileScreen> {
  List<VehicleProfile> _profiles = [];
  VehicleProfile? _active;

  @override
  void initState() { super.initState(); _load(); }

  void _load() {
    _profiles = widget.profileService.getAll();
    _active   = widget.profileService.getActive();
    setState(() {});
  }

  void _add() {
    final name   = TextEditingController(text: 'Мой X-Trail');
    final make   = TextEditingController(text: 'Nissan');
    final model  = TextEditingController(text: 'X-Trail T30');
    final year   = TextEditingController(text: '2004');
    final engine = TextEditingController(text: 'QR20DE');
    final disp   = TextEditingController(text: '2.0');

    showDialog(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Новый профиль'),
      content: SingleChildScrollView(child: Column(mainAxisSize: MainAxisSize.min, children: [
        TextField(controller: name, decoration: const InputDecoration(labelText: 'Название')),
        TextField(controller: make, decoration: const InputDecoration(labelText: 'Марка')),
        TextField(controller: model, decoration: const InputDecoration(labelText: 'Модель')),
        TextField(controller: year, decoration: const InputDecoration(labelText: 'Год')),
        TextField(controller: engine, decoration: const InputDecoration(labelText: 'Двигатель')),
        TextField(controller: disp, decoration: const InputDecoration(labelText: 'Объём (л)'),
          keyboardType: TextInputType.number),
      ])),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c), child: const Text('Отмена')),
        TextButton(
          onPressed: () async {
            if (name.text.isNotEmpty) {
              await widget.profileService.create(
                name: name.text, make: make.text, model: model.text,
                year: year.text, engine: engine.text,
                displacement: double.tryParse(disp.text) ?? 2.0);
              Navigator.pop(c);
              _load();
            }
          },
          style: TextButton.styleFrom(foregroundColor: Colors.green),
          child: const Text('Создать')),
      ]));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Профили'), backgroundColor: const Color(0xFF16213E),
        actions: [IconButton(icon: const Icon(Icons.add), onPressed: _add)]),
      body: _profiles.isEmpty
        ? Center(child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
            const Icon(Icons.directions_car, size: 80, color: Colors.white24),
            const SizedBox(height: 16),
            const Text('Нет профилей', style: TextStyle(color: Colors.white54, fontSize: 18)),
            const SizedBox(height: 8),
            ElevatedButton.icon(onPressed: _add,
              icon: const Icon(Icons.add), label: const Text('Создать')),
          ]))
        : ListView.builder(
            padding: const EdgeInsets.all(12), itemCount: _profiles.length,
            itemBuilder: (_, i) {
              final p = _profiles[i];
              final active = _active?.id == p.id;
              return Card(
                color: active ? Colors.green.withOpacity(0.2) : const Color(0xFF16213E),
                child: ListTile(
                  leading: CircleAvatar(
                    backgroundColor: active ? Colors.green : const Color(0xFF0F3460),
                    child: const Icon(Icons.directions_car, color: Colors.white)),
                  title: Text(p.name, style: const TextStyle(fontWeight: FontWeight.bold)),
                  subtitle: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                    Text('${p.make} ${p.model} ${p.year}'),
                    Text('${p.engine} | ${p.displacement.toStringAsFixed(1)}L | PID: ${p.customPids.length}',
                      style: const TextStyle(fontSize: 11)),
                    Text('MAF ×${p.mafMultiplier.toStringAsFixed(2)} | Speed ×${p.speedMultiplier.toStringAsFixed(2)} | Fuel ×${p.fuelCorrection.toStringAsFixed(1)}',
                      style: const TextStyle(fontSize: 10, color: Colors.white54)),
                  ]),
                  trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                    if (active) const Icon(Icons.check_circle, color: Colors.green)
                    else IconButton(icon: const Icon(Icons.check, color: Colors.blue),
                      onPressed: () async {
                        await widget.profileService.setActive(p.id);
                        _load();
                      }),
                    IconButton(icon: const Icon(Icons.delete, color: Colors.red),
                      onPressed: () async {
                        await widget.profileService.delete(p.id);
                        _load();
                      }),
                  ])));
            }),
    );
  }
}
''')
print("✅ profile_screen.dart")

# ================================================================
# rom_compare_screen.dart — сравнение двух ROM
# ================================================================
with open('lib/screens/rom_compare_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import '../services/rom_map_reader.dart';
import '../models/tuning_map.dart';
import '../widgets/map_table_view.dart';

class RomCompareScreen extends StatefulWidget {
  const RomCompareScreen({super.key});
  @override
  State<RomCompareScreen> createState() => _RomCompareScreenState();
}

class _RomCompareScreenState extends State<RomCompareScreen> {
  final _romA = RomMapReader();
  final _romB = RomMapReader();
  List<RomDiff>? _diffs;

  Future<void> _loadA() async {
    final ok = await _romA.pickAndLoad();
    if (ok) setState(() => _diffs = null);
  }

  Future<void> _loadB() async {
    final ok = await _romB.pickAndLoad();
    if (ok) setState(() => _diffs = null);
  }

  void _compare() {
    if (!_romA.isLoaded || !_romB.isLoaded) return;
    final diffs = RomMapReader.compareRoms(_romA, _romB);
    setState(() => _diffs = diffs);
  }

  void _viewMap(TuningMap mapA, TuningMap mapB, String title) {
    Navigator.push(context, MaterialPageRoute(builder: (_) => _DualMapView(
      title: title, mapA: mapA, mapB: mapB)));
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('ROM Сравнение'), backgroundColor: const Color(0xFF16213E)),
      body: SingleChildScrollView(
        padding: const EdgeInsets.all(12),
        child: Column(children: [
          // Загрузка файлов
          Row(children: [
            Expanded(child: ElevatedButton.icon(
              onPressed: _loadA,
              icon: const Icon(Icons.folder_open, size: 16),
              label: Text(_romA.isLoaded ? _romA.fileName! : 'Оригинал',
                style: const TextStyle(fontSize: 11)),
              style: ElevatedButton.styleFrom(
                backgroundColor: _romA.isLoaded ? Colors.green.shade800 : const Color(0xFF0F3460),
                minimumSize: const Size.fromHeight(42)))),
            const SizedBox(width: 8),
            Expanded(child: ElevatedButton.icon(
              onPressed: _loadB,
              icon: const Icon(Icons.folder_open, size: 16),
              label: Text(_romB.isLoaded ? _romB.fileName! : 'Модифициров.',
                style: const TextStyle(fontSize: 11)),
              style: ElevatedButton.styleFrom(
                backgroundColor: _romB.isLoaded ? Colors.blue.shade800 : const Color(0xFF0F3460),
                minimumSize: const Size.fromHeight(42)))),
          ]),
          const SizedBox(height: 8),
          SizedBox(width: double.infinity,
            child: ElevatedButton.icon(
              onPressed: _romA.isLoaded && _romB.isLoaded ? _compare : null,
              icon: const Icon(Icons.compare_arrows),
              label: const Text('СРАВНИТЬ', style: TextStyle(fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(
                backgroundColor: Colors.cyan, minimumSize: const Size.fromHeight(44)))),
          if (_romA.isLoaded && _romB.isLoaded) ...[
            const SizedBox(height: 4),
            Text('A: ${_romA.fileName} (${_romA.length} B)   B: ${_romB.fileName} (${_romB.length} B)',
              style: const TextStyle(color: Colors.white54, fontSize: 10)),
          ],
          const SizedBox(height: 12),
          // Результаты
          if (_diffs != null) ..._diffs!.map((d) => Card(
            color: d.hasDiffs ? Colors.orange.withOpacity(0.15) : const Color(0xFF16213E),
            child: ListTile(
              leading: Icon(d.hasDiffs ? Icons.difference : Icons.check_circle,
                color: d.hasDiffs ? Colors.orange : Colors.green),
              title: Text(d.mapName, style: const TextStyle(fontWeight: FontWeight.bold)),
              subtitle: Text(d.hasDiffs
                ? '${d.cellDiffs.length} различий'
                : 'Идентичны',
                style: TextStyle(color: d.hasDiffs ? Colors.orange : Colors.green, fontSize: 12)),
              trailing: d.hasDiffs
                ? ElevatedButton(
                    onPressed: () => _viewMap(d.mapA, d.mapB, d.mapName),
                    style: ElevatedButton.styleFrom(backgroundColor: Colors.cyan),
                    child: const Text('СМОТРЕТЬ', style: TextStyle(fontSize: 11)))
                : null,
            ),
          )),
        ]),
      ),
    );
  }
}

/// Показывает две карты рядом с подсветкой различий
class _DualMapView extends StatelessWidget {
  final String title;
  final TuningMap mapA, mapB;
  const _DualMapView({required this.title, required this.mapA, required this.mapB});

  @override
  Widget build(BuildContext context) {
    return DefaultTabController(length: 2, child: Scaffold(
      backgroundColor: const Color(0xFF1A1A2E),
      appBar: AppBar(
        title: Text(title), backgroundColor: const Color(0xFF16213E),
        bottom: const TabBar(tabs: [
          Tab(text: 'Оригинал (A)'),
          Tab(text: 'Модифицир. (B)'),
        ])),
      body: TabBarView(children: [
        MapTableView(originalMap: mapA, changes: const [], isFullscreen: true),
        MapTableView(originalMap: mapB, changes: const [], isFullscreen: true),
      ]),
    ));
  }
}
''')
print("✅ rom_compare_screen.dart (загрузка двух ROM + сравнение + просмотр)")

# ================================================================
# ecu_read_screen.dart — АВТО + UNLOCK + чтение карт + запись в .bin + терминал
# ================================================================
with open('lib/screens/ecu_read_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'package:intl/intl.dart';
import '../models/custom_map_def.dart';
import '../models/tuning_map.dart';
import '../services/obd_service.dart';
import '../services/ecu_map_reader.dart';
import '../services/nissan_unlock.dart';
import '../services/rom_map_reader.dart';
import '../services/map_storage_service.dart';
import '../widgets/fps_indicator.dart';
import '../widgets/map_table_view.dart';

class EcuReadScreen extends StatefulWidget {
  final OBDService obdService;
  const EcuReadScreen({super.key, required this.obdService});
  @override
  State<EcuReadScreen> createState() => _EcuReadScreenState();
}

class _EcuReadScreenState extends State<EcuReadScreen>
    with SingleTickerProviderStateMixin {
  late final TabController _tab;
  late final EcuMapReaderService _reader;
  final RomMapReader _rom = RomMapReader();

  bool _detecting = false, _unlocking = false, _unlocked = false;
  MemReadCommand? _detectedCmd;
  final List<String> _log = [];
  final Map<String, double> _readProg = {};
  final Map<String, EcuMapReadResult> _readRes = {};

  // Терминал
  final _termCmd = TextEditingController();
  final List<String> _termLog = [];

  static final _stdMaps = [
    CustomMapDef(id:'torque', name:'Engine Torque', address:0x7C3C,
      rows:16, cols:16, isUInt16:true, formula:'(X-32768)/10.24', units:'Nm', createdAt:DateTime(2025)),
    CustomMapDef(id:'spark', name:'Spark Advance WOT', address:0x6EBC,
      rows:16, cols:16, isUInt16:true, formula:'(X-16384)/128', units:'deg', createdAt:DateTime(2025)),
    CustomMapDef(id:'ve', name:'Fresh Air Rate / VE', address:0xA754,
      rows:16, cols:15, isUInt16:true, formula:'X/256', units:'%', createdAt:DateTime(2025)),
    CustomMapDef(id:'vtc', name:'VTC Intake Cam', address:0x6BF1,
      rows:8, cols:8, isUInt16:false, formula:'(X-128)/2', units:'deg', createdAt:DateTime(2025)),
    CustomMapDef(id:'force', name:'Powertrain Force', address:0xA2BC,
      rows:16, cols:16, isUInt16:true, formula:'X-32768', units:'N', createdAt:DateTime(2025)),
    CustomMapDef(id:'lambda', name:'Enrichment Lambda', address:0x6521,
      rows:8, cols:8, isUInt16:false, formula:'X/128', units:'Lambda', createdAt:DateTime(2025)),
  ];

  @override
  void initState() {
    super.initState();
    _tab = TabController(length: 3, vsync: this);
    _reader = EcuMapReaderService(widget.obdService);
    _detectedCmd = _reader.getSavedCommand();
  }
  @override
  void dispose() { _tab.dispose(); _termCmd.dispose(); super.dispose(); }

  // ── АВТО детект ──────────────────────────────────────────────
  Future<void> _autoDetect() async {
    if (!widget.obdService.ecuResponds) { _snack('Нет ЭБУ', Colors.red); return; }
    setState(() { _detecting = true; _log.clear(); });
    final cmd = await _reader.autoDetectCommand(
      onProgress: (m) => setState(() => _log.add(m)));
    setState(() { _detecting = false; _detectedCmd = cmd; });
    _snack(cmd != null ? '✅ ${cmd.description}' : '❌ Не найдено',
      cmd != null ? Colors.green : Colors.red);
  }

  // ── UNLOCK ───────────────────────────────────────────────────
  Future<void> _unlock() async {
    if (!widget.obdService.ecuResponds) { _snack('Нет ЭБУ', Colors.red); return; }
    setState(() { _unlocking = true; _log.clear(); _log.add('=== UNLOCK ==='); });

    final seedResp = await widget.obdService.sendCommand('2701', timeout: 3000);
    final clean    = seedResp.replaceAll(' ', '').toUpperCase();
    _log.add('<<< $clean');
    setState(() {});

    if (!clean.contains('6701') || clean.length < clean.indexOf('6701') + 12) {
      _log.add('❌ Seed не получен');
      setState(() => _unlocking = false);
      return;
    }

    final idx     = clean.indexOf('6701');
    final seedHex = clean.substring(idx + 4, idx + 12);
    final seed    = <int>[];
    for (int i = 0; i < 8; i += 2) seed.add(int.parse(seedHex.substring(i, i + 2), radix: 16));
    _log.add('SEED: ${NissanUnlock.seedToHex(seed)}');
    setState(() {});

    final key    = NissanUnlock.calcKey(seed);
    final keyHex = NissanUnlock.keyToHex(key);
    _log.add('KEY:  $keyHex');
    setState(() {});

    final resp   = await widget.obdService.sendCommand('2702$keyHex', timeout: 3000);
    final rClean = resp.replaceAll(' ', '').toUpperCase();
    _log.add('<<< $rClean');

    if (rClean.startsWith('6702')) {
      _log.add('✅✅✅ UNLOCK SUCCESS ✅✅✅');
      setState(() { _unlocked = true; _unlocking = false; });
      _snack('РАЗБЛОКИРОВАНО!', Colors.green);
    } else {
      _log.add('❌ ${rClean.startsWith("7F2736") ? "Много попыток (10 сек)" : "Неверный key"}');
      setState(() => _unlocking = false);
      _snack('Не удалось', Colors.red);
    }
  }

  // ── Чтение карты из ЭБУ ──────────────────────────────────────
  Future<void> _readMap(CustomMapDef def) async {
    if (_detectedCmd == null) { _snack('Сначала АВТО', Colors.orange); return; }
    setState(() => _readProg[def.addressHex] = 0);
    final res = await _reader.readMap(
      def: def, command: _detectedCmd,
      onProgress: (d, t) => setState(() => _readProg[def.addressHex] = d / t * 100));
    setState(() {
      _readProg.remove(def.addressHex);
      if (res != null) _readRes[def.addressHex] = res;
    });
  }

  void _viewMap(EcuMapReadResult r) {
    final m = TuningMap(
      name: '${r.def.name} (ЭБУ)', address: r.def.addressHex,
      rows: r.def.rows, cols: r.def.cols,
      rpmAxis: List.generate(r.def.rows, (i) => (i + 1) * 400.0),
      loadAxis: List.generate(r.def.cols, (i) => i * 10.0),
      data: r.data, units: r.def.units);
    Navigator.push(context, MaterialPageRoute(builder: (_) =>
      MapFullscreenView(originalMap: m, changes: const [])));
  }

  Future<void> _saveAsDefault(EcuMapReadResult r) async {
    final m = TuningMap(
      name: r.def.name, address: r.def.addressHex,
      rows: r.def.rows, cols: r.def.cols,
      rpmAxis: List.generate(r.def.rows, (i) => (i + 1) * 400.0),
      loadAxis: List.generate(r.def.cols, (i) => i * 10.0),
      data: r.data, units: r.def.units);
    await MapStorageService.saveMap(m);
    _snack('Сохранено как дефолт', Colors.green);
  }

  // ── Загрузка .bin + запись правок ────────────────────────────
  Future<void> _loadRom() async {
    final ok = await _rom.pickAndLoad();
    if (!ok) return;
    setState(() => _log.add('ROM: ${_rom.fileName} (${_rom.length} B)'));
    _snack('Загружен: ${_rom.fileName}', Colors.green);
  }

  Future<void> _saveRomWithEdits() async {
    if (!_rom.isLoaded) { _snack('Сначала загрузи .bin', Colors.orange); return; }

    // Показываем список карт для выбора
    final maps = _rom.readAllMaps();
    if (maps.isEmpty) { _snack('Не удалось прочитать карты', Colors.red); return; }

    final selected = await showDialog<Set<int>>(context: context, builder: (c) {
      final sel = <int>{};
      return StatefulBuilder(builder: (c, setD) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: const Text('Записать правки в ROM'),
        content: SizedBox(width: double.maxFinite,
          child: Column(mainAxisSize: MainAxisSize.min, children: [
            Text('ROM: ${_rom.fileName}', style: const TextStyle(color: Colors.white70, fontSize: 11)),
            const SizedBox(height: 8),
            ...maps.asMap().entries.map((e) => CheckboxListTile(
              dense: true,
              title: Text(e.value.name, style: const TextStyle(fontSize: 12)),
              subtitle: Text('${e.value.rows}x${e.value.cols} ${e.value.units}',
                style: const TextStyle(fontSize: 10)),
              value: sel.contains(e.key),
              onChanged: (v) => setD(() {
                if (v == true) sel.add(e.key);
                else sel.remove(e.key);
              }))),
          ])),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c), child: const Text('Отмена')),
          TextButton(onPressed: () => Navigator.pop(c, sel),
            style: TextButton.styleFrom(foregroundColor: Colors.green),
            child: const Text('ЗАПИСАТЬ')),
        ]));
    });

    if (selected == null || selected.isEmpty) return;

    // Получаем сохранённые правки и записываем в ROM буфер
    int count = 0;
    for (final i in selected) {
      final map = maps[i];
      // Находим соответствующий RomMapDef
      final def = RomMapReader.standardMaps.firstWhere(
        (d) => d.addressHex == map.address,
        orElse: () => RomMapReader.standardMaps.first);

      // Загружаем правки из MapStorage
      final edited = await MapStorageService.loadMapData(map.address);
      if (edited != null) {
        final editedMap = map.copy();
        editedMap.data.clear();
        editedMap.data.addAll(edited);
        _rom.writeMapToBuffer(editedMap, def);
        count++;
      } else {
        // Записываем оригинальную карту из ROM
        _rom.writeMapToBuffer(map, def);
        count++;
      }
    }

    // Сохраняем файл
    final path = await _rom.saveModifiedRom(modIndex: 1);
    if (path != null) {
      _snack('NLP_MOD1 сохранён ($count карт)', Colors.green);
      setState(() => _log.add('Записано: $path'));
    }
  }

  // ── Терминал ─────────────────────────────────────────────────
  Future<void> _sendTerm() async {
    final c = _termCmd.text.trim().toUpperCase();
    if (c.isEmpty || !widget.obdService.isConnected) return;
    setState(() => _termLog.add('>>> $c'));
    final r = await _reader.testRawCommand(c);
    setState(() => _termLog.add('<<< $r'));
    _termCmd.clear();
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(m), backgroundColor: c));
  }

  // ── Build ────────────────────────────────────────────────────
  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('ЭБУ Карты'), backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)],
        bottom: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.list), text: 'Карты'),
          Tab(icon: Icon(Icons.sd_card), text: 'ROM'),
          Tab(icon: Icon(Icons.terminal), text: 'Терминал'),
        ])),
      body: Column(children: [
        _statusPanel(),
        Expanded(child: TabBarView(controller: _tab, children: [
          _mapsTab(),
          _romTab(),
          _termTab(),
        ])),
      ]),
    );
  }

  Widget _statusPanel() {
    final conn = widget.obdService.ecuResponds;
    return Container(
      padding: const EdgeInsets.all(6), color: const Color(0xFF16213E),
      child: Column(children: [
        Row(children: [
          Icon(conn ? Icons.check_circle : Icons.error,
            color: conn ? Colors.green : Colors.red, size: 16),
          const SizedBox(width: 4),
          Expanded(child: Text(conn ? 'ЭБУ подключён' : 'Нет ЭБУ',
            style: TextStyle(color: conn ? Colors.green : Colors.red, fontSize: 12))),
          if (_unlocked) Container(
            padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
            decoration: BoxDecoration(color: Colors.green, borderRadius: BorderRadius.circular(4)),
            child: const Text('UNLOCKED', style: TextStyle(color: Colors.white, fontSize: 9, fontWeight: FontWeight.bold))),
        ]),
        const SizedBox(height: 6),
        Row(children: [
          Expanded(child: ElevatedButton.icon(
            onPressed: (_detecting || !conn) ? null : _autoDetect,
            icon: _detecting
              ? const SizedBox(width: 12, height: 12, child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
              : const Icon(Icons.search, size: 14),
            label: const Text('АВТО', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(backgroundColor: Colors.cyan, padding: const EdgeInsets.symmetric(vertical: 6)))),
          const SizedBox(width: 3),
          Expanded(child: ElevatedButton.icon(
            onPressed: (_unlocking || !conn) ? null : _unlock,
            icon: _unlocking
              ? const SizedBox(width: 12, height: 12, child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
              : Icon(_unlocked ? Icons.lock_open : Icons.lock, size: 14),
            label: Text(_unlocked ? 'OK' : 'UNLOCK', style: const TextStyle(fontSize: 10, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: _unlocked ? Colors.green : Colors.red,
              padding: const EdgeInsets.symmetric(vertical: 6)))),
        ]),
        if (_log.isNotEmpty) Container(
          margin: const EdgeInsets.only(top: 4), padding: const EdgeInsets.all(4),
          constraints: const BoxConstraints(maxHeight: 100),
          decoration: BoxDecoration(color: Colors.black, borderRadius: BorderRadius.circular(4)),
          child: SingleChildScrollView(reverse: true,
            child: Text(_log.join('\n'),
              style: const TextStyle(fontFamily: 'monospace', fontSize: 9, color: Colors.cyan)))),
      ]),
    );
  }

  Widget _mapsTab() {
    return ListView.builder(
      padding: const EdgeInsets.all(8), itemCount: _stdMaps.length,
      itemBuilder: (_, i) => _mapCard(_stdMaps[i]));
  }

  Widget _mapCard(CustomMapDef def) {
    final prog = _readProg[def.addressHex];
    final res  = _readRes[def.addressHex];
    return Card(color: const Color(0xFF16213E), margin: const EdgeInsets.symmetric(vertical: 3),
      child: Padding(padding: const EdgeInsets.all(10), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          Row(children: [
            Expanded(child: Text(def.name, style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 13))),
            Container(
              padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
              decoration: BoxDecoration(color: const Color(0xFF0F3460), borderRadius: BorderRadius.circular(4)),
              child: Text(def.addressHex, style: const TextStyle(fontFamily: 'monospace', fontSize: 10, color: Colors.cyan))),
          ]),
          Text('${def.rows}x${def.cols} ${def.isUInt16 ? "U16" : "U8"}',
            style: const TextStyle(color: Colors.white70, fontSize: 10)),
          const SizedBox(height: 6),
          if (prog != null) ...[
            LinearProgressIndicator(value: prog / 100, color: Colors.cyan),
            Text('${prog.toInt()}%', style: const TextStyle(color: Colors.cyan, fontSize: 11)),
          ] else if (res != null) ...[
            Text('Прочитано: ${DateFormat("HH:mm:ss").format(res.readAt)}',
              style: const TextStyle(color: Colors.green, fontSize: 11)),
            const SizedBox(height: 6),
            Row(children: [
              Expanded(child: ElevatedButton.icon(
                onPressed: () => _viewMap(res),
                icon: const Icon(Icons.visibility, size: 14), label: const Text('Просмотр'),
                style: ElevatedButton.styleFrom(backgroundColor: Colors.blue))),
              const SizedBox(width: 4),
              Expanded(child: ElevatedButton.icon(
                onPressed: () => _saveAsDefault(res),
                icon: const Icon(Icons.save, size: 14), label: const Text('В дефолт'),
                style: ElevatedButton.styleFrom(backgroundColor: Colors.orange))),
            ]),
          ] else
            SizedBox(width: double.infinity, child: ElevatedButton.icon(
              onPressed: _detectedCmd == null ? null : () => _readMap(def),
              icon: const Icon(Icons.download, size: 16),
              label: Text(_detectedCmd == null ? 'Сначала АВТО' : 'ЧИТАТЬ',
                style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(backgroundColor: Colors.cyan))),
        ])));
  }

  Widget _romTab() {
    return SingleChildScrollView(padding: const EdgeInsets.all(8), child: Column(children: [
      // Загрузка .bin
      SizedBox(width: double.infinity, child: ElevatedButton.icon(
        onPressed: _loadRom,
        icon: const Icon(Icons.folder_open),
        label: Text(_rom.isLoaded
          ? '${_rom.fileName} (${_rom.length} B)'
          : 'ЗАГРУЗИТЬ .BIN ПРОШИВКУ',
          style: const TextStyle(fontWeight: FontWeight.bold)),
        style: ElevatedButton.styleFrom(
          backgroundColor: _rom.isLoaded ? Colors.green.shade800 : Colors.deepPurple,
          minimumSize: const Size.fromHeight(44)))),
      if (_rom.isLoaded) ...[
        const SizedBox(height: 8),
        // Предпросмотр карт
        ..._rom.readAllMaps().map((m) => Card(color: const Color(0xFF16213E),
          child: ListTile(dense: true,
            leading: const Icon(Icons.grid_on, color: Colors.cyan, size: 18),
            title: Text(m.name, style: const TextStyle(fontSize: 12)),
            subtitle: Text('${m.rows}x${m.cols} avg=${m.avgValue.toStringAsFixed(1)} ${m.units}',
              style: const TextStyle(fontSize: 10, color: Colors.white70)),
            trailing: IconButton(
              icon: const Icon(Icons.visibility, color: Colors.blue, size: 18),
              onPressed: () => Navigator.push(context, MaterialPageRoute(builder: (_) =>
                MapFullscreenView(originalMap: m, changes: const [])))),
          ))),
        const SizedBox(height: 8),
        // Сохранить как дефолт
        SizedBox(width: double.infinity, child: ElevatedButton.icon(
          onPressed: () async {
            final count = await _rom.saveAllAsDefaults();
            _snack('Сохранено $count карт как дефолтные', Colors.green);
          },
          icon: const Icon(Icons.save_alt),
          label: const Text('СОХРАНИТЬ ВСЕ КАК ДЕФОЛТ'),
          style: ElevatedButton.styleFrom(backgroundColor: Colors.orange,
            minimumSize: const Size.fromHeight(40)))),
        const SizedBox(height: 8),
        // Записать правки в .bin
        SizedBox(width: double.infinity, child: ElevatedButton.icon(
          onPressed: _saveRomWithEdits,
          icon: const Icon(Icons.edit_note),
          label: const Text('ЗАПИСАТЬ ПРАВКИ В ROM → NLP_MOD1',
            style: TextStyle(fontWeight: FontWeight.bold)),
          style: ElevatedButton.styleFrom(backgroundColor: Colors.red.shade800,
            minimumSize: const Size.fromHeight(44)))),
        const SizedBox(height: 8),
        const Text('Выбери карты → правки из анализатора запишутся в .bin\n'
            'Файл сохранится как NLP_MOD1_<имя>.bin',
          style: TextStyle(color: Colors.white54, fontSize: 10), textAlign: TextAlign.center),
      ],
    ]));
  }

  Widget _termTab() {
    return Column(children: [
      // Быстрые кнопки
      Container(color: const Color(0xFF16213E), padding: const EdgeInsets.all(6),
        child: Wrap(spacing: 4, runSpacing: 4, children: [
          for (final e in [['1081','Default'],['1085','Prog'],['2701','Seed'],
                            ['237C3C04','KWP'],['217C3C','Consult'],['1A81','ECU ID']])
            ElevatedButton(
              onPressed: () { _termCmd.text = e[0]; _sendTerm(); },
              style: ElevatedButton.styleFrom(
                backgroundColor: const Color(0xFF0F3460),
                padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 3)),
              child: Column(mainAxisSize: MainAxisSize.min, children: [
                Text(e[0], style: const TextStyle(fontFamily: 'monospace', fontSize: 10)),
                Text(e[1], style: const TextStyle(fontSize: 8, color: Colors.white70)),
              ])),
        ])),
      // Лог
      Expanded(child: Container(color: Colors.black, padding: const EdgeInsets.all(8),
        child: SingleChildScrollView(
          child: SelectableText(_termLog.join('\n'),
            style: const TextStyle(fontFamily: 'monospace', fontSize: 11, color: Colors.green))))),
      // Ввод
      Container(color: const Color(0xFF16213E), padding: const EdgeInsets.all(6),
        child: Row(children: [
          Expanded(child: TextField(controller: _termCmd,
            style: const TextStyle(fontFamily: 'monospace'),
            decoration: const InputDecoration(hintText: 'Команда...',
              border: OutlineInputBorder(),
              contentPadding: EdgeInsets.symmetric(horizontal: 8, vertical: 6), isDense: true),
            textCapitalization: TextCapitalization.characters,
            onSubmitted: (_) => _sendTerm())),
          IconButton(icon: const Icon(Icons.send, color: Colors.green), onPressed: _sendTerm),
          IconButton(icon: const Icon(Icons.clear_all, color: Colors.white54),
            onPressed: () => setState(() => _termLog.clear())),
        ])),
    ]);
  }
}
''')
print("✅ ecu_read_screen.dart (АВТО + UNLOCK + чтение + .bin запись NLP_MOD + терминал)")

# ================================================================
# service_screen.dart — тесты + обучения
# ================================================================
with open('lib/screens/service_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import '../services/obd_service.dart';
import '../widgets/fps_indicator.dart';

class _Test {
  final String name, cmdBase, unit, category;
  final double min, max, step;
  final bool isBool;
  final String Function(double) fmt;
  final int Function(double) toRaw;
  bool active;
  _Test({required this.name, required this.cmdBase, required this.unit,
    required this.min, required this.max, required this.step,
    this.isBool = false, required this.category,
    required this.fmt, required this.toRaw, this.active = false});
  String startCmd(double v) => cmdBase + toRaw(v).toRadixString(16).padLeft(2,'0').toUpperCase() + '00';
  String get stopCmd => '${cmdBase}0000';
}

class _Learn {
  final String name, cmd, desc;
  final bool danger;
  const _Learn({required this.name, required this.cmd, required this.desc, this.danger = false});
}

class ServiceScreen extends StatefulWidget {
  final OBDService obdService;
  const ServiceScreen({super.key, required this.obdService});
  @override
  State<ServiceScreen> createState() => _ServiceScreenState();
}

class _ServiceScreenState extends State<ServiceScreen>
    with SingleTickerProviderStateMixin {
  late final TabController _tab;
  bool _busy = false;
  String _result = '';
  final Map<String, Timer> _timers = {};

  final _tests = <_Test>[
    _Test(name:'Симуляция ОЖ', cmdBase:'3001', unit:'°C', min:0, max:125, step:1, category:'temp',
      fmt:(v) => '${v.toStringAsFixed(0)}°C', toRaw:(v) => (v+50).toInt().clamp(0,255)),
    _Test(name:'Коррекция впрыска', cmdBase:'3002', unit:'%', min:75, max:125, step:1, category:'fuel',
      fmt:(v) => '${v.toStringAsFixed(0)}%', toRaw:(v) => v.toInt().clamp(0,255)),
    _Test(name:'Угол зажигания', cmdBase:'3003', unit:'°', min:-10, max:0, step:1, category:'ignition',
      fmt:(v) => '${v.toStringAsFixed(0)}°', toRaw:(v) => v.toInt().clamp(-128,127) & 0xFF),
    _Test(name:'Клапан ХХ', cmdBase:'3005', unit:'%', min:0, max:127, step:1, category:'idle',
      fmt:(v) => '${v.toStringAsFixed(0)}%', toRaw:(v) => (v+50).toInt().clamp(0,255)),
    _Test(name:'VTC угол', cmdBase:'3019', unit:'°', min:-64, max:63, step:0.5, category:'vtc',
      fmt:(v) => '${v.toStringAsFixed(1)}°', toRaw:(v) => (v*2).toInt().clamp(-128,127) & 0xFF),
    _Test(name:'Вентилятор HIGH', cmdBase:'300D', unit:'', min:0, max:1, step:1, isBool:true, category:'fan',
      fmt:(v) => v>0?'ВКЛ':'ВЫКЛ', toRaw:(v) => v>0?1:0),
    _Test(name:'Реле бензонасоса', cmdBase:'302D', unit:'', min:0, max:1, step:1, isBool:true, category:'fuel',
      fmt:(v) => v>0?'ВКЛ':'ВЫКЛ', toRaw:(v) => v>0?1:0),
  ];

  static const _learns = <_Learn>[
    _Learn(name:'Обучение подачи воздуха ХХ', cmd:'3103', desc:'Двигатель прогрет!', danger:true),
    _Learn(name:'Обучение дросселя', cmd:'3104', desc:'После чистки дросселя!', danger:true),
    _Learn(name:'Обучение коленвала', cmd:'3105', desc:'После замены ГРМ'),
    _Learn(name:'СБРОС АДАПТАЦИЙ', cmd:'3106', desc:'Сбросить ВСЕ!', danger:true),
    _Learn(name:'Обучение VTC', cmd:'310A', desc:'Обучение фаз', danger:true),
    _Learn(name:'Обучение A/F', cmd:'3114', desc:'После замены лямбды!', danger:true),
  ];

  @override
  void initState() { super.initState(); _tab = TabController(length: 2, vsync: this); }
  @override
  void dispose() {
    for (final t in _timers.values) t.cancel();
    _tab.dispose();
    super.dispose();
  }

  Future<void> _start(_Test t, double v) async {
    if (!widget.obdService.isConnected) return;
    setState(() => _busy = true);
    final cmd = t.startCmd(v);
    final r   = await widget.obdService.sendCommand(cmd, timeout: 3000);
    setState(() { _busy = false; _result = '>>> $cmd <<< $r'; t.active = true; });
    _timers[t.cmdBase]?.cancel();
    _timers[t.cmdBase] = Timer.periodic(const Duration(seconds: 2), (_) async {
      if (t.active && widget.obdService.isConnected) {
        await widget.obdService.sendCommand(cmd, timeout: 1500);
      }
    });
  }

  Future<void> _stop(_Test t) async {
    _timers[t.cmdBase]?.cancel();
    _timers.remove(t.cmdBase);
    setState(() => _busy = true);
    await widget.obdService.sendCommand(t.stopCmd, timeout: 2000);
    await widget.obdService.sendCommand('30000000', timeout: 2000);
    await widget.obdService.sendCommand('1081', timeout: 2000);
    await widget.obdService.sendCommand('ATSH8110FC', timeout: 1000);
    await widget.obdService.sendCommand('ATFI', timeout: 3000);
    setState(() { _busy = false; t.active = false; });
  }

  Future<void> _stopAll() async {
    setState(() => _busy = true);
    for (final t in _tests.where((t) => t.active)) {
      _timers[t.cmdBase]?.cancel();
      await widget.obdService.sendCommand(t.stopCmd, timeout: 1500);
      t.active = false;
    }
    _timers.clear();
    await widget.obdService.sendCommand('30000000', timeout: 2000);
    await widget.obdService.sendCommand('1081', timeout: 2000);
    await widget.obdService.sendCommand('ATSH8110FC', timeout: 1000);
    setState(() => _busy = false);
  }

  Future<void> _learn(_Learn l) async {
    if (!widget.obdService.isConnected) return;
    if (l.danger) {
      final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: Text(l.name),
        content: Text('${l.desc}\n\nУверены?', style: const TextStyle(color: Colors.white70)),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c, false), child: const Text('Отмена')),
          TextButton(onPressed: () => Navigator.pop(c, true),
            style: TextButton.styleFrom(foregroundColor: Colors.orange),
            child: const Text('ВЫПОЛНИТЬ')),
        ]));
      if (ok != true) return;
    }
    setState(() => _busy = true);
    final r = await widget.obdService.sendCommand(l.cmd, timeout: 5000);
    setState(() { _busy = false; _result = '>>> ${l.cmd} <<< $r'; });
    if (mounted) ScaffoldMessenger.of(context).showSnackBar(SnackBar(
      content: Text(r.contains('71') ? '${l.name} OK' : '${l.name}: $r'),
      backgroundColor: r.contains('71') ? Colors.green : Colors.orange));
  }

  @override
  Widget build(BuildContext context) {
    final hasActive = _tests.any((t) => t.active);
    return Scaffold(
      appBar: AppBar(title: const Text('Сервис'), backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: widget.obdService),
          if (hasActive) IconButton(
            icon: const Icon(Icons.stop_circle, color: Colors.red),
            onPressed: _busy ? null : _stopAll),
        ],
        bottom: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.build), text: 'Тесты'),
          Tab(icon: Icon(Icons.school), text: 'Обучения'),
        ])),
      body: Column(children: [
        if (!widget.obdService.ecuResponds)
          Container(width: double.infinity, padding: const EdgeInsets.all(8),
            color: Colors.red.withOpacity(0.3),
            child: const Text('⚠️ ЭБУ не подключён', textAlign: TextAlign.center,
              style: TextStyle(color: Colors.white, fontWeight: FontWeight.bold))),
        if (_busy) const LinearProgressIndicator(color: Color(0xFFE94560)),
        if (_result.isNotEmpty) Container(
          width: double.infinity, padding: const EdgeInsets.all(6), color: const Color(0xFF0F3460),
          child: Text(_result, style: const TextStyle(
            fontFamily: 'monospace', fontSize: 10, color: Colors.cyan))),
        Expanded(child: TabBarView(controller: _tab, children: [
          ListView.builder(
            padding: const EdgeInsets.all(8), itemCount: _tests.length,
            itemBuilder: (_, i) {
              final t = _tests[i];
              return Card(
                color: t.active ? Colors.green.withOpacity(0.2) : const Color(0xFF16213E),
                margin: const EdgeInsets.symmetric(vertical: 3),
                child: Padding(padding: const EdgeInsets.all(10), child: Column(
                  crossAxisAlignment: CrossAxisAlignment.start, children: [
                    Row(children: [
                      if (t.active) Container(width: 10, height: 10,
                        margin: const EdgeInsets.only(right: 6),
                        decoration: const BoxDecoration(color: Colors.green, shape: BoxShape.circle)),
                      Expanded(child: Text(t.name, style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 13))),
                      Text(t.cmdBase, style: const TextStyle(fontFamily: 'monospace', fontSize: 9, color: Colors.cyan)),
                    ]),
                    const SizedBox(height: 8),
                    if (t.isBool) Row(children: [
                      Expanded(child: ElevatedButton(
                        onPressed: _busy ? null : () => _start(t, 1),
                        style: ElevatedButton.styleFrom(backgroundColor: Colors.green,
                          minimumSize: const Size.fromHeight(42)),
                        child: const Text('СТАРТ', style: TextStyle(fontWeight: FontWeight.bold)))),
                      const SizedBox(width: 10),
                      Expanded(child: ElevatedButton(
                        onPressed: _busy ? null : () => _stop(t),
                        style: ElevatedButton.styleFrom(backgroundColor: Colors.red,
                          minimumSize: const Size.fromHeight(42)),
                        child: const Text('СТОП', style: TextStyle(fontWeight: FontWeight.bold)))),
                    ])
                    else _SliderTest(test: t,
                      onStart: _busy ? null : (v) => _start(t, v),
                      onStop: _busy ? null : () => _stop(t)),
                  ])));
            }),
          ListView.builder(
            padding: const EdgeInsets.all(8), itemCount: _learns.length,
            itemBuilder: (_, i) {
              final l = _learns[i];
              return Card(
                color: l.danger ? Colors.orange.withOpacity(0.15) : const Color(0xFF16213E),
                margin: const EdgeInsets.symmetric(vertical: 3),
                child: ListTile(
                  leading: Icon(l.danger ? Icons.warning : Icons.school,
                    color: l.danger ? Colors.orange : Colors.cyan),
                  title: Text(l.name, style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 13)),
                  subtitle: Text(l.desc, style: const TextStyle(color: Colors.white70, fontSize: 11)),
                  trailing: ElevatedButton(
                    onPressed: _busy ? null : () => _learn(l),
                    style: ElevatedButton.styleFrom(
                      backgroundColor: l.danger ? Colors.orange : Colors.cyan),
                    child: Text(l.danger ? 'ВЫПОЛН.' : 'СТАРТ', style: const TextStyle(fontSize: 11)))));
            }),
        ])),
      ]),
    );
  }
}

class _SliderTest extends StatefulWidget {
  final _Test test;
  final void Function(double)? onStart;
  final void Function()? onStop;
  const _SliderTest({required this.test, this.onStart, this.onStop});
  @override
  State<_SliderTest> createState() => _SliderTestState();
}

class _SliderTestState extends State<_SliderTest> {
  late double _val;
  @override
  void initState() { super.initState(); _val = (widget.test.min + widget.test.max) / 2; }
  @override
  Widget build(BuildContext context) {
    return Column(children: [
      Text(widget.test.fmt(_val),
        style: const TextStyle(color: Colors.yellow, fontWeight: FontWeight.bold, fontSize: 18)),
      Slider(
        value: _val.clamp(widget.test.min, widget.test.max),
        min: widget.test.min, max: widget.test.max,
        divisions: ((widget.test.max - widget.test.min) / widget.test.step).round(),
        onChanged: (v) => setState(() => _val = v)),
      Row(children: [
        Expanded(child: ElevatedButton(
          onPressed: widget.onStart != null ? () => widget.onStart!(_val) : null,
          style: ElevatedButton.styleFrom(backgroundColor: Colors.green,
            minimumSize: const Size.fromHeight(44)),
          child: const Text('СТАРТ', style: TextStyle(fontWeight: FontWeight.bold)))),
        const SizedBox(width: 10),
        Expanded(child: ElevatedButton(
          onPressed: widget.onStop,
          style: ElevatedButton.styleFrom(backgroundColor: Colors.red,
            minimumSize: const Size.fromHeight(44)),
          child: const Text('СТОП', style: TextStyle(fontWeight: FontWeight.bold)))),
      ]),
    ]);
  }
}
''')
print("✅ service_screen.dart (тесты + обучения)")

print()
print("=" * 60)
print("✅ Ячейка 4B готова! Все экраны созданы.")
print("=" * 60)
print()
print("Экраны:")
!ls lib/screens/
print()
print("Виджеты:")
!ls lib/widgets/

✅ main.dart
✅ fps_indicator.dart
✅ map_table_view.dart (градиент + история + save/reset)
✅ home_screen.dart (StreamSubscription fix)
✅ dashboard_screen.dart (настраиваемый + профиль + STFT/LTFT/режим)
  📝 Заглушка: graph_screen.dart
  📝 Заглушка: log_graph_screen.dart
  📝 Заглушка: logging_screen.dart
  📝 Заглушка: events_screen.dart
  📝 Заглушка: dtc_screen.dart
  📝 Заглушка: analyzer_screen.dart
  📝 Заглушка: ecu_read_screen.dart
  📝 Заглушка: service_screen.dart
  📝 Заглушка: performance_screen.dart
  📝 Заглушка: export_screen.dart
  📝 Заглушка: custom_pid_screen.dart
  📝 Заглушка: profile_screen.dart
  📝 Заглушка: rom_compare_screen.dart
  📝 Заглушка: terminal_screen.dart
  📝 Заглушка: settings_screen.dart

✅ Ячейка 4A готова!
  main.dart, home_screen, dashboard_screen
  fps_indicator, map_table_view
  + заглушки для 15 экранов (будут заменены в части B)
analyzer_screen.dart	export_screen.dart	 profile_screen.dart
custom_pid_screen.dart	graph_screen.dart	 rom_compare_screen.dart
da

In [ ]:
# @title 🔧 ФИКС: log_graph_screen.dart + nissan_pid_library.dart
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# ФИКС 1: log_graph_screen.dart
# Проблема: const _PI(...) с lambda и Color не могут быть const
# Решение: убираем const из класса и списка
# ================================================================
with open('lib/screens/log_graph_screen.dart', 'w') as f:
    f.write(r'''import 'dart:math';
import 'package:flutter/material.dart';
import 'package:fl_chart/fl_chart.dart';
import 'package:file_picker/file_picker.dart';
import '../models/obd_data.dart';
import '../services/analyzer_service.dart';

class _PI {
  final String key, label, unit;
  final Color color;
  final double Function(OBDData) get;
  final int digits;
  // НЕ const — lambda не может быть const
  _PI(this.key, this.label, this.unit, this.color, this.get, this.digits);
}

// НЕ const список — содержит lambda
final List<_PI> _params = [
  _PI('RPM',    'RPM',      'об/мин', Colors.blue,           (d) => d.rpm.toDouble(),         0),
  _PI('Speed',  'Скорость', 'км/ч',   Colors.cyan,           (d) => d.speed.toDouble(),        0),
  _PI('Timing', 'УОЗ',      '°',      Colors.lightGreen,     (d) => d.actualIgnition,          1),
  _PI('Knock',  'Knock',    '°',      Colors.deepOrange,     (d) => d.knockRetard,             1),
  _PI('Load',   'Нагрузка', '%',      Colors.amber,          (d) => d.engineLoad,              1),
  _PI('MAF',    'MAF',      'g/s',    Colors.purple,         (d) => d.mafGps,                  2),
  _PI('MAF_V',  'MAF V',    'V',      Colors.deepPurple,     (d) => d.mafVoltage,              3),
  _PI('AFR',    'AFR',      '',       Colors.yellow,         (d) => d.afr,                     2),
  _PI('ECT',    'ОЖ',       '°C',     Colors.red,            (d) => d.coolantTemp.toDouble(),  0),
  _PI('IAT',    'Впуск',    '°C',     Colors.cyan,           (d) => d.intakeTemp.toDouble(),   0),
  _PI('STFT',   'STFT',     '%',      Colors.lime,           (d) => d.shortFuelTrim,           1),
  _PI('LTFT',   'LTFT',     '%',      Colors.teal,           (d) => d.longFuelTrim,            1),
  _PI('VTC',    'VTC',      '°',      Colors.pink,           (d) => d.vtcActualAngle,          1),
  _PI('TPS',    'Дроссель', '%',      Colors.green,          (d) => d.throttlePos,             1),
  _PI('Batt',   'Батарея',  'V',      Colors.lightBlueAccent,(d) => d.batteryVoltage,          2),
  _PI('HP',     'HP',       'л.с.',   Colors.yellowAccent,   (d) => d.calculatedHP,            1),
  _PI('Torque', 'Момент',   'Нм',     Colors.orange,         (d) => d.calculatedTorqueNm,      0),
  _PI('VE',     'VE',       '%',      Colors.lightBlue,      (d) => d.volumetricEfficiency,    0),
  _PI('FuelLH', 'Расход',   'L/ч',    Colors.pink,           (d) => d.fuelFlowLph,             2),
  _PI('Fuel100','L/100км',  '',       Colors.pinkAccent,     (d) => d.fuelL100km,              1),
  _PI('Inj',    'Форсунки', 'ms',     Colors.amber,          (d) => d.injectorPulseWidth,      2),
  _PI('O2',     'O2',       'V',      Colors.indigo,         (d) => d.o2Voltage,               3),
];

_PI _p(String k) {
  try { return _params.firstWhere((p) => p.key == k); }
  catch (_) { return _params.first; }
}

class LogGraphScreen extends StatefulWidget {
  const LogGraphScreen({super.key});
  @override
  State<LogGraphScreen> createState() => _LogGraphScreenState();
}

class _LogGraphScreenState extends State<LogGraphScreen> {
  final _analyzer = AnalyzerService();
  List<OBDData>? _log;
  String? _name;
  bool _loading = false;
  Set<String> _sel = {'RPM', 'Timing', 'Knock'};
  double _rStart = 0, _rEnd = 1;
  int? _touchIdx;

  Future<void> _load() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv']);
      if (r == null) return;
      setState(() => _loading = true);
      _log  = await _analyzer.loadLogFromCSV(r.files.single.path!);
      _name = r.files.single.name;
      _rStart = 0; _rEnd = 1; _touchIdx = null;
      setState(() => _loading = false);
      if (mounted) ScaffoldMessenger.of(context).showSnackBar(SnackBar(
        content: Text('Загружено: ${_log!.length} записей'),
        backgroundColor: Colors.green));
    } catch (e) {
      setState(() => _loading = false);
    }
  }

  List<OBDData> get _range {
    if (_log == null) return [];
    final s = (_rStart * _log!.length).floor();
    final e = (_rEnd   * _log!.length).ceil().clamp(s + 1, _log!.length);
    return _log!.sublist(s, e);
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(
        title: const Text('Графики лога'),
        backgroundColor: const Color(0xFF16213E),
        actions: [
          if (_log != null)
            IconButton(
              icon: const Icon(Icons.zoom_out_map),
              tooltip: 'Сбросить zoom',
              onPressed: () => setState(() { _rStart = 0; _rEnd = 1; _touchIdx = null; }),
            ),
        ],
      ),
      body: SingleChildScrollView(
        child: Column(children: [
          // ── Кнопка загрузки ──────────────────────────────
          Padding(
            padding: const EdgeInsets.all(8),
            child: ElevatedButton.icon(
              onPressed: _loading ? null : _load,
              icon: _loading
                ? const SizedBox(width: 18, height: 18,
                    child: CircularProgressIndicator(strokeWidth: 2, color: Colors.white))
                : const Icon(Icons.folder_open),
              label: const Text('Загрузить CSV'),
              style: ElevatedButton.styleFrom(
                minimumSize: const Size.fromHeight(45),
                foregroundColor: Colors.white),
            ),
          ),
          if (_name != null)
            Padding(
              padding: const EdgeInsets.symmetric(horizontal: 8),
              child: Text('$_name — ${_log?.length ?? 0} записей',
                style: const TextStyle(color: Colors.green)),
            ),

          // ── Выбор параметров ─────────────────────────────
          if (_log != null)
            Padding(
              padding: const EdgeInsets.all(8),
              child: Column(
                crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Row(mainAxisAlignment: MainAxisAlignment.spaceBetween, children: [
                    const Text('ПАРАМЕТРЫ:',
                      style: TextStyle(color: Colors.white70, fontSize: 11,
                        fontWeight: FontWeight.bold)),
                    TextButton(
                      onPressed: () => setState(() => _sel = {'RPM'}),
                      child: const Text('Сброс', style: TextStyle(fontSize: 11))),
                  ]),
                  Wrap(spacing: 4, runSpacing: 4, children: _params.map((p) {
                    final sel = _sel.contains(p.key);
                    return FilterChip(
                      label: Text(p.label, style: TextStyle(fontSize: 11,
                        color: sel ? Colors.white : Colors.white70,
                        fontWeight: sel ? FontWeight.bold : FontWeight.normal)),
                      selected: sel,
                      onSelected: (s) => setState(() {
                        if (s) _sel.add(p.key);
                        else if (_sel.length > 1) _sel.remove(p.key);
                      }),
                      selectedColor: p.color.withOpacity(0.5),
                      backgroundColor: const Color(0xFF0F3460),
                      side: BorderSide(color: sel ? p.color : Colors.white24),
                      materialTapTargetSize: MaterialTapTargetSize.shrinkWrap,
                    );
                  }).toList()),
                ],
              ),
            ),

          // ── Контент (только если лог загружен) ───────────
          if (_log != null && _log!.isNotEmpty) ...[
            // Курсор
            if (_touchIdx != null) _buildTouchInfo(),
            // Статистика
            Padding(padding: const EdgeInsets.all(8), child: _buildStats()),
            // Совмещённый
            Padding(
              padding: const EdgeInsets.all(8),
              child: Card(color: const Color(0xFF16213E),
                child: Padding(padding: const EdgeInsets.all(10), child: Column(
                  crossAxisAlignment: CrossAxisAlignment.start,
                  children: [
                    const Text('СОВМЕЩЁННЫЙ (0-100%)',
                      style: TextStyle(color: Colors.white70, fontSize: 11,
                        fontWeight: FontWeight.bold)),
                    const Text('Тап — курсор',
                      style: TextStyle(color: Colors.cyan, fontSize: 9)),
                    const SizedBox(height: 8),
                    SizedBox(height: 300, child: _buildCombined()),
                  ],
                )),
              ),
            ),
            // Отдельные графики
            ..._sel.map((k) {
              final p = _p(k);
              return Padding(
                padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4),
                child: Card(color: const Color(0xFF16213E),
                  child: Padding(padding: const EdgeInsets.all(10), child: Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      Row(children: [
                        Container(width: 12, height: 12,
                          decoration: BoxDecoration(color: p.color,
                            borderRadius: BorderRadius.circular(2))),
                        const SizedBox(width: 6),
                        Text('${p.label} (${p.unit})', style: TextStyle(
                          color: p.color, fontSize: 13, fontWeight: FontWeight.bold)),
                        const Spacer(),
                        if (_log != null && _log!.isNotEmpty)
                          Text(p.get(_log!.last).toStringAsFixed(p.digits),
                            style: TextStyle(color: p.color, fontSize: 14,
                              fontWeight: FontWeight.bold)),
                      ]),
                      const SizedBox(height: 6),
                      SizedBox(height: 200, child: _buildSingle(p)),
                    ],
                  )),
                ),
              );
            }),
            // Zoom slider
            Padding(padding: const EdgeInsets.all(8), child: _buildSlider()),
            const SizedBox(height: 20),
          ],

          if (_log == null)
            const Padding(
              padding: EdgeInsets.all(40),
              child: Text('Загрузите CSV для просмотра графиков',
                style: TextStyle(color: Colors.white54, fontSize: 14),
                textAlign: TextAlign.center),
            ),
        ]),
      ),
    );
  }

  Widget _buildTouchInfo() {
    if (_touchIdx == null || _log == null || _touchIdx! >= _log!.length)
      return const SizedBox();
    final d = _log![_touchIdx!];
    return Card(
      color: Colors.cyan.withOpacity(0.15),
      margin: const EdgeInsets.all(8),
      child: Padding(padding: const EdgeInsets.all(8), child: Column(
        crossAxisAlignment: CrossAxisAlignment.start,
        children: [
          Row(children: [
            const Icon(Icons.touch_app, color: Colors.cyan, size: 16),
            const SizedBox(width: 4),
            Text('Точка #$_touchIdx',
              style: const TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold)),
            const Spacer(),
            GestureDetector(
              onTap: () => setState(() => _touchIdx = null),
              child: const Icon(Icons.close, size: 16, color: Colors.white54)),
          ]),
          const SizedBox(height: 4),
          Wrap(spacing: 10, runSpacing: 4, children: _sel.map((k) {
            final p = _p(k);
            return Row(mainAxisSize: MainAxisSize.min, children: [
              Container(width: 8, height: 8, color: p.color),
              const SizedBox(width: 3),
              Text('${p.label}: ${p.get(d).toStringAsFixed(p.digits)} ${p.unit}',
                style: TextStyle(color: p.color, fontSize: 11, fontWeight: FontWeight.bold)),
            ]);
          }).toList()),
        ],
      )),
    );
  }

  Widget _buildStats() {
    if (_log == null || _log!.isEmpty) return const SizedBox();
    return Card(color: const Color(0xFF16213E), child: Padding(
      padding: const EdgeInsets.all(8),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        const Text('СТАТИСТИКА:',
          style: TextStyle(color: Colors.white70, fontSize: 11, fontWeight: FontWeight.bold)),
        const SizedBox(height: 6),
        Table(
          border: TableBorder.all(color: Colors.white12, width: 0.5),
          columnWidths: const {
            0: FlexColumnWidth(2), 1: FlexColumnWidth(1.5),
            2: FlexColumnWidth(1.5), 3: FlexColumnWidth(1.5),
          },
          children: [
            const TableRow(
              decoration: BoxDecoration(color: Color(0xFF0F3460)),
              children: [
                Padding(padding: EdgeInsets.all(4),
                  child: Text('Параметр', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold))),
                Padding(padding: EdgeInsets.all(4),
                  child: Text('Мин', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
                Padding(padding: EdgeInsets.all(4),
                  child: Text('Сред', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
                Padding(padding: EdgeInsets.all(4),
                  child: Text('Макс', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold), textAlign: TextAlign.center)),
              ],
            ),
            ..._sel.map((k) {
              final p    = _p(k);
              final vals = _log!.map((d) => p.get(d)).toList();
              final minV = vals.reduce(min);
              final maxV = vals.reduce(max);
              final avgV = vals.reduce((a, b) => a + b) / vals.length;
              return TableRow(children: [
                Padding(padding: const EdgeInsets.all(4),
                  child: Row(children: [
                    Container(width: 8, height: 8, color: p.color),
                    const SizedBox(width: 4),
                    Expanded(child: Text(p.label,
                      style: const TextStyle(fontSize: 11))),
                  ])),
                Padding(padding: const EdgeInsets.all(4),
                  child: Text(minV.toStringAsFixed(p.digits),
                    style: const TextStyle(fontSize: 11, fontFamily: 'monospace'),
                    textAlign: TextAlign.center)),
                Padding(padding: const EdgeInsets.all(4),
                  child: Text(avgV.toStringAsFixed(p.digits),
                    style: const TextStyle(fontSize: 11, fontFamily: 'monospace'),
                    textAlign: TextAlign.center)),
                Padding(padding: const EdgeInsets.all(4),
                  child: Text(maxV.toStringAsFixed(p.digits),
                    style: const TextStyle(fontSize: 11, fontFamily: 'monospace'),
                    textAlign: TextAlign.center)),
              ]);
            }),
          ],
        ),
      ]),
    ));
  }

  Widget _buildCombined() {
    final data = _range;
    if (data.isEmpty) return const SizedBox();
    final step  = (data.length / 300).ceil().clamp(1, 100);
    final lines = <LineChartBarData>[];

    for (final k in _sel) {
      final p    = _p(k);
      final vals = data.map((d) => p.get(d)).toList();
      final minV = vals.reduce(min);
      final maxV = vals.reduce(max);
      final r    = (maxV - minV).abs() < 0.001 ? 1.0 : maxV - minV;
      final spots = <FlSpot>[];
      for (int i = 0; i < data.length; i += step) {
        spots.add(FlSpot(i.toDouble(), (p.get(data[i]) - minV) / r * 100));
      }
      lines.add(LineChartBarData(
        spots: spots, isCurved: false, color: p.color,
        barWidth: 1.5, dotData: const FlDotData(show: false)));
    }

    return LineChart(LineChartData(
      gridData: FlGridData(show: true, drawVerticalLine: false,
        horizontalInterval: 25,
        getDrawingHorizontalLine: (v) =>
          FlLine(color: Colors.white.withOpacity(0.1), strokeWidth: 1)),
      titlesData: FlTitlesData(
        rightTitles:  const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        topTitles:    const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        bottomTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        leftTitles: AxisTitles(sideTitles: SideTitles(showTitles: true,
          reservedSize: 32, interval: 25,
          getTitlesWidget: (v, _) => Text('${v.toInt()}%',
            style: const TextStyle(color: Colors.white54, fontSize: 9))))),
      borderData: FlBorderData(show: true,
        border: Border.all(color: Colors.white.withOpacity(0.1))),
      minX: 0, maxX: data.length.toDouble(),
      minY: 0, maxY: 100,
      lineBarsData: lines,
      lineTouchData: LineTouchData(
        enabled: true,
        touchCallback: (event, response) {
          if (response?.lineBarSpots != null &&
              response!.lineBarSpots!.isNotEmpty) {
            final si = (_rStart * (_log?.length ?? 0)).floor();
            final idx = response.lineBarSpots!.first.x.toInt();
            if (mounted) setState(() => _touchIdx = si + idx);
          }
        },
        touchTooltipData: LineTouchTooltipData(
          getTooltipColor: (_) => Colors.black87,
          getTooltipItems: (spots) {
            final selList = _sel.toList();
            return spots.asMap().entries.map((e) {
              if (e.key >= selList.length) return null;
              final p   = _p(selList[e.key]);
              final idx = e.value.x.toInt().clamp(0, data.length - 1);
              return LineTooltipItem(
                '${p.label}: ${p.get(data[idx]).toStringAsFixed(p.digits)} ${p.unit}',
                TextStyle(color: p.color, fontSize: 10, fontWeight: FontWeight.bold));
            }).toList();
          },
        ),
      ),
    ));
  }

  Widget _buildSingle(_PI p) {
    final data = _range;
    if (data.isEmpty) return const SizedBox();
    final step = (data.length / 300).ceil().clamp(1, 100);
    final spots = <FlSpot>[];
    double minY = double.infinity, maxY = -double.infinity;
    for (int i = 0; i < data.length; i += step) {
      final v = p.get(data[i]);
      spots.add(FlSpot(i.toDouble(), v));
      if (v < minY) minY = v;
      if (v > maxY) maxY = v;
    }
    if (minY == maxY) { minY -= 1; maxY += 1; }
    final margin = (maxY - minY) * 0.05;
    minY -= margin; maxY += margin;
    final interval = (maxY - minY) / 4;

    return LineChart(LineChartData(
      gridData: FlGridData(show: true, drawVerticalLine: false,
        horizontalInterval: interval == 0 ? 1 : interval,
        getDrawingHorizontalLine: (v) =>
          FlLine(color: Colors.white.withOpacity(0.1), strokeWidth: 1)),
      titlesData: FlTitlesData(
        rightTitles:  const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        topTitles:    const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        bottomTitles: const AxisTitles(sideTitles: SideTitles(showTitles: false)),
        leftTitles: AxisTitles(sideTitles: SideTitles(showTitles: true,
          reservedSize: 48,
          interval: interval == 0 ? 1 : interval,
          getTitlesWidget: (v, _) => Text(v.toStringAsFixed(p.digits),
            style: const TextStyle(color: Colors.white54, fontSize: 9))))),
      borderData: FlBorderData(show: true,
        border: Border.all(color: Colors.white.withOpacity(0.1))),
      minX: 0, maxX: data.length.toDouble(),
      minY: minY, maxY: maxY,
      lineBarsData: [
        LineChartBarData(spots: spots, isCurved: false, color: p.color,
          barWidth: 1.5, dotData: const FlDotData(show: false),
          belowBarData: BarAreaData(show: true, color: p.color.withOpacity(0.15))),
      ],
      lineTouchData: LineTouchData(
        enabled: true,
        touchTooltipData: LineTouchTooltipData(
          getTooltipColor: (_) => Colors.black87,
          getTooltipItems: (spots) => spots.map((s) => LineTooltipItem(
            '${p.label}: ${s.y.toStringAsFixed(p.digits)} ${p.unit}',
            TextStyle(color: p.color, fontSize: 11, fontWeight: FontWeight.bold))).toList()),
      ),
    ));
  }

  Widget _buildSlider() {
    if (_log == null || _log!.length < 10) return const SizedBox();
    return Card(color: const Color(0xFF16213E),
      child: Padding(padding: const EdgeInsets.all(8), child: Column(children: [
        Text(
          'ZOOM: ${(_rStart * _log!.length).toInt()} — '
          '${(_rEnd * _log!.length).toInt()} / ${_log!.length}',
          style: const TextStyle(color: Colors.white70, fontSize: 11)),
        RangeSlider(
          values: RangeValues(_rStart, _rEnd),
          min: 0, max: 1, divisions: 100,
          activeColor: const Color(0xFFE94560),
          inactiveColor: Colors.white24,
          onChanged: (v) => setState(() {
            if (v.end - v.start >= 0.02) {
              _rStart = v.start; _rEnd = v.end; _touchIdx = null;
            }
          }),
        ),
      ])));
  }
}
''')
print("✅ log_graph_screen.dart — убран const, добавлен тач-курсор + MAF_V + Torque + VE")

# ================================================================
# ФИКС 2: nissan_pid_library.dart
# Проблема: CustomPid.defaultUnchanged не существует
# toCustomPid() метод содержит некорректный код
# Решение: убрать некорректный метод toCustomPid() из NissanPidDef
# ================================================================
with open('lib/services/nissan_pid_library.dart', 'w') as f:
    f.write(r'''// ignore: depend_on_referenced_packages

class NissanPidDef {
  final String id;
  final String cmd;
  final String answer;
  final String name;
  final String desc;
  final String unit;
  final int    bytesCount;
  final double Function(List<int>) formula;
  final double minVal;
  final double maxVal;
  final int    priority;
  final String category;

  const NissanPidDef({
    required this.id,
    required this.cmd,
    required this.answer,
    required this.name,
    required this.desc,
    required this.unit,
    required this.bytesCount,
    required this.formula,
    this.minVal   = 0,
    this.maxVal   = 255,
    this.priority = 3,
    this.category = 'other',
  });
}

// ignore: avoid_classes_with_only_static_members
class NissanPidLibrary {
  static final List<NissanPidDef> all = [
    // ── Приоритет 1: быстрые (каждый цикл) ─────────────────────
    NissanPidDef(id:'RPM',    cmd:'2212010401', answer:'621201',
      name:'RPM',    desc:'Обороты',         unit:'RPM',   bytesCount:2,
      priority:1, category:'engine',    minVal:0, maxVal:8000,
      formula: (b) => (b[0]*256+b[1])*12.5),

    NissanPidDef(id:'TIMING', cmd:'22110A0401', answer:'62110A',
      name:'TIMING', desc:'УОЗ факт',        unit:'°BTDC', bytesCount:1,
      priority:1, category:'ignition',  minVal:-20, maxVal:60,
      formula: (b) => (110-b[0]).toDouble()),

    NissanPidDef(id:'KNOCK',  cmd:'22112D0401', answer:'62112D',
      name:'KNOCK',  desc:'Корр.УОЗ',        unit:'°',     bytesCount:1,
      priority:1, category:'ignition',  minVal:-30, maxVal:30,
      formula: (b) { int v=b[0]; if(v>=128) v-=256; return v.toDouble(); }),

    NissanPidDef(id:'TPS',    cmd:'22111E0401', answer:'62111E',
      name:'TPS',    desc:'Дроссель',         unit:'%',     bytesCount:1,
      priority:1, category:'throttle',  minVal:0, maxVal:100,
      formula: (b) => b[0]*0.35),

    /// MAF возвращает ВОЛЬТАЖ.
    /// Конвертация V→g/s через таблицу Hitachi в OBDService.
    NissanPidDef(id:'MAF_V',  cmd:'2212040401', answer:'621204',
      name:'MAF_V',  desc:'MAF напряжение',   unit:'V',     bytesCount:2,
      priority:1, category:'air',       minVal:0, maxVal:5,
      formula: (b) => (b[0]*256+b[1])*0.005),

    NissanPidDef(id:'ECT',    cmd:'2211010401', answer:'621101',
      name:'ECT',    desc:'Темп. ОЖ',         unit:'°C',    bytesCount:1,
      priority:1, category:'temp',      minVal:-30, maxVal:130,
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'LOAD',   cmd:'2211170401', answer:'621117',
      name:'LOAD',   desc:'Нагрузка',          unit:'%',     bytesCount:1,
      priority:1, category:'engine',    minVal:0, maxVal:100,
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'SPEED',  cmd:'2211020401', answer:'621102',
      name:'SPEED',  desc:'Скорость',          unit:'км/ч',  bytesCount:1,
      priority:1, category:'engine',    minVal:0, maxVal:200,
      formula: (b) => b[0]*2.0),

    NissanPidDef(id:'VTC_ACT',cmd:'2211350401', answer:'621135',
      name:'VTC_ACT',desc:'VTC факт B1',       unit:'°CA',   bytesCount:1,
      priority:1, category:'vtc',       minVal:-10, maxVal:50,
      formula: (b) => b[0]*0.5-64),

    NissanPidDef(id:'STFT',   cmd:'2211230401', answer:'621123',
      name:'STFT',   desc:'STFT B1',           unit:'%',     bytesCount:1,
      priority:1, category:'fuel',      minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'LTFT',   cmd:'2211250401', answer:'621125',
      name:'LTFT',   desc:'LTFT B1',           unit:'%',     bytesCount:1,
      priority:1, category:'fuel',      minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'INJ_B1', cmd:'2212060401', answer:'621206',
      name:'INJ_B1', desc:'Впрыск B1',         unit:'ms',    bytesCount:2,
      priority:1, category:'fuel',      minVal:0, maxVal:30,
      formula: (b) => (b[0]*256+b[1])*0.01),

    NissanPidDef(id:'O2_B1S1',cmd:'2211180401', answer:'621118',
      name:'O2_B1S1',desc:'O2 B1S1',          unit:'V',     bytesCount:1,
      priority:1, category:'fuel',      minVal:0, maxVal:1,
      formula: (b) => b[0]*0.01),

    // ── Приоритет 2: средние (каждый 3-й цикл) ──────────────────
    NissanPidDef(id:'PEDAL',  cmd:'22117C0401', answer:'62117C',
      name:'PEDAL',  desc:'Педаль газа',       unit:'%',     bytesCount:1,
      priority:2, category:'throttle',
      formula: (b) => b[0]*0.5),

    NissanPidDef(id:'BATT',   cmd:'2211030401', answer:'621103',
      name:'BATT',   desc:'Напряжение',         unit:'V',     bytesCount:1,
      priority:2, category:'electric',  minVal:8, maxVal:16,
      formula: (b) => b[0]*0.08),

    NissanPidDef(id:'IAT',    cmd:'2211060401', answer:'621106',
      name:'IAT',    desc:'Темп. впуска',       unit:'°C',    bytesCount:1,
      priority:2, category:'temp',      minVal:-30, maxVal:100,
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'MAP_V',  cmd:'22112A0401', answer:'62112A',
      name:'MAP_V',  desc:'MAP датчик',         unit:'V',     bytesCount:1,
      priority:2, category:'air',
      formula: (b) => b[0]*0.02),

    NissanPidDef(id:'IACV',   cmd:'22110B0401', answer:'62110B',
      name:'IACV',   desc:'Клапан ХХ',          unit:'%',     bytesCount:1,
      priority:2, category:'idle',
      formula: (b) => b[0]*0.5),

    NissanPidDef(id:'VTC_SOL',cmd:'2211380401', answer:'621138',
      name:'VTC_SOL',desc:'VTC Sol B1',         unit:'%',     bytesCount:1,
      priority:2, category:'vtc',
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'STFT_B2',cmd:'2211240401', answer:'621124',
      name:'STFT_B2',desc:'STFT B2',            unit:'%',     bytesCount:1,
      priority:2, category:'fuel',      minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'LTFT_B2',cmd:'2211260401', answer:'621126',
      name:'LTFT_B2',desc:'LTFT B2',            unit:'%',     bytesCount:1,
      priority:2, category:'fuel',      minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'INJ_BASE',cmd:'2212080401',answer:'621208',
      name:'INJ_BASE',desc:'Впрыск баз.',        unit:'ms',    bytesCount:2,
      priority:2, category:'fuel',
      formula: (b) => (b[0]*256+b[1])/2048.0),

    NissanPidDef(id:'VTC_DUTY',cmd:'22122D0401',answer:'62122D',
      name:'VTC_DUTY',desc:'VTC Duty B1',        unit:'%',     bytesCount:2,
      priority:2, category:'vtc',
      formula: (b) => (b[0]*256+b[1])*3200.0/32768.0),

    NissanPidDef(id:'POWER',  cmd:'2212570401', answer:'621257',
      name:'POWER',  desc:'Мощность запр.',      unit:'kW',    bytesCount:2,
      priority:2, category:'engine',
      formula: (b) => (b[0]*256+b[1])*0.03125),

    NissanPidDef(id:'TORQUE', cmd:'2212280401', answer:'621228',
      name:'TORQUE', desc:'Момент',              unit:'Nm',    bytesCount:2,
      priority:2, category:'engine',
      formula: (b) {
        int v=b[0]*256+b[1];
        if(v>=32768) v-=65536;
        return v/4.0;
      }),

    // ── Приоритет 3: медленные (каждый 10-й цикл) ───────────────
    NissanPidDef(id:'OIL_T',  cmd:'22111F0401', answer:'62111F',
      name:'OIL_T',  desc:'Темп. масла',         unit:'°C',    bytesCount:1,
      priority:3, category:'temp',
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'FUEL_T', cmd:'2211040401', answer:'621104',
      name:'FUEL_T', desc:'Темп. топлива',        unit:'°C',    bytesCount:1,
      priority:3, category:'temp',
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'O2_B2S1',cmd:'2211190401', answer:'621119',
      name:'O2_B2S1',desc:'O2 B2S1',             unit:'V',     bytesCount:1,
      priority:3, category:'fuel',
      formula: (b) => b[0]*0.01),

    NissanPidDef(id:'O2_B1S2',cmd:'22111A0401', answer:'62111A',
      name:'O2_B1S2',desc:'O2 B1S2',             unit:'V',     bytesCount:1,
      priority:3, category:'fuel',
      formula: (b) => b[0]*0.01),

    NissanPidDef(id:'AF_B1S1',cmd:'2212250401', answer:'621225',
      name:'AF_B1S1',desc:'A/F B1S1',            unit:'V',     bytesCount:2,
      priority:3, category:'fuel',
      formula: (b) => (b[0]*256+b[1])*0.005),

    NissanPidDef(id:'BARO',   cmd:'2211290401', answer:'621129',
      name:'BARO',   desc:'Атм. давление',        unit:'V',     bytesCount:1,
      priority:3, category:'air',
      formula: (b) => b[0]*0.02),

    NissanPidDef(id:'ALT_SPD',cmd:'2211900401', answer:'621190',
      name:'ALT_SPD',desc:'Об. генератора',       unit:'RPM',   bytesCount:1,
      priority:3, category:'electric',
      formula: (b) => b[0]*0.75),

    NissanPidDef(id:'BAT_SOC',cmd:'2211510401', answer:'621151',
      name:'BAT_SOC',desc:'Заряд АКБ',            unit:'%',     bytesCount:1,
      priority:3, category:'electric',
      formula: (b) => b[0].toDouble()),

    NissanPidDef(id:'FAN',    cmd:'2211470401', answer:'621147',
      name:'FAN',    desc:'Вентилятор',            unit:'%',     bytesCount:1,
      priority:3, category:'other',
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'IDLE_BS',cmd:'22110D0401', answer:'62110D',
      name:'IDLE_BS',desc:'Базовые ХХ',           unit:'RPM',   bytesCount:1,
      priority:3, category:'idle',    maxVal:3200,
      formula: (b) => b[0]*12.5),

    NissanPidDef(id:'CAT_T',  cmd:'2213020401', answer:'621302',
      name:'CAT_T',  desc:'Темп. катал.',          unit:'°C',    bytesCount:2,
      priority:3, category:'temp',    minVal:0, maxVal:1000,
      formula: (b) => (b[0]*256+b[1])*0.1-40),

    NissanPidDef(id:'AMBIENT',cmd:'22130A0401', answer:'62130A',
      name:'AMBIENT',desc:'Темп. воздуха',         unit:'°C',    bytesCount:2,
      priority:3, category:'temp',    minVal:-40, maxVal:60,
      formula: (b) => (b[0]*256+b[1])*0.1-40),

    NissanPidDef(id:'MISFIRE1',cmd:'2212220401',answer:'621222',
      name:'MISFIRE1',desc:'Пропуски Ц1',          unit:'cnt',   bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),

    NissanPidDef(id:'MISFIRE2',cmd:'2212230401',answer:'621223',
      name:'MISFIRE2',desc:'Пропуски Ц2',          unit:'cnt',   bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),

    NissanPidDef(id:'MISFIRE3',cmd:'2212240401',answer:'621224',
      name:'MISFIRE3',desc:'Пропуски Ц3',          unit:'cnt',   bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),

    NissanPidDef(id:'MISFIRE4',cmd:'2212250401',answer:'621225',
      name:'MISFIRE4',desc:'Пропуски Ц4',          unit:'cnt',   bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),
  ];

  static List<NissanPidDef> byPriority(int p) =>
      all.where((x) => x.priority == p).toList();

  static NissanPidDef? byId(String id) {
    try { return all.firstWhere((p) => p.id == id); }
    catch (_) { return null; }
  }

  static List<NissanPidDef> byCategory(String c) =>
      all.where((x) => x.category == c).toList();

  static List<String> get categories =>
      all.map((p) => p.category).toSet().toList()..sort();
}
''')
print("✅ nissan_pid_library.dart — убран некорректный toCustomPid()")



✅ log_graph_screen.dart — убран const, добавлен тач-курсор + MAF_V + Torque + VE
✅ nissan_pid_library.dart — убран некорректный toCustomPid()


In [ ]:
# @title 🔧 ФИКС: VE карта + чтение CSV из V5/V6
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# ФИКС 1: OBDData.fromCsvRow — гибкий парсер (V5 старый + V6 новый)
# ================================================================
with open('lib/models/obd_data.dart', 'w') as f:
    f.write(r'''import '../constants.dart';

class OBDData {
  final DateTime timestamp;
  final int    rpm;
  final int    speed;
  final double engineLoad;
  final int    coolantTemp;
  final int    intakeTemp;
  final double mafVoltage;
  final double mafGps;
  final double throttlePos;
  final double ignitionTiming;
  final double actualIgnition;
  final double shortFuelTrim;
  final double longFuelTrim;
  final double o2Voltage;
  final double afr;
  final double vtcTargetAngle;
  final double vtcActualAngle;
  final double knockRetard;
  final int    knockCount;
  final double injectorDuty;
  final double injectorPulseWidth;
  final double requestedTorque;
  final double actualTorque;
  final double oilTemp;
  final double afrTarget;
  final double lambda;
  final double manifoldPressure;
  final double acceleratorPedal;
  final double throttleActual;
  final double batteryVoltage;
  final double engineDisplacement;
  final double tripFuelL;

  const OBDData({
    required this.timestamp,
    this.rpm = 0, this.speed = 0, this.engineLoad = 0,
    this.coolantTemp = 0, this.intakeTemp = 0,
    this.mafVoltage = 0, this.mafGps = 0,
    this.throttlePos = 0, this.ignitionTiming = 0, this.actualIgnition = 0,
    this.shortFuelTrim = 0, this.longFuelTrim = 0,
    this.o2Voltage = 0, this.afr = 14.7,
    this.vtcTargetAngle = 0, this.vtcActualAngle = 0,
    this.knockRetard = 0, this.knockCount = 0,
    this.injectorDuty = 0, this.injectorPulseWidth = 0,
    this.requestedTorque = 0, this.actualTorque = 0,
    this.oilTemp = 0, this.afrTarget = 14.7,
    this.lambda = 1.0, this.manifoldPressure = 0,
    this.acceleratorPedal = 0, this.throttleActual = 0,
    this.batteryVoltage = 0, this.engineDisplacement = 2.0,
    this.tripFuelL = 0,
  });

  double get calculatedHP {
    if (mafGps <= 0 || rpm <= 0 || afr <= 0) return 0;
    final mafFuelGps = mafGps / afr;
    final powerKW    = mafFuelGps * 3600.0 / 250.0;
    return (powerKW * 1.3596).clamp(0, 500);
  }

  double get calculatedTorqueNm {
    if (calculatedHP <= 0 || rpm <= 0) return 0;
    final powerW = calculatedHP / 1.3596 * 1000;
    final omega  = rpm * 2 * 3.14159 / 60;
    return (powerW / omega).clamp(0, 400);
  }

  double get volumetricEfficiency {
    if (rpm <= 0 || mafGps <= 0) return 0;
    const airDensity = 1.184;
    final theoretical = rpm * engineDisplacement * airDensity / 120.0;
    if (theoretical <= 0) return 0;
    return (mafGps / theoretical * 100).clamp(0, 150);
  }

  double get fuelMassFlowGps => mafGps > 0 && afr > 0 ? mafGps / afr : 0;
  double get fuelFlowLph =>
      fuelMassFlowGps * 3600.0 / AppConstants.gasolineDensity;
  double get fuelL100km {
    if (speed < 5) return 0;
    return fuelFlowLph / speed * 100.0;
  }

  double get totalFuelTrim => shortFuelTrim + longFuelTrim;
  double get vtcError => (vtcTargetAngle - vtcActualAngle).abs();

  String get engineMode {
    if (rpm < 100)                        return 'STOP';
    if (rpm < 900 && throttlePos < 5)     return 'IDLE';
    if (throttlePos > 80)                 return 'WOT';
    if (throttlePos < 10 && speed > 0)    return 'COAST';
    return 'CRUISE';
  }

  List<dynamic> toCsvRow() => [
    timestamp.millisecondsSinceEpoch,
    rpm, speed,
    engineLoad.toStringAsFixed(2),
    coolantTemp, intakeTemp,
    mafVoltage.toStringAsFixed(4),
    mafGps.toStringAsFixed(3),
    throttlePos.toStringAsFixed(2),
    ignitionTiming.toStringAsFixed(2),
    shortFuelTrim.toStringAsFixed(2),
    longFuelTrim.toStringAsFixed(2),
    o2Voltage.toStringAsFixed(4),
    afr.toStringAsFixed(3),
    vtcTargetAngle.toStringAsFixed(2),
    vtcActualAngle.toStringAsFixed(2),
    knockRetard.toStringAsFixed(2),
    knockCount,
    actualIgnition.toStringAsFixed(2),
    injectorDuty.toStringAsFixed(2),
    injectorPulseWidth.toStringAsFixed(2),
    requestedTorque.toStringAsFixed(2),
    actualTorque.toStringAsFixed(2),
    oilTemp.toStringAsFixed(1),
    afrTarget.toStringAsFixed(3),
    lambda.toStringAsFixed(4),
    manifoldPressure.toStringAsFixed(2),
    acceleratorPedal.toStringAsFixed(2),
    throttleActual.toStringAsFixed(2),
    calculatedHP.toStringAsFixed(2),
    calculatedTorqueNm.toStringAsFixed(2),
    batteryVoltage.toStringAsFixed(2),
    volumetricEfficiency.toStringAsFixed(1),
    fuelFlowLph.toStringAsFixed(3),
    fuelL100km.toStringAsFixed(2),
    tripFuelL.toStringAsFixed(3),
  ];

  static List<String> csvHeaders() => [
    'Timestamp_ms', 'RPM', 'Speed_kmh', 'EngineLoad_pct',
    'CoolantTemp_C', 'IntakeTemp_C', 'MAF_V', 'MAF_gps',
    'ThrottlePos_pct', 'IgnitionTiming_deg',
    'STFT_pct', 'LTFT_pct', 'O2Voltage_V', 'AFR',
    'VTC_Target_deg', 'VTC_Actual_deg', 'KnockRetard_deg', 'KnockCount',
    'ActualIgnition_deg', 'InjectorDuty_pct', 'InjectorPW_ms',
    'RequestedTorque_Nm', 'ActualTorque_Nm', 'OilTemp_C',
    'AFR_Target', 'Lambda', 'ManifoldPressure_kPa',
    'AcceleratorPedal_pct', 'ThrottleActual_pct',
    'EstimatedHP', 'EstimatedTorque_Nm',
    'BatteryVoltage_V', 'VE_pct',
    'FuelFlow_Lph', 'FuelConsumption_L100km', 'TripFuel_L',
  ];

  /// Гибкий парсер CSV — работает с логами V5 (34 колонки) и V6 (36 колонок).
  /// V5 headers: 'Timestamp_ms','RPM','Speed_kmh','EngineLoad_pct',
  ///             'CoolantTemp_C','IntakeTemp_C','MAF_gs','ThrottlePos_pct',
  ///             'IgnitionTiming_deg','STFT_pct','LTFT_pct','O2Voltage_V','AFR',
  ///             'VTC_Target_deg','VTC_Actual_deg','KnockRetard_deg','KnockCount',
  ///             'ActualIgnition_deg','InjectorDuty_pct','RequestedTorque_Nm',...
  /// V6 headers: 'Timestamp_ms','RPM','Speed_kmh','EngineLoad_pct',
  ///             'CoolantTemp_C','IntakeTemp_C','MAF_V','MAF_gps',
  ///             'ThrottlePos_pct','IgnitionTiming_deg',...
  ///
  /// Определяем формат по наличию 'MAF_V' в заголовке (передаётся отдельно).
  static OBDData? fromCsvRow(List<dynamic> row, {bool isV6Format = true}) {
    try {
      if (row.isEmpty) return null;

      double _d(int i, [double def = 0]) {
        if (i < 0 || i >= row.length) return def;
        return double.tryParse(row[i].toString()) ?? def;
      }

      int _i(int i, [int def = 0]) {
        if (i < 0 || i >= row.length) return def;
        return int.tryParse(row[i].toString()) ?? def;
      }

      if (isV6Format) {
        // V6: [0]ts [1]rpm [2]speed [3]load [4]ect [5]iat
        //     [6]MAF_V [7]MAF_gps [8]tps [9]timing ...
        return OBDData(
          timestamp:       DateTime.fromMillisecondsSinceEpoch(_i(0)),
          rpm:             _i(1),
          speed:           _i(2),
          engineLoad:      _d(3),
          coolantTemp:     _i(4),
          intakeTemp:      _i(5),
          mafVoltage:      _d(6),
          mafGps:          _d(7),
          throttlePos:     _d(8),
          ignitionTiming:  _d(9),
          shortFuelTrim:   _d(10),
          longFuelTrim:    _d(11),
          o2Voltage:       _d(12),
          afr:             _d(13, 14.7),
          vtcTargetAngle:  _d(14),
          vtcActualAngle:  _d(15),
          knockRetard:     _d(16),
          knockCount:      _i(17),
          actualIgnition:  _d(18),
          injectorDuty:    _d(19),
          injectorPulseWidth: _d(20),
          requestedTorque: _d(21),
          actualTorque:    _d(22),
          oilTemp:         _d(23),
          afrTarget:       _d(24, 14.7),
          lambda:          _d(25, 1.0),
          manifoldPressure: _d(26),
          acceleratorPedal: _d(27),
          throttleActual:   _d(28),
          batteryVoltage:   _d(31),
          tripFuelL:        _d(35),
        );
      } else {
        // V5: [0]ts [1]rpm [2]speed [3]load [4]ect [5]iat
        //     [6]MAF_gs [7]tps [8]timing [9]stft [10]ltft
        //     [11]o2 [12]afr [13]vtc_tgt [14]vtc_act
        //     [15]knock [16]knock_cnt [17]actual_ign [18]inj_duty ...
        final mafGs = _d(6);
        return OBDData(
          timestamp:       DateTime.fromMillisecondsSinceEpoch(_i(0)),
          rpm:             _i(1),
          speed:           _i(2),
          engineLoad:      _d(3),
          coolantTemp:     _i(4),
          intakeTemp:      _i(5),
          mafVoltage:      0,       // в V5 не было
          mafGps:          mafGs,   // в V5 это уже g/s
          throttlePos:     _d(7),
          ignitionTiming:  _d(8),
          shortFuelTrim:   _d(9),
          longFuelTrim:    _d(10),
          o2Voltage:       _d(11),
          afr:             _d(12, 14.7),
          vtcTargetAngle:  _d(13),
          vtcActualAngle:  _d(14),
          knockRetard:     _d(15),
          knockCount:      _i(16),
          actualIgnition:  _d(17),
          injectorDuty:    _d(18),
          injectorPulseWidth: _d(18) / 100.0 * 20.0, // грубая оценка
          requestedTorque: _d(19),
          actualTorque:    _d(20),
          oilTemp:         _d(21),
          afrTarget:       _d(22, 14.7),
          lambda:          _d(23, 1.0),
          manifoldPressure: _d(24),
          acceleratorPedal: _d(25),
          throttleActual:   _d(26),
          batteryVoltage:   _d(29),
          tripFuelL:        _d(33),
        );
      }
    } catch (_) {
      return null;
    }
  }
}
''')
print("✅ obd_data.dart — гибкий парсер CSV (V5 + V6)")

# ================================================================
# ФИКС 2: analyzer_service.dart — автоопределение формата CSV
# ================================================================
with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:math';
import 'package:csv/csv.dart';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

enum TuningPatternType { maxPower, economy, stability }

class TuningPattern {
  final TuningPatternType type;
  final String name, description;
  final double knockTolerance, afrLean, afrRich;
  final double timingAggression, fuelTrimThreshold, vtcAggression;

  const TuningPattern({
    required this.type,
    required this.name,
    required this.description,
    this.knockTolerance    = 1.0,
    this.afrLean           = 15.0,
    this.afrRich           = 12.5,
    this.timingAggression  = 0.5,
    this.fuelTrimThreshold = 3.0,
    this.vtcAggression     = 0.5,
  });

  static const maxPower = TuningPattern(
    type: TuningPatternType.maxPower,
    name: 'Максимальная мощность',
    description: 'Агрессивный УОЗ, богатая WOT смесь',
    knockTolerance: 0.3, afrLean: 14.0, afrRich: 11.5,
    timingAggression: 0.9, fuelTrimThreshold: 2.0, vtcAggression: 0.9);

  static const economy = TuningPattern(
    type: TuningPatternType.economy,
    name: 'Минимальный расход',
    description: 'Бедная смесь на круизе',
    knockTolerance: 0.5, afrLean: 15.5, afrRich: 13.5,
    timingAggression: 0.3, fuelTrimThreshold: 5.0, vtcAggression: 0.3);

  static const stability = TuningPattern(
    type: TuningPatternType.stability,
    name: 'Стабильная работа',
    description: 'Консервативные настройки',
    knockTolerance: 2.0, afrLean: 14.7, afrRich: 13.0,
    timingAggression: 0.1, fuelTrimThreshold: 3.0, vtcAggression: 0.2);

  static const List<TuningPattern> all = [maxPower, economy, stability];
}

class AnalyzerService {
  static const _minSamples = 2;
  static const _minConf    = 0.4;

  Future<AnalysisResult> analyzeSparkMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) return _empty('Spark Advance', log.length,
      'Мало данных: ${valid.length}');

    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;
      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];
      final avgTiming = _avg(samples.map((d) => d.actualIgnition));
      final avgKnock  = _avg(samples.map((d) => d.knockRetard));
      final avgAFR    = _avg(samples.map((d) => d.afr));
      double sug = cur, conf = 0;
      String why = '';
      if (avgKnock > pattern.knockTolerance * 2) {
        sug = cur - min(avgKnock, 3.0);
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}°';
        conf = 0.9;
      } else if (avgKnock > pattern.knockTolerance) {
        sug = cur - (1.0 * (1.0 - pattern.timingAggression));
        why = 'Knock>${pattern.knockTolerance.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgTiming != 0 && (avgTiming - cur).abs() > 2) {
        sug = avgTiming + (pattern.timingAggression * 2 - 1);
        why = 'ЭБУ: ${avgTiming.toStringAsFixed(1)}°';
        conf = 0.6;
      } else if (avgKnock < pattern.knockTolerance * 0.2 &&
                 avgAFR > pattern.afrRich && avgAFR < pattern.afrLean) {
        sug = cur + pattern.timingAggression * 2;
        why = '+${(pattern.timingAggression * 2).toStringAsFixed(1)}° (стабильно)';
        conf = 0.5;
      }
      sug = sug.clamp(-5.0, 45.0);
      if ((sug - cur).abs() >= 0.5 && conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why));
      }
    }
    return AnalysisResult(mapName: 'Spark Advance', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('Spark', changes, valid.length, grouped.length));
  }

  Future<AnalysisResult> analyzeFuelMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 &&
      d.engineLoad > 0 && d.longFuelTrim.abs() < 30).toList();
    if (valid.length < 10) return _empty('Fuel Map / VE', log.length,
      'Мало данных: ${valid.length}');

    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;
      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];
      final avgSTFT = _avg(samples.map((d) => d.shortFuelTrim));
      final avgLTFT = _avg(samples.map((d) => d.longFuelTrim));
      final total = avgSTFT + avgLTFT;
      final avgAFR = _avg(samples.map((d) => d.afr));
      final avgLoad = _avg(samples.map((d) => d.engineLoad));
      double sug = cur, conf = 0;
      String why = '';
      if (total > pattern.fuelTrimThreshold) {
        sug = cur * (1 + total / 100.0);
        why = 'Trim +${total.toStringAsFixed(1)}% (бедно)';
        conf = min(0.9, total.abs() / 10);
      } else if (total < -pattern.fuelTrimThreshold) {
        sug = cur * (1 + total / 100.0);
        why = 'Trim ${total.toStringAsFixed(1)}% (богато)';
        conf = min(0.9, total.abs() / 10);
      } else if (avgAFR > pattern.afrLean && avgLoad > 50) {
        sug = cur * 1.04;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} бедно';
        conf = 0.6;
      } else if (avgAFR < pattern.afrRich && avgLoad > 50) {
        sug = cur * 0.96;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} богато';
        conf = 0.6;
      }
      final maxChange = cur * 0.15;
      final delta = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + delta;
      if ((sug - cur).abs() / (cur.abs() + 0.001) >= 0.01 && conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why));
      }
    }
    return AnalysisResult(mapName: 'Fuel Map / VE', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('Fuel', changes, valid.length, grouped.length));
  }

  Future<AnalysisResult> analyzeVTCMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    final valid = log.where((d) =>
      d.rpm > 800 && d.rpm < 7000 && d.engineLoad > 5).toList();
    if (valid.length < 10) return _empty('VTC Map', log.length, 'Мало данных');
    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;
      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];
      final avgAct = _avg(samples.map((d) => d.vtcActualAngle));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));
      double sug = cur, conf = 0;
      String why = '';
      if ((avgAct - cur).abs() > 3) {
        sug = avgAct + (pattern.vtcAggression * 2 - 1);
        why = 'ЭБУ: ${avgAct.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgKnock > pattern.knockTolerance && cur > 15) {
        sug = max(0, cur - 5);
        why = 'Детонация — снизить VTC';
        conf = 0.75;
      }
      sug = sug.clamp(0.0, 45.0);
      if ((sug - cur).abs() >= 2 && conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why));
      }
    }
    return AnalysisResult(mapName: 'VTC Map', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('VTC', changes, valid.length, grouped.length));
  }

  Future<AnalysisResult> analyzeTorqueMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) return _empty('Engine Torque', log.length, 'Мало данных');
    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;
      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];
      final avgCalc = _avg(samples.map((d) => d.calculatedTorqueNm));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));
      double sug = cur, conf = 0;
      String why = '';
      if (avgCalc > 0 && (avgCalc - cur).abs() > cur.abs() * 0.15) {
        sug = avgCalc;
        why = 'Расч. момент ${avgCalc.toStringAsFixed(1)} Нм';
        conf = 0.5;
      }
      if (avgKnock > pattern.knockTolerance * 2 && cur > 50) {
        sug = cur * 0.9;
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}°';
        conf = 0.7;
      }
      if ((sug - cur).abs() >= 5 && conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why));
      }
    }
    return AnalysisResult(mapName: 'Engine Torque', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('Torque', changes, valid.length, grouped.length));
  }

  /// Автоматически определяет формат CSV (V5 или V6) по заголовку.
  Future<List<OBDData>> loadLogFromCSV(String path) async {
    final content = await File(path).readAsString();
    final rows    = const CsvToListConverter().convert(content);
    if (rows.length < 2) return [];

    // Определяем формат по первой ячейке заголовка после MAF
    // V5: 'MAF_gs' на позиции 6, дальше сразу 'ThrottlePos_pct' на 7
    // V6: 'MAF_V'  на позиции 6, 'MAF_gps' на 7, 'ThrottlePos_pct' на 8
    bool isV6 = true;
    if (rows[0].length >= 8) {
      final h6 = rows[0][6].toString().toUpperCase();
      final h7 = rows[0][7].toString().toUpperCase();
      // V5: MAF_gs, MAF_GS, MAF_G/S
      // V6: MAF_V, MAF_gps
      if (h6.contains('MAF_G') && !h7.contains('MAF')) {
        isV6 = false;
      }
    }

    final result = <OBDData>[];
    for (int i = 1; i < rows.length; i++) {
      final d = OBDData.fromCsvRow(rows[i], isV6Format: isV6);
      if (d != null) result.add(d);
    }
    return result;
  }

  /// Объединяет несколько логов с усреднением и удалением дублей.
  Future<List<OBDData>> mergeLogs(List<List<OBDData>> logs) async {
    final all = <OBDData>[];
    for (final log in logs) all.addAll(log);
    all.sort((a, b) => a.timestamp.compareTo(b.timestamp));
    final deduped = <OBDData>[];
    for (final d in all) {
      if (deduped.isEmpty) { deduped.add(d); continue; }
      final diff = d.timestamp.difference(deduped.last.timestamp)
                    .inMilliseconds.abs();
      if (diff > 200) deduped.add(d);
    }
    return deduped;
  }

  Map<String, List<OBDData>> _group(List<OBDData> data, TuningMap map) {
    final result = <String, List<OBDData>>{};
    for (final d in data) {
      final ri = _closest(map.rpmAxis,  d.rpm.toDouble());
      final li = _closest(map.loadAxis, d.engineLoad);
      result.putIfAbsent('$ri,$li', () => []).add(d);
    }
    return result;
  }

  int _closest(List<double> axis, double v) {
    int idx = 0;
    double best = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  double _avg(Iterable<num> vals) {
    if (vals.isEmpty) return 0;
    return vals.reduce((a, b) => a + b) / vals.length;
  }

  AnalysisResult _empty(String name, int total, String msg) =>
      AnalysisResult(mapName: name, analyzedAt: DateTime.now(),
        totalSamples: total, changes: const [], summary: msg);

  String _summary(String type, List<MapCell> ch, int total, int cells) {
    if (ch.isEmpty) return '$type оптимальна\nДанных: $total | Клеток: $cells';
    final inc = ch.where((c) => c.delta > 0).length;
    final dec = ch.where((c) => c.delta < 0).length;
    final avg = ch.map((c) => c.delta.abs()).reduce((a, b) => a + b) / ch.length;
    final maxD = ch.map((c) => c.delta.abs()).reduce((a, b) => a > b ? a : b);
    final trend = inc > dec * 2 ? '↑ увеличение'
                : dec > inc * 2 ? '↓ уменьшение' : '↕ смешанное';
    return 'Данных: $total | Клеток: $cells\n'
           'Правок: ${ch.length} (+$inc -$dec) $trend\n'
           'Среднее: ${avg.toStringAsFixed(2)} | Макс: ${maxD.toStringAsFixed(2)}';
  }
}
''')
print("✅ analyzer_service.dart — автоопределение V5/V6 формата CSV")

# ================================================================
# ФИКС 3: rom_map_reader.dart — исправляем VE
# Проблема: rows=16 cols=15 + transpose = размеры не сходятся с осями
# Решение: правильно определяем rows/cols И меняем оси при transpose
# ================================================================
with open('lib/services/rom_map_reader.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:typed_data';
import 'package:file_picker/file_picker.dart';
import 'package:path_provider/path_provider.dart';
import '../models/tuning_map.dart';
import 'formula_evaluator.dart';
import 'map_storage_service.dart';

class RomMapDef {
  final String name;
  final String addressHex;
  final int    address;
  final int    rows, cols;
  final bool   isU16;
  final String formula, units;
  final double minVal, maxVal;
  final List<double> rpmAxis, loadAxis;
  final bool   transpose;

  const RomMapDef({
    required this.name,
    required this.addressHex,
    required this.address,
    required this.rows,
    required this.cols,
    required this.isU16,
    required this.formula,
    required this.units,
    required this.rpmAxis,
    required this.loadAxis,
    this.minVal    = -100,
    this.maxVal    = 400,
    this.transpose = false,
  });

  int get bytesPerCell => isU16 ? 2 : 1;
  int get totalBytes   => rows * cols * bytesPerCell;
}

class RomMapReader {
  Uint8List? _data;
  String? _fileName;
  String? _filePath;

  bool    get isLoaded => _data != null;
  String? get fileName => _fileName;
  String? get filePath => _filePath;
  int     get length   => _data?.length ?? 0;

  /// ВАЖНО: rows/cols в определении означают КАК ЛЕЖИТ В ROM,
  /// а не как отображается в приложении!
  /// VE в ROM: 15 строк по 16 значений (15x16), нужно транспонировать в 16x15.
  static final List<RomMapDef> standardMaps = [
    RomMapDef(
      name:'Spark Advance WOT', addressHex:'0x06EBC', address:0x6EBC,
      rows:16, cols:16, isU16:true, formula:'(X-16384)/128', units:'deg',
      minVal:-5, maxVal:45,
      rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
      loadAxis: [6,13,19,25,31,38,44,50,56,63,69,75,81,88,94,100],
    ),
    RomMapDef(
      name:'Engine Torque', addressHex:'0x07C3C', address:0x7C3C,
      rows:16, cols:16, isU16:true, formula:'(X-32768)/10.24', units:'Nm',
      minVal:-100, maxVal:200,
      rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
      loadAxis: [6,13,19,25,31,38,44,50,56,63,69,75,81,88,94,100],
    ),
    // VE: в ROM хранится КАК 15 строк (Load 10..100) x 16 значений (RPM 400..6400)
    // При transpose получится 16x15 (RPM x Load) как нужно для отображения
    RomMapDef(
      name:'Fresh Air Rate / VE', addressHex:'0x0A754', address:0xA754,
      rows:15, cols:16, isU16:true, formula:'X/256', units:'%',
      minVal:15, maxVal:250, transpose:true,
      rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
      loadAxis: [10,20,30,40,50,60,70,75,80,85,90,92,94,96,100],
    ),
    RomMapDef(
      name:'VTC Intake', addressHex:'0x06BF1', address:0x6BF1,
      rows:8, cols:8, isU16:false, formula:'(X-128)/2', units:'deg',
      minVal:0, maxVal:40,
      rpmAxis:  [800,1600,2400,3200,4000,4800,5600,6400],
      loadAxis: [0,15,30,45,60,75,90,100],
    ),
    RomMapDef(
      name:'Powertrain Force', addressHex:'0x0A2BC', address:0xA2BC,
      rows:16, cols:16, isU16:true, formula:'X-32768', units:'N',
      minVal:-5000, maxVal:5000,
      rpmAxis:  [400,800,1200,1600,2000,2400,2800,3200,3600,4000,4400,4800,5200,5600,6000,6400],
      loadAxis: [6,13,19,25,31,38,44,50,56,63,69,75,81,88,94,100],
    ),
    RomMapDef(
      name:'Enrichment Lambda', addressHex:'0x06521', address:0x6521,
      rows:8, cols:8, isU16:false, formula:'X/128', units:'Lambda',
      minVal:0.5, maxVal:1.5,
      rpmAxis:  [800,1600,2400,3200,4000,4800,5600,6400],
      loadAxis: [0,15,30,45,60,75,90,100],
    ),
  ];

  Future<bool> pickAndLoad() async {
    try {
      final result = await FilePicker.platform.pickFiles(
        type: FileType.custom,
        allowedExtensions: ['bin', 'rom', 'hex', 'dat'],
      );
      if (result == null || result.files.single.path == null) return false;
      _filePath = result.files.single.path!;
      _fileName = result.files.single.name;
      _data     = await File(_filePath!).readAsBytes();
      return true;
    } catch (_) { return false; }
  }

  int _u16BE(int addr) {
    if (_data == null || addr + 2 > _data!.length) return 0;
    return (_data![addr] << 8) | _data![addr + 1];
  }

  int _u8(int addr) {
    if (_data == null || addr >= _data!.length) return 0;
    return _data![addr];
  }

  TuningMap? readMap(RomMapDef def) {
    if (_data == null) return null;
    if (def.address + def.totalBytes > _data!.length) return null;

    final ev = FormulaEvaluator(def.formula);
    if (!ev.isValid) return null;

    // Читаем как в ROM: rows строк × cols значений
    var data = <List<double>>[];
    for (int r = 0; r < def.rows; r++) {
      final row = <double>[];
      for (int c = 0; c < def.cols; c++) {
        final off = (r * def.cols + c) * def.bytesPerCell;
        final raw = def.isU16
          ? _u16BE(def.address + off)
          : _u8(def.address + off);
        row.add(ev.evaluate(raw));
      }
      data.add(row);
    }

    // Транспонирование: превращаем [rows][cols] в [cols][rows]
    int finalRows = def.rows;
    int finalCols = def.cols;
    if (def.transpose && data.isNotEmpty) {
      final transposed = <List<double>>[];
      for (int c = 0; c < data[0].length; c++) {
        transposed.add([for (int r = 0; r < data.length; r++) data[r][c]]);
      }
      data      = transposed;
      finalRows = def.cols;
      finalCols = def.rows;
    }

    // Проверка соответствия размеров с осями
    if (finalRows != def.rpmAxis.length || finalCols != def.loadAxis.length) {
      // Ошибка в определении — не возвращаем карту
      return null;
    }

    return TuningMap(
      name:     def.name,
      address:  def.addressHex,
      rows:     finalRows,
      cols:     finalCols,
      rpmAxis:  def.rpmAxis,
      loadAxis: def.loadAxis,
      data:     data,
      units:    def.units,
      minValue: def.minVal,
      maxValue: def.maxVal,
    );
  }

  List<TuningMap> readAllMaps() {
    if (_data == null) return [];
    return standardMaps
        .map(readMap)
        .where((m) => m != null)
        .cast<TuningMap>()
        .toList();
  }

  List<int> _encodeValue(double value, RomMapDef def) {
    double raw;
    final f = def.formula.replaceAll(' ', '');
    if      (f == '(X-16384)/128')    raw = value * 128 + 16384;
    else if (f == '(X-32768)/10.24')  raw = value * 10.24 + 32768;
    else if (f == 'X/256')            raw = value * 256;
    else if (f == '(X-128)/2')        raw = value * 2 + 128;
    else if (f == 'X-32768')          raw = value + 32768;
    else if (f == 'X/128')            raw = value * 128;
    else                              raw = value;
    final rawInt = raw.round().clamp(0, def.isU16 ? 65535 : 255);
    if (def.isU16) return [(rawInt >> 8) & 0xFF, rawInt & 0xFF];
    return [rawInt & 0xFF];
  }

  bool writeMapToBuffer(TuningMap map, RomMapDef def) {
    if (_data == null) return false;
    var data = map.data;
    // Обратное транспонирование для VE
    if (def.transpose && data.isNotEmpty) {
      final orig = <List<double>>[];
      for (int c = 0; c < data[0].length; c++) {
        orig.add([for (int r = 0; r < data.length; r++) data[r][c]]);
      }
      data = orig;
    }
    for (int r = 0; r < def.rows; r++) {
      for (int c = 0; c < def.cols; c++) {
        if (r >= data.length || c >= data[r].length) continue;
        final off = def.address + (r * def.cols + c) * def.bytesPerCell;
        final bytes = _encodeValue(data[r][c], def);
        for (int i = 0; i < bytes.length; i++) {
          if (off + i < _data!.length) _data![off + i] = bytes[i];
        }
      }
    }
    return true;
  }

  Future<String?> saveModifiedRom({int modIndex = 1}) async {
    if (_data == null || _filePath == null) return null;
    try {
      final dir      = await getApplicationDocumentsDirectory();
      final origName = _fileName ?? 'rom.bin';
      final modName  = 'NLP_MOD${modIndex}_$origName';
      final outPath  = '${dir.path}/$modName';
      await File(outPath).writeAsBytes(_data!);
      return outPath;
    } catch (_) { return null; }
  }

  Future<int> saveAllAsDefaults() async {
    final maps = readAllMaps();
    for (final m in maps) await MapStorageService.saveMap(m);
    return maps.length;
  }

  static List<RomDiff> compareRoms(RomMapReader a, RomMapReader b) {
    final diffs = <RomDiff>[];
    for (final def in standardMaps) {
      final m1 = a.readMap(def);
      final m2 = b.readMap(def);
      if (m1 == null || m2 == null) continue;
      final cd = <RomCellDiff>[];
      for (int r = 0; r < m1.rows && r < m2.rows; r++) {
        for (int c = 0; c < m1.cols && c < m2.cols; c++) {
          final v1 = m1.data[r][c];
          final v2 = m2.data[r][c];
          if ((v1 - v2).abs() > 0.001) {
            cd.add(RomCellDiff(rpmIdx: r, loadIdx: c,
              rpm: m1.rpmAxis[r], load: m1.loadAxis[c],
              valueA: v1, valueB: v2));
          }
        }
      }
      diffs.add(RomDiff(mapName: def.name, mapDef: def,
        mapA: m1, mapB: m2, cellDiffs: cd));
    }
    return diffs;
  }
}

class RomCellDiff {
  final int rpmIdx, loadIdx;
  final double rpm, load, valueA, valueB;
  const RomCellDiff({required this.rpmIdx, required this.loadIdx,
    required this.rpm, required this.load,
    required this.valueA, required this.valueB});
  double get delta => valueB - valueA;
  double get deltaPercent => valueA != 0 ? (delta / valueA.abs()) * 100 : 0;
}

class RomDiff {
  final String mapName;
  final RomMapDef mapDef;
  final TuningMap mapA, mapB;
  final List<RomCellDiff> cellDiffs;
  const RomDiff({required this.mapName, required this.mapDef,
    required this.mapA, required this.mapB, required this.cellDiffs});
  bool get hasDiffs => cellDiffs.isNotEmpty;
}
''')
print("✅ rom_map_reader.dart — VE карта: rows=15 cols=16 в ROM → 16x15 после транспонирования")



✅ obd_data.dart — гибкий парсер CSV (V5 + V6)
✅ analyzer_service.dart — автоопределение V5/V6 формата CSV
✅ rom_map_reader.dart — VE карта: rows=15 cols=16 в ROM → 16x15 после транспонирования


In [ ]:
# @title 🔧 ФИКС: CSV парсер устойчивый + подсветка diff карт
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# ФИКС 1: analyzer_service.dart — используем split вместо CSV парсера
# Проблема: CsvToListConverter ломается на V5 файлах с \n перед данными
# ================================================================
with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:math';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

enum TuningPatternType { maxPower, economy, stability }

class TuningPattern {
  final TuningPatternType type;
  final String name, description;
  final double knockTolerance, afrLean, afrRich;
  final double timingAggression, fuelTrimThreshold, vtcAggression;

  const TuningPattern({
    required this.type,
    required this.name,
    required this.description,
    this.knockTolerance    = 1.0,
    this.afrLean           = 15.0,
    this.afrRich           = 12.5,
    this.timingAggression  = 0.5,
    this.fuelTrimThreshold = 3.0,
    this.vtcAggression     = 0.5,
  });

  static const maxPower = TuningPattern(
    type: TuningPatternType.maxPower,
    name: 'Максимальная мощность',
    description: 'Агрессивный УОЗ, богатая WOT смесь',
    knockTolerance: 0.3, afrLean: 14.0, afrRich: 11.5,
    timingAggression: 0.9, fuelTrimThreshold: 2.0, vtcAggression: 0.9);

  static const economy = TuningPattern(
    type: TuningPatternType.economy,
    name: 'Минимальный расход',
    description: 'Бедная смесь на круизе',
    knockTolerance: 0.5, afrLean: 15.5, afrRich: 13.5,
    timingAggression: 0.3, fuelTrimThreshold: 5.0, vtcAggression: 0.3);

  static const stability = TuningPattern(
    type: TuningPatternType.stability,
    name: 'Стабильная работа',
    description: 'Консервативные настройки',
    knockTolerance: 2.0, afrLean: 14.7, afrRich: 13.0,
    timingAggression: 0.1, fuelTrimThreshold: 3.0, vtcAggression: 0.2);

  static const List<TuningPattern> all = [maxPower, economy, stability];
}

class AnalyzerService {
  static const _minSamples = 2;
  static const _minConf    = 0.4;

  Future<AnalysisResult> analyzeSparkMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) return _empty('Spark Advance', log.length,
      'Мало данных: ${valid.length}');

    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;
      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];
      final avgTiming = _avg(samples.map((d) => d.actualIgnition));
      final avgKnock  = _avg(samples.map((d) => d.knockRetard));
      final avgAFR    = _avg(samples.map((d) => d.afr));
      double sug = cur, conf = 0;
      String why = '';
      if (avgKnock > pattern.knockTolerance * 2) {
        sug = cur - min(avgKnock, 3.0);
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}°';
        conf = 0.9;
      } else if (avgKnock > pattern.knockTolerance) {
        sug = cur - (1.0 * (1.0 - pattern.timingAggression));
        why = 'Knock>${pattern.knockTolerance.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgTiming != 0 && (avgTiming - cur).abs() > 2) {
        sug = avgTiming + (pattern.timingAggression * 2 - 1);
        why = 'ЭБУ: ${avgTiming.toStringAsFixed(1)}°';
        conf = 0.6;
      } else if (avgKnock < pattern.knockTolerance * 0.2 &&
                 avgAFR > pattern.afrRich && avgAFR < pattern.afrLean) {
        sug = cur + pattern.timingAggression * 2;
        why = '+${(pattern.timingAggression * 2).toStringAsFixed(1)}° (стабильно)';
        conf = 0.5;
      }
      sug = sug.clamp(-5.0, 45.0);
      if ((sug - cur).abs() >= 0.5 && conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why));
      }
    }
    return AnalysisResult(mapName: 'Spark Advance', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('Spark', changes, valid.length, grouped.length));
  }

  Future<AnalysisResult> analyzeFuelMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 &&
      d.engineLoad > 0 && d.longFuelTrim.abs() < 30).toList();
    if (valid.length < 10) return _empty('Fuel Map / VE', log.length,
      'Мало данных: ${valid.length}');

    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;
      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];
      final avgSTFT = _avg(samples.map((d) => d.shortFuelTrim));
      final avgLTFT = _avg(samples.map((d) => d.longFuelTrim));
      final total = avgSTFT + avgLTFT;
      final avgAFR = _avg(samples.map((d) => d.afr));
      final avgLoad = _avg(samples.map((d) => d.engineLoad));
      double sug = cur, conf = 0;
      String why = '';
      if (total > pattern.fuelTrimThreshold) {
        sug = cur * (1 + total / 100.0);
        why = 'Trim +${total.toStringAsFixed(1)}% (бедно)';
        conf = min(0.9, total.abs() / 10);
      } else if (total < -pattern.fuelTrimThreshold) {
        sug = cur * (1 + total / 100.0);
        why = 'Trim ${total.toStringAsFixed(1)}% (богато)';
        conf = min(0.9, total.abs() / 10);
      } else if (avgAFR > pattern.afrLean && avgLoad > 50) {
        sug = cur * 1.04;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} бедно';
        conf = 0.6;
      } else if (avgAFR < pattern.afrRich && avgLoad > 50) {
        sug = cur * 0.96;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} богато';
        conf = 0.6;
      }
      final maxChange = cur * 0.15;
      final delta = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + delta;
      if ((sug - cur).abs() / (cur.abs() + 0.001) >= 0.01 && conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why));
      }
    }
    return AnalysisResult(mapName: 'Fuel Map / VE', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('Fuel', changes, valid.length, grouped.length));
  }

  Future<AnalysisResult> analyzeVTCMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    final valid = log.where((d) =>
      d.rpm > 800 && d.rpm < 7000 && d.engineLoad > 5).toList();
    if (valid.length < 10) return _empty('VTC Map', log.length, 'Мало данных');
    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;
      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];
      final avgAct = _avg(samples.map((d) => d.vtcActualAngle));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));
      double sug = cur, conf = 0;
      String why = '';
      if ((avgAct - cur).abs() > 3) {
        sug = avgAct + (pattern.vtcAggression * 2 - 1);
        why = 'ЭБУ: ${avgAct.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgKnock > pattern.knockTolerance && cur > 15) {
        sug = max(0, cur - 5);
        why = 'Детонация — снизить VTC';
        conf = 0.75;
      }
      sug = sug.clamp(0.0, 45.0);
      if ((sug - cur).abs() >= 2 && conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why));
      }
    }
    return AnalysisResult(mapName: 'VTC Map', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('VTC', changes, valid.length, grouped.length));
  }

  Future<AnalysisResult> analyzeTorqueMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) return _empty('Engine Torque', log.length, 'Мало данных');
    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;
      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];
      final avgCalc = _avg(samples.map((d) => d.calculatedTorqueNm));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));
      double sug = cur, conf = 0;
      String why = '';
      if (avgCalc > 0 && (avgCalc - cur).abs() > cur.abs() * 0.15) {
        sug = avgCalc;
        why = 'Расч. момент ${avgCalc.toStringAsFixed(1)} Нм';
        conf = 0.5;
      }
      if (avgKnock > pattern.knockTolerance * 2 && cur > 50) {
        sug = cur * 0.9;
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}°';
        conf = 0.7;
      }
      if ((sug - cur).abs() >= 5 && conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why));
      }
    }
    return AnalysisResult(mapName: 'Engine Torque', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('Torque', changes, valid.length, grouped.length));
  }

  /// УСТОЙЧИВЫЙ ПАРСЕР CSV — работает с V5 и V6 логами.
  /// V5 (34 колонки): ts,rpm,speed,load,ect,iat,MAF_gs,tps,timing,stft,ltft,...
  /// V6 (36 колонок): ts,rpm,speed,load,ect,iat,MAF_V,MAF_gps,tps,timing,stft,ltft,...
  Future<List<OBDData>> loadLogFromCSV(String path) async {
    final file    = File(path);
    final content = await file.readAsString();
    final lines   = content.split('\n');
    if (lines.length < 2) return [];

    // Определяем формат по заголовку
    final header    = lines[0].toUpperCase();
    final headCols  = header.split(',');
    bool isV6 = false;

    // V6 если колонка [6] = MAF_V и [7] = MAF_GPS
    if (headCols.length >= 8) {
      final h6 = headCols[6].trim();
      final h7 = headCols[7].trim();
      isV6 = h6.contains('MAF_V') && h7.contains('MAF_GPS');
    }

    final result = <OBDData>[];
    for (int i = 1; i < lines.length; i++) {
      final line = lines[i].trim();
      if (line.isEmpty) continue;
      final parts = line.split(',');
      if (parts.length < 10) continue;

      try {
        double _d(int idx, [double def = 0]) {
          if (idx < 0 || idx >= parts.length) return def;
          final s = parts[idx].trim();
          if (s.isEmpty) return def;
          return double.tryParse(s) ?? def;
        }
        int _i(int idx, [int def = 0]) {
          if (idx < 0 || idx >= parts.length) return def;
          final s = parts[idx].trim();
          if (s.isEmpty) return def;
          return int.tryParse(s) ?? (double.tryParse(s)?.toInt() ?? def);
        }

        final ts = _i(0);
        if (ts == 0) continue;

        if (isV6) {
          // V6 формат
          result.add(OBDData(
            timestamp:          DateTime.fromMillisecondsSinceEpoch(ts),
            rpm:                _i(1),
            speed:              _i(2),
            engineLoad:         _d(3),
            coolantTemp:        _i(4),
            intakeTemp:         _i(5),
            mafVoltage:         _d(6),
            mafGps:             _d(7),
            throttlePos:        _d(8),
            ignitionTiming:     _d(9),
            shortFuelTrim:      _d(10),
            longFuelTrim:       _d(11),
            o2Voltage:          _d(12),
            afr:                _d(13, 14.7),
            vtcTargetAngle:     _d(14),
            vtcActualAngle:     _d(15),
            knockRetard:        _d(16),
            knockCount:         _i(17),
            actualIgnition:     _d(18),
            injectorDuty:       _d(19),
            injectorPulseWidth: _d(20),
            requestedTorque:    _d(21),
            actualTorque:       _d(22),
            oilTemp:            _d(23),
            afrTarget:          _d(24, 14.7),
            lambda:             _d(25, 1.0),
            manifoldPressure:   _d(26),
            acceleratorPedal:   _d(27),
            throttleActual:     _d(28),
            batteryVoltage:     _d(31),
            tripFuelL:          _d(35),
          ));
        } else {
          // V5 формат: MAF_gs на позиции 6, дальше сдвиг на -1
          result.add(OBDData(
            timestamp:          DateTime.fromMillisecondsSinceEpoch(ts),
            rpm:                _i(1),
            speed:              _i(2),
            engineLoad:         _d(3),
            coolantTemp:        _i(4),
            intakeTemp:         _i(5),
            mafVoltage:         0,       // в V5 не было
            mafGps:             _d(6),   // в V5 это уже g/s
            throttlePos:        _d(7),
            ignitionTiming:     _d(8),
            shortFuelTrim:      _d(9),
            longFuelTrim:       _d(10),
            o2Voltage:          _d(11),
            afr:                _d(12, 14.7),
            vtcTargetAngle:     _d(13),
            vtcActualAngle:     _d(14),
            knockRetard:        _d(15),
            knockCount:         _i(16),
            actualIgnition:     _d(17),
            injectorDuty:       _d(18),
            injectorPulseWidth: _d(18) / 100 * 20,
            requestedTorque:    _d(19),
            actualTorque:       _d(20),
            oilTemp:            _d(21),
            afrTarget:          _d(22, 14.7),
            lambda:             _d(23, 1.0),
            manifoldPressure:   _d(24),
            acceleratorPedal:   _d(25),
            throttleActual:     _d(26),
            batteryVoltage:     _d(29),
            tripFuelL:          _d(33),
          ));
        }
      } catch (_) { continue; }
    }
    return result;
  }

  Future<List<OBDData>> mergeLogs(List<List<OBDData>> logs) async {
    final all = <OBDData>[];
    for (final log in logs) all.addAll(log);
    all.sort((a, b) => a.timestamp.compareTo(b.timestamp));
    final deduped = <OBDData>[];
    for (final d in all) {
      if (deduped.isEmpty) { deduped.add(d); continue; }
      final diff = d.timestamp.difference(deduped.last.timestamp)
                    .inMilliseconds.abs();
      if (diff > 200) deduped.add(d);
    }
    return deduped;
  }

  Map<String, List<OBDData>> _group(List<OBDData> data, TuningMap map) {
    final result = <String, List<OBDData>>{};
    for (final d in data) {
      final ri = _closest(map.rpmAxis,  d.rpm.toDouble());
      final li = _closest(map.loadAxis, d.engineLoad);
      result.putIfAbsent('$ri,$li', () => []).add(d);
    }
    return result;
  }

  int _closest(List<double> axis, double v) {
    int idx = 0;
    double best = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) { best = d; idx = i; }
    }
    return idx;
  }

  double _avg(Iterable<num> vals) {
    if (vals.isEmpty) return 0;
    return vals.reduce((a, b) => a + b) / vals.length;
  }

  AnalysisResult _empty(String name, int total, String msg) =>
      AnalysisResult(mapName: name, analyzedAt: DateTime.now(),
        totalSamples: total, changes: const [], summary: msg);

  String _summary(String type, List<MapCell> ch, int total, int cells) {
    if (ch.isEmpty) return '$type оптимальна\nДанных: $total | Клеток: $cells';
    final inc = ch.where((c) => c.delta > 0).length;
    final dec = ch.where((c) => c.delta < 0).length;
    final avg = ch.map((c) => c.delta.abs()).reduce((a, b) => a + b) / ch.length;
    final maxD = ch.map((c) => c.delta.abs()).reduce((a, b) => a > b ? a : b);
    final trend = inc > dec * 2 ? '↑ увеличение'
                : dec > inc * 2 ? '↓ уменьшение' : '↕ смешанное';
    return 'Данных: $total | Клеток: $cells\n'
           'Правок: ${ch.length} (+$inc -$dec) $trend\n'
           'Среднее: ${avg.toStringAsFixed(2)} | Макс: ${maxD.toStringAsFixed(2)}';
  }
}
''')
print("✅ analyzer_service.dart — split-парсер CSV (V5 + V6, без csv библиотеки)")

# ================================================================
# ФИКС 2: rom_compare_screen.dart — режим OVERLAY с подсветкой diff
# ================================================================
with open('lib/screens/rom_compare_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import '../services/rom_map_reader.dart';
import '../models/tuning_map.dart';

class RomCompareScreen extends StatefulWidget {
  const RomCompareScreen({super.key});
  @override
  State<RomCompareScreen> createState() => _RomCompareScreenState();
}

class _RomCompareScreenState extends State<RomCompareScreen> {
  final _romA = RomMapReader();
  final _romB = RomMapReader();
  List<RomDiff>? _diffs;

  Future<void> _loadA() async {
    final ok = await _romA.pickAndLoad();
    if (ok) setState(() => _diffs = null);
  }

  Future<void> _loadB() async {
    final ok = await _romB.pickAndLoad();
    if (ok) setState(() => _diffs = null);
  }

  void _compare() {
    if (!_romA.isLoaded || !_romB.isLoaded) return;
    setState(() => _diffs = RomMapReader.compareRoms(_romA, _romB));
  }

  void _viewDiff(RomDiff diff) {
    Navigator.push(context, MaterialPageRoute(builder: (_) =>
      _DiffMapView(diff: diff)));
  }

  @override
  Widget build(BuildContext context) {
    final totalDiffs = _diffs?.fold<int>(0, (s, d) => s + d.cellDiffs.length) ?? 0;
    final mapsChanged = _diffs?.where((d) => d.hasDiffs).length ?? 0;

    return Scaffold(
      appBar: AppBar(title: const Text('ROM Сравнение'),
        backgroundColor: const Color(0xFF16213E)),
      body: SingleChildScrollView(
        padding: const EdgeInsets.all(12),
        child: Column(children: [
          // Загрузка файлов
          Row(children: [
            Expanded(child: ElevatedButton.icon(
              onPressed: _loadA,
              icon: const Icon(Icons.folder_open, size: 16),
              label: Text(_romA.isLoaded ? _romA.fileName! : 'Оригинал (A)',
                style: const TextStyle(fontSize: 11), overflow: TextOverflow.ellipsis),
              style: ElevatedButton.styleFrom(
                backgroundColor: _romA.isLoaded ? Colors.green.shade800 : const Color(0xFF0F3460),
                foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(42)))),
            const SizedBox(width: 8),
            Expanded(child: ElevatedButton.icon(
              onPressed: _loadB,
              icon: const Icon(Icons.folder_open, size: 16),
              label: Text(_romB.isLoaded ? _romB.fileName! : 'Модифициров. (B)',
                style: const TextStyle(fontSize: 11), overflow: TextOverflow.ellipsis),
              style: ElevatedButton.styleFrom(
                backgroundColor: _romB.isLoaded ? Colors.blue.shade800 : const Color(0xFF0F3460),
                foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(42)))),
          ]),
          const SizedBox(height: 8),
          SizedBox(width: double.infinity,
            child: ElevatedButton.icon(
              onPressed: _romA.isLoaded && _romB.isLoaded ? _compare : null,
              icon: const Icon(Icons.compare_arrows),
              label: const Text('СРАВНИТЬ', style: TextStyle(fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(
                backgroundColor: Colors.cyan, foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(44)))),

          // Сводка сравнения
          if (_diffs != null) ...[
            const SizedBox(height: 8),
            Card(color: const Color(0xFF16213E),
              child: Padding(padding: const EdgeInsets.all(12),
                child: Column(children: [
                  Row(mainAxisAlignment: MainAxisAlignment.spaceAround, children: [
                    Column(children: [
                      Text('$mapsChanged / ${_diffs!.length}',
                        style: const TextStyle(fontSize: 24, fontWeight: FontWeight.bold,
                          color: Colors.orange)),
                      const Text('Изменённых карт', style: TextStyle(fontSize: 11, color: Colors.white70)),
                    ]),
                    Column(children: [
                      Text('$totalDiffs',
                        style: const TextStyle(fontSize: 24, fontWeight: FontWeight.bold,
                          color: Colors.red)),
                      const Text('Изменённых ячеек', style: TextStyle(fontSize: 11, color: Colors.white70)),
                    ]),
                  ]),
                ]))),
          ],

          const SizedBox(height: 8),

          // Список карт с diff
          if (_diffs != null) ..._diffs!.map((d) => Card(
            color: d.hasDiffs ? Colors.orange.withOpacity(0.15) : const Color(0xFF16213E),
            child: ListTile(
              leading: Icon(d.hasDiffs ? Icons.difference : Icons.check_circle,
                color: d.hasDiffs ? Colors.orange : Colors.green),
              title: Text(d.mapName, style: const TextStyle(fontWeight: FontWeight.bold)),
              subtitle: Text(
                d.hasDiffs
                  ? '${d.cellDiffs.length} различий из ${d.mapA.rows * d.mapA.cols}'
                  : 'Идентичны',
                style: TextStyle(color: d.hasDiffs ? Colors.orange : Colors.green, fontSize: 12)),
              trailing: d.hasDiffs
                ? ElevatedButton(
                    onPressed: () => _viewDiff(d),
                    style: ElevatedButton.styleFrom(
                      backgroundColor: Colors.cyan, foregroundColor: Colors.white),
                    child: const Text('DIFF', style: TextStyle(fontSize: 11)))
                : null,
            ),
          )),
        ]),
      ),
    );
  }
}

/// Экран просмотра различий — режимы: A / B / DIFF (наложение)
class _DiffMapView extends StatefulWidget {
  final RomDiff diff;
  const _DiffMapView({required this.diff});
  @override
  State<_DiffMapView> createState() => _DiffMapViewState();
}

enum _ViewMode { diff, mapA, mapB }

class _DiffMapViewState extends State<_DiffMapView> {
  _ViewMode _mode = _ViewMode.diff;
  double _cellSize = 60;
  int? _selRow, _selCol;

  Map<String, RomCellDiff> get _diffMap {
    final m = <String, RomCellDiff>{};
    for (final c in widget.diff.cellDiffs) {
      m['${c.rpmIdx}_${c.loadIdx}'] = c;
    }
    return m;
  }

  @override
  Widget build(BuildContext context) {
    final map = _mode == _ViewMode.mapB ? widget.diff.mapB : widget.diff.mapA;
    final diffs = _diffMap;
    final selDiff = (_selRow != null && _selCol != null)
      ? diffs['${_selRow}_$_selCol']
      : null;

    return Scaffold(
      backgroundColor: const Color(0xFF1A1A2E),
      appBar: AppBar(
        title: Text(widget.diff.mapName),
        backgroundColor: const Color(0xFF16213E),
      ),
      body: Column(children: [
        // Переключатель режимов
        Container(
          color: const Color(0xFF16213E),
          padding: const EdgeInsets.all(6),
          child: Row(children: [
            Expanded(child: _modeBtn(_ViewMode.mapA, 'A: Оригинал', Colors.green)),
            const SizedBox(width: 4),
            Expanded(child: _modeBtn(_ViewMode.mapB, 'B: Модиф.',   Colors.blue)),
            const SizedBox(width: 4),
            Expanded(child: _modeBtn(_ViewMode.diff, 'DIFF',        Colors.orange)),
          ]),
        ),
        // Zoom
        Container(color: const Color(0xFF16213E), padding: const EdgeInsets.symmetric(horizontal: 8),
          child: Row(children: [
            Text('${map.rows}x${map.cols}',
              style: const TextStyle(color: Colors.white54, fontSize: 11)),
            const Spacer(),
            GestureDetector(
              onTap: () => setState(() => _cellSize = (_cellSize - 8).clamp(32, 90)),
              child: const Padding(padding: EdgeInsets.all(6),
                child: Icon(Icons.remove_circle_outline, color: Colors.white70))),
            Text('${_cellSize.toInt()}',
              style: const TextStyle(color: Colors.white54, fontSize: 11)),
            GestureDetector(
              onTap: () => setState(() => _cellSize = (_cellSize + 8).clamp(32, 90)),
              child: const Padding(padding: EdgeInsets.all(6),
                child: Icon(Icons.add_circle_outline, color: Colors.white70))),
            const SizedBox(width: 8),
            Container(
              padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 3),
              decoration: BoxDecoration(color: Colors.orange.withOpacity(0.3),
                borderRadius: BorderRadius.circular(4)),
              child: Text('${widget.diff.cellDiffs.length} правок',
                style: const TextStyle(color: Colors.orange, fontSize: 11,
                  fontWeight: FontWeight.bold))),
          ])),

        // Выбранная ячейка
        if (selDiff != null)
          Container(
            padding: const EdgeInsets.all(8),
            margin: const EdgeInsets.all(4),
            decoration: BoxDecoration(
              color: Colors.cyan.withOpacity(0.15),
              borderRadius: BorderRadius.circular(6),
              border: Border.all(color: Colors.cyan.withOpacity(0.5))),
            child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
              Row(children: [
                Text('RPM ${selDiff.rpm.toInt()} | Load ${selDiff.load.toInt()}${map.units.contains("%") || map.name.contains("VE") ? "%" : ""}',
                  style: const TextStyle(color: Colors.cyan, fontWeight: FontWeight.bold, fontSize: 13)),
                const Spacer(),
                GestureDetector(
                  onTap: () => setState(() { _selRow = null; _selCol = null; }),
                  child: const Icon(Icons.close, size: 16, color: Colors.white54)),
              ]),
              const SizedBox(height: 4),
              Row(children: [
                Expanded(child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                  const Text('Оригинал (A):', style: TextStyle(color: Colors.green, fontSize: 10)),
                  Text('${selDiff.valueA.toStringAsFixed(2)} ${map.units}',
                    style: const TextStyle(color: Colors.green, fontSize: 15, fontWeight: FontWeight.bold)),
                ])),
                const Icon(Icons.arrow_forward, color: Colors.white54),
                Expanded(child: Column(crossAxisAlignment: CrossAxisAlignment.end, children: [
                  const Text('Модифицир. (B):', style: TextStyle(color: Colors.blue, fontSize: 10)),
                  Text('${selDiff.valueB.toStringAsFixed(2)} ${map.units}',
                    style: const TextStyle(color: Colors.blue, fontSize: 15, fontWeight: FontWeight.bold)),
                ])),
              ]),
              const SizedBox(height: 4),
              Center(child: Text(
                'Δ ${selDiff.delta >= 0 ? "+" : ""}${selDiff.delta.toStringAsFixed(2)} '
                '(${selDiff.deltaPercent >= 0 ? "+" : ""}${selDiff.deltaPercent.toStringAsFixed(1)}%)',
                style: TextStyle(
                  color: selDiff.delta > 0 ? Colors.greenAccent : Colors.orangeAccent,
                  fontSize: 14, fontWeight: FontWeight.bold))),
            ]),
          ),

        // Таблица
        Expanded(
          child: InteractiveViewer(
            constrained: false,
            minScale: 0.5, maxScale: 3.0,
            boundaryMargin: const EdgeInsets.all(20),
            child: _buildTable(map, diffs),
          ),
        ),

        // Легенда
        Container(
          color: const Color(0xFF16213E),
          padding: const EdgeInsets.all(6),
          child: Row(mainAxisAlignment: MainAxisAlignment.spaceAround, children: [
            _legend(Colors.grey.shade700, 'Одинаково'),
            _legend(Colors.yellow.shade700, '<3%'),
            _legend(Colors.orange.shade800, '<8%'),
            _legend(Colors.red.shade800, '>8%'),
          ]),
        ),
      ]),
    );
  }

  Widget _modeBtn(_ViewMode m, String label, Color color) {
    final sel = _mode == m;
    return ElevatedButton(
      onPressed: () => setState(() => _mode = m),
      style: ElevatedButton.styleFrom(
        backgroundColor: sel ? color : const Color(0xFF0F3460),
        foregroundColor: Colors.white,
        padding: const EdgeInsets.symmetric(vertical: 8)),
      child: Text(label, style: TextStyle(
        fontSize: 11,
        fontWeight: sel ? FontWeight.bold : FontWeight.normal)),
    );
  }

  Widget _legend(Color c, String label) => Row(mainAxisSize: MainAxisSize.min, children: [
    Container(width: 14, height: 14,
      decoration: BoxDecoration(color: c, border: Border.all(color: Colors.white24))),
    const SizedBox(width: 4),
    Text(label, style: const TextStyle(color: Colors.white70, fontSize: 10)),
  ]);

  Widget _buildTable(TuningMap map, Map<String, RomCellDiff> diffs) {
    const headerH = 28.0;
    return Padding(
      padding: const EdgeInsets.all(4),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        // Заголовок
        Row(children: [
          Container(width: _cellSize, height: headerH,
            decoration: BoxDecoration(color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: const Center(child: Text('RPM/%',
              style: TextStyle(fontSize: 9, color: Colors.white70)))),
          ...List.generate(map.cols, (j) => Container(
            width: _cellSize, height: headerH,
            decoration: BoxDecoration(color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: Center(child: Text(map.loadAxis[j].toStringAsFixed(0),
              style: const TextStyle(fontSize: 10, color: Colors.white70))))),
        ]),
        // Строки
        ...List.generate(map.rows, (i) => Row(children: [
          Container(width: _cellSize, height: _cellSize,
            decoration: BoxDecoration(color: const Color(0xFF0F3460),
              border: Border.all(color: Colors.white24, width: 0.5)),
            child: Center(child: Text(map.rpmAxis[i].toStringAsFixed(0),
              style: const TextStyle(fontSize: 10, color: Colors.white70)))),
          ...List.generate(map.cols, (j) => _cell(i, j, map, diffs)),
        ])),
      ]),
    );
  }

  Widget _cell(int i, int j, TuningMap map, Map<String, RomCellDiff> diffs) {
    final diff  = diffs['${i}_$j'];
    final isSel = _selRow == i && _selCol == j;
    final valA  = widget.diff.mapA.data[i][j];
    final valB  = widget.diff.mapB.data[i][j];

    Color bgColor;
    if (diff == null) {
      // Нет различий — серый градиент по значению
      bgColor = Colors.grey.shade800.withOpacity(0.6);
    } else {
      final pct = diff.deltaPercent.abs();
      if (pct < 3) {
        bgColor = diff.delta > 0 ? Colors.yellow.shade700 : Colors.lightBlue.shade700;
      } else if (pct < 8) {
        bgColor = diff.delta > 0 ? Colors.orange.shade800 : Colors.blue.shade800;
      } else {
        bgColor = diff.delta > 0 ? Colors.red.shade800 : Colors.blueAccent.shade700;
      }
    }

    return GestureDetector(
      onTap: () => setState(() { _selRow = i; _selCol = j; }),
      child: Container(
        width: _cellSize, height: _cellSize,
        decoration: BoxDecoration(
          color: bgColor,
          border: Border.all(
            color: isSel ? Colors.cyan : (diff != null ? Colors.white54 : Colors.white12),
            width: isSel ? 2 : (diff != null ? 1 : 0.5)),
        ),
        child: _cellContent(diff, valA, valB),
      ),
    );
  }

  Widget _cellContent(RomCellDiff? diff, double valA, double valB) {
    if (_mode == _ViewMode.mapA) {
      return Center(child: Text(valA.toStringAsFixed(1),
        style: const TextStyle(fontSize: 11, color: Colors.white)));
    }
    if (_mode == _ViewMode.mapB) {
      return Center(child: Text(valB.toStringAsFixed(1),
        style: const TextStyle(fontSize: 11, color: Colors.white)));
    }
    // DIFF режим — показываем A → B
    if (diff == null) {
      return Center(child: Text(valA.toStringAsFixed(1),
        style: const TextStyle(fontSize: 10, color: Colors.white54)));
    }
    return Column(mainAxisAlignment: MainAxisAlignment.center, children: [
      Text(valA.toStringAsFixed(1),
        style: TextStyle(fontSize: 9, color: Colors.white.withOpacity(0.7),
          decoration: TextDecoration.lineThrough)),
      Text(valB.toStringAsFixed(1),
        style: TextStyle(fontSize: 11, fontWeight: FontWeight.bold,
          color: diff.delta > 0 ? Colors.greenAccent : Colors.yellowAccent)),
    ]);
  }
}
''')
print("✅ rom_compare_screen.dart — режимы A/B/DIFF + подсветка + детали ячейки")



✅ analyzer_service.dart — split-парсер CSV (V5 + V6, без csv библиотеки)
✅ rom_compare_screen.dart — режимы A/B/DIFF + подсветка + детали ячейки


In [ ]:
# @title 🔧 Кнопка "В прошивку" прямо в анализаторе + улучшение UX
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# 1. Добавляем глобальный RomMapReader (singleton)
#    чтобы .bin был доступен из анализатора и из EcuReadScreen
# ================================================================
with open('lib/services/rom_holder.dart', 'w') as f:
    f.write(r'''import 'rom_map_reader.dart';

/// Глобальный держатель загруженной .bin прошивки.
/// Один экземпляр на всё приложение — доступен из любого экрана.
class RomHolder {
  static final RomHolder instance = RomHolder._();
  RomHolder._();

  final RomMapReader reader = RomMapReader();

  bool get isLoaded => reader.isLoaded;
  String? get fileName => reader.fileName;
}
''')
print("✅ rom_holder.dart — глобальный держатель ROM")

# ================================================================
# 2. Обновляем ecu_read_screen.dart — используем RomHolder
# ================================================================
ecu_path = 'lib/screens/ecu_read_screen.dart'
with open(ecu_path, 'r') as f:
    code = f.read()

# Заменяем локальный _rom на RomHolder
code = code.replace(
    "import '../services/rom_map_reader.dart';",
    "import '../services/rom_map_reader.dart';\nimport '../services/rom_holder.dart';"
)
code = code.replace(
    "  final RomMapReader _rom = RomMapReader();",
    "  RomMapReader get _rom => RomHolder.instance.reader;"
)

with open(ecu_path, 'w') as f:
    f.write(code)
print("✅ ecu_read_screen.dart — использует RomHolder")

# ================================================================
# 3. Обновляем analyzer_screen.dart — добавляем кнопку "В ПРОШИВКУ"
# ================================================================
with open('lib/screens/analyzer_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:file_picker/file_picker.dart';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';
import '../services/analyzer_service.dart';
import '../services/tuning_service.dart';
import '../services/export_service.dart';
import '../services/obd_service.dart';
import '../services/rom_holder.dart';
import '../services/rom_map_reader.dart';
import '../widgets/fps_indicator.dart';
import '../widgets/map_table_view.dart';

class AnalyzerScreen extends StatefulWidget {
  final OBDService obdService;
  const AnalyzerScreen({super.key, required this.obdService});
  @override
  State<AnalyzerScreen> createState() => _AnalyzerScreenState();
}

class _AnalyzerScreenState extends State<AnalyzerScreen>
    with SingleTickerProviderStateMixin {
  final _analyzer = AnalyzerService();
  final _tuning   = TuningService();
  final _export   = ExportService();
  late final TabController _tab;

  String _map = 'Spark Advance';
  TuningPattern _pattern = TuningPattern.stability;
  TuningMap? _orig, _upd;

  List<OBDData>? _log;
  String? _logName;
  AnalysisResult? _logResult;
  bool _busy = false;

  bool _recording = false;
  final List<OBDData> _buf = [];
  AnalysisResult? _onlineResult;
  StreamSubscription? _dataSub;
  Timer? _autoTimer;

  @override
  void initState() { super.initState(); _tab = TabController(length: 2, vsync: this); }
  @override
  void dispose() {
    _tab.dispose();
    _dataSub?.cancel();
    _autoTimer?.cancel();
    super.dispose();
  }

  Future<(AnalysisResult, TuningMap, TuningMap)?> _doAnalysis(List<OBDData> data) async {
    TuningMap m;
    AnalysisResult r;
    switch (_map) {
      case 'Fuel Map / VE':
        m = await _tuning.getFuelMap();
        r = await _analyzer.analyzeFuelMap(data, m, pattern: _pattern);
      case 'VTC':
        m = await _tuning.getVTCMap();
        r = await _analyzer.analyzeVTCMap(data, m, pattern: _pattern);
      case 'Torque':
        m = await _tuning.getEngineTorqueMap();
        r = await _analyzer.analyzeTorqueMap(data, m, pattern: _pattern);
      default:
        m = await _tuning.getSparkAdvanceMap();
        r = await _analyzer.analyzeSparkMap(data, m, pattern: _pattern);
    }
    final u = m.copy();
    for (final c in r.changes) u.data[c.rpmIndex][c.loadIndex] = c.suggestedValue;
    return (r, m, u);
  }

  void _openMap() {
    if (_orig == null) return;
    Navigator.push(context, MaterialPageRoute(builder: (_) => MapFullscreenView(
      originalMap: _orig!, updatedMap: _upd,
      changes: (_logResult?.changes ?? _onlineResult?.changes) ?? [])));
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(m), backgroundColor: c));
  }

  // ── ЗАПИСЬ ПРАВОК В ПРОШИВКУ ─────────────────────────────────
  Future<void> _writeToRom() async {
    // 1. Проверяем что .bin загружен
    if (!RomHolder.instance.isLoaded) {
      // Предлагаем загрузить
      final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: const Text('Прошивка не загружена'),
        content: const Text(
          'Для записи правок в .bin файл прошивки нужно её загрузить.\n\n'
          'Загрузить сейчас?',
          style: TextStyle(color: Colors.white70)),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c, false),
            child: const Text('Отмена')),
          TextButton(onPressed: () => Navigator.pop(c, true),
            style: TextButton.styleFrom(foregroundColor: Colors.green),
            child: const Text('ЗАГРУЗИТЬ')),
        ]));
      if (ok != true) return;
      final loaded = await RomHolder.instance.reader.pickAndLoad();
      if (!loaded) { _snack('Файл не выбран', Colors.orange); return; }
      setState(() {});
    }

    // 2. Определяем какой RomMapDef соответствует текущей карте
    if (_upd == null || _logResult == null) {
      _snack('Сначала выполните анализ', Colors.orange); return;
    }

    final rom = RomHolder.instance.reader;
    final def = RomMapReader.standardMaps.firstWhere(
      (d) => d.addressHex == _upd!.address,
      orElse: () => RomMapReader.standardMaps.first);

    // 3. Показываем диалог с деталями
    final confirm = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Записать правки в ROM'),
      content: Column(mainAxisSize: MainAxisSize.min,
        crossAxisAlignment: CrossAxisAlignment.start, children: [
          _row('ROM файл:',   rom.fileName ?? '?'),
          _row('Карта:',      _upd!.name),
          _row('Адрес:',      _upd!.address),
          _row('Размер:',     '${_upd!.rows}×${_upd!.cols}'),
          _row('Правок:',     '${_logResult!.changes.length}'),
          const Divider(color: Colors.white24),
          const Text('Файл будет сохранён как:',
            style: TextStyle(color: Colors.white70, fontSize: 12)),
          Text('NLP_MOD1_${rom.fileName}',
            style: const TextStyle(color: Colors.orange, fontSize: 12,
              fontFamily: 'monospace')),
          const SizedBox(height: 8),
          const Text('Контрольная сумма пересчитается в PCMflash при загрузке.',
            style: TextStyle(color: Colors.cyan, fontSize: 10)),
        ]),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false),
          child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.red),
          child: const Text('ЗАПИСАТЬ')),
      ]));

    if (confirm != true) return;

    // 4. Записываем в буфер и сохраняем файл
    setState(() => _busy = true);
    final ok = rom.writeMapToBuffer(_upd!, def);
    if (!ok) {
      setState(() => _busy = false);
      _snack('Ошибка записи в буфер', Colors.red);
      return;
    }

    final path = await rom.saveModifiedRom(modIndex: 1);
    setState(() => _busy = false);

    if (path != null) {
      // Показываем результат
      if (!mounted) return;
      showDialog(context: context, builder: (c) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: const Row(children: [
          Icon(Icons.check_circle, color: Colors.green),
          SizedBox(width: 8),
          Text('УСПЕШНО!'),
        ]),
        content: Column(mainAxisSize: MainAxisSize.min,
          crossAxisAlignment: CrossAxisAlignment.start, children: [
            const Text('Правки записаны в файл:',
              style: TextStyle(color: Colors.white70)),
            const SizedBox(height: 8),
            Container(
              padding: const EdgeInsets.all(8),
              decoration: BoxDecoration(
                color: Colors.black26,
                borderRadius: BorderRadius.circular(4)),
              child: Text(path,
                style: const TextStyle(color: Colors.green, fontSize: 11,
                  fontFamily: 'monospace')),
            ),
            const SizedBox(height: 8),
            const Text('Файл готов для прошивки через PCMflash.',
              style: TextStyle(color: Colors.cyan, fontSize: 11)),
          ]),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c),
            child: const Text('OK')),
        ]));
    } else {
      _snack('Ошибка сохранения файла', Colors.red);
    }
  }

  Widget _row(String l, String v) => Padding(
    padding: const EdgeInsets.symmetric(vertical: 2),
    child: Row(children: [
      SizedBox(width: 90, child: Text(l,
        style: const TextStyle(color: Colors.white54, fontSize: 12))),
      Expanded(child: Text(v,
        style: const TextStyle(color: Colors.white, fontSize: 12,
          fontWeight: FontWeight.bold))),
    ]),
  );

  // ── Онлайн запись ────────────────────────────────────────────
  void _startRec() {
    if (!widget.obdService.isConnected || !widget.obdService.ecuResponds) {
      _snack('Нет подключения', Colors.red); return;
    }
    setState(() { _recording = true; _buf.clear(); _onlineResult = null; });
    _dataSub = widget.obdService.dataStream.listen((d) {
      if (_recording) { _buf.add(d); if (_buf.length > 5000) _buf.removeRange(0, 1000); }
    });
    _autoTimer = Timer.periodic(const Duration(seconds: 5), (_) async {
      if (_buf.length >= 20 && mounted) {
        final r = await _doAnalysis(_buf);
        if (r != null && mounted) setState(() {
          _onlineResult = r.$1; _orig = r.$2; _upd = r.$3;
        });
      }
      if (mounted) setState(() {});
    });
  }

  void _stopRec() {
    setState(() => _recording = false);
    _dataSub?.cancel(); _autoTimer?.cancel();
  }

  // ── Загрузка лога ────────────────────────────────────────────
  Future<void> _loadLog() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv']);
      if (r == null) return;
      setState(() => _busy = true);
      _log = await _analyzer.loadLogFromCSV(r.files.single.path!);
      _logName = r.files.single.name;
      setState(() => _busy = false);
      _snack('Загружено: ${_log!.length}', Colors.green);
    } catch (_) { setState(() => _busy = false); }
  }

  Future<void> _analyzeLog() async {
    if (_log == null || _log!.isEmpty) return;
    setState(() => _busy = true);
    final r = await _doAnalysis(_log!);
    if (r != null) setState(() { _logResult = r.$1; _orig = r.$2; _upd = r.$3; _busy = false; });
    else setState(() => _busy = false);
  }

  Future<void> _mergeAndAnalyze() async {
    try {
      final r = await FilePicker.platform.pickFiles(
        type: FileType.custom, allowedExtensions: ['csv'], allowMultiple: true);
      if (r == null || r.files.length < 2) {
        _snack('Выберите 2+ файла', Colors.orange); return;
      }
      setState(() => _busy = true);
      final logs = <List<OBDData>>[];
      for (final f in r.files) {
        if (f.path != null) logs.add(await _analyzer.loadLogFromCSV(f.path!));
      }
      _log = await _analyzer.mergeLogs(logs);
      _logName = '${r.files.length} логов → ${_log!.length} записей';
      setState(() => _busy = false);
      _snack('Объединено: ${_log!.length}', Colors.green);
    } catch (_) { setState(() => _busy = false); }
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Анализатор'),
        backgroundColor: const Color(0xFF16213E),
        actions: [FpsIndicator(obdService: widget.obdService)],
        bottom: TabBar(controller: _tab, tabs: const [
          Tab(icon: Icon(Icons.wifi), text: 'Онлайн'),
          Tab(icon: Icon(Icons.folder_open), text: 'Из лога'),
        ])),
      body: TabBarView(controller: _tab, children: [
        _onlineTab(), _logTab(),
      ]),
    );
  }

  Widget _mapSelector() {
    return Column(children: [
      Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
        child: Row(children: [
          const Text('Карта:', style: TextStyle(color: Colors.white70, fontSize: 12)),
          const SizedBox(width: 8),
          Expanded(child: DropdownButtonFormField<String>(
            value: _map, dropdownColor: const Color(0xFF16213E), isDense: true,
            items: const [
              DropdownMenuItem(value: 'Spark Advance', child: Text('Зажигание')),
              DropdownMenuItem(value: 'Fuel Map / VE', child: Text('Топливо/VE')),
              DropdownMenuItem(value: 'VTC', child: Text('VTC')),
              DropdownMenuItem(value: 'Torque', child: Text('Момент')),
            ],
            onChanged: (v) => setState(() => _map = v!))),
        ]))),
      const SizedBox(height: 4),
      Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          const Text('Паттерн:',
            style: TextStyle(color: Colors.orange, fontSize: 11, fontWeight: FontWeight.bold)),
          const SizedBox(height: 4),
          Wrap(spacing: 6, runSpacing: 6, children: TuningPattern.all.map((p) {
            final sel = _pattern.type == p.type;
            return GestureDetector(
              onTap: () => setState(() => _pattern = p),
              child: Container(
                padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 6),
                decoration: BoxDecoration(
                  color: sel ? Colors.orange.withOpacity(0.3) : const Color(0xFF0F3460),
                  borderRadius: BorderRadius.circular(6),
                  border: Border.all(color: sel ? Colors.orange : Colors.white24)),
                child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                  Text(p.name, style: TextStyle(
                    color: sel ? Colors.orange : Colors.white70,
                    fontSize: 11, fontWeight: sel ? FontWeight.bold : FontWeight.normal)),
                  Text(p.description, style: const TextStyle(color: Colors.white38, fontSize: 8)),
                ])));
          }).toList()),
        ]))),
    ]);
  }

  Widget _onlineTab() {
    final conn = widget.obdService.isConnected && widget.obdService.ecuResponds;
    return SingleChildScrollView(
      padding: const EdgeInsets.all(6),
      child: Column(children: [
        Card(color: _recording ? Colors.green.withOpacity(0.2) : const Color(0xFF16213E),
          child: Padding(padding: const EdgeInsets.all(8), child: Column(children: [
            Row(children: [
              Icon(conn ? Icons.check_circle : Icons.error,
                color: conn ? Colors.green : Colors.red, size: 18),
              const SizedBox(width: 6),
              Expanded(child: Text(conn ? 'ЭБУ готов' : 'Нет подключения',
                style: const TextStyle(fontSize: 12))),
              if (_recording)
                Container(
                  padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
                  decoration: BoxDecoration(color: Colors.red, borderRadius: BorderRadius.circular(4)),
                  child: const Text('REC', style: TextStyle(
                    color: Colors.white, fontWeight: FontWeight.bold, fontSize: 10))),
            ]),
            const SizedBox(height: 4),
            Row(children: [
              _stat('Записей', _buf.length.toString(), Colors.blue),
              const SizedBox(width: 4),
              _stat('Правок', (_onlineResult?.changes.length ?? 0).toString(), Colors.orange),
            ]),
          ]))),
        const SizedBox(height: 4),
        _mapSelector(),
        const SizedBox(height: 4),
        Row(children: [
          Expanded(child: ElevatedButton.icon(
            onPressed: _recording ? _stopRec : (conn ? _startRec : null),
            icon: Icon(_recording ? Icons.stop : Icons.play_arrow, size: 18),
            label: Text(_recording ? 'СТОП' : 'ЗАПИСЬ', style: const TextStyle(fontSize: 12)),
            style: ElevatedButton.styleFrom(
              backgroundColor: _recording ? Colors.red : Colors.green,
              foregroundColor: Colors.white,
              minimumSize: const Size.fromHeight(40)))),
          const SizedBox(width: 4),
          Expanded(child: ElevatedButton.icon(
            onPressed: _buf.length >= 20 ? () async {
              final r = await _doAnalysis(_buf);
              if (r != null && mounted) setState(() {
                _onlineResult = r.$1; _orig = r.$2; _upd = r.$3;
              });
            } : null,
            icon: const Icon(Icons.refresh, size: 18),
            label: const Text('АНАЛИЗ', style: TextStyle(fontSize: 12)),
            style: ElevatedButton.styleFrom(
              foregroundColor: Colors.white,
              minimumSize: const Size.fromHeight(40)))),
        ]),
        if (_onlineResult != null) ...[
          const SizedBox(height: 8),
          _resultCard(_onlineResult!),
        ],
        if (_onlineResult == null)
          const Padding(padding: EdgeInsets.all(20),
            child: Text('Нажми ЗАПИСЬ → покатайся → правки автоматически',
              style: TextStyle(color: Colors.white54, fontSize: 12),
              textAlign: TextAlign.center)),
      ]),
    );
  }

  Widget _logTab() {
    return SingleChildScrollView(
      padding: const EdgeInsets.all(6),
      child: Column(children: [
        Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(8),
          child: Column(children: [
            Row(children: [
              Expanded(child: ElevatedButton.icon(
                onPressed: _busy ? null : _loadLog,
                icon: const Icon(Icons.folder_open, size: 18),
                label: const Text('CSV'),
                style: ElevatedButton.styleFrom(
                  foregroundColor: Colors.white,
                  minimumSize: const Size.fromHeight(38)))),
              const SizedBox(width: 4),
              Expanded(child: ElevatedButton.icon(
                onPressed: _busy ? null : _mergeAndAnalyze,
                icon: const Icon(Icons.merge_type, size: 18),
                label: const Text('Объединить'),
                style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.purple, foregroundColor: Colors.white,
                  minimumSize: const Size.fromHeight(38)))),
            ]),
            if (_logName != null) ...[
              const SizedBox(height: 4),
              Text(_logName!, style: const TextStyle(color: Colors.white70, fontSize: 11)),
              Text('Записей: ${_log?.length ?? 0}',
                style: const TextStyle(color: Colors.green, fontWeight: FontWeight.bold)),
            ],
          ]))),
        const SizedBox(height: 4),
        _mapSelector(),
        const SizedBox(height: 4),
        ElevatedButton.icon(
          onPressed: _busy || _log == null ? null : _analyzeLog,
          icon: const Icon(Icons.analytics, size: 18),
          label: const Text('АНАЛИЗИРОВАТЬ'),
          style: ElevatedButton.styleFrom(
            backgroundColor: Colors.green, foregroundColor: Colors.white,
            minimumSize: const Size.fromHeight(40))),
        if (_logResult != null) ...[
          const SizedBox(height: 8),
          _resultCard(_logResult!),
        ],
      ]),
    );
  }

  Widget _stat(String l, String v, Color c) => Expanded(
    child: Container(
      padding: const EdgeInsets.all(3),
      decoration: BoxDecoration(color: const Color(0xFF0F3460),
        borderRadius: BorderRadius.circular(4)),
      child: Column(children: [
        Text(l, style: const TextStyle(color: Colors.white70, fontSize: 9)),
        Text(v, style: TextStyle(color: c, fontSize: 13, fontWeight: FontWeight.bold)),
      ])));

  Widget _resultCard(AnalysisResult r) {
    final romLoaded = RomHolder.instance.isLoaded;

    return Card(color: const Color(0xFF16213E), child: Padding(padding: const EdgeInsets.all(6),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Row(children: [
          const Icon(Icons.assessment, color: Colors.green, size: 18),
          const SizedBox(width: 4),
          Expanded(child: Text(r.mapName,
            style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold))),
          Text('${r.changes.length} правок',
            style: const TextStyle(color: Colors.orange, fontSize: 12)),
        ]),
        if (r.patternName.isNotEmpty)
          Text('Паттерн: ${r.patternName}',
            style: const TextStyle(color: Colors.white54, fontSize: 10)),
        Text(r.summary, style: const TextStyle(color: Colors.white70, fontSize: 10)),
        const SizedBox(height: 6),

        // Кнопка ОТКРЫТЬ КАРТУ
        if (_orig != null) SizedBox(width: double.infinity,
          child: ElevatedButton.icon(
            onPressed: _openMap,
            icon: const Icon(Icons.grid_on, size: 20),
            label: const Text('ОТКРЫТЬ КАРТУ',
              style: TextStyle(fontSize: 13, fontWeight: FontWeight.bold)),
            style: ElevatedButton.styleFrom(
              backgroundColor: Colors.cyan.shade700, foregroundColor: Colors.white,
              minimumSize: const Size.fromHeight(42)))),

        // ★★★ КНОПКА: ЗАПИСАТЬ В ПРОШИВКУ ★★★
        if (r.changes.isNotEmpty) ...[
          const SizedBox(height: 4),
          SizedBox(width: double.infinity,
            child: ElevatedButton.icon(
              onPressed: _busy ? null : _writeToRom,
              icon: Icon(romLoaded ? Icons.edit_note : Icons.folder_open, size: 20),
              label: Text(
                romLoaded
                  ? 'ЗАПИСАТЬ ПРАВКИ В ROM'
                  : 'ЗАГРУЗИТЬ .BIN И ЗАПИСАТЬ',
                style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(
                backgroundColor: Colors.red.shade800, foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(46)))),
          if (romLoaded)
            Padding(padding: const EdgeInsets.only(top: 2),
              child: Text('ROM: ${RomHolder.instance.fileName}',
                style: const TextStyle(color: Colors.green, fontSize: 10),
                textAlign: TextAlign.center)),
        ],

        const SizedBox(height: 6),
        SizedBox(height: 250,
          child: r.changes.isEmpty
            ? const Center(child: Text('Правок нет',
                style: TextStyle(color: Colors.green, fontSize: 14)))
            : ListView.builder(
                itemCount: r.changes.length,
                itemBuilder: (_, i) {
                  final c = r.changes[i];
                  final dc = c.delta > 0 ? Colors.green : Colors.orange;
                  return Card(color: const Color(0xFF0F3460),
                    margin: const EdgeInsets.symmetric(vertical: 2),
                    child: ListTile(dense: true,
                      title: Text('RPM ${c.rpm.toInt()} | ${c.load.toInt()}%',
                        style: const TextStyle(fontSize: 11, fontWeight: FontWeight.bold)),
                      subtitle: Text(
                        '${c.currentValue.toStringAsFixed(1)} → ${c.suggestedValue.toStringAsFixed(1)} (${c.reason})',
                        style: TextStyle(color: dc, fontSize: 10)),
                      trailing: Text('${(c.confidence * 100).toInt()}%',
                        style: TextStyle(
                          color: c.confidence > 0.8 ? Colors.green : Colors.orange,
                          fontSize: 11, fontWeight: FontWeight.bold))));
                })),
        const SizedBox(height: 4),
        Wrap(spacing: 4, runSpacing: 4, children: [
          _expBtn('WinOLS', Colors.blue, () async {
            if (_upd != null) { final p = await _export.exportToWinOLS(_upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
          _expBtn('JSON', Colors.green, () async {
            if (_upd != null && _orig != null) {
              final p = await _export.exportToJson(r, _orig!, _upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
          _expBtn('HEX', Colors.orange, () async {
            if (_upd != null) { final p = await _export.exportHexPatch(_upd!); _snack('→ ${p.split("/").last}', Colors.green); }}),
        ]),
      ])));
  }

  Widget _expBtn(String l, Color c, VoidCallback onTap) => ElevatedButton.icon(
    onPressed: onTap, icon: const Icon(Icons.download, size: 12),
    label: Text(l, style: const TextStyle(fontSize: 10)),
    style: ElevatedButton.styleFrom(
      backgroundColor: c, foregroundColor: Colors.white,
      padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4)));
}
''')
print("✅ analyzer_screen.dart — кнопка ЗАПИСАТЬ ПРАВКИ В ROM прямо в анализаторе")



✅ rom_holder.dart — глобальный держатель ROM
✅ ecu_read_screen.dart — использует RomHolder
✅ analyzer_screen.dart — кнопка ЗАПИСАТЬ ПРАВКИ В ROM прямо в анализаторе


In [ ]:
# @title 🔧 ФИКС: Сохранение в Downloads + правильный Torque анализ
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# ФИКС 1: AndroidManifest — разрешение на запись + MANAGE_EXTERNAL_STORAGE
# ================================================================
with open('android/app/src/main/AndroidManifest.xml', 'w') as f:
    f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" />
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" />
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN"
        android:usesPermissionFlags="neverForLocation" />
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT" />
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" />
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" />
    <uses-permission android:name="android.permission.WRITE_EXTERNAL_STORAGE"
        android:maxSdkVersion="32" />
    <uses-permission android:name="android.permission.READ_EXTERNAL_STORAGE"
        android:maxSdkVersion="32" />
    <uses-permission android:name="android.permission.MANAGE_EXTERNAL_STORAGE"
        tools:ignore="ScopedStorage" />
    <uses-permission android:name="android.permission.VIBRATE" />
    <uses-permission android:name="android.permission.WAKE_LOCK" />
    <uses-permission android:name="android.permission.INTERNET" />
    <application
        android:label="Nissan Logger V6"
        android:name="${applicationName}"
        android:icon="@mipmap/ic_launcher"
        android:usesCleartextTraffic="true"
        android:requestLegacyExternalStorage="true"
        android:allowBackup="true"
        xmlns:tools="http://schemas.android.com/tools"
        tools:replace="android:allowBackup">
        <activity
            android:name=".MainActivity"
            android:exported="true"
            android:launchMode="singleTop"
            android:theme="@style/LaunchTheme"
            android:configChanges="orientation|keyboardHidden|keyboard|screenSize|smallestScreenSize|locale|layoutDirection|fontScale|screenLayout|density|uiMode"
            android:hardwareAccelerated="true"
            android:windowSoftInputMode="adjustResize">
            <meta-data android:name="io.flutter.embedding.android.NormalTheme"
                android:resource="@style/NormalTheme" />
            <intent-filter>
                <action android:name="android.intent.action.MAIN"/>
                <category android:name="android.intent.category.LAUNCHER"/>
            </intent-filter>
        </activity>
        <meta-data android:name="flutterEmbedding" android:value="2" />
    </application>
</manifest>
''')
print("✅ AndroidManifest.xml — разрешения на внешнюю память")

# ================================================================
# ФИКС 2: rom_map_reader.dart — сохраняем в исходную папку файла
# ================================================================
rom_path = 'lib/services/rom_map_reader.dart'
with open(rom_path, 'r') as f:
    code = f.read()

# Заменяем метод saveModifiedRom
old_save = '''  Future<String?> saveModifiedRom({int modIndex = 1}) async {
    if (_data == null || _filePath == null) return null;
    try {
      final dir      = await getApplicationDocumentsDirectory();
      final origName = _fileName ?? 'rom.bin';
      final modName  = 'NLP_MOD${modIndex}_$origName';
      final outPath  = '${dir.path}/$modName';
      await File(outPath).writeAsBytes(_data!);
      return outPath;
    } catch (_) { return null; }
  }'''

new_save = '''  /// Сохраняет модифицированную прошивку.
  /// Сначала пытается в ту же папку что и оригинал.
  /// Если нет прав — падает в Downloads.
  /// Если и туда нет — в папку приложения.
  Future<String?> saveModifiedRom({int modIndex = 1}) async {
    if (_data == null || _filePath == null) return null;

    final origName = _fileName ?? 'rom.bin';
    final modName  = 'NLP_MOD${modIndex}_$origName';

    // 1. Пробуем сохранить в ту же папку где был оригинал
    try {
      final origDir  = File(_filePath!).parent.path;
      final outPath  = '$origDir/$modName';
      final outFile  = File(outPath);
      await outFile.writeAsBytes(_data!, flush: true);
      // Проверяем что реально записалось
      if (await outFile.exists() && await outFile.length() == _data!.length) {
        return outPath;
      }
    } catch (_) {}

    // 2. Пробуем в /storage/emulated/0/Download
    try {
      const downloadsPath = '/storage/emulated/0/Download';
      final dlDir  = Directory(downloadsPath);
      if (await dlDir.exists()) {
        final outPath = '$downloadsPath/$modName';
        final outFile = File(outPath);
        await outFile.writeAsBytes(_data!, flush: true);
        if (await outFile.exists() && await outFile.length() == _data!.length) {
          return outPath;
        }
      }
    } catch (_) {}

    // 3. Последний вариант — папка приложения
    try {
      final dir = await getApplicationDocumentsDirectory();
      final outPath = '${dir.path}/$modName';
      await File(outPath).writeAsBytes(_data!, flush: true);
      return outPath;
    } catch (_) { return null; }
  }'''

code = code.replace(old_save, new_save)

with open(rom_path, 'w') as f:
    f.write(code)
print("✅ rom_map_reader.dart — сохраняет в папку оригинала → Downloads → app")

# ================================================================
# ФИКС 3: analyzer_service.dart — правильный Torque анализ
# ================================================================
# Проблема: analyzeTorqueMap использует d.calculatedTorqueNm
# который считается через MAF. В V5 логах MAF был неправильный →
# calculatedTorqueNm выдаёт мусор → правки бредовые.
#
# Решение:
# 1. Использовать actualTorque из ЭБУ (если есть, != 0)
# 2. Иначе — сравнивать по загрузке двигателя (Load%)
# 3. Никогда не менять > 30% за раз
# 4. Пропускать пустые ячейки (значения близкие к 0)

an_path = 'lib/services/analyzer_service.dart'
with open(an_path, 'r') as f:
    an_code = f.read()

# Заменяем метод analyzeTorqueMap полностью
old_torque_start = "  Future<AnalysisResult> analyzeTorqueMap("
start = an_code.find(old_torque_start)
if start != -1:
    # Ищем конец метода
    brace_count = 0
    found_first = False
    end = start
    for i in range(start, len(an_code)):
        if an_code[i] == '{':
            brace_count += 1
            found_first = True
        elif an_code[i] == '}':
            brace_count -= 1
            if found_first and brace_count == 0:
                end = i + 1
                break

    new_torque = '''  Future<AnalysisResult> analyzeTorqueMap(
    List<OBDData> log, TuningMap map,
    {TuningPattern pattern = TuningPattern.stability}) async
  {
    // Фильтр: только валидные данные с нагрузкой
    final valid = log.where((d) =>
      d.rpm > 800 && d.rpm < 7500 &&
      d.engineLoad > 5 && d.throttlePos > 3).toList();

    if (valid.length < 20) {
      return _empty('Engine Torque', log.length,
        'Мало данных для анализа момента: ${valid.length}');
    }

    // Проверяем есть ли реальный момент от ЭБУ
    final hasRealTorque = valid.any((d) => d.actualTorque.abs() > 5);

    if (!hasRealTorque) {
      // ЭБУ не передаёт момент — не можем корректно анализировать
      return _empty('Engine Torque', log.length,
        'ЭБУ не передаёт данные момента.\\n'
        'Расчёт по MAF ненадёжен для карты момента.\\n'
        'Используйте карту зажигания или VE.');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      // Требуем минимум 3 замера в ячейке для надёжности
      if (samples.length < 3) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      // Пропускаем ячейки с околонулевым значением (не рабочие зоны)
      if (cur.abs() < 5) continue;

      // Берём только валидный момент
      final validSamples = samples.where((d) => d.actualTorque.abs() > 5).toList();
      if (validSamples.length < 3) continue;

      final avgActual = _avg(validSamples.map((d) => d.actualTorque));
      final avgKnock  = _avg(samples.map((d) => d.knockRetard));

      double sug = cur;
      double conf = 0;
      String why = '';

      // Разница от текущего значения карты
      final diff = avgActual - cur;
      final diffPct = cur != 0 ? (diff / cur.abs()) * 100 : 0;

      if (diffPct.abs() > 10) {
        // Значительная разница — корректируем в сторону ЭБУ
        sug = cur + diff * 0.5; // берём половину пути
        why = 'ЭБУ: ${avgActual.toStringAsFixed(0)} Нм (расх. ${diffPct.toStringAsFixed(0)}%)';
        conf = 0.6;
      } else if (avgKnock > pattern.knockTolerance * 2 && cur > 50) {
        sug = cur * 0.9;
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}° — снизить на 10%';
        conf = 0.7;
      }

      // ЖЁСТКОЕ ограничение: не более 20% изменения за раз
      final maxChange = cur.abs() * 0.20;
      final finalDiff = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + finalDiff;

      // Показываем только реально значимые правки (>3 Нм и >5%)
      if ((sug - cur).abs() >= 3 &&
          (sug - cur).abs() / cur.abs() > 0.05 &&
          conf >= _minConf) {
        changes.add(MapCell(rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: validSamples.length, reason: why));
      }
    }

    return AnalysisResult(mapName: 'Engine Torque', analyzedAt: DateTime.now(),
      totalSamples: log.length, changes: changes, patternName: pattern.name,
      summary: _summary('Torque', changes, valid.length, grouped.length));
  }'''

    an_code = an_code[:start] + new_torque + an_code[end:]
    with open(an_path, 'w') as f:
        f.write(an_code)
    print("✅ analyzer_service.dart — Torque анализ: требует actualTorque от ЭБУ")

# ================================================================
# ФИКС 4: Диалог результата — красивый путь, кнопка "Открыть папку"
# ================================================================
an_screen = 'lib/screens/analyzer_screen.dart'
with open(an_screen, 'r') as f:
    ac = f.read()

# Улучшаем диалог результата — сокращаем путь и добавляем поясняющий текст
old_dialog = '''      showDialog(context: context, builder: (c) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: const Row(children: [
          Icon(Icons.check_circle, color: Colors.green),
          SizedBox(width: 8),
          Text('УСПЕШНО!'),
        ]),
        content: Column(mainAxisSize: MainAxisSize.min,
          crossAxisAlignment: CrossAxisAlignment.start, children: [
            const Text('Правки записаны в файл:',
              style: TextStyle(color: Colors.white70)),
            const SizedBox(height: 8),
            Container(
              padding: const EdgeInsets.all(8),
              decoration: BoxDecoration(
                color: Colors.black26,
                borderRadius: BorderRadius.circular(4)),
              child: Text(path,
                style: const TextStyle(color: Colors.green, fontSize: 11,
                  fontFamily: 'monospace')),
            ),
            const SizedBox(height: 8),
            const Text('Файл готов для прошивки через PCMflash.',
              style: TextStyle(color: Colors.cyan, fontSize: 11)),
          ]),
        actions: [
          TextButton(onPressed: () => Navigator.pop(c),
            child: const Text('OK')),
        ]));'''

new_dialog = '''      // Определяем куда сохранилось
      String location = 'внутренняя папка';
      String shortPath = path;
      if (path.contains('/Download')) {
        location = 'Downloads';
        shortPath = 'Downloads/${path.split("/").last}';
      } else if (path.startsWith('/storage/emulated/0/')) {
        location = 'внешняя память';
        shortPath = path.replaceFirst('/storage/emulated/0/', '');
      } else if (path.contains('/data/user/')) {
        location = 'папка приложения (нужно вытащить через adb)';
        shortPath = path.split('/').last;
      }

      showDialog(context: context, builder: (c) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: const Row(children: [
          Icon(Icons.check_circle, color: Colors.green, size: 28),
          SizedBox(width: 8),
          Text('УСПЕШНО!'),
        ]),
        content: Column(mainAxisSize: MainAxisSize.min,
          crossAxisAlignment: CrossAxisAlignment.start, children: [
            Text('Файл сохранён в $location:',
              style: const TextStyle(color: Colors.white70, fontSize: 13)),
            const SizedBox(height: 8),
            Container(
              padding: const EdgeInsets.all(10),
              decoration: BoxDecoration(
                color: Colors.black26,
                borderRadius: BorderRadius.circular(6),
                border: Border.all(color: Colors.green.withOpacity(0.5))),
              child: Text(shortPath,
                style: const TextStyle(color: Colors.green, fontSize: 12,
                  fontFamily: 'monospace', fontWeight: FontWeight.bold)),
            ),
            const SizedBox(height: 12),
            Container(
              padding: const EdgeInsets.all(8),
              decoration: BoxDecoration(
                color: Colors.cyan.withOpacity(0.1),
                borderRadius: BorderRadius.circular(4)),
              child: const Row(children: [
                Icon(Icons.info_outline, color: Colors.cyan, size: 16),
                SizedBox(width: 6),
                Expanded(child: Text(
                  'Открой в PCMflash — контрольная сумма пересчитается автоматически.',
                  style: TextStyle(color: Colors.cyan, fontSize: 11))),
              ]),
            ),
            const SizedBox(height: 8),
            SelectableText(
              'Полный путь:\\n$path',
              style: const TextStyle(color: Colors.white38, fontSize: 9,
                fontFamily: 'monospace')),
          ]),
        actions: [
          TextButton(
            onPressed: () async {
              await Clipboard.setData(ClipboardData(text: path));
              if (mounted) ScaffoldMessenger.of(context).showSnackBar(
                const SnackBar(content: Text('Путь скопирован'),
                  backgroundColor: Colors.green));
            },
            child: const Text('КОПИРОВАТЬ ПУТЬ')),
          TextButton(onPressed: () => Navigator.pop(c),
            style: TextButton.styleFrom(foregroundColor: Colors.green),
            child: const Text('OK')),
        ]));'''

if old_dialog in ac:
    ac = ac.replace(old_dialog, new_dialog)

    # Добавляем import Clipboard если нет
    if "import 'package:flutter/services.dart';" not in ac:
        ac = ac.replace(
            "import 'package:flutter/material.dart';",
            "import 'package:flutter/material.dart';\nimport 'package:flutter/services.dart';"
        )

    with open(an_screen, 'w') as f:
        f.write(ac)
    print("✅ analyzer_screen.dart — красивый диалог + копирование пути")
else:
    print("⚠️ Диалог не найден для замены")



✅ AndroidManifest.xml — разрешения на внешнюю память
✅ rom_map_reader.dart — сохраняет в папку оригинала → Downloads → app
✅ analyzer_service.dart — Torque анализ: требует actualTorque от ЭБУ
✅ analyzer_screen.dart — красивый диалог + копирование пути


In [ ]:
# @title 🔧 ФИКС: analyzer_service.dart переписан полностью
import os
os.chdir('/content/nissan_logger_v6')

with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:math';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

// ── Паттерны настройки ──────────────────────────────────────
enum TuningPatternType { maxPower, economy, stability }

class TuningPattern {
  final TuningPatternType type;
  final String name, description;
  final double knockTolerance, afrLean, afrRich;
  final double timingAggression, fuelTrimThreshold, vtcAggression;

  const TuningPattern({
    required this.type,
    required this.name,
    required this.description,
    this.knockTolerance    = 1.0,
    this.afrLean           = 15.0,
    this.afrRich           = 12.5,
    this.timingAggression  = 0.5,
    this.fuelTrimThreshold = 3.0,
    this.vtcAggression     = 0.5,
  });

  static const maxPower = TuningPattern(
    type: TuningPatternType.maxPower,
    name: 'Максимальная мощность',
    description: 'Агрессивный УОЗ, богатая WOT смесь',
    knockTolerance: 0.3, afrLean: 14.0, afrRich: 11.5,
    timingAggression: 0.9, fuelTrimThreshold: 2.0, vtcAggression: 0.9);

  static const economy = TuningPattern(
    type: TuningPatternType.economy,
    name: 'Минимальный расход',
    description: 'Бедная смесь на круизе',
    knockTolerance: 0.5, afrLean: 15.5, afrRich: 13.5,
    timingAggression: 0.3, fuelTrimThreshold: 5.0, vtcAggression: 0.3);

  static const stability = TuningPattern(
    type: TuningPatternType.stability,
    name: 'Стабильная работа',
    description: 'Консервативные настройки',
    knockTolerance: 2.0, afrLean: 14.7, afrRich: 13.0,
    timingAggression: 0.1, fuelTrimThreshold: 3.0, vtcAggression: 0.2);

  static const List<TuningPattern> all = [maxPower, economy, stability];
}

class AnalyzerService {
  static const _minSamples = 2;
  static const _minConf    = 0.4;

  // ─────────────────────────────────────────────────────────────
  // SPARK ADVANCE
  // ─────────────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeSparkMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) {
      return _empty('Spark Advance', log.length,
        'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgTiming = _avg(samples.map((d) => d.actualIgnition));
      final avgKnock  = _avg(samples.map((d) => d.knockRetard));
      final avgAFR    = _avg(samples.map((d) => d.afr));

      double sug = cur;
      double conf = 0;
      String why = '';

      if (avgKnock > pattern.knockTolerance * 2) {
        sug = cur - min(avgKnock, 3.0);
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}°';
        conf = 0.9;
      } else if (avgKnock > pattern.knockTolerance) {
        sug = cur - (1.0 * (1.0 - pattern.timingAggression));
        why = 'Knock>${pattern.knockTolerance.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgTiming != 0 && (avgTiming - cur).abs() > 2) {
        sug = avgTiming + (pattern.timingAggression * 2 - 1);
        why = 'ЭБУ: ${avgTiming.toStringAsFixed(1)}°';
        conf = 0.6;
      } else if (avgKnock < pattern.knockTolerance * 0.2 &&
                 avgAFR > pattern.afrRich &&
                 avgAFR < pattern.afrLean) {
        sug = cur + pattern.timingAggression * 2;
        why = '+${(pattern.timingAggression * 2).toStringAsFixed(1)}° (стаб.)';
        conf = 0.5;
      }

      sug = sug.clamp(-5.0, 45.0);
      if ((sug - cur).abs() >= 0.5 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'Spark Advance',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('Spark', changes, valid.length, grouped.length),
    );
  }

  // ─────────────────────────────────────────────────────────────
  // FUEL MAP / VE
  // ─────────────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeFuelMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 &&
      d.engineLoad > 0 && d.longFuelTrim.abs() < 30).toList();
    if (valid.length < 10) {
      return _empty('Fuel Map / VE', log.length,
        'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgSTFT = _avg(samples.map((d) => d.shortFuelTrim));
      final avgLTFT = _avg(samples.map((d) => d.longFuelTrim));
      final total = avgSTFT + avgLTFT;
      final avgAFR = _avg(samples.map((d) => d.afr));
      final avgLoad = _avg(samples.map((d) => d.engineLoad));

      double sug = cur;
      double conf = 0;
      String why = '';

      if (total > pattern.fuelTrimThreshold) {
        sug = cur * (1 + total / 100.0);
        why = 'Trim +${total.toStringAsFixed(1)}% (бедно)';
        conf = min(0.9, total.abs() / 10);
      } else if (total < -pattern.fuelTrimThreshold) {
        sug = cur * (1 + total / 100.0);
        why = 'Trim ${total.toStringAsFixed(1)}% (богато)';
        conf = min(0.9, total.abs() / 10);
      } else if (avgAFR > pattern.afrLean && avgLoad > 50) {
        sug = cur * 1.04;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} бедно';
        conf = 0.6;
      } else if (avgAFR < pattern.afrRich && avgLoad > 50) {
        sug = cur * 0.96;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} богато';
        conf = 0.6;
      }

      final maxChange = cur * 0.15;
      final delta = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + delta;

      if ((sug - cur).abs() / (cur.abs() + 0.001) >= 0.01 &&
          conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'Fuel Map / VE',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('Fuel', changes, valid.length, grouped.length),
    );
  }

  // ─────────────────────────────────────────────────────────────
  // VTC
  // ─────────────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeVTCMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 800 && d.rpm < 7000 && d.engineLoad > 5).toList();
    if (valid.length < 10) {
      return _empty('VTC Map', log.length, 'Мало данных');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgAct = _avg(samples.map((d) => d.vtcActualAngle));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));

      double sug = cur;
      double conf = 0;
      String why = '';

      if ((avgAct - cur).abs() > 3) {
        sug = avgAct + (pattern.vtcAggression * 2 - 1);
        why = 'ЭБУ: ${avgAct.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgKnock > pattern.knockTolerance && cur > 15) {
        sug = max(0, cur - 5);
        why = 'Детонация — снизить VTC';
        conf = 0.75;
      }

      sug = sug.clamp(0.0, 45.0);
      if ((sug - cur).abs() >= 2 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'VTC Map',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('VTC', changes, valid.length, grouped.length),
    );
  }

  // ─────────────────────────────────────────────────────────────
  // TORQUE — только по реальным данным ЭБУ, никакого MAF расчёта
  // ─────────────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeTorqueMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 800 && d.rpm < 7500 &&
      d.engineLoad > 5 && d.throttlePos > 3).toList();

    if (valid.length < 20) {
      return _empty('Engine Torque', log.length,
        'Мало данных: ${valid.length}');
    }

    // Проверяем, передаёт ли ЭБУ реальный момент
    final hasRealTorque = valid.any((d) => d.actualTorque.abs() > 5);
    if (!hasRealTorque) {
      return _empty('Engine Torque', log.length,
        'ЭБУ не передаёт данные момента.\n'
        'Используйте карту зажигания или VE.');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < 3) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      // Пропускаем пустые/нерабочие ячейки
      if (cur.abs() < 5) continue;

      // Берём только замеры где есть валидный момент от ЭБУ
      final validSamples =
          samples.where((d) => d.actualTorque.abs() > 5).toList();
      if (validSamples.length < 3) continue;

      final avgActual = _avg(validSamples.map((d) => d.actualTorque));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));

      double sug = cur;
      double conf = 0;
      String why = '';

      final diff = avgActual - cur;
      final diffPct = cur != 0 ? (diff / cur.abs()) * 100 : 0;

      if (diffPct.abs() > 10) {
        sug = cur + diff * 0.5;
        why = 'ЭБУ: ${avgActual.toStringAsFixed(0)} Нм (расх. '
              '${diffPct.toStringAsFixed(0)}%)';
        conf = 0.6;
      } else if (avgKnock > pattern.knockTolerance * 2 && cur > 50) {
        sug = cur * 0.9;
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}° — снизить 10%';
        conf = 0.7;
      }

      // Не более 20% изменения за раз
      final maxChange = cur.abs() * 0.20;
      final finalDiff = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + finalDiff;

      // Только значимые правки: > 3 Нм и > 5%
      if ((sug - cur).abs() >= 3 &&
          (sug - cur).abs() / cur.abs() > 0.05 &&
          conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: validSamples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'Engine Torque',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('Torque', changes, valid.length, grouped.length),
    );
  }

  // ─────────────────────────────────────────────────────────────
  // Загрузка CSV — V5 и V6 автоопределение
  // ─────────────────────────────────────────────────────────────
  Future<List<OBDData>> loadLogFromCSV(String path) async {
    final file    = File(path);
    final content = await file.readAsString();
    final lines   = content.split('\n');
    if (lines.length < 2) return [];

    final header   = lines[0].toUpperCase();
    final headCols = header.split(',');
    bool isV6 = false;
    if (headCols.length >= 8) {
      final h6 = headCols[6].trim();
      final h7 = headCols[7].trim();
      isV6 = h6.contains('MAF_V') && h7.contains('MAF_GPS');
    }

    final result = <OBDData>[];
    for (int i = 1; i < lines.length; i++) {
      final line = lines[i].trim();
      if (line.isEmpty) continue;
      final parts = line.split(',');
      if (parts.length < 10) continue;

      try {
        double _d(int idx, [double def = 0]) {
          if (idx < 0 || idx >= parts.length) return def;
          final s = parts[idx].trim();
          if (s.isEmpty) return def;
          return double.tryParse(s) ?? def;
        }
        int _i(int idx, [int def = 0]) {
          if (idx < 0 || idx >= parts.length) return def;
          final s = parts[idx].trim();
          if (s.isEmpty) return def;
          return int.tryParse(s) ?? (double.tryParse(s)?.toInt() ?? def);
        }

        final ts = _i(0);
        if (ts == 0) continue;

        if (isV6) {
          result.add(OBDData(
            timestamp:          DateTime.fromMillisecondsSinceEpoch(ts),
            rpm:                _i(1),
            speed:              _i(2),
            engineLoad:         _d(3),
            coolantTemp:        _i(4),
            intakeTemp:         _i(5),
            mafVoltage:         _d(6),
            mafGps:             _d(7),
            throttlePos:        _d(8),
            ignitionTiming:     _d(9),
            shortFuelTrim:      _d(10),
            longFuelTrim:       _d(11),
            o2Voltage:          _d(12),
            afr:                _d(13, 14.7),
            vtcTargetAngle:     _d(14),
            vtcActualAngle:     _d(15),
            knockRetard:        _d(16),
            knockCount:         _i(17),
            actualIgnition:     _d(18),
            injectorDuty:       _d(19),
            injectorPulseWidth: _d(20),
            requestedTorque:    _d(21),
            actualTorque:       _d(22),
            oilTemp:            _d(23),
            afrTarget:          _d(24, 14.7),
            lambda:             _d(25, 1.0),
            manifoldPressure:   _d(26),
            acceleratorPedal:   _d(27),
            throttleActual:     _d(28),
            batteryVoltage:     _d(31),
            tripFuelL:          _d(35),
          ));
        } else {
          // V5 формат
          result.add(OBDData(
            timestamp:          DateTime.fromMillisecondsSinceEpoch(ts),
            rpm:                _i(1),
            speed:              _i(2),
            engineLoad:         _d(3),
            coolantTemp:        _i(4),
            intakeTemp:         _i(5),
            mafVoltage:         0,
            mafGps:             _d(6),
            throttlePos:        _d(7),
            ignitionTiming:     _d(8),
            shortFuelTrim:      _d(9),
            longFuelTrim:       _d(10),
            o2Voltage:          _d(11),
            afr:                _d(12, 14.7),
            vtcTargetAngle:     _d(13),
            vtcActualAngle:     _d(14),
            knockRetard:        _d(15),
            knockCount:         _i(16),
            actualIgnition:     _d(17),
            injectorDuty:       _d(18),
            injectorPulseWidth: _d(18) / 100 * 20,
            requestedTorque:    _d(19),
            actualTorque:       _d(20),
            oilTemp:            _d(21),
            afrTarget:          _d(22, 14.7),
            lambda:             _d(23, 1.0),
            manifoldPressure:   _d(24),
            acceleratorPedal:   _d(25),
            throttleActual:     _d(26),
            batteryVoltage:     _d(29),
            tripFuelL:          _d(33),
          ));
        }
      } catch (_) {
        continue;
      }
    }
    return result;
  }

  // ─────────────────────────────────────────────────────────────
  // Слияние логов
  // ─────────────────────────────────────────────────────────────
  Future<List<OBDData>> mergeLogs(List<List<OBDData>> logs) async {
    final all = <OBDData>[];
    for (final log in logs) all.addAll(log);
    all.sort((a, b) => a.timestamp.compareTo(b.timestamp));
    final deduped = <OBDData>[];
    for (final d in all) {
      if (deduped.isEmpty) {
        deduped.add(d);
        continue;
      }
      final diff = d.timestamp
          .difference(deduped.last.timestamp)
          .inMilliseconds
          .abs();
      if (diff > 200) deduped.add(d);
    }
    return deduped;
  }

  // ─────────────────────────────────────────────────────────────
  // Вспомогательные
  // ─────────────────────────────────────────────────────────────
  Map<String, List<OBDData>> _group(List<OBDData> data, TuningMap map) {
    final result = <String, List<OBDData>>{};
    for (final d in data) {
      final ri = _closest(map.rpmAxis,  d.rpm.toDouble());
      final li = _closest(map.loadAxis, d.engineLoad);
      result.putIfAbsent('$ri,$li', () => []).add(d);
    }
    return result;
  }

  int _closest(List<double> axis, double v) {
    int idx = 0;
    double best = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) {
        best = d;
        idx = i;
      }
    }
    return idx;
  }

  double _avg(Iterable<num> vals) {
    if (vals.isEmpty) return 0;
    return vals.reduce((a, b) => a + b) / vals.length;
  }

  AnalysisResult _empty(String name, int total, String msg) =>
      AnalysisResult(
        mapName: name,
        analyzedAt: DateTime.now(),
        totalSamples: total,
        changes: const [],
        summary: msg,
      );

  String _summary(String type, List<MapCell> ch, int total, int cells) {
    if (ch.isEmpty) {
      return '$type оптимальна\nДанных: $total | Клеток: $cells';
    }
    final inc = ch.where((c) => c.delta > 0).length;
    final dec = ch.where((c) => c.delta < 0).length;
    final avg = ch.map((c) => c.delta.abs()).reduce((a, b) => a + b) /
        ch.length;
    final maxD =
        ch.map((c) => c.delta.abs()).reduce((a, b) => a > b ? a : b);
    final trend = inc > dec * 2
        ? '↑ увеличение'
        : dec > inc * 2
            ? '↓ уменьшение'
            : '↕ смешанное';
    return 'Данных: $total | Клеток: $cells\n'
           'Правок: ${ch.length} (+$inc -$dec) $trend\n'
           'Среднее: ${avg.toStringAsFixed(2)} | Макс: ${maxD.toStringAsFixed(2)}';
  }
}
''')
print("✅ analyzer_service.dart переписан полностью (без синтаксических ошибок)")

# ── Также проверим AndroidManifest, добавим xmlns:tools в корень
manifest_path = 'android/app/src/main/AndroidManifest.xml'
with open(manifest_path, 'r') as f:
    mf = f.read()

# Проблема прошлого варианта: xmlns:tools был в <application>, а нужен в корне <manifest>
if 'xmlns:tools=' not in mf.split('<application')[0]:
    mf = mf.replace(
        '<manifest xmlns:android="http://schemas.android.com/apk/res/android">',
        '<manifest xmlns:android="http://schemas.android.com/apk/res/android"\n'
        '    xmlns:tools="http://schemas.android.com/tools">'
    )
    # Убираем дубли из <application>
    mf = mf.replace(
        '        xmlns:tools="http://schemas.android.com/tools"\n'
        '        tools:replace="android:allowBackup">',
        '        tools:replace="android:allowBackup">'
    )
    with open(manifest_path, 'w') as f:
        f.write(mf)
    print("✅ AndroidManifest.xml — xmlns:tools в корневой manifest")



✅ analyzer_service.dart переписан полностью (без синтаксических ошибок)
✅ AndroidManifest.xml — xmlns:tools в корневой manifest


In [ ]:
# @title 🔧 ФИКС: Разрешения + выбор папки + пакетная запись карт
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# 1. pubspec.yaml — добавляем зависимости для выбора папки
# ================================================================
with open('pubspec.yaml', 'w') as f:
    f.write('''name: nissan_logger_v6
description: Nissan X-Trail T30 QR20DE Tuning Logger V6
version: 6.0.0+1
publish_to: none

environment:
  sdk: ">=3.0.0 <4.0.0"
  flutter: ">=3.10.0"

dependencies:
  flutter:
    sdk: flutter
  cupertino_icons: ^1.0.6
  fl_chart: 0.68.0
  flutter_bluetooth_serial: 0.4.0
  permission_handler: 11.3.1
  path_provider: 2.1.4
  path: ^1.9.0
  csv: 6.0.0
  shared_preferences: 2.3.2
  file_picker: 8.1.2
  share_plus: 10.0.2
  intl: ^0.19.0
  uuid: 4.5.0
  vibration: 2.0.0
  math_expressions: 2.5.0

dev_dependencies:
  flutter_test:
    sdk: flutter
  flutter_lints: ^4.0.0

flutter:
  uses-material-design: true
''')
print("✅ pubspec.yaml")

# ================================================================
# 2. Новый сервис — накопитель правок + запись пакетом
# ================================================================
with open('lib/services/pending_edits.dart', 'w') as f:
    f.write(r'''import '../models/tuning_map.dart';
import '../models/analysis_result.dart';
import 'rom_map_reader.dart';

/// Хранит правки нескольких карт до записи в ROM пакетом.
class PendingEdits {
  static final PendingEdits instance = PendingEdits._();
  PendingEdits._();

  /// address -> (original map, updated map, analysis)
  final Map<String, _PendingEntry> _edits = {};

  int get count => _edits.length;
  bool get isEmpty => _edits.isEmpty;
  List<_PendingEntry> get all => _edits.values.toList();

  /// Добавить/обновить правки для карты
  void addOrUpdate(TuningMap original, TuningMap updated, AnalysisResult result) {
    if (result.changes.isEmpty) return;
    _edits[original.address] = _PendingEntry(
      original: original,
      updated: updated,
      result: result,
    );
  }

  void remove(String address) => _edits.remove(address);
  void clear() => _edits.clear();

  _PendingEntry? get(String address) => _edits[address];
  bool has(String address) => _edits.containsKey(address);

  /// Найти RomMapDef для карты
  RomMapDef? findDef(String address) {
    try {
      return RomMapReader.standardMaps.firstWhere(
        (d) => d.addressHex == address);
    } catch (_) { return null; }
  }
}

class _PendingEntry {
  final TuningMap original;
  final TuningMap updated;
  final AnalysisResult result;
  bool selected;

  _PendingEntry({
    required this.original,
    required this.updated,
    required this.result,
    this.selected = true,
  });

  int get changeCount => result.changes.length;
  String get name => original.name;
  String get address => original.address;
}
''')
print("✅ pending_edits.dart — накопитель правок")

# ================================================================
# 3. Сервис сохранения — с выбором папки + разрешениями
# ================================================================
with open('lib/services/rom_saver.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'package:file_picker/file_picker.dart';
import 'package:path_provider/path_provider.dart';
import 'package:permission_handler/permission_handler.dart';
import 'package:shared_preferences/shared_preferences.dart';

/// Работа с разрешениями и выбор папки для сохранения ROM.
class RomSaver {
  static const _kSaveDirKey = 'rom_save_directory';

  /// Запрашивает MANAGE_EXTERNAL_STORAGE (Android 11+).
  /// Возвращает true если доступ есть.
  static Future<bool> requestStoragePermission() async {
    // Android 11+ — нужен MANAGE_EXTERNAL_STORAGE
    if (await Permission.manageExternalStorage.isGranted) return true;

    final status = await Permission.manageExternalStorage.request();
    if (status.isGranted) return true;

    // Fallback: обычные разрешения (Android < 11)
    if (await Permission.storage.isGranted) return true;
    final s2 = await Permission.storage.request();
    return s2.isGranted;
  }

  /// Показать где текущая папка сохранения
  static Future<String?> getSavedDirectory() async {
    final p = await SharedPreferences.getInstance();
    return p.getString(_kSaveDirKey);
  }

  static Future<void> setSavedDirectory(String path) async {
    final p = await SharedPreferences.getInstance();
    await p.setString(_kSaveDirKey, path);
  }

  /// Открывает диалог выбора папки.
  /// Возвращает путь или null.
  static Future<String?> pickDirectory() async {
    try {
      final path = await FilePicker.platform.getDirectoryPath(
        dialogTitle: 'Выбери папку для сохранения .bin',
      );
      if (path != null) await setSavedDirectory(path);
      return path;
    } catch (_) { return null; }
  }

  /// Пробует сохранить файл. Порядок:
  /// 1. Заданная пользователем папка (SharedPrefs)
  /// 2. Downloads
  /// 3. Папка приложения
  static Future<SaveResult> saveFile(
    List<int> bytes,
    String fileName, {
    String? preferredDir,
  }) async {
    final tried = <String>[];

    // 1. Пользовательская папка
    final userDir = preferredDir ?? await getSavedDirectory();
    if (userDir != null) {
      tried.add(userDir);
      final r = await _tryWrite('$userDir/$fileName', bytes);
      if (r != null) return SaveResult(path: r, location: 'выбранная папка');
    }

    // 2. Downloads
    const downloads = '/storage/emulated/0/Download';
    if (await Directory(downloads).exists()) {
      tried.add(downloads);
      final r = await _tryWrite('$downloads/$fileName', bytes);
      if (r != null) return SaveResult(path: r, location: 'Downloads');
    }

    // 3. Приложение
    try {
      final dir = await getApplicationDocumentsDirectory();
      tried.add(dir.path);
      final r = await _tryWrite('${dir.path}/$fileName', bytes);
      if (r != null) return SaveResult(
        path: r,
        location: 'папка приложения (нет разрешений на внешнюю память)',
      );
    } catch (_) {}

    return SaveResult.error('Не удалось сохранить. Попытки:\n${tried.join("\n")}');
  }

  static Future<String?> _tryWrite(String path, List<int> bytes) async {
    try {
      final f = File(path);
      await f.writeAsBytes(bytes, flush: true);
      if (await f.exists() && await f.length() == bytes.length) return path;
    } catch (_) {}
    return null;
  }
}

class SaveResult {
  final String? path;
  final String location;
  final String? error;
  bool get ok => path != null;

  SaveResult({required this.path, required this.location}) : error = null;
  SaveResult.error(this.error) : path = null, location = '';
}
''')
print("✅ rom_saver.dart — разрешения + выбор папки + фолбэки")

# ================================================================
# 4. Экран-диалог для пакетной записи правок
# ================================================================
with open('lib/screens/write_rom_screen.dart', 'w') as f:
    f.write(r'''import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import '../services/rom_holder.dart';
import '../services/rom_map_reader.dart';
import '../services/pending_edits.dart';
import '../services/rom_saver.dart';

class WriteRomScreen extends StatefulWidget {
  const WriteRomScreen({super.key});
  @override
  State<WriteRomScreen> createState() => _WriteRomScreenState();
}

class _WriteRomScreenState extends State<WriteRomScreen> {
  final _pending = PendingEdits.instance;
  String? _saveDir;
  bool _busy = false;
  bool _hasPermission = false;

  @override
  void initState() {
    super.initState();
    _init();
  }

  Future<void> _init() async {
    _hasPermission = await RomSaver.requestStoragePermission();
    _saveDir = await RomSaver.getSavedDirectory();
    if (mounted) setState(() {});
  }

  Future<void> _pickDir() async {
    final path = await RomSaver.pickDirectory();
    if (path != null && mounted) setState(() => _saveDir = path);
  }

  Future<void> _loadRom() async {
    if (!RomHolder.instance.isLoaded) {
      final ok = await RomHolder.instance.reader.pickAndLoad();
      if (!ok) _snack('Файл не выбран', Colors.orange);
      if (mounted) setState(() {});
    }
  }

  Future<void> _writeSelected() async {
    if (!RomHolder.instance.isLoaded) {
      _snack('Сначала загрузи .bin прошивку', Colors.orange);
      return;
    }

    final selected = _pending.all.where((e) => e.selected).toList();
    if (selected.isEmpty) {
      _snack('Выбери хотя бы одну карту', Colors.orange);
      return;
    }

    setState(() => _busy = true);

    final rom = RomHolder.instance.reader;
    int written = 0;
    final writtenNames = <String>[];

    for (final entry in selected) {
      final def = _pending.findDef(entry.address);
      if (def == null) continue;
      if (rom.writeMapToBuffer(entry.updated, def)) {
        written++;
        writtenNames.add(entry.name);
      }
    }

    if (written == 0) {
      setState(() => _busy = false);
      _snack('Не удалось записать', Colors.red);
      return;
    }

    // Формируем имя файла с номером MOD
    final origName = rom.fileName ?? 'rom.bin';
    final baseName = origName.replaceAll('.bin', '');
    int modIdx = 1;
    // Инкремент если уже есть NLP_MODx в имени
    final match = RegExp(r'NLP_MOD(\d+)').firstMatch(baseName);
    if (match != null) {
      modIdx = int.parse(match.group(1)!) + 1;
    }
    final cleanName = baseName.replaceAll(RegExp(r'NLP_MOD\d+_'), '');
    final outName = 'NLP_MOD${modIdx}_$cleanName.bin';

    // Сохраняем
    final result = await RomSaver.saveFile(
      rom.reader,
      outName,
      preferredDir: _saveDir,
    );

    setState(() => _busy = false);

    if (result.ok) {
      // Очищаем правки (или предлагаем)
      _showSuccess(result, written, writtenNames);
    } else {
      _showError(result.error ?? 'Неизвестная ошибка');
    }
  }

  void _showSuccess(SaveResult result, int count, List<String> names) {
    showDialog(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Row(children: [
        Icon(Icons.check_circle, color: Colors.green, size: 28),
        SizedBox(width: 8),
        Text('УСПЕШНО!'),
      ]),
      content: SingleChildScrollView(
        child: Column(
          mainAxisSize: MainAxisSize.min,
          crossAxisAlignment: CrossAxisAlignment.start,
          children: [
            Text('Записано карт: $count',
              style: const TextStyle(color: Colors.green, fontSize: 14,
                fontWeight: FontWeight.bold)),
            const SizedBox(height: 4),
            ...names.map((n) => Padding(
              padding: const EdgeInsets.only(left: 8, top: 2),
              child: Text('• $n', style: const TextStyle(fontSize: 12)))),
            const Divider(color: Colors.white24, height: 20),
            Text('Сохранено в ${result.location}:',
              style: const TextStyle(color: Colors.white70, fontSize: 12)),
            const SizedBox(height: 6),
            Container(
              padding: const EdgeInsets.all(8),
              decoration: BoxDecoration(
                color: Colors.black26,
                borderRadius: BorderRadius.circular(4),
                border: Border.all(color: Colors.green.withOpacity(0.5))),
              child: SelectableText(result.path!,
                style: const TextStyle(color: Colors.green, fontSize: 11,
                  fontFamily: 'monospace')),
            ),
            const SizedBox(height: 8),
            Container(
              padding: const EdgeInsets.all(8),
              decoration: BoxDecoration(
                color: Colors.cyan.withOpacity(0.1),
                borderRadius: BorderRadius.circular(4)),
              child: const Row(children: [
                Icon(Icons.info_outline, color: Colors.cyan, size: 16),
                SizedBox(width: 6),
                Expanded(child: Text(
                  'Открой в PCMflash — контрольная сумма пересчитается.',
                  style: TextStyle(color: Colors.cyan, fontSize: 11))),
              ]),
            ),
          ],
        ),
      ),
      actions: [
        TextButton(
          onPressed: () async {
            await Clipboard.setData(ClipboardData(text: result.path!));
            if (mounted) _snack('Скопировано', Colors.green);
          },
          child: const Text('КОПИРОВАТЬ ПУТЬ')),
        TextButton(
          onPressed: () {
            Navigator.pop(c);
            _pending.clear();
            if (mounted) setState(() {});
          },
          style: TextButton.styleFrom(foregroundColor: Colors.orange),
          child: const Text('ОЧИСТИТЬ ПРАВКИ')),
        TextButton(
          onPressed: () => Navigator.pop(c),
          style: TextButton.styleFrom(foregroundColor: Colors.green),
          child: const Text('OK')),
      ]));
  }

  void _showError(String msg) {
    showDialog(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Row(children: [
        Icon(Icons.error, color: Colors.red),
        SizedBox(width: 8),
        Text('Ошибка'),
      ]),
      content: Text(msg, style: const TextStyle(color: Colors.white70)),
      actions: [
        TextButton(
          onPressed: () async {
            await openAppSettings();
          },
          child: const Text('НАСТРОЙКИ ПРИЛОЖЕНИЯ')),
        TextButton(
          onPressed: () => Navigator.pop(c),
          child: const Text('OK')),
      ]));
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(
      SnackBar(content: Text(m), backgroundColor: c));
  }

  @override
  Widget build(BuildContext context) {
    final selected = _pending.all.where((e) => e.selected).length;
    final total = _pending.count;

    return Scaffold(
      appBar: AppBar(
        title: const Text('Запись правок в ROM'),
        backgroundColor: const Color(0xFF16213E),
      ),
      body: Column(children: [
        // ── Статус ─────────────────────────────────
        Container(
          padding: const EdgeInsets.all(12),
          color: const Color(0xFF16213E),
          child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
            // Разрешения
            Row(children: [
              Icon(_hasPermission ? Icons.check_circle : Icons.warning,
                color: _hasPermission ? Colors.green : Colors.orange, size: 20),
              const SizedBox(width: 8),
              Expanded(child: Text(
                _hasPermission
                  ? 'Разрешение на память есть'
                  : 'Нет разрешения на память',
                style: TextStyle(
                  color: _hasPermission ? Colors.green : Colors.orange,
                  fontSize: 12))),
              if (!_hasPermission)
                TextButton(
                  onPressed: () async {
                    _hasPermission = await RomSaver.requestStoragePermission();
                    if (mounted) setState(() {});
                  },
                  style: TextButton.styleFrom(foregroundColor: Colors.orange),
                  child: const Text('ЗАПРОСИТЬ')),
            ]),
            const SizedBox(height: 8),
            // ROM файл
            Row(children: [
              Icon(RomHolder.instance.isLoaded ? Icons.check_circle : Icons.warning,
                color: RomHolder.instance.isLoaded ? Colors.green : Colors.orange,
                size: 20),
              const SizedBox(width: 8),
              Expanded(child: Text(
                RomHolder.instance.isLoaded
                  ? 'ROM: ${RomHolder.instance.fileName}'
                  : 'ROM не загружен',
                style: TextStyle(
                  color: RomHolder.instance.isLoaded ? Colors.green : Colors.orange,
                  fontSize: 12), overflow: TextOverflow.ellipsis)),
              if (!RomHolder.instance.isLoaded)
                TextButton(
                  onPressed: _loadRom,
                  child: const Text('ЗАГРУЗИТЬ')),
            ]),
            const SizedBox(height: 8),
            // Папка сохранения
            Row(children: [
              const Icon(Icons.folder, color: Colors.cyan, size: 20),
              const SizedBox(width: 8),
              Expanded(child: Text(
                _saveDir ?? 'Downloads (по умолчанию)',
                style: const TextStyle(color: Colors.cyan, fontSize: 11),
                overflow: TextOverflow.ellipsis)),
              TextButton(
                onPressed: _pickDir,
                child: const Text('ВЫБРАТЬ')),
            ]),
          ]),
        ),

        // ── Список правок ──────────────────────────
        Expanded(child: _pending.isEmpty
          ? const Center(child: Padding(padding: EdgeInsets.all(20),
              child: Text(
                'Пока нет правок.\n\n'
                'Проанализируй логи в Анализаторе — правки автоматически\n'
                'сохранятся сюда для пакетной записи.',
                textAlign: TextAlign.center,
                style: TextStyle(color: Colors.white54, fontSize: 13))))
          : ListView(padding: const EdgeInsets.all(8), children: [
              // Заголовок
              Padding(padding: const EdgeInsets.all(8),
                child: Row(children: [
                  Text('Правок: $selected из $total',
                    style: const TextStyle(fontWeight: FontWeight.bold)),
                  const Spacer(),
                  TextButton(
                    onPressed: () {
                      final allSel = _pending.all.every((e) => e.selected);
                      for (final e in _pending.all) e.selected = !allSel;
                      setState(() {});
                    },
                    child: Text(
                      _pending.all.every((e) => e.selected)
                        ? 'СНЯТЬ ВСЕ' : 'ВЫБРАТЬ ВСЕ',
                      style: const TextStyle(fontSize: 11))),
                ])),
              // Карты
              ..._pending.all.map((e) => Card(
                color: e.selected
                  ? Colors.green.withOpacity(0.1)
                  : const Color(0xFF16213E),
                child: CheckboxListTile(
                  value: e.selected,
                  onChanged: (v) {
                    e.selected = v ?? false;
                    setState(() {});
                  },
                  activeColor: Colors.green,
                  title: Text(e.name,
                    style: const TextStyle(fontWeight: FontWeight.bold)),
                  subtitle: Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      Text('${e.changeCount} правок • ${e.address}',
                        style: const TextStyle(fontSize: 11, color: Colors.white54)),
                      Text('Паттерн: ${e.result.patternName}',
                        style: const TextStyle(fontSize: 10, color: Colors.orange)),
                    ]),
                  secondary: IconButton(
                    icon: const Icon(Icons.delete_outline, color: Colors.red),
                    onPressed: () {
                      _pending.remove(e.address);
                      setState(() {});
                    }),
                ))),
            ])),

        // ── Кнопка записи ──────────────────────────
        if (_pending.isNotEmpty)
          Container(
            padding: const EdgeInsets.all(12),
            color: const Color(0xFF16213E),
            child: SizedBox(
              width: double.infinity,
              child: ElevatedButton.icon(
                onPressed: _busy || selected == 0 ? null : _writeSelected,
                icon: _busy
                  ? const SizedBox(width: 20, height: 20,
                      child: CircularProgressIndicator(color: Colors.white, strokeWidth: 2))
                  : const Icon(Icons.save, size: 24),
                label: Text(
                  _busy
                    ? 'ЗАПИСЬ...'
                    : 'ЗАПИСАТЬ $selected КАРТ В ROM',
                  style: const TextStyle(fontSize: 15, fontWeight: FontWeight.bold)),
                style: ElevatedButton.styleFrom(
                  backgroundColor: Colors.red.shade800,
                  foregroundColor: Colors.white,
                  minimumSize: const Size.fromHeight(50)),
              ),
            ),
          ),
      ]),
    );
  }
}
''')
print("✅ write_rom_screen.dart — пакетная запись с чекбоксами")

# ================================================================
# 5. Обновляем analyzer_screen.dart — добавлять правки в PendingEdits
# ================================================================
an_screen = 'lib/screens/analyzer_screen.dart'
with open(an_screen, 'r') as f:
    code = f.read()

# Добавляем импорты
if "'../services/pending_edits.dart'" not in code:
    code = code.replace(
        "import '../services/rom_holder.dart';",
        "import '../services/rom_holder.dart';\n"
        "import '../services/pending_edits.dart';\n"
        "import 'write_rom_screen.dart';"
    )

# Заменяем метод _writeToRom
old_write_start = "  Future<void> _writeToRom() async {"
start = code.find(old_write_start)
if start != -1:
    brace_count = 0
    found_first = False
    end = start
    for i in range(start, len(code)):
        if code[i] == '{':
            brace_count += 1
            found_first = True
        elif code[i] == '}':
            brace_count -= 1
            if found_first and brace_count == 0:
                end = i + 1
                break

    new_write = '''  Future<void> _writeToRom() async {
    if (_upd == null || _orig == null) return;
    final result = _logResult ?? _onlineResult;
    if (result == null) return;

    // Добавляем в накопитель правок
    PendingEdits.instance.addOrUpdate(_orig!, _upd!, result);
    _snack('Правки добавлены в очередь на запись', Colors.green);

    // Открываем экран записи
    if (!mounted) return;
    await Navigator.push(context, MaterialPageRoute(
      builder: (_) => const WriteRomScreen()));
    if (mounted) setState(() {});
  }'''

    code = code[:start] + new_write + code[end:]

# Также в _resultCard меняем текст кнопки на понятный
code = code.replace(
    "romLoaded\n                  ? 'ЗАПИСАТЬ ПРАВКИ В ROM'\n                  : 'ЗАГРУЗИТЬ .BIN И ЗАПИСАТЬ',",
    "'В ОЧЕРЕДЬ НА ЗАПИСЬ В ROM (${PendingEdits.instance.count + 1})',"
)

with open(an_screen, 'w') as f:
    f.write(code)
print("✅ analyzer_screen.dart — правки идут в PendingEdits + переход на экран записи")

# ================================================================
# 6. Регистрируем экран в навигации (home_screen.dart)
# ================================================================
home_path = 'lib/screens/home_screen.dart'
with open(home_path, 'r') as f:
    hcode = f.read()

# Добавляем import
if "write_rom_screen" not in hcode:
    hcode = hcode.replace(
        "import 'terminal_screen.dart';",
        "import 'terminal_screen.dart';\nimport 'write_rom_screen.dart';"
    )

# Добавляем пункт в _items (перед 'Настройки')
if "'ROM Запись'" not in hcode:
    hcode = hcode.replace(
        "_NavItem(Icons.settings, 'Настройки',",
        "_NavItem(Icons.save_alt, 'ROM Запись',\n"
        "        const WriteRomScreen()),\n"
        "      _NavItem(Icons.settings, 'Настройки',"
    )

with open(home_path, 'w') as f:
    f.write(hcode)
print("✅ home_screen.dart — добавлена вкладка 'ROM Запись'")

# ================================================================
# 7. RomMapReader — добавляем геттер reader (для доступа к bytes)
# ================================================================
rom_path = 'lib/services/rom_map_reader.dart'
with open(rom_path, 'r') as f:
    rcode = f.read()

# Добавляем геттер после других геттеров
if "List<int> get reader" not in rcode:
    rcode = rcode.replace(
        "int     get length   => _data?.length ?? 0;",
        "int     get length   => _data?.length ?? 0;\n"
        "  List<int> get reader => _data ?? [];"
    )
    with open(rom_path, 'w') as f:
        f.write(rcode)
    print("✅ rom_map_reader.dart — геттер reader для bytes")
# @title 🔧 ФИКС двух ошибок компиляции
import os
os.chdir('/content/nissan_logger_v6')

# ─── Фикс 1: import openAppSettings в write_rom_screen.dart ─────
path = 'lib/screens/write_rom_screen.dart'
with open(path, 'r') as f:
    code = f.read()

if "import 'package:permission_handler/permission_handler.dart';" not in code:
    code = code.replace(
        "import 'package:flutter/services.dart';",
        "import 'package:flutter/services.dart';\n"
        "import 'package:permission_handler/permission_handler.dart';"
    )
    with open(path, 'w') as f:
        f.write(code)
    print("✅ write_rom_screen.dart — добавлен import permission_handler")

# ─── Фикс 2: геттер isNotEmpty в pending_edits.dart ─────────────
pe_path = 'lib/services/pending_edits.dart'
with open(pe_path, 'r') as f:
    pe = f.read()

if "bool get isNotEmpty" not in pe:
    pe = pe.replace(
        "bool get isEmpty => _edits.isEmpty;",
        "bool get isEmpty => _edits.isEmpty;\n"
        "  bool get isNotEmpty => _edits.isNotEmpty;"
    )
    with open(pe_path, 'w') as f:
        f.write(pe)
    print("✅ pending_edits.dart — добавлен геттер isNotEmpty")




✅ pubspec.yaml
✅ pending_edits.dart — накопитель правок
✅ rom_saver.dart — разрешения + выбор папки + фолбэки
✅ write_rom_screen.dart — пакетная запись с чекбоксами
✅ analyzer_screen.dart — правки идут в PendingEdits + переход на экран записи
✅ home_screen.dart — добавлена вкладка 'ROM Запись'
✅ rom_map_reader.dart — геттер reader для bytes
✅ write_rom_screen.dart — добавлен import permission_handler
✅ pending_edits.dart — добавлен геттер isNotEmpty


In [ ]:
# @title 🔧 ФИКС: диагностика MAF + Torque анализ с фолбэком на нагрузку
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# ФИКС 1: OBDService — правильно писать И vольтаж И g/s в лог
# Проблема: если PID возвращает g/s напрямую (V5 совместимость)
# то mafVoltage=0, а mafGps идёт как есть.
# Если PID возвращает вольтаж (V6) — считаем через таблицу.
# Логгер пишет ОБА значения.
# ================================================================
obd_path = 'lib/services/obd_service.dart'
with open(obd_path, 'r') as f:
    code = f.read()

# Ищем метод _publish
old_publish_start = "  void _publish() {"
start = code.find(old_publish_start)
if start != -1:
    brace_count = 0
    found_first = False
    end = start
    for i in range(start, len(code)):
        if code[i] == '{':
            brace_count += 1
            found_first = True
        elif code[i] == '}':
            brace_count -= 1
            if found_first and brace_count == 0:
                end = i + 1
                break

    new_publish = '''  void _publish() {
    // MAF PID: определяем что вернуло — вольтаж или g/s
    final rawMaf = _v('MAF_V');
    double mafVolt = 0;
    double mafGpsRaw = 0;

    if (rawMaf > 0 && rawMaf < 5.5) {
      // Похоже на вольтаж (0.5-5V диапазон Hitachi)
      mafVolt   = rawMaf;
      mafGpsRaw = AppConstants.mafVoltToGps(rawMaf);
    } else if (rawMaf >= 5.5) {
      // Похоже на прямые g/s (старый формат)
      mafVolt   = 0;
      mafGpsRaw = rawMaf;
    }

    final mafMult   = _profile?.mafMultiplier ?? 1.0;
    final mafGps    = mafGpsRaw * mafMult;

    final speedRaw  = _v('SPEED');
    final speedMult = _profile?.speedMultiplier ?? 1.0;
    final speed     = (speedRaw * speedMult).toInt().clamp(0, 300);
    final disp      = _profile?.displacement ?? 2.0;

    final o2   = _v('O2_B1S1');
    final stft = _v('STFT');
    final afr  = _calcAfr(o2, stft);

    final data = OBDData(
      timestamp:          DateTime.now(),
      rpm:                _v('RPM').toInt().clamp(0, 9999),
      speed:              speed,
      engineLoad:         _v('LOAD').clamp(0, 100),
      coolantTemp:        _v('ECT').toInt().clamp(-40, 200),
      intakeTemp:         _v('IAT').toInt().clamp(-40, 100),
      mafVoltage:         mafVolt,
      mafGps:             mafGps,
      throttlePos:        _v('TPS').clamp(0, 100),
      ignitionTiming:     _v('TIMING'),
      actualIgnition:     _v('TIMING'),
      vtcActualAngle:     _v('VTC_ACT'),
      knockRetard:        _v('KNOCK').abs(),
      shortFuelTrim:      _v('STFT').clamp(-100, 100),
      longFuelTrim:       _v('LTFT').clamp(-100, 100),
      o2Voltage:          o2,
      afr:                afr,
      injectorPulseWidth: _v('INJ_B1'),
      injectorDuty:       (_v('INJ_B1') / 20.0 * 100).clamp(0, 100),
      manifoldPressure:   _v('MAP_V') * 40,
      acceleratorPedal:   _v('PEDAL'),
      throttleActual:     _v('TPS'),
      batteryVoltage:     _v('BATT'),
      engineDisplacement: disp,
      tripFuelL:          _tripFuelL,
      actualTorque:       _v('TORQUE'),
      requestedTorque:    _v('POWER'),
    );

    _updateTripFuel(data.fuelFlowLph);
    _dataCtrl.add(data);
  }'''

    code = code[:start] + new_publish + code[end:]
    with open(obd_path, 'w') as f:
        f.write(code)
    print("✅ obd_service.dart — авто-детект вольтаж vs g/s + запись actualTorque")

# ================================================================
# ФИКС 2: analyzer_service.dart — Torque анализ полностью через
# оценку по Load% + учёт knock (не требует ЭБУ-момента)
# ================================================================
an_path = 'lib/services/analyzer_service.dart'
with open(an_path, 'r') as f:
    code = f.read()

old_start = "  Future<AnalysisResult> analyzeTorqueMap("
start = code.find(old_start)
if start != -1:
    brace_count = 0
    found_first = False
    end = start
    for i in range(start, len(code)):
        if code[i] == '{':
            brace_count += 1
            found_first = True
        elif code[i] == '}':
            brace_count -= 1
            if found_first and brace_count == 0:
                end = i + 1
                break

    new_torque = '''  Future<AnalysisResult> analyzeTorqueMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 800 && d.rpm < 7500 && d.engineLoad > 3).toList();

    if (valid.length < 20) {
      return _empty('Engine Torque', log.length,
        'Мало данных: ${valid.length} (нужно 20+)');
    }

    // Проверяем есть ли реальный момент от ЭБУ
    final withTorque = valid.where((d) => d.actualTorque.abs() > 5).length;
    final torqueRatio = withTorque / valid.length;
    final useEcuTorque = torqueRatio > 0.3;

    // Для QR20DE максимум ~192 Нм при 4000 RPM
    // Простая модель: T(load, rpm) = Load% * 192 * bell(RPM)
    double estimateTorque(double load, double rpm) {
      final loadFactor = load / 100.0;
      // Колокол вокруг 4000 RPM, спад к 800 и 7500
      final rpmNorm = (rpm - 4000).abs() / 3500;
      final rpmFactor = (1.0 - rpmNorm * 0.5).clamp(0.3, 1.0);
      return loadFactor * 192.0 * rpmFactor;
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    int cellsAnalyzed = 0;
    int cellsSkipped = 0;

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < 2) { cellsSkipped++; continue; }

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      // Пропускаем ячейки с нулевым/отрицательным значением
      if (cur < 3) { cellsSkipped++; continue; }
      cellsAnalyzed++;

      final avgKnock = _avg(samples.map((d) => d.knockRetard));
      final avgLoad = _avg(samples.map((d) => d.engineLoad));
      final avgRpm = _avg(samples.map((d) => d.rpm.toDouble()));

      double target;
      String source;

      if (useEcuTorque) {
        // Используем момент от ЭБУ
        final withT = samples.where((d) => d.actualTorque.abs() > 5).toList();
        if (withT.length >= 2) {
          target = _avg(withT.map((d) => d.actualTorque));
          source = 'ЭБУ';
        } else {
          target = estimateTorque(avgLoad, avgRpm);
          source = 'оценка';
        }
      } else {
        target = estimateTorque(avgLoad, avgRpm);
        source = 'оценка';
      }

      double sug = cur;
      double conf = 0;
      String why = '';

      final diff = target - cur;
      final diffPct = cur != 0 ? (diff / cur.abs()) * 100 : 0;

      if (diffPct.abs() > 15) {
        // Значительное расхождение → корректируем частично
        sug = cur + diff * 0.3;
        why = '$source: ${target.toStringAsFixed(0)} Нм '
              '(${diffPct > 0 ? "+" : ""}${diffPct.toStringAsFixed(0)}%)';
        conf = useEcuTorque ? 0.6 : 0.45;
      } else if (avgKnock > pattern.knockTolerance * 2 && cur > 50) {
        sug = cur * 0.9;
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}° — снизить';
        conf = 0.7;
      }

      // Ограничиваем ±20%
      final maxChange = cur.abs() * 0.20;
      final finalDiff = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + finalDiff;

      if ((sug - cur).abs() >= 3 &&
          (sug - cur).abs() / cur.abs() > 0.05 &&
          conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    final srcName = useEcuTorque
      ? 'ЭБУ (${(torqueRatio * 100).toStringAsFixed(0)}%)'
      : 'оценка Load/RPM';

    String summary = 'Источник: $srcName\\n'
                     'Проанализировано: $cellsAnalyzed из ${grouped.length} ячеек\\n';
    if (cellsSkipped > 0) {
      summary += 'Пропущено (пустые/малые): $cellsSkipped\\n';
    }
    if (changes.isEmpty) {
      summary += 'Правок не требуется — момент близок к карте';
    } else {
      summary += 'Правок: ${changes.length}';
    }

    return AnalysisResult(
      mapName: 'Engine Torque',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: summary,
    );
  }'''

    code = code[:start] + new_torque + code[end:]
    with open(an_path, 'w') as f:
        f.write(code)
    print("✅ analyzer_service.dart — Torque всегда работает (ЭБУ или оценка)")

# ================================================================
# ФИКС 3: Показывать в дашборде и логгере что MAF — вольтаж
# добавляем колонку MAF_V в CSV если не 0
# уже сделано в OBDData.toCsvRow — просто проверим что она есть
# ================================================================
data_path = 'lib/models/obd_data.dart'
with open(data_path, 'r') as f:
    dc = f.read()

# Убеждаемся что requestedTorque и actualTorque в CSV
if 'requestedTorque.toStringAsFixed(2)' not in dc:
    print("⚠️ Проверь ручной toCsvRow — actualTorque должен писаться")
else:
    print("✅ obd_data.dart — momentum поля пишутся в CSV")

# ================================================================
# ФИКС 4: На экране Приборы показать MAF_V явно
# Проверяем что есть параметр maf_v в дашборде
# ================================================================
dash_path = 'lib/screens/dashboard_screen.dart'
with open(dash_path, 'r') as f:
    dashcode = f.read()

if "'maf_v'" in dashcode:
    print("✅ dashboard_screen.dart — MAF V уже доступен для отображения")
else:
    print("⚠️ dashboard_screen.dart — нужно добавить maf_v")



✅ obd_service.dart — авто-детект вольтаж vs g/s + запись actualTorque
✅ analyzer_service.dart — Torque всегда работает (ЭБУ или оценка)
✅ obd_data.dart — momentum поля пишутся в CSV
✅ dashboard_screen.dart — MAF V уже доступен для отображения


In [ ]:
# @title 🔧 ФИКС: analyzer_service.dart переписан ЦЕЛИКОМ с нуля
import os
os.chdir('/content/nissan_logger_v6')

with open('lib/services/analyzer_service.dart', 'w') as f:
    f.write(r'''import 'dart:io';
import 'dart:math';
import '../models/obd_data.dart';
import '../models/tuning_map.dart';
import '../models/analysis_result.dart';

// ═══════════════════════════════════════════════════════════════
// ПАТТЕРНЫ НАСТРОЙКИ
// ═══════════════════════════════════════════════════════════════
enum TuningPatternType { maxPower, economy, stability }

class TuningPattern {
  final TuningPatternType type;
  final String name, description;
  final double knockTolerance, afrLean, afrRich;
  final double timingAggression, fuelTrimThreshold, vtcAggression;

  const TuningPattern({
    required this.type,
    required this.name,
    required this.description,
    this.knockTolerance    = 1.0,
    this.afrLean           = 15.0,
    this.afrRich           = 12.5,
    this.timingAggression  = 0.5,
    this.fuelTrimThreshold = 3.0,
    this.vtcAggression     = 0.5,
  });

  static const maxPower = TuningPattern(
    type: TuningPatternType.maxPower,
    name: 'Максимальная мощность',
    description: 'Агрессивный УОЗ, богатая WOT смесь',
    knockTolerance: 0.3, afrLean: 14.0, afrRich: 11.5,
    timingAggression: 0.9, fuelTrimThreshold: 2.0, vtcAggression: 0.9,
  );

  static const economy = TuningPattern(
    type: TuningPatternType.economy,
    name: 'Минимальный расход',
    description: 'Бедная смесь на круизе',
    knockTolerance: 0.5, afrLean: 15.5, afrRich: 13.5,
    timingAggression: 0.3, fuelTrimThreshold: 5.0, vtcAggression: 0.3,
  );

  static const stability = TuningPattern(
    type: TuningPatternType.stability,
    name: 'Стабильная работа',
    description: 'Консервативные настройки',
    knockTolerance: 2.0, afrLean: 14.7, afrRich: 13.0,
    timingAggression: 0.1, fuelTrimThreshold: 3.0, vtcAggression: 0.2,
  );

  static const List<TuningPattern> all = [maxPower, economy, stability];
}

// ═══════════════════════════════════════════════════════════════
// СЕРВИС АНАЛИЗА
// ═══════════════════════════════════════════════════════════════
class AnalyzerService {
  static const _minSamples = 2;
  static const _minConf    = 0.4;

  // ─────────────────────────────────────────────────────────────
  // SPARK ADVANCE
  // ─────────────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeSparkMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 && d.engineLoad > 0).toList();
    if (valid.length < 10) {
      return _empty('Spark Advance', log.length,
        'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgTiming = _avg(samples.map((d) => d.actualIgnition));
      final avgKnock  = _avg(samples.map((d) => d.knockRetard));
      final avgAFR    = _avg(samples.map((d) => d.afr));

      double sug = cur;
      double conf = 0;
      String why = '';

      if (avgKnock > pattern.knockTolerance * 2) {
        sug = cur - min(avgKnock, 3.0);
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}°';
        conf = 0.9;
      } else if (avgKnock > pattern.knockTolerance) {
        sug = cur - (1.0 * (1.0 - pattern.timingAggression));
        why = 'Knock>${pattern.knockTolerance.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgTiming != 0 && (avgTiming - cur).abs() > 2) {
        sug = avgTiming + (pattern.timingAggression * 2 - 1);
        why = 'ЭБУ: ${avgTiming.toStringAsFixed(1)}°';
        conf = 0.6;
      } else if (avgKnock < pattern.knockTolerance * 0.2 &&
                 avgAFR > pattern.afrRich &&
                 avgAFR < pattern.afrLean) {
        sug = cur + pattern.timingAggression * 2;
        why = '+${(pattern.timingAggression * 2).toStringAsFixed(1)}° (стаб.)';
        conf = 0.5;
      }

      sug = sug.clamp(-5.0, 45.0);
      if ((sug - cur).abs() >= 0.5 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'Spark Advance',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('Spark', changes, valid.length, grouped.length),
    );
  }

  // ─────────────────────────────────────────────────────────────
  // FUEL MAP / VE
  // ─────────────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeFuelMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 400 && d.rpm < 7500 &&
      d.engineLoad > 0 && d.longFuelTrim.abs() < 30).toList();
    if (valid.length < 10) {
      return _empty('Fuel Map / VE', log.length,
        'Мало данных: ${valid.length}');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgSTFT = _avg(samples.map((d) => d.shortFuelTrim));
      final avgLTFT = _avg(samples.map((d) => d.longFuelTrim));
      final total = avgSTFT + avgLTFT;
      final avgAFR = _avg(samples.map((d) => d.afr));
      final avgLoad = _avg(samples.map((d) => d.engineLoad));

      double sug = cur;
      double conf = 0;
      String why = '';

      if (total > pattern.fuelTrimThreshold) {
        sug = cur * (1 + total / 100.0);
        why = 'Trim +${total.toStringAsFixed(1)}% (бедно)';
        conf = min(0.9, total.abs() / 10);
      } else if (total < -pattern.fuelTrimThreshold) {
        sug = cur * (1 + total / 100.0);
        why = 'Trim ${total.toStringAsFixed(1)}% (богато)';
        conf = min(0.9, total.abs() / 10);
      } else if (avgAFR > pattern.afrLean && avgLoad > 50) {
        sug = cur * 1.04;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} бедно';
        conf = 0.6;
      } else if (avgAFR < pattern.afrRich && avgLoad > 50) {
        sug = cur * 0.96;
        why = 'AFR ${avgAFR.toStringAsFixed(1)} богато';
        conf = 0.6;
      }

      final maxChange = cur * 0.15;
      final delta = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + delta;

      if ((sug - cur).abs() / (cur.abs() + 0.001) >= 0.01 &&
          conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'Fuel Map / VE',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('Fuel', changes, valid.length, grouped.length),
    );
  }

  // ─────────────────────────────────────────────────────────────
  // VTC
  // ─────────────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeVTCMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 800 && d.rpm < 7000 && d.engineLoad > 5).toList();
    if (valid.length < 10) {
      return _empty('VTC Map', log.length, 'Мало данных');
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < _minSamples) continue;

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      final avgAct = _avg(samples.map((d) => d.vtcActualAngle));
      final avgKnock = _avg(samples.map((d) => d.knockRetard));

      double sug = cur;
      double conf = 0;
      String why = '';

      if ((avgAct - cur).abs() > 3) {
        sug = avgAct + (pattern.vtcAggression * 2 - 1);
        why = 'ЭБУ: ${avgAct.toStringAsFixed(1)}°';
        conf = 0.7;
      } else if (avgKnock > pattern.knockTolerance && cur > 15) {
        sug = max(0, cur - 5);
        why = 'Детонация — снизить VTC';
        conf = 0.75;
      }

      sug = sug.clamp(0.0, 45.0);
      if ((sug - cur).abs() >= 2 && conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    return AnalysisResult(
      mapName: 'VTC Map',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: _summary('VTC', changes, valid.length, grouped.length),
    );
  }

  // ─────────────────────────────────────────────────────────────
  // ENGINE TORQUE — работает всегда: либо ЭБУ, либо оценка
  // ─────────────────────────────────────────────────────────────
  Future<AnalysisResult> analyzeTorqueMap(
    List<OBDData> log,
    TuningMap map, {
    TuningPattern pattern = TuningPattern.stability,
  }) async {
    final valid = log.where((d) =>
      d.rpm > 800 && d.rpm < 7500 && d.engineLoad > 3).toList();

    if (valid.length < 20) {
      return _empty('Engine Torque', log.length,
        'Мало данных: ${valid.length} (нужно 20+)');
    }

    // Определяем источник момента
    final withTorque = valid.where((d) => d.actualTorque.abs() > 5).length;
    final torqueRatio = withTorque / valid.length;
    final useEcuTorque = torqueRatio > 0.3;

    // Оценка момента для QR20DE (пик ~192 Нм при 4000 RPM)
    double estimateTorque(double load, double rpm) {
      final loadFactor = load / 100.0;
      final rpmNorm = (rpm - 4000).abs() / 3500;
      final rpmFactor = (1.0 - rpmNorm * 0.5).clamp(0.3, 1.0);
      return loadFactor * 192.0 * rpmFactor;
    }

    final grouped = _group(valid, map);
    final changes = <MapCell>[];
    int cellsAnalyzed = 0;
    int cellsSkipped = 0;

    for (final entry in grouped.entries) {
      final samples = entry.value;
      if (samples.length < 2) { cellsSkipped++; continue; }

      final parts = entry.key.split(',');
      final ri = int.parse(parts[0]);
      final li = int.parse(parts[1]);
      final cur = map.data[ri][li];

      if (cur < 3) { cellsSkipped++; continue; }
      cellsAnalyzed++;

      final avgKnock = _avg(samples.map((d) => d.knockRetard));
      final avgLoad = _avg(samples.map((d) => d.engineLoad));
      final avgRpm = _avg(samples.map((d) => d.rpm.toDouble()));

      double target;
      String source;

      if (useEcuTorque) {
        final withT = samples.where((d) => d.actualTorque.abs() > 5).toList();
        if (withT.length >= 2) {
          target = _avg(withT.map((d) => d.actualTorque));
          source = 'ЭБУ';
        } else {
          target = estimateTorque(avgLoad, avgRpm);
          source = 'оценка';
        }
      } else {
        target = estimateTorque(avgLoad, avgRpm);
        source = 'оценка';
      }

      double sug = cur;
      double conf = 0;
      String why = '';

      final diff = target - cur;
      final diffPct = cur != 0 ? (diff / cur.abs()) * 100 : 0;

      if (diffPct.abs() > 15) {
        sug = cur + diff * 0.3;
        why = '$source: ${target.toStringAsFixed(0)} Нм '
              '(${diffPct > 0 ? "+" : ""}${diffPct.toStringAsFixed(0)}%)';
        conf = useEcuTorque ? 0.6 : 0.45;
      } else if (avgKnock > pattern.knockTolerance * 2 && cur > 50) {
        sug = cur * 0.9;
        why = 'Детонация ${avgKnock.toStringAsFixed(1)}° — снизить';
        conf = 0.7;
      }

      final maxChange = cur.abs() * 0.20;
      final finalDiff = (sug - cur).clamp(-maxChange, maxChange);
      sug = cur + finalDiff;

      if ((sug - cur).abs() >= 3 &&
          (sug - cur).abs() / cur.abs() > 0.05 &&
          conf >= _minConf) {
        changes.add(MapCell(
          rpmIndex: ri, loadIndex: li,
          rpm: map.rpmAxis[ri], load: map.loadAxis[li],
          currentValue: cur, suggestedValue: sug,
          confidence: conf, sampleCount: samples.length, reason: why,
        ));
      }
    }

    final srcName = useEcuTorque
      ? 'ЭБУ (${(torqueRatio * 100).toStringAsFixed(0)}%)'
      : 'оценка Load/RPM';

    String summary = 'Источник: $srcName\n'
                     'Проанализировано: $cellsAnalyzed из ${grouped.length} ячеек\n';
    if (cellsSkipped > 0) {
      summary += 'Пропущено (пустые/малые): $cellsSkipped\n';
    }
    if (changes.isEmpty) {
      summary += 'Правок не требуется — момент близок к карте';
    } else {
      summary += 'Правок: ${changes.length}';
    }

    return AnalysisResult(
      mapName: 'Engine Torque',
      analyzedAt: DateTime.now(),
      totalSamples: log.length,
      changes: changes,
      patternName: pattern.name,
      summary: summary,
    );
  }

  // ─────────────────────────────────────────────────────────────
  // ЗАГРУЗКА CSV (V5 и V6 автоопределение)
  // ─────────────────────────────────────────────────────────────
  Future<List<OBDData>> loadLogFromCSV(String path) async {
    final file    = File(path);
    final content = await file.readAsString();
    final lines   = content.split('\n');
    if (lines.length < 2) return [];

    final header   = lines[0].toUpperCase();
    final headCols = header.split(',');
    bool isV6 = false;
    if (headCols.length >= 8) {
      final h6 = headCols[6].trim();
      final h7 = headCols[7].trim();
      isV6 = h6.contains('MAF_V') && h7.contains('MAF_GPS');
    }

    final result = <OBDData>[];
    for (int i = 1; i < lines.length; i++) {
      final line = lines[i].trim();
      if (line.isEmpty) continue;
      final parts = line.split(',');
      if (parts.length < 10) continue;

      try {
        double _d(int idx, [double def = 0]) {
          if (idx < 0 || idx >= parts.length) return def;
          final s = parts[idx].trim();
          if (s.isEmpty) return def;
          return double.tryParse(s) ?? def;
        }
        int _i(int idx, [int def = 0]) {
          if (idx < 0 || idx >= parts.length) return def;
          final s = parts[idx].trim();
          if (s.isEmpty) return def;
          return int.tryParse(s) ?? (double.tryParse(s)?.toInt() ?? def);
        }

        final ts = _i(0);
        if (ts == 0) continue;

        if (isV6) {
          result.add(OBDData(
            timestamp:          DateTime.fromMillisecondsSinceEpoch(ts),
            rpm:                _i(1),
            speed:              _i(2),
            engineLoad:         _d(3),
            coolantTemp:        _i(4),
            intakeTemp:         _i(5),
            mafVoltage:         _d(6),
            mafGps:             _d(7),
            throttlePos:        _d(8),
            ignitionTiming:     _d(9),
            shortFuelTrim:      _d(10),
            longFuelTrim:       _d(11),
            o2Voltage:          _d(12),
            afr:                _d(13, 14.7),
            vtcTargetAngle:     _d(14),
            vtcActualAngle:     _d(15),
            knockRetard:        _d(16),
            knockCount:         _i(17),
            actualIgnition:     _d(18),
            injectorDuty:       _d(19),
            injectorPulseWidth: _d(20),
            requestedTorque:    _d(21),
            actualTorque:       _d(22),
            oilTemp:            _d(23),
            afrTarget:          _d(24, 14.7),
            lambda:             _d(25, 1.0),
            manifoldPressure:   _d(26),
            acceleratorPedal:   _d(27),
            throttleActual:     _d(28),
            batteryVoltage:     _d(31),
            tripFuelL:          _d(35),
          ));
        } else {
          // V5: MAF_gs — уже в g/s
          final mafRaw = _d(6);
          // Автодетект: если < 5.5 значит вольтаж, конвертируем
          double mafGps;
          double mafVolt;
          if (mafRaw < 5.5 && mafRaw > 0) {
            mafVolt = mafRaw;
            mafGps = _mafVoltToGps(mafRaw);
          } else {
            mafVolt = 0;
            mafGps = mafRaw;
          }

          result.add(OBDData(
            timestamp:          DateTime.fromMillisecondsSinceEpoch(ts),
            rpm:                _i(1),
            speed:              _i(2),
            engineLoad:         _d(3),
            coolantTemp:        _i(4),
            intakeTemp:         _i(5),
            mafVoltage:         mafVolt,
            mafGps:             mafGps,
            throttlePos:        _d(7),
            ignitionTiming:     _d(8),
            shortFuelTrim:      _d(9),
            longFuelTrim:       _d(10),
            o2Voltage:          _d(11),
            afr:                _d(12, 14.7),
            vtcTargetAngle:     _d(13),
            vtcActualAngle:     _d(14),
            knockRetard:        _d(15),
            knockCount:         _i(16),
            actualIgnition:     _d(17),
            injectorDuty:       _d(18),
            injectorPulseWidth: _d(18) / 100 * 20,
            requestedTorque:    _d(19),
            actualTorque:       _d(20),
            oilTemp:            _d(21),
            afrTarget:          _d(22, 14.7),
            lambda:             _d(23, 1.0),
            manifoldPressure:   _d(24),
            acceleratorPedal:   _d(25),
            throttleActual:     _d(26),
            batteryVoltage:     _d(29),
            tripFuelL:          _d(33),
          ));
        }
      } catch (_) {
        continue;
      }
    }
    return result;
  }

  /// Дублирует таблицу из AppConstants (для парсинга старых V5 логов)
  double _mafVoltToGps(double v) {
    // Упрощённая аппроксимация Hitachi для QR20DE
    const table = [
      [0.5, 0.0], [0.7, 0.8], [0.9, 2.0], [1.0, 2.8],
      [1.2, 5.2], [1.5, 11.5], [1.8, 21.7], [2.0, 31.3],
      [2.3, 51.0], [2.5, 68.3], [3.0, 129.0], [3.5, 216.5],
      [4.0, 320.0], [5.0, 570.0],
    ];
    if (v <= table.first[0]) return 0;
    if (v >= table.last[0])  return table.last[1];
    for (int i = 0; i < table.length - 1; i++) {
      if (v >= table[i][0] && v <= table[i + 1][0]) {
        final ratio = (v - table[i][0]) / (table[i + 1][0] - table[i][0]);
        return table[i][1] + ratio * (table[i + 1][1] - table[i][1]);
      }
    }
    return 0;
  }

  // ─────────────────────────────────────────────────────────────
  // MERGE LOGS
  // ─────────────────────────────────────────────────────────────
  Future<List<OBDData>> mergeLogs(List<List<OBDData>> logs) async {
    final all = <OBDData>[];
    for (final log in logs) all.addAll(log);
    all.sort((a, b) => a.timestamp.compareTo(b.timestamp));
    final deduped = <OBDData>[];
    for (final d in all) {
      if (deduped.isEmpty) {
        deduped.add(d);
        continue;
      }
      final diff = d.timestamp
          .difference(deduped.last.timestamp)
          .inMilliseconds
          .abs();
      if (diff > 200) deduped.add(d);
    }
    return deduped;
  }

  // ─────────────────────────────────────────────────────────────
  // ВСПОМОГАТЕЛЬНОЕ
  // ─────────────────────────────────────────────────────────────
  Map<String, List<OBDData>> _group(List<OBDData> data, TuningMap map) {
    final result = <String, List<OBDData>>{};
    for (final d in data) {
      final ri = _closest(map.rpmAxis,  d.rpm.toDouble());
      final li = _closest(map.loadAxis, d.engineLoad);
      result.putIfAbsent('$ri,$li', () => []).add(d);
    }
    return result;
  }

  int _closest(List<double> axis, double v) {
    int idx = 0;
    double best = double.infinity;
    for (int i = 0; i < axis.length; i++) {
      final d = (axis[i] - v).abs();
      if (d < best) {
        best = d;
        idx = i;
      }
    }
    return idx;
  }

  double _avg(Iterable<num> vals) {
    if (vals.isEmpty) return 0;
    return vals.reduce((a, b) => a + b) / vals.length;
  }

  AnalysisResult _empty(String name, int total, String msg) =>
      AnalysisResult(
        mapName: name,
        analyzedAt: DateTime.now(),
        totalSamples: total,
        changes: const [],
        summary: msg,
      );

  String _summary(String type, List<MapCell> ch, int total, int cells) {
    if (ch.isEmpty) {
      return '$type оптимальна\nДанных: $total | Клеток: $cells';
    }
    final inc = ch.where((c) => c.delta > 0).length;
    final dec = ch.where((c) => c.delta < 0).length;
    final avg = ch.map((c) => c.delta.abs()).reduce((a, b) => a + b) /
        ch.length;
    final maxD =
        ch.map((c) => c.delta.abs()).reduce((a, b) => a > b ? a : b);
    final trend = inc > dec * 2
        ? '↑ увеличение'
        : dec > inc * 2
            ? '↓ уменьшение'
            : '↕ смешанное';
    return 'Данных: $total | Клеток: $cells\n'
           'Правок: ${ch.length} (+$inc -$dec) $trend\n'
           'Среднее: ${avg.toStringAsFixed(2)} | Макс: ${maxD.toStringAsFixed(2)}';
  }
}
''')
print("✅ analyzer_service.dart переписан ЦЕЛИКОМ, чисто и без regex")



✅ analyzer_service.dart переписан ЦЕЛИКОМ, чисто и без regex


In [ ]:
# @title 🔍 ФИКС: Кастомные PID на дашборде + аудит библиотеки
import os
os.chdir('/content/nissan_logger_v6')

# ═══════════════════════════════════════════════════════════════
# ЧАСТЬ 1: Полная библиотека PID (все что были в V5 + дополнительные)
# ═══════════════════════════════════════════════════════════════
with open('lib/services/nissan_pid_library.dart', 'w') as f:
    f.write(r'''class NissanPidDef {
  final String id;
  final String cmd;
  final String answer;
  final String name;
  final String desc;
  final String unit;
  final int    bytesCount;
  final double Function(List<int>) formula;
  final double minVal;
  final double maxVal;
  final int    priority;
  final String category;

  const NissanPidDef({
    required this.id,
    required this.cmd,
    required this.answer,
    required this.name,
    required this.desc,
    required this.unit,
    required this.bytesCount,
    required this.formula,
    this.minVal   = 0,
    this.maxVal   = 255,
    this.priority = 3,
    this.category = 'other',
  });
}

class NissanPidLibrary {
  static final List<NissanPidDef> all = [
    // ═══ ПРИОРИТЕТ 1: КРИТИЧЕСКИ ВАЖНЫЕ (каждый цикл) ═══════════

    NissanPidDef(id:'RPM', cmd:'2212010401', answer:'621201',
      name:'RPM', desc:'Обороты', unit:'RPM', bytesCount:2,
      priority:1, category:'engine', minVal:0, maxVal:8000,
      formula: (b) => (b[0]*256+b[1])*12.5),

    NissanPidDef(id:'TIMING', cmd:'22110A0401', answer:'62110A',
      name:'TIMING', desc:'УОЗ факт', unit:'°BTDC', bytesCount:1,
      priority:1, category:'ignition', minVal:-20, maxVal:60,
      formula: (b) => (110-b[0]).toDouble()),

    NissanPidDef(id:'KNOCK', cmd:'22112D0401', answer:'62112D',
      name:'KNOCK', desc:'Корр.УОЗ (детонация)', unit:'°', bytesCount:1,
      priority:1, category:'ignition', minVal:-30, maxVal:30,
      formula: (b) { int v=b[0]; if(v>=128) v-=256; return v.toDouble(); }),

    NissanPidDef(id:'TPS', cmd:'22111E0401', answer:'62111E',
      name:'TPS', desc:'Дроссель %', unit:'%', bytesCount:1,
      priority:1, category:'throttle', minVal:0, maxVal:100,
      formula: (b) => b[0]*0.35),

    NissanPidDef(id:'MAF_V', cmd:'2212040401', answer:'621204',
      name:'MAF_V', desc:'MAF вольтаж', unit:'V', bytesCount:2,
      priority:1, category:'air', minVal:0, maxVal:5,
      formula: (b) => (b[0]*256+b[1])*0.005),

    NissanPidDef(id:'MAF_GS', cmd:'2212090401', answer:'621209',
      name:'MAF_GS', desc:'MAF g/s (если ЭБУ даёт)', unit:'g/s', bytesCount:2,
      priority:1, category:'air', minVal:0, maxVal:500,
      formula: (b) => (b[0]*256+b[1])*0.01),

    NissanPidDef(id:'ECT', cmd:'2211010401', answer:'621101',
      name:'ECT', desc:'Темп. ОЖ', unit:'°C', bytesCount:1,
      priority:1, category:'temp', minVal:-30, maxVal:130,
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'LOAD', cmd:'2211170401', answer:'621117',
      name:'LOAD', desc:'Нагрузка двигателя', unit:'%', bytesCount:1,
      priority:1, category:'engine', minVal:0, maxVal:100,
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'SPEED', cmd:'2211020401', answer:'621102',
      name:'SPEED', desc:'Скорость', unit:'км/ч', bytesCount:1,
      priority:1, category:'engine', minVal:0, maxVal:200,
      formula: (b) => b[0]*2.0),

    NissanPidDef(id:'VTC_ACT', cmd:'2211350401', answer:'621135',
      name:'VTC_ACT', desc:'VTC факт B1', unit:'°CA', bytesCount:1,
      priority:1, category:'vtc', minVal:-10, maxVal:50,
      formula: (b) => b[0]*0.5-64),

    NissanPidDef(id:'STFT', cmd:'2211230401', answer:'621123',
      name:'STFT', desc:'STFT B1 (краткая коррекция)', unit:'%', bytesCount:1,
      priority:1, category:'fuel', minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'LTFT', cmd:'2211250401', answer:'621125',
      name:'LTFT', desc:'LTFT B1 (долгая коррекция)', unit:'%', bytesCount:1,
      priority:1, category:'fuel', minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'INJ_B1', cmd:'2212060401', answer:'621206',
      name:'INJ_B1', desc:'Впрыск B1', unit:'ms', bytesCount:2,
      priority:1, category:'fuel', minVal:0, maxVal:30,
      formula: (b) => (b[0]*256+b[1])*0.01),

    NissanPidDef(id:'O2_B1S1', cmd:'2211180401', answer:'621118',
      name:'O2_B1S1', desc:'O2 датчик B1S1', unit:'V', bytesCount:1,
      priority:1, category:'fuel', minVal:0, maxVal:1,
      formula: (b) => b[0]*0.01),

    // ═══ ПРИОРИТЕТ 2: ВАЖНЫЕ (каждый 3-й цикл) ═════════════════

    NissanPidDef(id:'PEDAL', cmd:'22110E0401', answer:'62110E',
      name:'PEDAL', desc:'Педаль газа', unit:'%', bytesCount:1,
      priority:2, category:'throttle',
      formula: (b) => b[0]*0.5),

    NissanPidDef(id:'BATT', cmd:'2211030401', answer:'621103',
      name:'BATT', desc:'Напряжение АКБ', unit:'V', bytesCount:1,
      priority:2, category:'electric', minVal:8, maxVal:16,
      formula: (b) => b[0]*0.08),

    NissanPidDef(id:'IAT', cmd:'2211060401', answer:'621106',
      name:'IAT', desc:'Темп. впуска', unit:'°C', bytesCount:1,
      priority:2, category:'temp', minVal:-30, maxVal:100,
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'MAP_V', cmd:'22112A0401', answer:'62112A',
      name:'MAP_V', desc:'MAP датчик', unit:'V', bytesCount:1,
      priority:2, category:'air',
      formula: (b) => b[0]*0.02),

    NissanPidDef(id:'IACV', cmd:'22110B0401', answer:'62110B',
      name:'IACV', desc:'Клапан ХХ (IACV)', unit:'%', bytesCount:1,
      priority:2, category:'idle',
      formula: (b) => b[0]*0.5),

    NissanPidDef(id:'IDLE_BASE', cmd:'22110D0401', answer:'62110D',
      name:'IDLE_BASE', desc:'Базовые ХХ', unit:'RPM', bytesCount:1,
      priority:2, category:'idle', maxVal:3200,
      formula: (b) => b[0]*12.5),

    NissanPidDef(id:'INJ_BASE', cmd:'2212080401', answer:'621208',
      name:'INJ_BASE', desc:'Впрыск базовый', unit:'ms', bytesCount:2,
      priority:2, category:'fuel',
      formula: (b) => (b[0]*256+b[1])/2048.0),

    NissanPidDef(id:'VTC_SOL', cmd:'2211380401', answer:'621138',
      name:'VTC_SOL', desc:'VTC соленоид B1', unit:'%', bytesCount:1,
      priority:2, category:'vtc',
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'VTC_DUTY', cmd:'22122D0401', answer:'62122D',
      name:'VTC_DUTY', desc:'VTC Duty B1', unit:'%', bytesCount:2,
      priority:2, category:'vtc',
      formula: (b) => (b[0]*256+b[1])*3200.0/32768.0),

    NissanPidDef(id:'STFT_B2', cmd:'2211240401', answer:'621124',
      name:'STFT_B2', desc:'STFT B2', unit:'%', bytesCount:1,
      priority:2, category:'fuel', minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'LTFT_B2', cmd:'2211260401', answer:'621126',
      name:'LTFT_B2', desc:'LTFT B2', unit:'%', bytesCount:1,
      priority:2, category:'fuel', minVal:-100, maxVal:100,
      formula: (b) => (b[0]-100).toDouble()),

    NissanPidDef(id:'POWER_KW', cmd:'2212570401', answer:'621257',
      name:'POWER_KW', desc:'Запрошенная мощность', unit:'kW', bytesCount:2,
      priority:2, category:'engine',
      formula: (b) => (b[0]*256+b[1])*0.03125),

    NissanPidDef(id:'TORQUE', cmd:'2212280401', answer:'621228',
      name:'TORQUE', desc:'Момент двигателя', unit:'Nm', bytesCount:2,
      priority:2, category:'engine',
      formula: (b) {
        int v=b[0]*256+b[1];
        if(v>=32768) v-=65536;
        return v/4.0;
      }),

    // ═══ ПРИОРИТЕТ 3: ПОЛЕЗНЫЕ (каждый 10-й цикл) ══════════════

    NissanPidDef(id:'OIL_T', cmd:'22111F0401', answer:'62111F',
      name:'OIL_T', desc:'Темп. масла', unit:'°C', bytesCount:1,
      priority:3, category:'temp',
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'FUEL_T', cmd:'2211040401', answer:'621104',
      name:'FUEL_T', desc:'Темп. топлива', unit:'°C', bytesCount:1,
      priority:3, category:'temp',
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'RAD_T', cmd:'22114B0401', answer:'62114B',
      name:'RAD_T', desc:'Темп. радиатора', unit:'°C', bytesCount:1,
      priority:3, category:'temp',
      formula: (b) => (b[0]-50).toDouble()),

    NissanPidDef(id:'AMBIENT', cmd:'22130A0401', answer:'62130A',
      name:'AMBIENT', desc:'Темп. воздуха', unit:'°C', bytesCount:2,
      priority:3, category:'temp', minVal:-40, maxVal:60,
      formula: (b) => (b[0]*256+b[1])*0.1-40),

    NissanPidDef(id:'CAT_T', cmd:'2213020401', answer:'621302',
      name:'CAT_T', desc:'Темп. катализатора', unit:'°C', bytesCount:2,
      priority:3, category:'temp', minVal:0, maxVal:1000,
      formula: (b) => (b[0]*256+b[1])*0.1-40),

    NissanPidDef(id:'O2_B2S1', cmd:'2211190401', answer:'621119',
      name:'O2_B2S1', desc:'O2 B2S1', unit:'V', bytesCount:1,
      priority:3, category:'fuel',
      formula: (b) => b[0]*0.01),

    NissanPidDef(id:'O2_B1S2', cmd:'22111A0401', answer:'62111A',
      name:'O2_B1S2', desc:'O2 B1S2 (после кат.)', unit:'V', bytesCount:1,
      priority:3, category:'fuel',
      formula: (b) => b[0]*0.01),

    NissanPidDef(id:'O2_B2S2', cmd:'22111B0401', answer:'62111B',
      name:'O2_B2S2', desc:'O2 B2S2', unit:'V', bytesCount:1,
      priority:3, category:'fuel',
      formula: (b) => b[0]*0.01),

    NissanPidDef(id:'AF_B1S1', cmd:'2212250401', answer:'621225',
      name:'AF_B1S1', desc:'A/F датчик B1S1', unit:'V', bytesCount:2,
      priority:3, category:'fuel',
      formula: (b) => (b[0]*256+b[1])*0.005),

    NissanPidDef(id:'FUEL_LVL', cmd:'2211140401', answer:'621114',
      name:'FUEL_LVL', desc:'Уровень топлива', unit:'V', bytesCount:1,
      priority:3, category:'fuel',
      formula: (b) => b[0]*0.04),

    NissanPidDef(id:'BARO', cmd:'2211290401', answer:'621129',
      name:'BARO', desc:'Атм. давление', unit:'V', bytesCount:1,
      priority:3, category:'air',
      formula: (b) => b[0]*0.02),

    NissanPidDef(id:'ALT_SPD', cmd:'2211900401', answer:'621190',
      name:'ALT_SPD', desc:'Обороты генератора', unit:'RPM', bytesCount:1,
      priority:3, category:'electric',
      formula: (b) => b[0]*0.75),

    NissanPidDef(id:'BAT_SOC', cmd:'2211510401', answer:'621151',
      name:'BAT_SOC', desc:'Заряд АКБ SOC', unit:'%', bytesCount:1,
      priority:3, category:'electric',
      formula: (b) => b[0].toDouble()),

    NissanPidDef(id:'FAN', cmd:'2211470401', answer:'621147',
      name:'FAN', desc:'Вентилятор охлаждения', unit:'%', bytesCount:1,
      priority:3, category:'other',
      formula: (b) => b[0]*100.0/256.0),

    NissanPidDef(id:'MISFIRE1', cmd:'2212220401', answer:'621222',
      name:'MISFIRE1', desc:'Пропуски цилиндр 1', unit:'cnt', bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),

    NissanPidDef(id:'MISFIRE2', cmd:'2212230401', answer:'621223',
      name:'MISFIRE2', desc:'Пропуски цилиндр 2', unit:'cnt', bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),

    NissanPidDef(id:'MISFIRE3', cmd:'2212240401', answer:'621224',
      name:'MISFIRE3', desc:'Пропуски цилиндр 3', unit:'cnt', bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),

    NissanPidDef(id:'MISFIRE4', cmd:'2212260401', answer:'621226',
      name:'MISFIRE4', desc:'Пропуски цилиндр 4', unit:'cnt', bytesCount:2,
      priority:3, category:'ignition',
      formula: (b) => (b[0]*256+b[1]).toDouble()),

    NissanPidDef(id:'PEDAL_V2', cmd:'22120D0401', answer:'62120D',
      name:'PEDAL_V2', desc:'Педаль S1 вольтаж', unit:'V', bytesCount:2,
      priority:3, category:'throttle',
      formula: (b) => (b[0]*256+b[1])*0.005),
  ];

  static List<NissanPidDef> byPriority(int p) =>
      all.where((x) => x.priority == p).toList();

  static NissanPidDef? byId(String id) {
    try { return all.firstWhere((p) => p.id == id); }
    catch (_) { return null; }
  }

  static List<NissanPidDef> byCategory(String c) =>
      all.where((x) => x.category == c).toList();

  static List<String> get categories =>
      all.map((p) => p.category).toSet().toList()..sort();
}
''')
print("✅ nissan_pid_library.dart — полная библиотека (48 PID)")

# ═══════════════════════════════════════════════════════════════
# ЧАСТЬ 2: Dashboard — читает ВСЕ пиды (стандартные + кастомные)
# ═══════════════════════════════════════════════════════════════
with open('lib/screens/dashboard_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import '../models/obd_data.dart';
import '../models/alert.dart';
import '../models/vehicle_profile.dart';
import '../models/custom_pid.dart';
import '../services/obd_service.dart';
import '../services/alert_service.dart';
import '../services/profile_service.dart';
import '../services/nissan_pid_library.dart';
import '../widgets/fps_indicator.dart';

/// Универсальное описание параметра для дашборда.
/// Работает и с полями OBDData (стандартные), и с PID из библиотеки/профиля.
class _P {
  final String id, label, unit;
  final double Function(OBDData data, Map<String, double> pidValues) get;
  final int digits;
  final Color color;
  final String source; // 'obd', 'pid', 'custom'
  const _P({
    required this.id,
    required this.label,
    required this.unit,
    required this.get,
    required this.digits,
    required this.color,
    this.source = 'obd',
  });
}

/// Собирает список ВСЕХ доступных параметров:
/// 1. Стандартные (RPM, скорость и т.д. — из OBDData)
/// 2. Все PID из библиотеки (по имени в pidValues)
/// 3. Кастомные PID из профиля
List<_P> _buildAllParams(VehicleProfile profile) {
  final result = <_P>[
    // ── Стандартные из OBDData ─────────────────────────
    _P(id: 'rpm', label: 'RPM', unit: '',
      get: (d, _) => d.rpm.toDouble(),
      digits: 0, color: Colors.blue),
    _P(id: 'speed', label: 'Скорость', unit: 'км/ч',
      get: (d, _) => d.speed.toDouble(),
      digits: 0, color: Colors.cyan),
    _P(id: 'timing', label: 'Зажигание', unit: '°',
      get: (d, _) => d.actualIgnition, digits: 1, color: Colors.green),
    _P(id: 'knock', label: 'Knock', unit: '°',
      get: (d, _) => d.knockRetard, digits: 1, color: Colors.red),
    _P(id: 'vtc', label: 'VTC', unit: '°',
      get: (d, _) => d.vtcActualAngle, digits: 1, color: Colors.cyan),
    _P(id: 'load', label: 'Нагрузка', unit: '%',
      get: (d, _) => d.engineLoad, digits: 0, color: Colors.orange),
    _P(id: 'throttle', label: 'Дроссель', unit: '%',
      get: (d, _) => d.throttlePos, digits: 0, color: Colors.green),
    _P(id: 'maf_v', label: 'MAF V', unit: 'V',
      get: (d, _) => d.mafVoltage, digits: 3, color: Colors.purple),
    _P(id: 'maf_gps', label: 'MAF', unit: 'g/s',
      get: (d, _) => d.mafGps, digits: 2, color: Colors.purple),
    _P(id: 'afr', label: 'AFR', unit: '',
      get: (d, _) => d.afr, digits: 2, color: Colors.teal),
    _P(id: 'ect', label: 'ОЖ', unit: '°C',
      get: (d, _) => d.coolantTemp.toDouble(),
      digits: 0, color: Colors.red),
    _P(id: 'iat', label: 'Впуск', unit: '°C',
      get: (d, _) => d.intakeTemp.toDouble(),
      digits: 0, color: Colors.cyan),
    _P(id: 'batt', label: 'Батарея', unit: 'V',
      get: (d, _) => d.batteryVoltage, digits: 2, color: Colors.yellow),
    _P(id: 'inj', label: 'Форсунки', unit: 'ms',
      get: (d, _) => d.injectorPulseWidth, digits: 2, color: Colors.amber),
    _P(id: 'stft', label: 'STFT', unit: '%',
      get: (d, _) => d.shortFuelTrim, digits: 1, color: Colors.lime),
    _P(id: 'ltft', label: 'LTFT', unit: '%',
      get: (d, _) => d.longFuelTrim, digits: 1, color: Colors.teal),
    _P(id: 'o2', label: 'O2', unit: 'V',
      get: (d, _) => d.o2Voltage, digits: 3, color: Colors.indigo),
    _P(id: 'hp', label: 'Мощность', unit: 'л.с.',
      get: (d, _) => d.calculatedHP, digits: 1, color: Colors.yellow),
    _P(id: 'torque', label: 'Момент', unit: 'Нм',
      get: (d, _) => d.calculatedTorqueNm, digits: 0, color: Colors.orange),
    _P(id: 've', label: 'VE', unit: '%',
      get: (d, _) => d.volumetricEfficiency,
      digits: 0, color: Colors.lightBlue),
    _P(id: 'fuel_lh', label: 'Расход', unit: 'L/ч',
      get: (d, _) => d.fuelFlowLph, digits: 2, color: Colors.pink),
    _P(id: 'fuel_100', label: 'L/100км', unit: '',
      get: (d, _) => d.fuelL100km, digits: 1, color: Colors.pinkAccent),
    _P(id: 'pedal', label: 'Педаль', unit: '%',
      get: (d, _) => d.acceleratorPedal, digits: 0, color: Colors.green),
    _P(id: 'injduty', label: 'Впрыск%', unit: '%',
      get: (d, _) => d.injectorDuty, digits: 0, color: Colors.deepOrange),
    _P(id: 'trip', label: 'Поездка', unit: 'L',
      get: (d, _) => d.tripFuelL, digits: 3, color: Colors.orange),
    _P(id: 'actual_torque', label: 'Момент ЭБУ', unit: 'Нм',
      get: (d, _) => d.actualTorque, digits: 0, color: Colors.deepOrange),
  ];

  // ── ВСЕ PID из библиотеки (доступны когда ЭБУ подключен) ──
  final defaultColors = [
    Colors.tealAccent, Colors.amberAccent, Colors.lightGreenAccent,
    Colors.deepPurpleAccent, Colors.pinkAccent, Colors.cyanAccent,
  ];
  int ci = 0;
  final deletedSet = profile.deletedPidIds.toSet();
  final modifiedIds = profile.customPids
      .where((p) => p.status == PidStatus.defaultModified)
      .map((p) => p.originalId)
      .whereType<String>()
      .toSet();

  for (final pid in NissanPidLibrary.all) {
    if (deletedSet.contains(pid.id)) continue;
    if (modifiedIds.contains(pid.id)) continue; // изменённые будут ниже как кастомные

    // Не дублируем те что уже как OBDData поля
    final builtInIds = {
      'RPM','SPEED','TIMING','KNOCK','TPS','MAF_V','MAF_GS','ECT','LOAD',
      'VTC_ACT','STFT','LTFT','INJ_B1','O2_B1S1','PEDAL','BATT','IAT',
      'MAP_V','TORQUE',
    };
    if (builtInIds.contains(pid.id)) continue;

    result.add(_P(
      id: 'pid_${pid.id}',
      label: pid.name,
      unit: pid.unit,
      get: (_, values) => values[pid.id] ?? 0,
      digits: pid.unit == 'V' || pid.unit == 'g/s' ? 2 : 1,
      color: defaultColors[ci++ % defaultColors.length],
      source: 'pid',
    ));
  }

  // ── Кастомные / изменённые PID из профиля ──
  for (final custom in profile.customPids) {
    if (custom.status == PidStatus.defaultDeleted) continue;
    result.add(_P(
      id: 'custom_${custom.id}',
      label: custom.name,
      unit: custom.unit,
      get: (_, values) => values[custom.id] ?? 0,
      digits: custom.unit == 'V' ? 3 : 1,
      color: Colors.orangeAccent,
      source: 'custom',
    ));
  }

  return result;
}

class DashboardScreen extends StatefulWidget {
  final OBDService obdService;
  final AlertService alertService;
  final ProfileService profileService;
  const DashboardScreen({
    super.key,
    required this.obdService,
    required this.alertService,
    required this.profileService,
  });
  @override
  State<DashboardScreen> createState() => _DashboardScreenState();
}

class _DashboardScreenState extends State<DashboardScreen> {
  OBDData _data = OBDData(timestamp: DateTime.now());
  Map<String, double> _pidValues = {};
  List<Alert> _alerts = [];
  List<String> _layout = [];
  List<_P> _allParams = [];
  StreamSubscription? _sub;
  Timer? _refreshTimer;

  @override
  void initState() {
    super.initState();
    _reload();
    _sub = widget.obdService.dataStream.listen((data) {
      if (mounted) setState(() {
        _data = data;
        _pidValues = Map.from(widget.obdService.pidValues);
        _alerts = widget.alertService.recentAlerts.take(3).toList();
      });
    });
    // Обновляем каждые 500мс даже если данных нет (для offline pid значений)
    _refreshTimer = Timer.periodic(const Duration(milliseconds: 500), (_) {
      if (mounted && widget.obdService.pidValues.isNotEmpty) {
        setState(() {
          _pidValues = Map.from(widget.obdService.pidValues);
        });
      }
    });
  }

  @override
  void dispose() {
    _sub?.cancel();
    _refreshTimer?.cancel();
    super.dispose();
  }

  void _reload() {
    final profile = widget.profileService.getActiveOrDefault();
    _allParams = _buildAllParams(profile);
    _layout = List<String>.from(profile.dashboardLayout);
    if (_layout.length < 6) {
      _layout = ['timing','knock','vtc','load','throttle','maf_gps',
                  'afr','ect','iat','batt','inj','fuel_lh'];
    }
    setState(() {});
  }

  Future<void> _saveLayout() async {
    final profile = widget.profileService.getActiveOrDefault();
    await widget.profileService.update(
      profile.copyWith(dashboardLayout: _layout));
  }

  _P _getParam(String id) {
    try { return _allParams.firstWhere((p) => p.id == id); }
    catch (_) { return _allParams.first; }
  }

  void _pickParam(int index) {
    // Группируем параметры по source для удобства
    final builtIn = _allParams.where((p) => p.source == 'obd').toList();
    final pids = _allParams.where((p) => p.source == 'pid').toList();
    final custom = _allParams.where((p) => p.source == 'custom').toList();

    showModalBottomSheet(
      context: context,
      backgroundColor: const Color(0xFF16213E),
      isScrollControlled: true,
      builder: (c) => DraggableScrollableSheet(
        expand: false,
        initialChildSize: 0.7,
        maxChildSize: 0.9,
        builder: (_, scrollController) => Container(
          padding: const EdgeInsets.all(12),
          child: Column(children: [
            Row(children: [
              const Icon(Icons.tune, color: Colors.cyan),
              const SizedBox(width: 8),
              const Text('Выбери параметр',
                style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
              const Spacer(),
              Text('${_allParams.length} доступно',
                style: const TextStyle(color: Colors.white54, fontSize: 11)),
            ]),
            const SizedBox(height: 8),
            Expanded(child: ListView(
              controller: scrollController,
              children: [
                if (builtIn.isNotEmpty) ...[
                  _sectionHeader('Стандартные', Colors.cyan),
                  _paramGrid(builtIn, index, c),
                ],
                if (pids.isNotEmpty) ...[
                  const SizedBox(height: 12),
                  _sectionHeader('PID библиотеки', Colors.tealAccent),
                  _paramGrid(pids, index, c),
                ],
                if (custom.isNotEmpty) ...[
                  const SizedBox(height: 12),
                  _sectionHeader('Кастомные / изменённые', Colors.orange),
                  _paramGrid(custom, index, c),
                ],
              ],
            )),
          ]),
        ),
      ),
    );
  }

  Widget _sectionHeader(String title, Color color) => Padding(
    padding: const EdgeInsets.symmetric(vertical: 4),
    child: Row(children: [
      Container(width: 4, height: 16, color: color),
      const SizedBox(width: 6),
      Text(title.toUpperCase(),
        style: TextStyle(color: color, fontSize: 11,
          fontWeight: FontWeight.bold)),
    ]),
  );

  Widget _paramGrid(List<_P> params, int index, BuildContext ctx) {
    return GridView.builder(
      shrinkWrap: true,
      physics: const NeverScrollableScrollPhysics(),
      gridDelegate: const SliverGridDelegateWithFixedCrossAxisCount(
        crossAxisCount: 3, childAspectRatio: 2.2,
        crossAxisSpacing: 6, mainAxisSpacing: 6),
      itemCount: params.length,
      itemBuilder: (_, i) {
        final p = params[i];
        final sel = _layout.contains(p.id);
        final val = p.get(_data, _pidValues);
        return GestureDetector(
          onTap: () {
            setState(() => _layout[index] = p.id);
            _saveLayout();
            Navigator.pop(ctx);
          },
          child: Container(
            decoration: BoxDecoration(
              color: sel ? p.color.withOpacity(0.3) : const Color(0xFF0F3460),
              borderRadius: BorderRadius.circular(6),
              border: Border.all(color: p.color.withOpacity(0.5))),
            child: Center(child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                Text(p.label, style: TextStyle(
                  color: p.color, fontSize: 10, fontWeight: FontWeight.bold),
                  textAlign: TextAlign.center, maxLines: 1,
                  overflow: TextOverflow.ellipsis),
                Text('${val.toStringAsFixed(p.digits)} ${p.unit}',
                  style: TextStyle(
                    color: p.color.withOpacity(0.8), fontSize: 9)),
              ],
            )),
          ),
        );
      },
    );
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(
        title: const Text('Приборная панель'),
        backgroundColor: const Color(0xFF16213E),
        actions: [
          IconButton(
            icon: const Icon(Icons.refresh),
            tooltip: 'Обновить список PID',
            onPressed: _reload),
          FpsIndicator(obdService: widget.obdService),
        ],
      ),
      body: SingleChildScrollView(
        padding: const EdgeInsets.all(8),
        child: Column(crossAxisAlignment: CrossAxisAlignment.stretch, children: [
          if (_alerts.isNotEmpty)
            Card(
              color: _alerts.first.level == AlertLevel.danger
                ? Colors.red.withOpacity(0.3)
                : Colors.orange.withOpacity(0.3),
              child: Padding(
                padding: const EdgeInsets.all(8),
                child: Column(
                  crossAxisAlignment: CrossAxisAlignment.start,
                  children: _alerts.map((a) => Text(a.message,
                    style: const TextStyle(
                      fontWeight: FontWeight.bold, fontSize: 11,
                      color: Colors.white))).toList(),
                ),
              ),
            ),
          Row(children: [
            Expanded(child: _bigGauge('RPM', _data.rpm.toString(),
              _data.rpm > 6500 ? Colors.red
                : _data.rpm > 5500 ? Colors.orange : Colors.green)),
            const SizedBox(width: 6),
            Expanded(child: _bigGauge('KM/H', _data.speed.toString(), Colors.blue)),
          ]),
          const SizedBox(height: 6),
          GridView.builder(
            shrinkWrap: true,
            physics: const NeverScrollableScrollPhysics(),
            gridDelegate: const SliverGridDelegateWithFixedCrossAxisCount(
              crossAxisCount: 3, childAspectRatio: 1.8,
              crossAxisSpacing: 4, mainAxisSpacing: 4),
            itemCount: _layout.length,
            itemBuilder: (_, i) {
              final p = _getParam(_layout[i]);
              final val = p.get(_data, _pidValues);
              return GestureDetector(
                onLongPress: () => _pickParam(i),
                child: _paramCard(
                  p.label,
                  val.toStringAsFixed(p.digits),
                  p.unit,
                  p.color,
                  isCustom: p.source == 'custom',
                  isPid: p.source == 'pid'),
              );
            },
          ),
          const SizedBox(height: 6),
          Card(
            color: const Color(0xFF16213E),
            child: Padding(
              padding: const EdgeInsets.all(10),
              child: Row(children: [
                _bigStat('HP', _data.calculatedHP.toStringAsFixed(1), Colors.yellow),
                _bigStat('Нм', _data.calculatedTorqueNm.toStringAsFixed(0), Colors.orange),
                _bigStat('VE%', _data.volumetricEfficiency.toStringAsFixed(0), Colors.lightBlue),
              ]),
            ),
          ),
          const SizedBox(height: 6),
          Card(
            color: const Color(0xFF16213E),
            child: Padding(
              padding: const EdgeInsets.all(10),
              child: Column(children: [
                const Text('ТОПЛИВНЫЕ КОРРЕКЦИИ',
                  style: TextStyle(color: Colors.white70, fontSize: 11)),
                const SizedBox(height: 6),
                Row(children: [
                  Expanded(child: _trim('STFT', _data.shortFuelTrim)),
                  Expanded(child: _trim('LTFT', _data.longFuelTrim)),
                ]),
              ]),
            ),
          ),
          const SizedBox(height: 6),
          Card(
            color: const Color(0xFF16213E),
            child: Padding(
              padding: const EdgeInsets.all(10),
              child: Column(children: [
                const Text('РЕЖИМ РАБОТЫ',
                  style: TextStyle(color: Colors.white70, fontSize: 11)),
                Text(_data.engineMode, style: const TextStyle(
                  fontSize: 16, color: Colors.cyan, fontWeight: FontWeight.bold)),
              ]),
            ),
          ),
          const SizedBox(height: 4),
          const Center(child: Text(
            'Удерживай ячейку — выбор из ${''}любых PID/кастомных',
            style: TextStyle(color: Colors.white30, fontSize: 10))),
          const SizedBox(height: 8),
        ]),
      ),
    );
  }

  Widget _bigGauge(String label, String value, Color color) => Card(
    color: const Color(0xFF16213E),
    child: Padding(
      padding: const EdgeInsets.all(12),
      child: Column(children: [
        Text(label, style: const TextStyle(color: Colors.white70, fontSize: 12)),
        const SizedBox(height: 4),
        FittedBox(child: Text(value, style: TextStyle(
          color: color, fontSize: 38, fontWeight: FontWeight.bold))),
      ]),
    ),
  );

  Widget _paramCard(
    String label, String value, String unit, Color color,
    {bool isCustom = false, bool isPid = false}
  ) => Card(
    color: const Color(0xFF16213E),
    shape: RoundedRectangleBorder(
      borderRadius: BorderRadius.circular(8),
      side: BorderSide(
        color: isCustom
          ? Colors.orange.withOpacity(0.6)
          : isPid
            ? Colors.tealAccent.withOpacity(0.4)
            : color.withOpacity(0.2),
        width: (isCustom || isPid) ? 1.5 : 1)),
    child: Stack(children: [
      Padding(
        padding: const EdgeInsets.all(4),
        child: Column(mainAxisAlignment: MainAxisAlignment.center, children: [
          Text(label, style: const TextStyle(color: Colors.white54, fontSize: 9),
            textAlign: TextAlign.center, maxLines: 1, overflow: TextOverflow.ellipsis),
          FittedBox(child: Row(
            mainAxisSize: MainAxisSize.min,
            crossAxisAlignment: CrossAxisAlignment.baseline,
            textBaseline: TextBaseline.alphabetic,
            children: [
              Text(value, style: TextStyle(
                color: color, fontSize: 15, fontWeight: FontWeight.bold)),
              if (unit.isNotEmpty)
                Text(' $unit', style: TextStyle(
                  color: color.withOpacity(0.6), fontSize: 9)),
            ],
          )),
        ]),
      ),
      // Индикатор источника (маленькая точка в углу)
      if (isCustom || isPid)
        Positioned(
          top: 2, right: 2,
          child: Container(
            width: 6, height: 6,
            decoration: BoxDecoration(
              color: isCustom ? Colors.orange : Colors.tealAccent,
              shape: BoxShape.circle))),
    ]),
  );

  Widget _bigStat(String label, String value, Color color) => Expanded(
    child: Column(children: [
      Text(label, style: const TextStyle(color: Colors.white70, fontSize: 11)),
      Text(value, style: TextStyle(
        fontSize: 20, color: color, fontWeight: FontWeight.bold)),
    ]),
  );

  Widget _trim(String label, double value) {
    final c = value.abs() > 15 ? Colors.red
            : value.abs() > 10 ? Colors.orange
            : Colors.green;
    return Column(children: [
      Text(label, style: const TextStyle(color: Colors.white70, fontSize: 11)),
      Text('${value.toStringAsFixed(1)}%', style: TextStyle(
        color: c, fontSize: 18, fontWeight: FontWeight.bold)),
    ]);
  }
}
''')
print("✅ dashboard_screen.dart — все PID + кастомные + автообновление")



✅ nissan_pid_library.dart — полная библиотека (48 PID)
✅ dashboard_screen.dart — все PID + кастомные + автообновление


In [ ]:
# @title 🔧 ФИКС: Сброс кеша при загрузке нового ROM + Подсказка в Анализаторе
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# 1. rom_holder.dart — при загрузке нового ROM сбрасываем старый кеш
# ================================================================
with open('lib/services/rom_holder.dart', 'w') as f:
    f.write(r'''import 'rom_map_reader.dart';
import 'map_storage_service.dart';

/// Глобальный держатель загруженной .bin прошивки.
class RomHolder {
  static final RomHolder instance = RomHolder._();
  RomHolder._();

  final RomMapReader reader = RomMapReader();

  bool get isLoaded => reader.isLoaded;
  String? get fileName => reader.fileName;

  /// Загрузка нового ROM с автоматической очисткой старых правок
  Future<bool> loadNewRom() async {
    final ok = await reader.pickAndLoad();
    if (ok) {
      // Сбрасываем старый кеш правок, чтобы метки "С правками"
      // не накладывались на новую прошивку
      await MapStorageService.resetAll();
    }
    return ok;
  }
}
''')
print("✅ rom_holder.dart — сброс кеша при загрузке нового ROM")

# ================================================================
# 2. analyzer_screen.dart — добавлено пояснение про старый/новый лог
# ================================================================
an_screen = 'lib/screens/analyzer_screen.dart'
with open(an_screen, 'r') as f:
    code = f.read()

# Меняем вызов pickAndLoad на loadNewRom
code = code.replace(
    "final loaded = await RomHolder.instance.reader.pickAndLoad();",
    "final loaded = await RomHolder.instance.loadNewRom();"
)

# Добавляем информационную карточку-совет в подвал результатов
old_summary_row = "Text(r.summary, style: const TextStyle(color: Colors.white70, fontSize: 10)),"
new_summary_row = '''Text(r.summary, style: const TextStyle(color: Colors.white70, fontSize: 10)),
        const SizedBox(height: 6),
        Container(
          padding: const EdgeInsets.all(6),
          decoration: BoxDecoration(
            color: Colors.amber.withOpacity(0.12),
            borderRadius: BorderRadius.circular(4),
            border: Border.all(color: Colors.amber.withOpacity(0.4)),
          ),
          child: const Row(children: [
            Icon(Icons.lightbulb_outline, color: Colors.amber, size: 16),
            SizedBox(width: 6),
            Expanded(
              child: Text(
                'Важно: Для проверки результатов прошивки запишите НОВЫЙ лог на машине с этой прошивкой!',
                style: TextStyle(color: Colors.amber, fontSize: 9.5),
              ),
            ),
          ]),
        ),'''

if old_summary_row in code:
    code = code.replace(old_summary_row, new_summary_row)

with open(an_screen, 'w') as f:
    f.write(code)
print("✅ analyzer_screen.dart — добавлена подсказка про новый лог")

# ================================================================
# 3. ecu_read_screen.dart — замена на loadNewRom
# ================================================================
ecu_screen = 'lib/screens/ecu_read_screen.dart'
with open(ecu_screen, 'r') as f:
    ecode = f.read()

ecode = ecode.replace(
    "final ok = await _rom.pickAndLoad();",
    "final ok = await RomHolder.instance.loadNewRom();"
)

with open(ecu_screen, 'w') as f:
    f.write(ecode)
print("✅ ecu_read_screen.dart — замена на loadNewRom")



✅ rom_holder.dart — сброс кеша при загрузке нового ROM
✅ analyzer_screen.dart — добавлена подсказка про новый лог
✅ ecu_read_screen.dart — замена на loadNewRom


In [ ]:
# @title 🔧 ФИКС: Сквозное редактирование и сохранение кастомных PID
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# 1. obd_service.dart — поддержка кастомных/изменённых PID из профиля
# ================================================================
obd_path = 'lib/services/obd_service.dart'
with open(obd_path, 'r') as f:
    code = f.read()

# Добавляем импорт формул и профиля если их нет
if "import 'formula_evaluator.dart';" not in code:
    code = "import 'formula_evaluator.dart';\n" + code

# Обновляем метод _buildPidLists с учётом кастомных PID из профиля
old_build = "  void _buildPidLists() {\n    _fastPids = _activePids.where((p) => p.priority == 1).toList();\n    _medPids  = _activePids.where((p) => p.priority == 2).toList();\n    _slowPids = _activePids.where((p) => p.priority == 3).toList();\n  }"

new_build = r'''  void _buildPidLists() {
    final p = _profile;
    if (p != null && (p.customPids.isNotEmpty || p.deletedPidIds.isNotEmpty)) {
      final customMap = {for (var cp in p.customPids) cp.id: cp};
      final deletedSet = p.deletedPidIds.toSet();
      final effective = <NissanPidDef>[];

      for (final def in _activePids) {
        if (deletedSet.contains(def.id)) continue;
        if (customMap.containsKey(def.id)) {
          final c = customMap[def.id]!;
          final ev = FormulaEvaluator(c.formula);
          effective.add(NissanPidDef(
            id: c.id, cmd: c.cmd, answer: c.answer,
            name: c.name, desc: c.desc, unit: c.unit,
            bytesCount: c.bytesCount,
            formula: (b) {
              final raw = c.bytesCount == 2 && b.length >= 2
                  ? (b[0] * 256 + b[1]) : (b.isNotEmpty ? b[0] : 0);
              return ev.evaluate(raw);
            },
            minVal: c.minVal, maxVal: c.maxVal,
            priority: c.priority, category: c.category,
          ));
        } else {
          effective.add(def);
        }
      }

      // Добавляем новые кастомные PID пользователя
      for (final c in p.customPids) {
        if (c.status == PidStatus.userAdded && !_activePids.any((def) => def.id == c.id)) {
          final ev = FormulaEvaluator(c.formula);
          effective.add(NissanPidDef(
            id: c.id, cmd: c.cmd, answer: c.answer,
            name: c.name, desc: c.desc, unit: c.unit,
            bytesCount: c.bytesCount,
            formula: (b) {
              final raw = c.bytesCount == 2 && b.length >= 2
                  ? (b[0] * 256 + b[1]) : (b.isNotEmpty ? b[0] : 0);
              return ev.evaluate(raw);
            },
            minVal: c.minVal, maxVal: c.maxVal,
            priority: c.priority, category: c.category,
          ));
        }
      }
      _activePids = effective;
    }

    _fastPids = _activePids.where((p) => p.priority == 1).toList();
    _medPids  = _activePids.where((p) => p.priority == 2).toList();
    _slowPids = _activePids.where((p) => p.priority == 3).toList();
  }'''

if old_build in code:
    code = code.replace(old_build, new_build)

with open(obd_path, 'w') as f:
    f.write(code)
print("✅ obd_service.dart — интегрированы кастомные PID из профиля в опрос ЭБУ")

# ================================================================
# 2. custom_pid_screen.dart — Переписан полностью с рабочим тап/edit/save
# ================================================================
with open('lib/screens/custom_pid_screen.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'package:uuid/uuid.dart';
import '../models/custom_pid.dart';
import '../services/obd_service.dart';
import '../services/profile_service.dart';
import '../services/nissan_pid_library.dart';
import '../widgets/fps_indicator.dart';

class CustomPIDScreen extends StatefulWidget {
  final OBDService obdService;
  final ProfileService profileService;
  const CustomPIDScreen({
    super.key,
    required this.obdService,
    required this.profileService,
  });
  @override
  State<CustomPIDScreen> createState() => _CustomPIDScreenState();
}

class _CustomPIDScreenState extends State<CustomPIDScreen> {
  Timer? _t;
  String _cat = 'all';
  String _search = '';

  @override
  void initState() {
    super.initState();
    _t = Timer.periodic(const Duration(milliseconds: 500), (_) {
      if (mounted) setState(() {});
    });
  }

  @override
  void dispose() {
    _t?.cancel();
    super.dispose();
  }

  // ── Диалог создания/редактирования PID ────────────────────────
  void _editPidDialog({CustomPid? existing, NissanPidDef? fromDefault}) {
    final isNew = existing == null && fromDefault == null;
    final isEditingDefault = fromDefault != null;

    final cmdCtrl = TextEditingController(
      text: existing?.cmd ?? fromDefault?.cmd ?? '2211FF0401');
    final answerCtrl = TextEditingController(
      text: existing?.answer ?? fromDefault?.answer ?? '6211FF');
    final nameCtrl = TextEditingController(
      text: existing?.name ?? fromDefault?.name ?? '');
    final descCtrl = TextEditingController(
      text: existing?.desc ?? fromDefault?.desc ?? '');
    final unitCtrl = TextEditingController(
      text: existing?.unit ?? fromDefault?.unit ?? '');
    final formulaCtrl = TextEditingController(
      text: existing?.formula ?? 'X');
    final bytesCtrl = TextEditingController(
      text: (existing?.bytesCount ?? fromDefault?.bytesCount ?? 1).toString());
    final minCtrl = TextEditingController(
      text: (existing?.minVal ?? fromDefault?.minVal ?? 0).toString());
    final maxCtrl = TextEditingController(
      text: (existing?.maxVal ?? fromDefault?.maxVal ?? 255).toString());

    String category = existing?.category ?? fromDefault?.category ?? 'other';
    int priority = existing?.priority ?? fromDefault?.priority ?? 3;

    showDialog(context: context, builder: (c) => StatefulBuilder(
      builder: (c, setD) => AlertDialog(
        backgroundColor: const Color(0xFF16213E),
        title: Text(
          isNew ? 'Добавить PID'
            : isEditingDefault ? 'Изменить дефолтный PID'
            : 'Редактировать PID',
          style: const TextStyle(fontSize: 16, fontWeight: FontWeight.bold),
        ),
        content: SingleChildScrollView(
          child: Column(mainAxisSize: MainAxisSize.min, children: [
            if (isEditingDefault)
              Container(
                padding: const EdgeInsets.all(6),
                margin: const EdgeInsets.only(bottom: 8),
                decoration: BoxDecoration(
                  color: Colors.orange.withOpacity(0.15),
                  borderRadius: BorderRadius.circular(4),
                  border: Border.all(color: Colors.orange.withOpacity(0.4))),
                child: const Text(
                  'Сохранится как кастомная копия. Оригинал можно будет сбросить.',
                  style: TextStyle(color: Colors.orange, fontSize: 10)),
              ),
            TextField(
              controller: cmdCtrl,
              decoration: const InputDecoration(
                labelText: 'Команда (hex)', hintText: '2211FF0401', isDense: true),
              style: const TextStyle(fontFamily: 'monospace')),
            const SizedBox(height: 6),
            TextField(
              controller: answerCtrl,
              decoration: const InputDecoration(
                labelText: 'Ответ (hex префикс)', hintText: '6211FF', isDense: true),
              style: const TextStyle(fontFamily: 'monospace')),
            const SizedBox(height: 6),
            TextField(
              controller: nameCtrl,
              decoration: const InputDecoration(
                labelText: 'Имя / ID', hintText: 'MY_TEMP', isDense: true)),
            const SizedBox(height: 6),
            TextField(
              controller: descCtrl,
              decoration: const InputDecoration(
                labelText: 'Описание', hintText: 'Моя температура', isDense: true)),
            const SizedBox(height: 6),
            TextField(
              controller: unitCtrl,
              decoration: const InputDecoration(
                labelText: 'Единица измерения', hintText: '°C', isDense: true)),
            const SizedBox(height: 6),
            TextField(
              controller: formulaCtrl,
              decoration: const InputDecoration(
                labelText: 'Формула',
                hintText: 'X - 50  или  X * 0.01',
                helperText: 'X = raw число из байт ответа',
                isDense: true),
              style: const TextStyle(fontFamily: 'monospace')),
            const SizedBox(height: 6),
            Row(children: [
              Expanded(child: TextField(
                controller: bytesCtrl,
                decoration: const InputDecoration(labelText: 'Байт (1 или 2)', isDense: true),
                keyboardType: TextInputType.number)),
              const SizedBox(width: 6),
              Expanded(child: TextField(
                controller: minCtrl,
                decoration: const InputDecoration(labelText: 'Мин', isDense: true),
                keyboardType: TextInputType.number)),
              const SizedBox(width: 6),
              Expanded(child: TextField(
                controller: maxCtrl,
                decoration: const InputDecoration(labelText: 'Макс', isDense: true),
                keyboardType: TextInputType.number)),
            ]),
            const SizedBox(height: 10),
            Row(children: [
              const Text('Приоритет:', style: TextStyle(fontSize: 11)),
              const SizedBox(width: 6),
              DropdownButton<int>(
                value: priority,
                isDense: true,
                dropdownColor: const Color(0xFF16213E),
                items: const [
                  DropdownMenuItem(value: 1, child: Text('1 - Быстрый')),
                  DropdownMenuItem(value: 2, child: Text('2 - Средний')),
                  DropdownMenuItem(value: 3, child: Text('3 - Медленный')),
                ],
                onChanged: (v) => setD(() => priority = v ?? 3)),
            ]),
            Row(children: [
              const Text('Категория:', style: TextStyle(fontSize: 11)),
              const SizedBox(width: 6),
              Expanded(child: DropdownButton<String>(
                value: category,
                isDense: true,
                isExpanded: true,
                dropdownColor: const Color(0xFF16213E),
                items: ['engine', 'ignition', 'fuel', 'air', 'temp', 'vtc', 'throttle', 'idle', 'electric', 'other']
                    .map((cat) => DropdownMenuItem(value: cat, child: Text(cat)))
                    .toList(),
                onChanged: (v) => setD(() => category = v ?? 'other'))),
            ]),
          ]),
        ),
        actions: [
          if (existing != null && existing.status == PidStatus.userAdded)
            TextButton(
              onPressed: () async {
                await widget.profileService.deletePid(existing.id);
                if (mounted) Navigator.pop(c);
                _applyToOBD();
                setState(() {});
                _snack('Удалён', Colors.orange);
              },
              style: TextButton.styleFrom(foregroundColor: Colors.red),
              child: const Text('УДАЛИТЬ')),
          TextButton(
            onPressed: () => Navigator.pop(c),
            child: const Text('Отмена')),
          TextButton(
            onPressed: () async {
              if (cmdCtrl.text.trim().isEmpty || nameCtrl.text.trim().isEmpty) {
                _snack('Заполните команду и имя', Colors.red);
                return;
              }

              final pid = CustomPid(
                id: existing?.id ?? (isEditingDefault ? fromDefault.id : const Uuid().v4()),
                cmd: cmdCtrl.text.trim().toUpperCase(),
                answer: answerCtrl.text.trim().toUpperCase(),
                name: nameCtrl.text.trim(),
                desc: descCtrl.text.trim().isEmpty ? nameCtrl.text.trim() : descCtrl.text.trim(),
                unit: unitCtrl.text.trim(),
                bytesCount: int.tryParse(bytesCtrl.text) ?? 1,
                formula: formulaCtrl.text.trim().isEmpty ? 'X' : formulaCtrl.text.trim(),
                minVal: double.tryParse(minCtrl.text) ?? 0,
                maxVal: double.tryParse(maxCtrl.text) ?? 255,
                priority: priority,
                category: category,
                status: isEditingDefault ? PidStatus.defaultModified : (existing?.status ?? PidStatus.userAdded),
                originalId: isEditingDefault ? fromDefault.id : existing?.originalId,
              );

              await widget.profileService.saveCustomPid(pid);
              _applyToOBD();
              if (mounted) Navigator.pop(c);
              setState(() {});
              _snack('Сохранён PID: ${pid.name}', Colors.green);
            },
            style: TextButton.styleFrom(foregroundColor: Colors.green),
            child: Text(isNew ? 'ДОБАВИТЬ' : 'СОХРАНИТЬ')),
        ],
      ),
    ));
  }

  void _applyToOBD() {
    final activeProfile = widget.profileService.getActiveOrDefault();
    widget.obdService.applyProfile(activeProfile);
  }

  Future<void> _hideDefaultPid(NissanPidDef pid) async {
    final ok = await showDialog<bool>(context: context, builder: (c) => AlertDialog(
      backgroundColor: const Color(0xFF16213E),
      title: const Text('Скрыть PID?'),
      content: Text('${pid.name} (${pid.desc})\n\nЕго можно будет восстановить кнопкой сброса.',
        style: const TextStyle(color: Colors.white70)),
      actions: [
        TextButton(onPressed: () => Navigator.pop(c, false), child: const Text('Отмена')),
        TextButton(onPressed: () => Navigator.pop(c, true),
          style: TextButton.styleFrom(foregroundColor: Colors.red),
          child: const Text('СКРЫТЬ')),
      ]));

    if (ok == true) {
      await widget.profileService.deletePid(pid.id, isDefault: true);
      _applyToOBD();
      setState(() {});
      _snack('PID ${pid.name} скрыт', Colors.orange);
    }
  }

  Future<void> _restoreDefaults() async {
    final profile = widget.profileService.getActiveOrDefault();
    for (final id in profile.deletedPidIds) {
      await widget.profileService.restoreDefaultPid(id);
    }
    _applyToOBD();
    setState(() {});
    _snack('Все дефолтные PID восстановлены', Colors.green);
  }

  void _snack(String m, Color c) {
    if (!mounted) return;
    ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text(m), backgroundColor: c));
  }

  @override
  Widget build(BuildContext context) {
    final obd = widget.obdService;
    final profile = widget.profileService.getActiveOrDefault();
    final customPids = profile.customPids;
    final deletedIds = profile.deletedPidIds.toSet();

    final modifiedOrigIds = customPids
        .where((p) => p.status == PidStatus.defaultModified)
        .map((p) => p.originalId)
        .whereType<String>()
        .toSet();

    // Фильтруем список
    var defaults = NissanPidLibrary.all
        .where((p) => !deletedIds.contains(p.id))
        .where((p) => !modifiedOrigIds.contains(p.id))
        .toList();

    if (_cat != 'all') {
      defaults = defaults.where((p) => p.category == _cat).toList();
    }

    if (_search.isNotEmpty) {
      final q = _search.toLowerCase();
      defaults = defaults.where((p) =>
        p.name.toLowerCase().contains(q) ||
        p.desc.toLowerCase().contains(q) ||
        p.cmd.toLowerCase().contains(q)).toList();
    }

    final cats = ['all', ...NissanPidLibrary.categories];

    return Scaffold(
      appBar: AppBar(
        title: const Text('PID редактор'),
        backgroundColor: const Color(0xFF16213E),
        actions: [
          FpsIndicator(obdService: obd),
          IconButton(
            icon: const Icon(Icons.add_circle, color: Colors.green, size: 26),
            tooltip: 'Добавить PID',
            onPressed: () => _editPidDialog()),
          if (deletedIds.isNotEmpty)
            IconButton(
              icon: const Icon(Icons.restore, color: Colors.orange),
              tooltip: 'Восстановить скрытые',
              onPressed: _restoreDefaults),
        ],
      ),
      body: Column(children: [
        // Верхняя инфо-панель
        Container(
          padding: const EdgeInsets.all(8),
          color: const Color(0xFF16213E),
          child: Column(children: [
            Row(children: [
              const Icon(Icons.memory, color: Colors.cyan, size: 16),
              const SizedBox(width: 6),
              Expanded(child: Text(
                obd.ecuId.isNotEmpty
                  ? 'ECU: ${obd.ecuId} | В опросе: ${obd.activePids.length}'
                  : 'ECU: OFFLINE | Всего в базе: ${NissanPidLibrary.all.length}',
                style: const TextStyle(color: Colors.cyan, fontSize: 11))),
              if (customPids.isNotEmpty)
                Container(
                  padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
                  decoration: BoxDecoration(color: Colors.orange.withOpacity(0.3), borderRadius: BorderRadius.circular(4)),
                  child: Text('Своих: ${customPids.length}',
                    style: const TextStyle(color: Colors.orange, fontSize: 10, fontWeight: FontWeight.bold))),
            ]),
            const SizedBox(height: 6),
            TextField(
              decoration: const InputDecoration(
                hintText: 'Поиск по имени, команде, описанию...',
                prefixIcon: Icon(Icons.search, size: 18),
                isDense: true,
                border: OutlineInputBorder(),
                contentPadding: EdgeInsets.symmetric(horizontal: 8, vertical: 6)),
              style: const TextStyle(fontSize: 12),
              onChanged: (v) => setState(() => _search = v)),
          ]),
        ),

        // Категории
        SizedBox(
          height: 38,
          child: ListView.builder(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 6),
            itemCount: cats.length,
            itemBuilder: (_, i) => Padding(
              padding: const EdgeInsets.only(right: 4),
              child: FilterChip(
                label: Text(cats[i], style: const TextStyle(fontSize: 10)),
                selected: _cat == cats[i],
                onSelected: (_) => setState(() => _cat = cats[i]),
                backgroundColor: const Color(0xFF0F3460),
                selectedColor: const Color(0xFFE94560).withOpacity(0.5)))),
        ),

        // Список PID
        Expanded(child: ListView(padding: const EdgeInsets.all(6), children: [
          // ── КАСТОМНЫЕ ПИДЫ ──
          if (customPids.isNotEmpty) ...[
            const Padding(padding: EdgeInsets.symmetric(horizontal: 4, vertical: 2),
              child: Text('ПОЛЬЗОВАТЕЛЬСКИЕ И ИЗМЕНЁННЫЕ:',
                style: TextStyle(color: Colors.orange, fontSize: 11, fontWeight: FontWeight.bold))),
            ...customPids.map((p) {
              final val = obd.pidValues[p.name] ?? obd.pidValues[p.id] ?? 0;
              final isMod = p.status == PidStatus.defaultModified;
              return Card(
                color: Colors.orange.withOpacity(0.12),
                margin: const EdgeInsets.symmetric(vertical: 2),
                child: ListTile(
                  dense: true,
                  onTap: () => _editPidDialog(existing: p),
                  leading: CircleAvatar(
                    radius: 14,
                    backgroundColor: isMod ? Colors.deepOrange : Colors.orange,
                    child: Icon(isMod ? Icons.edit : Icons.star, size: 14, color: Colors.white)),
                  title: Row(children: [
                    Text(p.desc, style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)),
                    const SizedBox(width: 6),
                    Text('[${p.name}]', style: const TextStyle(color: Colors.orangeAccent, fontSize: 10)),
                  ]),
                  subtitle: Text('${p.cmd} | ${p.formula} | Пприор.${p.priority}',
                    style: const TextStyle(fontSize: 10, color: Colors.white54, fontFamily: 'monospace')),
                  trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                    Text('${val.toStringAsFixed(1)} ${p.unit}',
                      style: const TextStyle(color: Colors.yellow, fontWeight: FontWeight.bold, fontSize: 12)),
                    const SizedBox(width: 4),
                    IconButton(
                      icon: const Icon(Icons.edit, size: 16, color: Colors.cyan),
                      onPressed: () => _editPidDialog(existing: p)),
                    IconButton(
                      icon: const Icon(Icons.delete, size: 16, color: Colors.red),
                      onPressed: () async {
                        await widget.profileService.deletePid(p.id);
                        _applyToOBD();
                        setState(() {});
                        _snack('Удалён ${p.name}', Colors.orange);
                      }),
                  ]),
                ));
            }),
            const Divider(color: Colors.white24, height: 16),
          ],

          // ── ДЕФОЛТНЫЕ ПИДЫ ──
          const Padding(padding: EdgeInsets.symmetric(horizontal: 4, vertical: 2),
            child: Text('СТАНДАРТНЫЕ PID (нажмите для редактирования):',
              style: TextStyle(color: Colors.cyan, fontSize: 11, fontWeight: FontWeight.bold))),
          ...defaults.map((p) {
            final val = obd.pidValues[p.name] ?? obd.pidValues[p.id] ?? 0;
            return Card(
              color: const Color(0xFF16213E),
              margin: const EdgeInsets.symmetric(vertical: 2),
              child: ListTile(
                dense: true,
                onTap: () => _editPidDialog(fromDefault: p),
                leading: CircleAvatar(
                  radius: 14,
                  backgroundColor: p.priority == 1 ? Colors.red : p.priority == 2 ? Colors.orange : Colors.grey,
                  child: Text('P${p.priority}', style: const TextStyle(fontSize: 9, color: Colors.white))),
                title: Row(children: [
                  Text(p.desc, style: const TextStyle(fontSize: 12, fontWeight: FontWeight.bold)),
                  const SizedBox(width: 6),
                  Text('[${p.name}]', style: const TextStyle(color: Colors.cyan, fontSize: 10)),
                ]),
                subtitle: Text('${p.cmd} | ${p.category}',
                  style: const TextStyle(fontFamily: 'monospace', fontSize: 10, color: Colors.white54)),
                trailing: Row(mainAxisSize: MainAxisSize.min, children: [
                  Text('${val.toStringAsFixed(1)} ${p.unit}',
                    style: const TextStyle(color: Colors.yellow, fontWeight: FontWeight.bold, fontSize: 12)),
                  const SizedBox(width: 4),
                  IconButton(
                    icon: const Icon(Icons.edit, size: 16, color: Colors.cyan),
                    tooltip: 'Изменить',
                    onPressed: () => _editPidDialog(fromDefault: p)),
                  IconButton(
                    icon: const Icon(Icons.visibility_off, size: 16, color: Colors.red),
                    tooltip: 'Скрыть',
                    onPressed: () => _hideDefaultPid(p)),
                ]),
              ));
          }),
        ])),

        // Нижняя кнопка Добавить
        Container(
          padding: const EdgeInsets.all(8),
          color: const Color(0xFF16213E),
          child: SizedBox(
            width: double.infinity,
            child: ElevatedButton.icon(
              onPressed: () => _editPidDialog(),
              icon: const Icon(Icons.add, size: 20),
              label: const Text('ДОБАВИТЬ НОВЫЙ PID', style: TextStyle(fontWeight: FontWeight.bold)),
              style: ElevatedButton.styleFrom(
                backgroundColor: Colors.green, foregroundColor: Colors.white,
                minimumSize: const Size.fromHeight(42)))),
        ),
      ]),
    );
  }
}
''')
print("✅ custom_pid_screen.dart — полностью рабочий редактор ПИДов")
# @title 🔧 ФИКС: Добавление импорта custom_pid.dart в obd_service.dart
import os
os.chdir('/content/nissan_logger_v6')

# Добавляем импорт custom_pid.dart в obd_service.dart
obd_path = 'lib/services/obd_service.dart'
with open(obd_path, 'r') as f:
    code = f.read()

if "import '../models/custom_pid.dart';" not in code:
    code = "import '../models/custom_pid.dart';\n" + code
    with open(obd_path, 'w') as f:
        f.write(code)
    print("✅ obd_service.dart — добавлен импорт custom_pid.dart")
else:
    print("⚠️ Импорт уже присутствует")


✅ obd_service.dart — интегрированы кастомные PID из профиля в опрос ЭБУ
✅ custom_pid_screen.dart — полностью рабочий редактор ПИДов
✅ obd_service.dart — добавлен импорт custom_pid.dart


In [ ]:
# @title 🔧 ФИКС: 100% рабочая система кастомных PID (ProfileService + OBDService + Dashboard)
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# 1. profile_service.dart — гарантированное сохранение активного профиля
# ================================================================
with open('lib/services/profile_service.dart', 'w') as f:
    f.write(r'''import 'package:uuid/uuid.dart';
import '../models/vehicle_profile.dart';
import '../models/custom_pid.dart';
import 'settings_service.dart';

class ProfileService {
  static const _uuid = Uuid();

  List<VehicleProfile> getAll() {
    final list = SettingsService.profilesJson
        .map((s) => VehicleProfile.fromJsonString(s))
        .toList();

    if (list.isEmpty) {
      final def = _defaultProfile();
      SettingsService.setProfilesJson([def.toJsonString()]);
      SettingsService.setActiveProfileId(def.id);
      return [def];
    }
    return list;
  }

  VehicleProfile? getActive() {
    final id = SettingsService.activeProfileId;
    final all = getAll();
    if (id == null && all.isNotEmpty) {
      SettingsService.setActiveProfileId(all.first.id);
      return all.first;
    }
    try {
      return all.firstWhere((p) => p.id == id);
    } catch (_) {
      if (all.isNotEmpty) {
        SettingsService.setActiveProfileId(all.first.id);
        return all.first;
      }
      return null;
    }
  }

  VehicleProfile getActiveOrDefault() {
    return getActive() ?? _defaultProfile();
  }

  Future<VehicleProfile> create({
    required String name,
    required String make,
    required String model,
    required String year,
    required String engine,
    double displacement = 2.0,
    String ecuFirmware  = '',
  }) async {
    final profile = VehicleProfile(
      id:           _uuid.v4(),
      name:         name,
      make:         make,
      model:        model,
      year:         year,
      engine:       engine,
      displacement: displacement,
      ecuFirmware:  ecuFirmware,
      createdAt:    DateTime.now(),
    );
    final all = getAll()..add(profile);
    await _save(all);
    await SettingsService.setActiveProfileId(profile.id);
    return profile;
  }

  Future<void> update(VehicleProfile profile) async {
    final all = getAll();
    final idx = all.indexWhere((p) => p.id == profile.id);
    if (idx >= 0) {
      all[idx] = profile;
    } else {
      all.add(profile);
    }
    await _save(all);
    await SettingsService.setActiveProfileId(profile.id);
  }

  Future<void> delete(String id) async {
    final all = getAll()..removeWhere((p) => p.id == id);
    await _save(all);
    if (SettingsService.activeProfileId == id) {
      await SettingsService.setActiveProfileId(
          all.isNotEmpty ? all.first.id : null);
    }
  }

  Future<void> setActive(String id) async {
    await SettingsService.setActiveProfileId(id);
  }

  // ── PID персонализация ───────────────────────────────────────

  Future<void> saveCustomPid(CustomPid pid) async {
    final profile = getActiveOrDefault();
    final pids = List<CustomPid>.from(profile.customPids);
    final idx = pids.indexWhere((p) => p.id == pid.id);
    if (idx >= 0) {
      pids[idx] = pid;
    } else {
      pids.add(pid);
    }
    await update(profile.copyWith(customPids: pids));
  }

  Future<void> deletePid(String pidId, {bool isDefault = false}) async {
    final profile = getActiveOrDefault();
    final pids    = List<CustomPid>.from(profile.customPids)
        ..removeWhere((p) => p.id == pidId);
    final deleted = List<String>.from(profile.deletedPidIds);
    if (isDefault && !deleted.contains(pidId)) deleted.add(pidId);
    await update(profile.copyWith(customPids: pids, deletedPidIds: deleted));
  }

  Future<void> restoreDefaultPid(String pidId) async {
    final profile = getActiveOrDefault();
    final deleted = List<String>.from(profile.deletedPidIds)
        ..remove(pidId);
    await update(profile.copyWith(deletedPidIds: deleted));
  }

  Future<void> _save(List<VehicleProfile> all) async {
    await SettingsService.setProfilesJson(
        all.map((p) => p.toJsonString()).toList());
  }

  VehicleProfile _defaultProfile() => VehicleProfile(
    id:        'default_t30',
    name:      'Nissan X-Trail T30',
    make:      'Nissan',
    model:     'X-Trail T30',
    year:      '2004',
    engine:    'QR20DE',
    createdAt: DateTime(2024),
  );
}
''')
print("✅ profile_service.dart — 100% надёжное сохранение профилей и PID")

# ================================================================
# 2. obd_service.dart — мгновенный rebuild PID при applyProfile
# ================================================================
with open('lib/services/obd_service.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import '../models/obd_data.dart';
import '../models/vehicle_profile.dart';
import '../models/custom_pid.dart';
import '../constants.dart';
import 'nissan_pid_library.dart';
import 'settings_service.dart';
import 'formula_evaluator.dart';

class OBDService {
  BluetoothConnection? _connection;
  StreamSubscription?  _inputSub;
  final StringBuffer   _rxBuf = StringBuffer();

  bool       _cmdInProgress = false;
  Completer<String>? _cmdCompleter;

  bool _isPolling    = false;
  bool _pollPaused   = false;
  int  _pollCounter  = 0;
  int  _mediumIdx    = 0;
  int  _slowIdx      = 0;
  double _pollFps    = 0.0;
  int    _lastPollMs = 0;

  bool   _ecuResponds  = false;
  bool   _initialized  = false;
  String _protocolInfo = '';
  String _ecuId        = '';

  List<NissanPidDef> _scannedPids = [];
  List<NissanPidDef> _activePids  = [];
  List<NissanPidDef> _fastPids    = [];
  List<NissanPidDef> _medPids     = [];
  List<NissanPidDef> _slowPids    = [];
  final Map<String, double>   _values  = {};
  final Map<String, List<int>>_rawData = {};

  VehicleProfile? _profile;

  double    _tripFuelL  = 0;
  DateTime? _lastFuelTs;

  bool   _autoReconnect   = true;
  int    _reconnectTries  = 0;
  String? _lastAddress;
  Timer?  _reconnectTimer;
  static const _maxReconnect = 3;

  final _dataCtrl = StreamController<OBDData>.broadcast();
  final _logCtrl  = StreamController<String>.broadcast();

  Stream<OBDData> get dataStream => _dataCtrl.stream;
  Stream<String>  get logStream  => _logCtrl.stream;

  bool   get isConnected   => _connection?.isConnected ?? false;
  bool   get isInitialized => _initialized;
  bool   get ecuResponds   => _ecuResponds;
  String get protocolInfo  => _protocolInfo;
  String get ecuId         => _ecuId;
  int    get pollFps       => _pollFps.toInt();
  int    get lastPollMs    => _lastPollMs;
  double get tripFuelL     => _tripFuelL;
  List<NissanPidDef>        get activePids  => _activePids;
  Map<String, double>        get pidValues   => Map.unmodifiable(_values);
  Map<String, List<int>>     get rawPidData  => Map.unmodifiable(_rawData);

  void applyProfile(VehicleProfile profile) {
    _profile = profile;
    _tripFuelL = SettingsService.tripFuelL;
    _rebuildPidLists();
  }

  Future<void> resetTripFuel() async {
    _tripFuelL = 0;
    _lastFuelTs = null;
    await SettingsService.resetTripFuel();
  }

  Future<void> loadTripFuel() async {
    _tripFuelL = SettingsService.tripFuelL;
  }

  void _log(String msg) { _logCtrl.add(msg); }

  Future<List<BluetoothDevice>> getBondedDevices() async {
    try { return await FlutterBluetoothSerial.instance.getBondedDevices(); }
    catch (_) { return []; }
  }

  Future<BluetoothState> getBluetoothState() async =>
      FlutterBluetoothSerial.instance.state;

  Future<bool?> requestEnable() async =>
      FlutterBluetoothSerial.instance.requestEnable();

  Future<bool> connect(String address) async {
    try {
      _log('=== BT $address ===');
      _initialized  = false;
      _ecuResponds  = false;
      _lastAddress  = address;
      _reconnectTries = 0;

      _connection = await BluetoothConnection.toAddress(address);
      _log('BT OK');

      _inputSub = _connection!.input!.listen(
        _onData,
        onDone:  _onDisconnected,
        onError: (e) => _log('BT Error: $e'),
      );

      await Future.delayed(const Duration(milliseconds: 1500));
      _rxBuf.clear();
      _cmdInProgress = false;
      _cmdCompleter  = null;

      _connection!.output.add(Uint8List.fromList([13, 13, 13]));
      await _connection!.output.allSent;
      await Future.delayed(const Duration(milliseconds: 500));
      _rxBuf.clear();

      final r = await sendCommand('ATZ', timeout: 5000);
      _log('ATZ: [$r]');
      await Future.delayed(const Duration(milliseconds: 1500));

      if (r.toUpperCase().contains('ELM')) {
        _initialized = true;
        await SettingsService.setLastBtDevice(address);
        _log('ELM OK');
        return true;
      }
      return false;
    } catch (e) {
      _log('ERR: $e');
      return false;
    }
  }

  Future<bool> initECU({bool useCache = true}) async {
    if (!isConnected) return false;
    _log('=== INIT ECU ===');
    _ecuResponds = false;
    _rawData.clear();
    _scannedPids.clear();

    await sendCommand('ATZ', timeout: 4000);
    await Future.delayed(const Duration(milliseconds: 1000));
    for (final cmd in ['ATE0','ATL0','ATS0','ATH0','ATAL','ATSW00',
                        'ATST19','ATAT2','ATIB10','ATSP5','ATSH8110FC']) {
      await sendCommand(cmd, timeout: 1500);
    }
    await sendCommand('ATFI', timeout: 3000);
    await Future.delayed(const Duration(milliseconds: 200));

    final r = await sendCommand('2211000401', timeout: 8000);
    _log('BUS: [$r]');
    if (!r.replaceAll(' ','').toUpperCase().contains('6211')) {
      _log('BUS FAIL');
      return false;
    }
    _log('ECU RESPONDS!');
    _ecuResponds = true;

    final idR = await sendCommand('1A81', timeout: 3000);
    final idC = idR.replaceAll(' ','').toUpperCase();
    if (idC.contains('5A')) {
      final idx = idC.indexOf('5A');
      _ecuId = _hexToAscii(idC.substring(idx + 2));
      _log('ECU ID: $_ecuId');
    }

    if (useCache) {
      final ce = SettingsService.cachedEcuId;
      final cp = SettingsService.cachedPidList;
      if (ce == _ecuId && cp.isNotEmpty) {
        _log('CACHE ${cp.length} pids');
        for (final name in cp) {
          final pid = NissanPidLibrary.byId(name);
          if (pid != null) _scannedPids.add(pid);
        }
        _rebuildPidLists();
        _protocolInfo = 'Nissan $_ecuId (кеш)';
        Future.delayed(const Duration(milliseconds: 300), startPolling);
        return true;
      }
    }

    _log('SCAN ${NissanPidLibrary.all.length} pids');
    for (final pid in NissanPidLibrary.all) {
      final resp = await sendCommand(pid.cmd, timeout: 600);
      final clean = resp.replaceAll(' ','').toUpperCase();
      if (clean.contains(pid.answer)) {
        final bytes = _extractBytes(resp, pid.answer);
        if (bytes.length >= pid.bytesCount) _scannedPids.add(pid);
      }
      await Future.delayed(const Duration(milliseconds: 20));
    }
    _rebuildPidLists();
    _log('Found ${_scannedPids.length} pids');

    if (_activePids.isEmpty) return false;

    await SettingsService.setCachedEcuId(_ecuId);
    await SettingsService.setCachedPidList(_scannedPids.map((p) => p.id).toList());

    _protocolInfo = 'Nissan $_ecuId (${_activePids.length} pid)';
    Future.delayed(const Duration(milliseconds: 300), startPolling);
    return true;
  }

  void _rebuildPidLists() {
    final p = _profile;
    final deletedSet = p?.deletedPidIds.toSet() ?? {};
    final customPids = p?.customPids ?? <CustomPid>[];
    final customMap  = {for (var cp in customPids) cp.id: cp};

    final baseList = _scannedPids.isNotEmpty ? _scannedPids : NissanPidLibrary.all;
    final effective = <NissanPidDef>[];

    for (final def in baseList) {
      if (deletedSet.contains(def.id)) continue;
      if (customMap.containsKey(def.id)) {
        effective.add(_customToDef(customMap[def.id]!));
      } else {
        effective.add(def);
      }
    }

    for (final c in customPids) {
      if (c.status == PidStatus.userAdded && !effective.any((def) => def.id == c.id)) {
        effective.add(_customToDef(c));
      }
    }

    _activePids = effective;
    _fastPids   = _activePids.where((p) => p.priority == 1).toList();
    _medPids    = _activePids.where((p) => p.priority == 2).toList();
    _slowPids   = _activePids.where((p) => p.priority == 3).toList();
  }

  NissanPidDef _customToDef(CustomPid c) {
    final ev = FormulaEvaluator(c.formula);
    return NissanPidDef(
      id:         c.id,
      cmd:        c.cmd,
      answer:     c.answer,
      name:       c.name,
      desc:       c.desc,
      unit:       c.unit,
      bytesCount: c.bytesCount,
      formula: (b) {
        final raw = c.bytesCount == 2 && b.length >= 2
            ? (b[0] * 256 + b[1]) : (b.isNotEmpty ? b[0] : 0);
        return ev.evaluate(raw);
      },
      minVal:     c.minVal,
      maxVal:     c.maxVal,
      priority:   c.priority,
      category:   c.category,
    );
  }

  void startPolling() {
    if (_isPolling || !_ecuResponds || _activePids.isEmpty) return;
    _isPolling = true;
    _log('POLL START');
    _pollLoop();
  }

  void stopPolling() { _isPolling = false; }

  Future<void> _pollLoop() async {
    final fpsTimer = Stopwatch()..start();
    int fpsCount = 0;

    while (_isPolling && isConnected && _ecuResponds) {
      while (_pollPaused && _isPolling) {
        await Future.delayed(const Duration(milliseconds: 10));
      }
      if (!_isPolling) break;

      _pollCounter++;
      final sw = Stopwatch()..start();

      for (final pid in _fastPids) {
        if (!_isPolling || _pollPaused) break;
        await _pollPid(pid);
      }

      if (_pollCounter % 3 == 0 && _medPids.isNotEmpty && !_pollPaused) {
        for (int i = 0; i < 2 && !_pollPaused; i++) {
          await _pollPid(_medPids[_mediumIdx % _medPids.length]);
          _mediumIdx++;
        }
      }

      if (_pollCounter % 10 == 0 && _slowPids.isNotEmpty && !_pollPaused) {
        await _pollPid(_slowPids[_slowIdx % _slowPids.length]);
        _slowIdx++;
      }

      sw.stop();
      _lastPollMs = sw.elapsedMilliseconds;
      fpsCount++;
      if (fpsTimer.elapsedMilliseconds >= 1000) {
        _pollFps = fpsCount * 1000.0 / fpsTimer.elapsedMilliseconds;
        fpsCount = 0;
        fpsTimer.reset();
      }

      _publish();

      final interval = _profile?.pollingInterval ?? SettingsService.pollingInterval;
      if (interval > 0) {
        await Future.delayed(Duration(milliseconds: interval));
      }
    }
  }

  Future<void> _pollPid(NissanPidDef pid) async {
    try {
      final r = await sendCommand(pid.cmd, timeout: 300, pausePolling: false);
      final bytes = _extractBytes(r, pid.answer);
      if (bytes.length >= pid.bytesCount) {
        final val = pid.formula(bytes);
        // Записываем значения И по ID, И по NAME
        _values[pid.id]   = val;
        _values[pid.name] = val;
        _rawData[pid.cmd]  = bytes;
      }
    } catch (_) {}
  }

  void _publish() {
    final mafVolt  = _v('MAF_V');
    final mafGpsRaw = AppConstants.mafVoltToGps(mafVolt);
    final mafMult   = _profile?.mafMultiplier ?? 1.0;
    final mafGps    = mafGpsRaw * mafMult;

    final speedRaw  = _v('SPEED');
    final speedMult = _profile?.speedMultiplier ?? 1.0;
    final speed     = (speedRaw * speedMult).toInt().clamp(0, 300);
    final disp      = _profile?.displacement ?? 2.0;

    final o2   = _v('O2_B1S1');
    final stft = _v('STFT');
    final afr  = _calcAfr(o2, stft);

    final data = OBDData(
      timestamp:          DateTime.now(),
      rpm:                _v('RPM').toInt().clamp(0, 9999),
      speed:              speed,
      engineLoad:         _v('LOAD').clamp(0, 100),
      coolantTemp:        _v('ECT').toInt().clamp(-40, 200),
      intakeTemp:         _v('IAT').toInt().clamp(-40, 100),
      mafVoltage:         mafVolt,
      mafGps:             mafGps,
      throttlePos:        _v('TPS').clamp(0, 100),
      ignitionTiming:     _v('TIMING'),
      actualIgnition:     _v('TIMING'),
      vtcActualAngle:     _v('VTC_ACT'),
      knockRetard:        _v('KNOCK').abs(),
      shortFuelTrim:      _v('STFT').clamp(-100, 100),
      longFuelTrim:       _v('LTFT').clamp(-100, 100),
      o2Voltage:          o2,
      afr:                afr,
      injectorPulseWidth: _v('INJ_B1'),
      injectorDuty:       (_v('INJ_B1') / 20.0 * 100).clamp(0, 100),
      manifoldPressure:   _v('MAP_V') * 40,
      acceleratorPedal:   _v('PEDAL'),
      throttleActual:     _v('TPS'),
      batteryVoltage:     _v('BATT'),
      engineDisplacement: disp,
      tripFuelL:          _tripFuelL,
      actualTorque:       _v('TORQUE'),
      requestedTorque:    _v('POWER_KW'),
    );

    _updateTripFuel(data.fuelFlowLph);
    _dataCtrl.add(data);
  }

  double _v(String key) => _values[key] ?? 0;

  double _calcAfr(double o2, double stft) {
    double lambda;
    if      (o2 > 0.85) lambda = 0.87;
    else if (o2 > 0.75) lambda = 0.92;
    else if (o2 > 0.60) lambda = 0.97;
    else if (o2 > 0.45) lambda = 1.00;
    else if (o2 > 0.30) lambda = 1.03;
    else if (o2 > 0.15) lambda = 1.05;
    else                lambda = 1.10;
    lambda *= (1 + stft / 100.0 * 0.3);
    return (lambda * 14.7).clamp(10.0, 20.0);
  }

  void _updateTripFuel(double fuelLph) {
    final now = DateTime.now();
    if (_lastFuelTs != null && fuelLph > 0) {
      final dt = now.difference(_lastFuelTs!).inMilliseconds / 1000.0;
      _tripFuelL += (fuelLph / 3600.0) * dt;
      if (_pollCounter % 100 == 0) {
        SettingsService.setTripFuelL(_tripFuelL);
      }
    }
    _lastFuelTs = now;
  }

  Future<String> sendCommand(
    String cmd, {
    int  timeout      = 1000,
    bool pausePolling = true,
  }) async {
    if (!isConnected) return '';

    if (pausePolling && _isPolling) {
      _pollPaused = true;
      int wait = 0;
      while (_cmdInProgress && wait < 30) {
        await Future.delayed(const Duration(milliseconds: 10));
        wait++;
      }
    }

    int guard = 0;
    while (_cmdInProgress && guard < 50) {
      await Future.delayed(const Duration(milliseconds: 5));
      guard++;
    }
    if (_cmdInProgress) {
      _cmdInProgress = false;
      _cmdCompleter  = null;
    }

    _rxBuf.clear();
    _cmdInProgress = true;
    _cmdCompleter  = Completer<String>();

    try {
      final bytes = [...cmd.codeUnits, 13];
      _connection!.output.add(Uint8List.fromList(bytes));
      await _connection!.output.allSent;

      String resp = '';
      try {
        resp = await _cmdCompleter!.future
            .timeout(Duration(milliseconds: timeout));
      } catch (_) {
        resp = _rxBuf.toString();
      }

      _cmdInProgress = false;
      _cmdCompleter  = null;

      if (pausePolling && _isPolling) {
        await Future.delayed(const Duration(milliseconds: 30));
        _pollPaused = false;
      }

      return _clean(resp);
    } catch (e) {
      _cmdInProgress = false;
      _cmdCompleter  = null;
      if (pausePolling) _pollPaused = false;
      return '';
    }
  }

  String _clean(String r) =>
      r.replaceAll('>','').replaceAll('\r',' ').replaceAll('\n',' ')
       .replaceAll('  ',' ').trim();

  void _onData(Uint8List data) {
    final s = String.fromCharCodes(data);
    _rxBuf.write(s);
    if (s.contains('>') && _cmdCompleter != null && !_cmdCompleter!.isCompleted) {
      _cmdCompleter!.complete(_rxBuf.toString());
    }
  }

  void _onDisconnected() {
    _log('BT DISCONNECTED');
    stopPolling();
    _initialized   = false;
    _ecuResponds   = false;
    _cmdInProgress = false;
    _cmdCompleter  = null;
    _connection    = null;

    if (_autoReconnect && _lastAddress != null &&
        _reconnectTries < _maxReconnect) {
      _scheduleReconnect();
    }
  }

  void _scheduleReconnect() {
    _reconnectTries++;
    _log('Reconnect attempt $_reconnectTries');
    _reconnectTimer?.cancel();
    _reconnectTimer = Timer(
      Duration(seconds: 3 * _reconnectTries),
      () async {
        if (_lastAddress == null) return;
        final ok = await connect(_lastAddress!);
        if (ok) await initECU(useCache: true);
        else if (_reconnectTries < _maxReconnect) _scheduleReconnect();
      },
    );
  }

  Future<void> disconnect() async {
    _autoReconnect = false;
    _reconnectTimer?.cancel();
    stopPolling();
    _initialized   = false;
    _ecuResponds   = false;
    _cmdInProgress = false;
    _cmdCompleter  = null;
    await _inputSub?.cancel();
    _inputSub = null;
    await _connection?.close();
    _connection = null;
    _autoReconnect = true;
    await SettingsService.setTripFuelL(_tripFuelL);
  }

  void dispose() {
    disconnect();
    _dataCtrl.close();
    _logCtrl.close();
  }

  List<int> _extractBytes(String resp, String prefix) {
    final s = resp.replaceAll(' ','').replaceAll('BUSINIT:OK','')
                  .toUpperCase();
    final idx = s.indexOf(prefix);
    if (idx < 0) return [];
    final hex = s.substring(idx + prefix.length);
    final result = <int>[];
    for (int i = 0; i + 1 < hex.length; i += 2) {
      final h = hex.substring(i, i + 2);
      if (!RegExp(r'^[0-9A-F]+$').hasMatch(h)) break;
      try { result.add(int.parse(h, radix: 16)); } catch (_) { break; }
    }
    return result;
  }

  String _hexToAscii(String hex) {
    final sb = StringBuffer();
    for (int i = 0; i + 1 < hex.length; i += 2) {
      try {
        final b = int.parse(hex.substring(i, i + 2), radix: 16);
        if (b >= 0x20 && b <= 0x7E) sb.writeCharCode(b);
      } catch (_) { break; }
    }
    return sb.toString();
  }
}
''')
print("✅ obd_service.dart — 100% надёжная работа с кастомными PID")



✅ profile_service.dart — 100% надёжное сохранение профилей и PID
✅ obd_service.dart — 100% надёжная работа с кастомными PID


In [ ]:
# @title 🚀 ФИКС: Интеграция Dual-Speed (57.6k Baud) в Nissan Logger V6
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# 1. obd_service.dart — логика переключения Dual-Speed (10400 -> 57600)
# ================================================================
with open('lib/services/obd_service.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import '../models/obd_data.dart';
import '../models/vehicle_profile.dart';
import '../models/custom_pid.dart';
import '../constants.dart';
import 'nissan_pid_library.dart';
import 'settings_service.dart';
import 'formula_evaluator.dart';

class OBDService {
  BluetoothConnection? _connection;
  StreamSubscription?  _inputSub;
  final StringBuffer   _rxBuf = StringBuffer();

  bool       _cmdInProgress = false;
  Completer<String>? _cmdCompleter;

  bool _isPolling    = false;
  bool _pollPaused   = false;
  int  _pollCounter  = 0;
  int  _mediumIdx    = 0;
  int  _slowIdx      = 0;
  double _pollFps    = 0.0;
  int    _lastPollMs = 0;

  bool   _ecuResponds  = false;
  bool   _initialized  = false;
  String _protocolInfo = '';
  String _ecuId        = '';
  bool   _isHighSpeed  = false; // Флаг работы на 57600 baud

  List<NissanPidDef> _scannedPids = [];
  List<NissanPidDef> _activePids  = [];
  List<NissanPidDef> _fastPids    = [];
  List<NissanPidDef> _medPids     = [];
  List<NissanPidDef> _slowPids    = [];
  final Map<String, double>    _values  = {};
  final Map<String, List<int>> _rawData = {};

  VehicleProfile? _profile;

  double    _tripFuelL  = 0;
  DateTime? _lastFuelTs;

  bool   _autoReconnect   = true;
  int    _reconnectTries  = 0;
  String? _lastAddress;
  Timer?  _reconnectTimer;
  static const _maxReconnect = 3;

  final _dataCtrl = StreamController<OBDData>.broadcast();
  final _logCtrl  = StreamController<String>.broadcast();

  Stream<OBDData> get dataStream => _dataCtrl.stream;
  Stream<String>  get logStream  => _logCtrl.stream;

  bool   get isConnected   => _connection?.isConnected ?? false;
  bool   get isInitialized => _initialized;
  bool   get ecuResponds   => _ecuResponds;
  String get protocolInfo  => _protocolInfo;
  String get ecuId         => _ecuId;
  bool   get isHighSpeed   => _isHighSpeed;
  int    get pollFps       => _pollFps.toInt();
  int    get lastPollMs    => _lastPollMs;
  double get tripFuelL     => _tripFuelL;
  List<NissanPidDef>    get activePids => _activePids;
  Map<String, double>    get pidValues  => Map.unmodifiable(_values);

  void applyProfile(VehicleProfile profile) {
    _profile = profile;
    _tripFuelL = SettingsService.tripFuelL;
    _rebuildPidLists();
  }

  void _log(String msg) => _logCtrl.add(msg);

  Future<List<BluetoothDevice>> getBondedDevices() async {
    try { return await FlutterBluetoothSerial.instance.getBondedDevices(); }
    catch (_) { return []; }
  }

  Future<BluetoothState> getBluetoothState() async =>
      FlutterBluetoothSerial.instance.state;

  Future<bool?> requestEnable() async =>
      FlutterBluetoothSerial.instance.requestEnable();

  Future<bool> connect(String address) async {
    try {
      _log('=== BT $address ===');
      _initialized  = false;
      _ecuResponds  = false;
      _lastAddress  = address;
      _reconnectTries = 0;

      _connection = await BluetoothConnection.toAddress(address);
      _log('BT OK');

      _inputSub = _connection!.input!.listen(
        _onData,
        onDone:  _onDisconnected,
        onError: (e) => _log('BT Error: $e'),
      );

      await Future.delayed(const Duration(milliseconds: 1500));
      _rxBuf.clear();
      _cmdInProgress = false;
      _cmdCompleter  = null;

      _connection!.output.add(Uint8List.fromList([13, 13, 13]));
      await _connection!.output.allSent;
      await Future.delayed(const Duration(milliseconds: 500));
      _rxBuf.clear();

      final r = await sendCommand('ATZ', timeout: 5000);
      _log('ATZ: [$r]');
      await Future.delayed(const Duration(milliseconds: 1500));

      if (r.toUpperCase().contains('ELM')) {
        _initialized = true;
        await SettingsService.setLastBtDevice(address);
        _log('ELM OK');
        return true;
      }
      return false;
    } catch (e) {
      _log('ERR: $e');
      return false;
    }
  }

  // ── ИНИЦИАЛИЗАЦИЯ С DUAL-SPEED ─────────────────────────────────
  Future<bool> initECU({bool useCache = true, bool enableHighSpeed = true}) async {
    if (!isConnected) return false;
    _log('=== INIT ECU (Nissan 1EQ010) ===');
    _ecuResponds = false;
    _isHighSpeed = false;
    _rawData.clear();
    _scannedPids.clear();

    // 1. Старт на стандартной скорости 10 400 бод
    await sendCommand('ATZ', timeout: 4000);
    await Future.delayed(const Duration(milliseconds: 1000));
    for (final cmd in ['ATE0','ATL0','ATS0','ATH0','ATAL','ATSW00',
                        'ATST19','ATAT2','ATIB10','ATSP5','ATSH8110FC']) {
      await sendCommand(cmd, timeout: 1500);
    }
    await sendCommand('ATFI', timeout: 3000);
    await Future.delayed(const Duration(milliseconds: 200));

    final r = await sendCommand('2211000401', timeout: 8000);
    if (!r.replaceAll(' ','').toUpperCase().contains('6211')) {
      _log('❌ ЭБУ не отвечает на 10 400 Baud');
      return false;
    }
    _log('✅ ЭБУ ответил на 10 400 Baud!');
    _ecuResponds = true;

    final idR = await sendCommand('1A81', timeout: 3000);
    final idC = idR.replaceAll(' ','').toUpperCase();
    if (idC.contains('5A')) {
      final idx = idC.indexOf('5A');
      _ecuId = _hexToAscii(idC.substring(idx + 2));
      _log('ECU ID: $_ecuId');
    }

    // 2. ПРОБУЕМ DUAL-SPEED РАЗГОН ДО 57 600 BAUD
    if (enableHighSpeed) {
      await _trySwitchTo57600();
    }

    // 3. Загрузка PID из кеша или сканирование
    if (useCache) {
      final ce = SettingsService.cachedEcuId;
      final cp = SettingsService.cachedPidList;
      if (ce == _ecuId && cp.isNotEmpty) {
        _log('CACHE ${cp.length} pids');
        for (final name in cp) {
          final pid = NissanPidLibrary.byId(name);
          if (pid != null) _scannedPids.add(pid);
        }
        _rebuildPidLists();
        _protocolInfo = 'Nissan $_ecuId ${_isHighSpeed ? "[57.6k Baud HighSpeed]" : "[10.4k Baud]"}';
        Future.delayed(const Duration(milliseconds: 300), startPolling);
        return true;
      }
    }

    _log('SCAN ${NissanPidLibrary.all.length} pids...');
    for (final pid in NissanPidLibrary.all) {
      final resp = await sendCommand(pid.cmd, timeout: _isHighSpeed ? 250 : 600);
      final clean = resp.replaceAll(' ','').toUpperCase();
      if (clean.contains(pid.answer)) {
        final bytes = _extractBytes(resp, pid.answer);
        if (bytes.length >= pid.bytesCount) _scannedPids.add(pid);
      }
      await Future.delayed(Duration(milliseconds: _isHighSpeed ? 5 : 20));
    }
    _rebuildPidLists();
    _log('Найдено ${_scannedPids.length} PID');

    if (_activePids.isEmpty) return false;

    await SettingsService.setCachedEcuId(_ecuId);
    await SettingsService.setCachedPidList(_scannedPids.map((p) => p.id).toList());

    _protocolInfo = 'Nissan $_ecuId (${_activePids.length} pid) ${_isHighSpeed ? "[57.6k HighSpeed]" : "[10.4k]"}';
    Future.delayed(const Duration(milliseconds: 300), startPolling);
    return true;
  }

  // ── Процедура перехода 10400 -> 57600 ─────────────────────────
  Future<bool> _trySwitchTo57600() async {
    _log('🚀 Запрос перехода на 57 600 Baud (команда 10 92)...');

    final resp = await sendCommand('1092', timeout: 1000);
    final clean = resp.replaceAll(' ','').toUpperCase();

    if (clean.contains('5092') || clean.contains('OK')) {
      _log('✅ ЭБУ подтвердил 50 92! Пауза 40 мс...');
      await Future.delayed(const Duration(milliseconds: 40));

      // Переключаем адаптер ELM327 на 57600 бод
      _log('Переключаем ELM327 (ATIB57)...');
      await sendCommand('ATIB57', timeout: 500, pausePolling: false);
      await Future.delayed(const Duration(milliseconds: 30));

      // Проверка отклика PID Оборотов на новой скорости
      final test = await sendCommand('2212010401', timeout: 400);
      if (test.contains('621201')) {
        _isHighSpeed = true;
        _log('🎉🎉🎉 СВЯЗЬ НА 57 600 BAUD УСТАНОВЛЕНА! (Скорость опроса ~90-120 Гц)');
        return true;
      } else {
        _log('⚠️ Сбой отклика на 57600, откат на 10400...');
        await sendCommand('ATIB10', timeout: 500, pausePolling: false);
      }
    } else {
      _log('ℹ️ Прошивка ЭБУ не ответила на 10 92 (работаем на 10 400 Baud)');
    }
    return false;
  }

  void _rebuildPidLists() {
    final p = _profile;
    final deletedSet = p?.deletedPidIds.toSet() ?? {};
    final customPids = p?.customPids ?? <CustomPid>[];
    final customMap  = {for (var cp in customPids) cp.id: cp};

    final baseList = _scannedPids.isNotEmpty ? _scannedPids : NissanPidLibrary.all;
    final effective = <NissanPidDef>[];

    for (final def in baseList) {
      if (deletedSet.contains(def.id)) continue;
      if (customMap.containsKey(def.id)) {
        effective.add(_customToDef(customMap[def.id]!));
      } else {
        effective.add(def);
      }
    }

    for (final c in customPids) {
      if (c.status == PidStatus.userAdded && !effective.any((def) => def.id == c.id)) {
        effective.add(_customToDef(c));
      }
    }

    _activePids = effective;
    _fastPids   = _activePids.where((p) => p.priority == 1).toList();
    _medPids    = _activePids.where((p) => p.priority == 2).toList();
    _slowPids   = _activePids.where((p) => p.priority == 3).toList();
  }

  NissanPidDef _customToDef(CustomPid c) {
    final ev = FormulaEvaluator(c.formula);
    return NissanPidDef(
      id: c.id, cmd: c.cmd, answer: c.answer,
      name: c.name, desc: c.desc, unit: c.unit,
      bytesCount: c.bytesCount,
      formula: (b) {
        final raw = c.bytesCount == 2 && b.length >= 2
            ? (b[0] * 256 + b[1]) : (b.isNotEmpty ? b[0] : 0);
        return ev.evaluate(raw);
      },
      minVal: c.minVal, maxVal: c.maxVal,
      priority: c.priority, category: c.category,
    );
  }

  void startPolling() {
    if (_isPolling || !_ecuResponds || _activePids.isEmpty) return;
    _isPolling = true;
    _log('POLL START (HighSpeed: $_isHighSpeed)');
    _pollLoop();
  }

  void stopPolling() { _isPolling = false; }

  Future<void> _pollLoop() async {
    final fpsTimer = Stopwatch()..start();
    int fpsCount = 0;

    while (_isPolling && isConnected && _ecuResponds) {
      while (_pollPaused && _isPolling) {
        await Future.delayed(const Duration(milliseconds: 5));
      }
      if (!_isPolling) break;

      _pollCounter++;
      final sw = Stopwatch()..start();

      for (final pid in _fastPids) {
        if (!_isPolling || _pollPaused) break;
        await _pollPid(pid);
      }

      if (_pollCounter % 2 == 0 && _medPids.isNotEmpty && !_pollPaused) {
        for (int i = 0; i < 2 && !_pollPaused; i++) {
          await _pollPid(_medPids[_mediumIdx % _medPids.length]);
          _mediumIdx++;
        }
      }

      if (_pollCounter % 5 == 0 && _slowPids.isNotEmpty && !_pollPaused) {
        await _pollPid(_slowPids[_slowIdx % _slowPids.length]);
        _slowIdx++;
      }

      sw.stop();
      _lastPollMs = sw.elapsedMilliseconds;
      fpsCount++;

      if (fpsTimer.elapsedMilliseconds >= 1000) {
        _pollFps = fpsCount * 1000.0 / fpsTimer.elapsedMilliseconds;
        fpsCount = 0;
        fpsTimer.reset();
      }

      _publish();

      // На высокой скорости пауза между циклами минимальная (5 мс)
      final baseInterval = _profile?.pollingInterval ?? SettingsService.pollingInterval;
      final actualInterval = _isHighSpeed ? 5 : baseInterval;

      if (actualInterval > 0) {
        await Future.delayed(Duration(milliseconds: actualInterval));
      }
    }
  }

  Future<void> _pollPid(NissanPidDef pid) async {
    try {
      final r = await sendCommand(pid.cmd, timeout: _isHighSpeed ? 150 : 300, pausePolling: false);
      final bytes = _extractBytes(r, pid.answer);
      if (bytes.length >= pid.bytesCount) {
        final val = pid.formula(bytes);
        _values[pid.id]   = val;
        _values[pid.name] = val;
        _rawData[pid.cmd]  = bytes;
      }
    } catch (_) {}
  }

  void _publish() {
    final mafVolt   = _v('MAF_V');
    final mafGpsRaw = AppConstants.mafVoltToGps(mafVolt);
    final mafMult   = _profile?.mafMultiplier ?? 1.0;
    final mafGps    = mafGpsRaw * mafMult;

    final speedRaw  = _v('SPEED');
    final speedMult = _profile?.speedMultiplier ?? 1.0;
    final speed     = (speedRaw * speedMult).toInt().clamp(0, 300);
    final disp      = _profile?.displacement ?? 2.0;

    final o2   = _v('O2_B1S1');
    final stft = _v('STFT');
    final afr  = _calcAfr(o2, stft);

    final data = OBDData(
      timestamp:          DateTime.now(),
      rpm:                _v('RPM').toInt().clamp(0, 9999),
      speed:              speed,
      engineLoad:         _v('LOAD').clamp(0, 100),
      coolantTemp:        _v('ECT').toInt().clamp(-40, 200),
      intakeTemp:         _v('IAT').toInt().clamp(-40, 100),
      mafVoltage:         mafVolt,
      mafGps:             mafGps,
      throttlePos:        _v('TPS').clamp(0, 100),
      ignitionTiming:     _v('TIMING'),
      actualIgnition:     _v('TIMING'),
      vtcActualAngle:     _v('VTC_ACT'),
      knockRetard:        _v('KNOCK').abs(),
      shortFuelTrim:      _v('STFT').clamp(-100, 100),
      longFuelTrim:       _v('LTFT').clamp(-100, 100),
      o2Voltage:          o2,
      afr:                afr,
      injectorPulseWidth: _v('INJ_B1'),
      injectorDuty:       (_v('INJ_B1') / 20.0 * 100).clamp(0, 100),
      manifoldPressure:   _v('MAP_V') * 40,
      acceleratorPedal:   _v('PEDAL'),
      throttleActual:     _v('TPS'),
      batteryVoltage:     _v('BATT'),
      engineDisplacement: disp,
      tripFuelL:          _tripFuelL,
      actualTorque:       _v('TORQUE'),
      requestedTorque:    _v('POWER_KW'),
    );

    _dataCtrl.add(data);
  }

  double _v(String key) => _values[key] ?? 0;

  double _calcAfr(double o2, double stft) {
    double lambda;
    if      (o2 > 0.85) lambda = 0.87;
    else if (o2 > 0.75) lambda = 0.92;
    else if (o2 > 0.60) lambda = 0.97;
    else if (o2 > 0.45) lambda = 1.00;
    else if (o2 > 0.30) lambda = 1.03;
    else if (o2 > 0.15) lambda = 1.05;
    else                lambda = 1.10;
    lambda *= (1 + stft / 100.0 * 0.3);
    return (lambda * 14.7).clamp(10.0, 20.0);
  }

  Future<String> sendCommand(
    String cmd, {
    int  timeout      = 1000,
    bool pausePolling = true,
  }) async {
    if (!isConnected) return '';

    if (pausePolling && _isPolling) {
      _pollPaused = true;
      int wait = 0;
      while (_cmdInProgress && wait < 30) {
        await Future.delayed(const Duration(milliseconds: 5));
        wait++;
      }
    }

    int guard = 0;
    while (_cmdInProgress && guard < 50) {
      await Future.delayed(const Duration(milliseconds: 2));
      guard++;
    }
    if (_cmdInProgress) {
      _cmdInProgress = false;
      _cmdCompleter  = null;
    }

    _rxBuf.clear();
    _cmdInProgress = true;
    _cmdCompleter  = Completer<String>();

    try {
      final bytes = [...cmd.codeUnits, 13];
      _connection!.output.add(Uint8List.fromList(bytes));
      await _connection!.output.allSent;

      String resp = '';
      try {
        resp = await _cmdCompleter!.future
            .timeout(Duration(milliseconds: timeout));
      } catch (_) {
        resp = _rxBuf.toString();
      }

      _cmdInProgress = false;
      _cmdCompleter  = null;

      if (pausePolling && _isPolling) {
        await Future.delayed(const Duration(milliseconds: 10));
        _pollPaused = false;
      }

      return _clean(resp);
    } catch (e) {
      _cmdInProgress = false;
      _cmdCompleter  = null;
      if (pausePolling) _pollPaused = false;
      return '';
    }
  }

  String _clean(String r) =>
      r.replaceAll('>','').replaceAll('\r',' ').replaceAll('\n',' ')
       .replaceAll('  ',' ').trim();

  void _onData(Uint8List data) {
    final s = String.fromCharCodes(data);
    _rxBuf.write(s);
    if (s.contains('>') && _cmdCompleter != null && !_cmdCompleter!.isCompleted) {
      _cmdCompleter!.complete(_rxBuf.toString());
    }
  }

  void _onDisconnected() {
    _log('BT DISCONNECTED');
    stopPolling();
    _initialized   = false;
    _ecuResponds   = false;
    _cmdInProgress = false;
    _cmdCompleter  = null;
    _connection    = null;

    if (_autoReconnect && _lastAddress != null && _reconnectTries < _maxReconnect) {
      _scheduleReconnect();
    }
  }

  void _scheduleReconnect() {
    _reconnectTries++;
    _reconnectTimer?.cancel();
    _reconnectTimer = Timer(
      Duration(seconds: 3 * _reconnectTries),
      () async {
        if (_lastAddress == null) return;
        final ok = await connect(_lastAddress!);
        if (ok) await initECU(useCache: true);
        else if (_reconnectTries < _maxReconnect) _scheduleReconnect();
      },
    );
  }

  Future<void> disconnect() async {
    _autoReconnect = false;
    _reconnectTimer?.cancel();
    stopPolling();
    _initialized   = false;
    _ecuResponds   = false;
    _cmdInProgress = false;
    _cmdCompleter  = null;
    await _inputSub?.cancel();
    _inputSub = null;
    await _connection?.close();
    _connection = null;
    _autoReconnect = true;
  }

  void dispose() {
    disconnect();
    _dataCtrl.close();
    _logCtrl.close();
  }

  List<int> _extractBytes(String resp, String prefix) {
    final s = resp.replaceAll(' ','').replaceAll('BUSINIT:OK','').toUpperCase();
    final idx = s.indexOf(prefix);
    if (idx < 0) return [];
    final hex = s.substring(idx + prefix.length);
    final result = <int>[];
    for (int i = 0; i + 1 < hex.length; i += 2) {
      final h = hex.substring(i, i + 2);
      if (!RegExp(r'^[0-9A-F]+$').hasMatch(h)) break;
      try { result.add(int.parse(h, radix: 16)); } catch (_) { break; }
    }
    return result;
  }

  String _hexToAscii(String hex) {
    final sb = StringBuffer();
    for (int i = 0; i + 1 < hex.length; i += 2) {
      try {
        final b = int.parse(hex.substring(i, i + 2), radix: 16);
        if (b >= 0x20 && b <= 0x7E) sb.writeCharCode(b);
      } catch (_) { break; }
    }
    return sb.toString();
  }
}
''')
print("✅ obd_service.dart — Полная реализация Dual-Speed (57.6k)!")



✅ obd_service.dart — Полная реализация Dual-Speed (57.6k)!

🔨 Пересборка APK...

⚠️ ОШИБКИ:
lib/screens/home_screen.dart:55:10: Error: The method 'loadTripFuel' isn't defined for the type 'OBDService'.
lib/screens/settings_screen.dart:284:41: Error: The method 'resetTripFuel' isn't defined for the type 'OBDService'.
Target kernel_snapshot_program failed: Exception
FAILURE: Build failed with an exception.

📋 Итог:
Target kernel_snapshot_program failed: Exception
FAILURE: Build failed with an exception.
Execution failed for task ':app:compileFlutterBuildRelease'.
BUILD FAILED in 36s
Gradle task assembleRelease failed with exit code 1

❌ APK НЕ СОБРАН
Running Gradle task 'assembleRelease'...                        
lib/screens/home_screen.dart:55:10: Error: The method 'loadTripFuel' isn't defined for the type 'OBDService'.
 - 'OBDService' is from 'package:nissan_logger_v6/services/obd_service.dart' ('lib/services/obd_service.dart').
Try correcting the name to the name of an existing metho

In [ ]:
# @title 🔧 ФИКС: Восстановление методов tripFuel + высокоскоростной старт ELM327
import os
os.chdir('/content/nissan_logger_v6')

# ================================================================
# obd_service.dart — со всеми методами и разгоном K-Line (57.6k)
# ================================================================
with open('lib/services/obd_service.dart', 'w') as f:
    f.write(r'''import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import '../models/obd_data.dart';
import '../models/vehicle_profile.dart';
import '../models/custom_pid.dart';
import '../constants.dart';
import 'nissan_pid_library.dart';
import 'settings_service.dart';
import 'formula_evaluator.dart';

class OBDService {
  BluetoothConnection? _connection;
  StreamSubscription?  _inputSub;
  final StringBuffer   _rxBuf = StringBuffer();

  bool       _cmdInProgress = false;
  Completer<String>? _cmdCompleter;

  bool _isPolling    = false;
  bool _pollPaused   = false;
  int  _pollCounter  = 0;
  int  _mediumIdx    = 0;
  int  _slowIdx      = 0;
  double _pollFps    = 0.0;
  int    _lastPollMs = 0;

  bool   _ecuResponds  = false;
  bool   _initialized  = false;
  String _protocolInfo = '';
  String _ecuId        = '';
  bool   _isHighSpeed  = false;

  List<NissanPidDef> _scannedPids = [];
  List<NissanPidDef> _activePids  = [];
  List<NissanPidDef> _fastPids    = [];
  List<NissanPidDef> _medPids     = [];
  List<NissanPidDef> _slowPids    = [];
  final Map<String, double>    _values  = {};
  final Map<String, List<int>> _rawData = {};

  VehicleProfile? _profile;

  double    _tripFuelL  = 0;
  DateTime? _lastFuelTs;

  bool   _autoReconnect   = true;
  int    _reconnectTries  = 0;
  String? _lastAddress;
  Timer?  _reconnectTimer;
  static const _maxReconnect = 3;

  final _dataCtrl = StreamController<OBDData>.broadcast();
  final _logCtrl  = StreamController<String>.broadcast();

  Stream<OBDData> get dataStream => _dataCtrl.stream;
  Stream<String>  get logStream  => _logCtrl.stream;

  bool   get isConnected   => _connection?.isConnected ?? false;
  bool   get isInitialized => _initialized;
  bool   get ecuResponds   => _ecuResponds;
  String get protocolInfo  => _protocolInfo;
  String get ecuId         => _ecuId;
  bool   get isHighSpeed   => _isHighSpeed;
  int    get pollFps       => _pollFps.toInt();
  int    get lastPollMs    => _lastPollMs;
  double get tripFuelL     => _tripFuelL;
  List<NissanPidDef>    get activePids => _activePids;
  Map<String, double>    get pidValues  => Map.unmodifiable(_values);

  void applyProfile(VehicleProfile profile) {
    _profile = profile;
    _tripFuelL = SettingsService.tripFuelL;
    _rebuildPidLists();
  }

  // ── Методы расчёта поездки ──────────────────────────────────
  Future<void> resetTripFuel() async {
    _tripFuelL = 0;
    _lastFuelTs = null;
    await SettingsService.resetTripFuel();
  }

  Future<void> loadTripFuel() async {
    await SettingsService.init();
    _tripFuelL = SettingsService.tripFuelL;
  }

  void _log(String msg) => _logCtrl.add(msg);

  Future<List<BluetoothDevice>> getBondedDevices() async {
    try { return await FlutterBluetoothSerial.instance.getBondedDevices(); }
    catch (_) { return []; }
  }

  Future<BluetoothState> getBluetoothState() async =>
      FlutterBluetoothSerial.instance.state;

  Future<bool?> requestEnable() async =>
      FlutterBluetoothSerial.instance.requestEnable();

  Future<bool> connect(String address) async {
    try {
      _log('=== BT $address ===');
      _initialized  = false;
      _ecuResponds  = false;
      _lastAddress  = address;
      _reconnectTries = 0;

      _connection = await BluetoothConnection.toAddress(address);
      _log('BT OK');

      _inputSub = _connection!.input!.listen(
        _onData,
        onDone:  _onDisconnected,
        onError: (e) => _log('BT Error: $e'),
      );

      await Future.delayed(const Duration(milliseconds: 1500));
      _rxBuf.clear();
      _cmdInProgress = false;
      _cmdCompleter  = null;

      _connection!.output.add(Uint8List.fromList([13, 13, 13]));
      await _connection!.output.allSent;
      await Future.delayed(const Duration(milliseconds: 500));
      _rxBuf.clear();

      final r = await sendCommand('ATZ', timeout: 5000);
      _log('ATZ: [$r]');
      await Future.delayed(const Duration(milliseconds: 1500));

      if (r.toUpperCase().contains('ELM')) {
        _initialized = true;
        await SettingsService.setLastBtDevice(address);
        _log('ELM OK');
        return true;
      }
      return false;
    } catch (e) {
      _log('ERR: $e');
      return false;
    }
  }

  // ── Инициализация ЭБУ ────────────────────────────────────────
  Future<bool> initECU({bool useCache = true, bool enableHighSpeed = true}) async {
    if (!isConnected) return false;
    _log('=== INIT ECU (Nissan 1EQ010) ===');
    _ecuResponds = false;
    _isHighSpeed = false;
    _rawData.clear();
    _scannedPids.clear();

    // 1. Установка параметров ELM327 и попытка разгона до 57600
    await sendCommand('ATZ', timeout: 4000);
    await Future.delayed(const Duration(milliseconds: 800));

    if (enableHighSpeed) {
      _log('🚀 Настройка ELM327 на 57 600 Baud (PP 2D)...');
      await sendCommand('ATSP5', timeout: 1000);
      await sendCommand('ATPP2DSV15', timeout: 1000);
      await sendCommand('ATPP2DON', timeout: 1000);
      await sendCommand('ATZ', timeout: 2000); // Ресет для применения PP
      await Future.delayed(const Duration(milliseconds: 800));
    }

    for (final cmd in ['ATE0','ATL0','ATS0','ATH0','ATAL','ATSW00',
                        'ATST19','ATAT2','ATSP5','ATSH8110FC']) {
      await sendCommand(cmd, timeout: 1500);
    }

    final initResp = await sendCommand('ATFI', timeout: 4000);
    _log('ATFI: [$initResp]');
    await Future.delayed(const Duration(milliseconds: 200));

    final r = await sendCommand('2211000401', timeout: 6000);
    if (!r.replaceAll(' ','').toUpperCase().contains('6211')) {
      _log('⚠️ Сбой соединения. Пробуем стандартную скорость 10400...');
      _isHighSpeed = false;
      await sendCommand('ATPP2DOFF', timeout: 1000);
      await sendCommand('ATZ', timeout: 2000);
      await Future.delayed(const Duration(milliseconds: 800));
      for (final cmd in ['ATE0','ATL0','ATS0','ATH0','ATAL','ATSW00',
                          'ATST19','ATAT2','ATIB10','ATSP5','ATSH8110FC']) {
        await sendCommand(cmd, timeout: 1200);
      }
      await sendCommand('ATFI', timeout: 3000);
      final r2 = await sendCommand('2211000401', timeout: 6000);
      if (!r2.replaceAll(' ','').toUpperCase().contains('6211')) {
        _log('❌ ЭБУ не отвечает');
        return false;
      }
    } else {
      if (enableHighSpeed) _isHighSpeed = true;
    }

    _log('✅ ЭБУ ОТВЕЧАЕТ! (HighSpeed: $_isHighSpeed)');
    _ecuResponds = true;

    final idR = await sendCommand('1A81', timeout: 3000);
    final idC = idR.replaceAll(' ','').toUpperCase();
    if (idC.contains('5A')) {
      final idx = idC.indexOf('5A');
      _ecuId = _hexToAscii(idC.substring(idx + 2));
      _log('ECU ID: $_ecuId');
    }

    if (useCache) {
      final ce = SettingsService.cachedEcuId;
      final cp = SettingsService.cachedPidList;
      if (ce == _ecuId && cp.isNotEmpty) {
        _log('CACHE ${cp.length} pids');
        for (final name in cp) {
          final pid = NissanPidLibrary.byId(name);
          if (pid != null) _scannedPids.add(pid);
        }
        _rebuildPidLists();
        _protocolInfo = 'Nissan $_ecuId ${_isHighSpeed ? "[57.6k HighSpeed]" : "[10.4k]"}';
        Future.delayed(const Duration(milliseconds: 300), startPolling);
        return true;
      }
    }

    _log('SCAN ${NissanPidLibrary.all.length} pids...');
    for (final pid in NissanPidLibrary.all) {
      final resp = await sendCommand(pid.cmd, timeout: _isHighSpeed ? 250 : 600);
      final clean = resp.replaceAll(' ','').toUpperCase();
      if (clean.contains(pid.answer)) {
        final bytes = _extractBytes(resp, pid.answer);
        if (bytes.length >= pid.bytesCount) _scannedPids.add(pid);
      }
      await Future.delayed(Duration(milliseconds: _isHighSpeed ? 5 : 20));
    }
    _rebuildPidLists();
    _log('Найдено ${_scannedPids.length} PID');

    if (_activePids.isEmpty) return false;

    await SettingsService.setCachedEcuId(_ecuId);
    await SettingsService.setCachedPidList(_scannedPids.map((p) => p.id).toList());

    _protocolInfo = 'Nissan $_ecuId (${_activePids.length} pid) ${_isHighSpeed ? "[57.6k HighSpeed]" : "[10.4k]"}';
    Future.delayed(const Duration(milliseconds: 300), startPolling);
    return true;
  }

  void _rebuildPidLists() {
    final p = _profile;
    final deletedSet = p?.deletedPidIds.toSet() ?? {};
    final customPids = p?.customPids ?? <CustomPid>[];
    final customMap  = {for (var cp in customPids) cp.id: cp};

    final baseList = _scannedPids.isNotEmpty ? _scannedPids : NissanPidLibrary.all;
    final effective = <NissanPidDef>[];

    for (final def in baseList) {
      if (deletedSet.contains(def.id)) continue;
      if (customMap.containsKey(def.id)) {
        effective.add(_customToDef(customMap[def.id]!));
      } else {
        effective.add(def);
      }
    }

    for (final c in customPids) {
      if (c.status == PidStatus.userAdded && !effective.any((def) => def.id == c.id)) {
        effective.add(_customToDef(c));
      }
    }

    _activePids = effective;
    _fastPids   = _activePids.where((p) => p.priority == 1).toList();
    _medPids    = _activePids.where((p) => p.priority == 2).toList();
    _slowPids   = _activePids.where((p) => p.priority == 3).toList();
  }

  NissanPidDef _customToDef(CustomPid c) {
    final ev = FormulaEvaluator(c.formula);
    return NissanPidDef(
      id: c.id, cmd: c.cmd, answer: c.answer,
      name: c.name, desc: c.desc, unit: c.unit,
      bytesCount: c.bytesCount,
      formula: (b) {
        final raw = c.bytesCount == 2 && b.length >= 2
            ? (b[0] * 256 + b[1]) : (b.isNotEmpty ? b[0] : 0);
        return ev.evaluate(raw);
      },
      minVal: c.minVal, maxVal: c.maxVal,
      priority: c.priority, category: c.category,
    );
  }

  void startPolling() {
    if (_isPolling || !_ecuResponds || _activePids.isEmpty) return;
    _isPolling = true;
    _log('POLL START (HighSpeed: $_isHighSpeed)');
    _pollLoop();
  }

  void stopPolling() { _isPolling = false; }

  Future<void> _pollLoop() async {
    final fpsTimer = Stopwatch()..start();
    int fpsCount = 0;

    while (_isPolling && isConnected && _ecuResponds) {
      while (_pollPaused && _isPolling) {
        await Future.delayed(const Duration(milliseconds: 5));
      }
      if (!_isPolling) break;

      _pollCounter++;
      final sw = Stopwatch()..start();

      for (final pid in _fastPids) {
        if (!_isPolling || _pollPaused) break;
        await _pollPid(pid);
      }

      if (_pollCounter % 2 == 0 && _medPids.isNotEmpty && !_pollPaused) {
        for (int i = 0; i < 2 && !_pollPaused; i++) {
          await _pollPid(_medPids[_mediumIdx % _medPids.length]);
          _mediumIdx++;
        }
      }

      if (_pollCounter % 5 == 0 && _slowPids.isNotEmpty && !_pollPaused) {
        await _pollPid(_slowPids[_slowIdx % _slowPids.length]);
        _slowIdx++;
      }

      sw.stop();
      _lastPollMs = sw.elapsedMilliseconds;
      fpsCount++;

      if (fpsTimer.elapsedMilliseconds >= 1000) {
        _pollFps = fpsCount * 1000.0 / fpsTimer.elapsedMilliseconds;
        fpsCount = 0;
        fpsTimer.reset();
      }

      _publish();

      final baseInterval = _profile?.pollingInterval ?? SettingsService.pollingInterval;
      final actualInterval = _isHighSpeed ? 5 : baseInterval;

      if (actualInterval > 0) {
        await Future.delayed(Duration(milliseconds: actualInterval));
      }
    }
  }

  Future<void> _pollPid(NissanPidDef pid) async {
    try {
      final r = await sendCommand(pid.cmd, timeout: _isHighSpeed ? 150 : 300, pausePolling: false);
      final bytes = _extractBytes(r, pid.answer);
      if (bytes.length >= pid.bytesCount) {
        final val = pid.formula(bytes);
        _values[pid.id]   = val;
        _values[pid.name] = val;
        _rawData[pid.cmd]  = bytes;
      }
    } catch (_) {}
  }

  void _publish() {
    final mafVolt   = _v('MAF_V');
    final mafGpsRaw = AppConstants.mafVoltToGps(mafVolt);
    final mafMult   = _profile?.mafMultiplier ?? 1.0;
    final mafGps    = mafGpsRaw * mafMult;

    final speedRaw  = _v('SPEED');
    final speedMult = _profile?.speedMultiplier ?? 1.0;
    final speed     = (speedRaw * speedMult).toInt().clamp(0, 300);
    final disp      = _profile?.displacement ?? 2.0;

    final o2   = _v('O2_B1S1');
    final stft = _v('STFT');
    final afr  = _calcAfr(o2, stft);

    final data = OBDData(
      timestamp:          DateTime.now(),
      rpm:                _v('RPM').toInt().clamp(0, 9999),
      speed:              speed,
      engineLoad:         _v('LOAD').clamp(0, 100),
      coolantTemp:        _v('ECT').toInt().clamp(-40, 200),
      intakeTemp:         _v('IAT').toInt().clamp(-40, 100),
      mafVoltage:         mafVolt,
      mafGps:             mafGps,
      throttlePos:        _v('TPS').clamp(0, 100),
      ignitionTiming:     _v('TIMING'),
      actualIgnition:     _v('TIMING'),
      vtcActualAngle:     _v('VTC_ACT'),
      knockRetard:        _v('KNOCK').abs(),
      shortFuelTrim:      _v('STFT').clamp(-100, 100),
      longFuelTrim:       _v('LTFT').clamp(-100, 100),
      o2Voltage:          o2,
      afr:                afr,
      injectorPulseWidth: _v('INJ_B1'),
      injectorDuty:       (_v('INJ_B1') / 20.0 * 100).clamp(0, 100),
      manifoldPressure:   _v('MAP_V') * 40,
      acceleratorPedal:   _v('PEDAL'),
      throttleActual:     _v('TPS'),
      batteryVoltage:     _v('BATT'),
      engineDisplacement: disp,
      tripFuelL:          _tripFuelL,
      actualTorque:       _v('TORQUE'),
      requestedTorque:    _v('POWER_KW'),
    );

    _dataCtrl.add(data);
  }

  double _v(String key) => _values[key] ?? 0;

  double _calcAfr(double o2, double stft) {
    double lambda;
    if      (o2 > 0.85) lambda = 0.87;
    else if (o2 > 0.75) lambda = 0.92;
    else if (o2 > 0.60) lambda = 0.97;
    else if (o2 > 0.45) lambda = 1.00;
    else if (o2 > 0.30) lambda = 1.03;
    else if (o2 > 0.15) lambda = 1.05;
    else                lambda = 1.10;
    lambda *= (1 + stft / 100.0 * 0.3);
    return (lambda * 14.7).clamp(10.0, 20.0);
  }

  Future<String> sendCommand(
    String cmd, {
    int  timeout      = 1000,
    bool pausePolling = true,
  }) async {
    if (!isConnected) return '';

    if (pausePolling && _isPolling) {
      _pollPaused = true;
      int wait = 0;
      while (_cmdInProgress && wait < 30) {
        await Future.delayed(const Duration(milliseconds: 5));
        wait++;
      }
    }

    int guard = 0;
    while (_cmdInProgress && guard < 50) {
      await Future.delayed(const Duration(milliseconds: 2));
      guard++;
    }
    if (_cmdInProgress) {
      _cmdInProgress = false;
      _cmdCompleter  = null;
    }

    _rxBuf.clear();
    _cmdInProgress = true;
    _cmdCompleter  = Completer<String>();

    try {
      final bytes = [...cmd.codeUnits, 13];
      _connection!.output.add(Uint8List.fromList(bytes));
      await _connection!.output.allSent;

      String resp = '';
      try {
        resp = await _cmdCompleter!.future
            .timeout(Duration(milliseconds: timeout));
      } catch (_) {
        resp = _rxBuf.toString();
      }

      _cmdInProgress = false;
      _cmdCompleter  = null;

      if (pausePolling && _isPolling) {
        await Future.delayed(const Duration(milliseconds: 10));
        _pollPaused = false;
      }

      return _clean(resp);
    } catch (e) {
      _cmdInProgress = false;
      _cmdCompleter  = null;
      if (pausePolling) _pollPaused = false;
      return '';
    }
  }

  String _clean(String r) =>
      r.replaceAll('>','').replaceAll('\r',' ').replaceAll('\n',' ')
       .replaceAll('  ',' ').trim();

  void _onData(Uint8List data) {
    final s = String.fromCharCodes(data);
    _rxBuf.write(s);
    if (s.contains('>') && _cmdCompleter != null && !_cmdCompleter!.isCompleted) {
      _cmdCompleter!.complete(_rxBuf.toString());
    }
  }

  void _onDisconnected() {
    _log('BT DISCONNECTED');
    stopPolling();
    _initialized   = false;
    _ecuResponds   = false;
    _cmdInProgress = false;
    _cmdCompleter  = null;
    _connection    = null;

    if (_autoReconnect && _lastAddress != null && _reconnectTries < _maxReconnect) {
      _scheduleReconnect();
    }
  }

  void _scheduleReconnect() {
    _reconnectTries++;
    _reconnectTimer?.cancel();
    _reconnectTimer = Timer(
      Duration(seconds: 3 * _reconnectTries),
      () async {
        if (_lastAddress == null) return;
        final ok = await connect(_lastAddress!);
        if (ok) await initECU(useCache: true);
        else if (_reconnectTries < _maxReconnect) _scheduleReconnect();
      },
    );
  }

  Future<void> disconnect() async {
    _autoReconnect = false;
    _reconnectTimer?.cancel();
    stopPolling();
    _initialized   = false;
    _ecuResponds   = false;
    _cmdInProgress = false;
    _cmdCompleter  = null;
    await _inputSub?.cancel();
    _inputSub = null;
    await _connection?.close();
    _connection = null;
    _autoReconnect = true;
  }

  void dispose() {
    disconnect();
    _dataCtrl.close();
    _logCtrl.close();
  }

  List<int> _extractBytes(String resp, String prefix) {
    final s = resp.replaceAll(' ','').replaceAll('BUSINIT:OK','').toUpperCase();
    final idx = s.indexOf(prefix);
    if (idx < 0) return [];
    final hex = s.substring(idx + prefix.length);
    final result = <int>[];
    for (int i = 0; i + 1 < hex.length; i += 2) {
      final h = hex.substring(i, i + 2);
      if (!RegExp(r'^[0-9A-F]+$').hasMatch(h)) break;
      try { result.add(int.parse(h, radix: 16)); } catch (_) { break; }
    }
    return result;
  }

  String _hexToAscii(String hex) {
    final sb = StringBuffer();
    for (int i = 0; i + 1 < hex.length; i += 2) {
      try {
        final b = int.parse(hex.substring(i, i + 2), radix: 16);
        if (b >= 0x20 && b <= 0x7E) sb.writeCharCode(b);
      } catch (_) { break; }
    }
    return sb.toString();
  }
}
''')
print("✅ obd_service.dart — методы loadTripFuel и resetTripFuel восстановлены")

# ── Пересборка ──────────────────────────────────────────────────
print()
print("=" * 60)
print("🔨 Пересборка APK...")
print("=" * 60)

apk = '/content/nissan_logger_v6/build/app/outputs/flutter-apk/app-release.apk'
if os.path.exists(apk):
    os.remove(apk)

os.environ['JAVA_HOME']          = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH']               = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH','')
os.environ['PATH']               = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['ANDROID_HOME']       = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT']   = '/content/android-sdk'
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

result = !JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64 \
  CMAKE_MAKE_PROGRAM=/usr/bin/ninja \
  flutter build apk --release --no-tree-shake-icons \
    --android-skip-build-dependency-validation 2>&1

errors = [l for l in result if 'error:' in l.lower() or 'exception' in l.lower()]
if errors:
    print("\n⚠️ ОШИБКИ:")
    for line in errors[-30:]:
        print(line)

important = [l for l in result if any(k in l.lower() for k in
  ['failed', 'built', 'app-release.apk'])]
print("\n📋 Итог:")
for line in important[-10:]:
    print(line)

if os.path.exists(apk):
    mb = os.path.getsize(apk) / 1048576
    print(f"\n🎉 APK СОБРАН! {mb:.1f} MB")
    !cp {apk} /content/NissanLoggerV6.apk
    from google.colab import files
    files.download('/content/NissanLoggerV6.apk')
else:
    print("\n❌ APK НЕ СОБРАН")
    for line in result[-60:]:
        print(line)

✅ obd_service.dart — методы loadTripFuel и resetTripFuel восстановлены

🔨 Пересборка APK...

📋 Итог:
✓ Built build/app/outputs/flutter-apk/app-release.apk (60.0MB)

🎉 APK СОБРАН! 57.2 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title 🔨 Ячейка 5/5: Сборка APK V6 (~10-15 мин)
import os
import glob

os.chdir('/content/nissan_logger_v6')

# ── Переменные окружения ─────────────────────────────────────
os.environ['JAVA_HOME']       = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH']            = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ.get('PATH','')
os.environ['PATH']            = '/content/flutter/bin:/content/flutter/bin/cache/dart-sdk/bin:' + os.environ['PATH']
os.environ['PUB_CACHE']       = '/content/.pub-cache'
os.environ['ANDROID_HOME']    = '/content/android-sdk'
os.environ['ANDROID_SDK_ROOT']= '/content/android-sdk'
os.environ['PATH']            = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + os.environ['PATH']
os.environ['CMAKE_MAKE_PROGRAM'] = '/usr/bin/ninja'

print("=" * 60)
print("Java:")
print("=" * 60)
!java -version 2>&1 | head -1

# ── ЭТАП 1: pub get ──────────────────────────────────────────
print()
print("=" * 60)
print("📦 ЭТАП 1: flutter clean + pub get")
print("=" * 60)

!flutter clean 2>&1 | tail -3
!rm -rf build
!rm -f pubspec.lock

pub_result = !flutter pub get 2>&1
for line in pub_result[-10:]:
    print(line)

has_error = any('error' in l.lower() for l in pub_result[-5:])
if has_error:
    print("\n⚠️ pub get с проблемами")
else:
    print("\n✅ pub get OK")

# ── ЭТАП 2: Патч Bluetooth плагина ────────────────────────────
print()
print("=" * 60)
print("🔧 ЭТАП 2: Патч flutter_bluetooth_serial")
print("=" * 60)

plugin_dirs = glob.glob('/content/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*')
if not plugin_dirs:
    plugin_dirs = glob.glob('/root/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-*')

for plugin_dir in plugin_dirs:
    # build.gradle
    with open(f'{plugin_dir}/android/build.gradle', 'w') as f:
        f.write('''group 'io.github.edufolly.flutterbluetoothserial'
version '1.0-SNAPSHOT'

buildscript {
    repositories {
        google()
        mavenCentral()
    }
    dependencies {
        classpath 'com.android.tools.build:gradle:8.6.0'
    }
}

allprojects {
    repositories {
        google()
        mavenCentral()
    }
}

apply plugin: 'com.android.library'

android {
    namespace 'io.github.edufolly.flutterbluetoothserial'
    compileSdk 36

    compileOptions {
        sourceCompatibility JavaVersion.VERSION_17
        targetCompatibility JavaVersion.VERSION_17
    }

    defaultConfig {
        minSdk 21
    }

    lintOptions {
        disable 'InvalidPackage'
        checkReleaseBuilds false
        abortOnError false
    }
}

dependencies {
    implementation 'androidx.core:core:1.13.1'
}
''')
    # AndroidManifest.xml
    with open(f'{plugin_dir}/android/src/main/AndroidManifest.xml', 'w') as f:
        f.write('''<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-feature android:name="android.hardware.bluetooth" android:required="true" />
    <uses-permission android:name="android.permission.BLUETOOTH" />
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" />
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT" />
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" />
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" />
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" />
</manifest>
''')
    print(f"✅ Bluetooth плагин пропатчен: {plugin_dir.split('/')[-1]}")

if not plugin_dirs:
    print("⚠️ Bluetooth плагин не найден в кеше")

# ── ЭТАП 3: Проверка структуры ────────────────────────────────
print()
print("=" * 60)
print("📁 ЭТАП 3: Проверка структуры проекта")
print("=" * 60)

required_files = [
    'lib/main.dart',
    'lib/constants.dart',
    'lib/models/obd_data.dart',
    'lib/models/tuning_map.dart',
    'lib/models/analysis_result.dart',
    'lib/models/alert.dart',
    'lib/models/dtc_code.dart',
    'lib/models/custom_pid.dart',
    'lib/models/vehicle_profile.dart',
    'lib/models/custom_map_def.dart',
    'lib/services/settings_service.dart',
    'lib/services/profile_service.dart',
    'lib/services/nissan_pid_library.dart',
    'lib/services/formula_evaluator.dart',
    'lib/services/obd_service.dart',
    'lib/services/logger_service.dart',
    'lib/services/dtc_service.dart',
    'lib/services/dtc_database.dart',
    'lib/services/alert_service.dart',
    'lib/services/map_storage_service.dart',
    'lib/services/map_history_service.dart',
    'lib/services/rom_map_reader.dart',
    'lib/services/analyzer_service.dart',
    'lib/services/tuning_service.dart',
    'lib/services/ecu_map_reader.dart',
    'lib/services/nissan_unlock.dart',
    'lib/services/performance_service.dart',
    'lib/services/export_service.dart',
    'lib/widgets/fps_indicator.dart',
    'lib/widgets/map_table_view.dart',
    'lib/screens/home_screen.dart',
    'lib/screens/dashboard_screen.dart',
    'lib/screens/settings_screen.dart',
    'lib/screens/graph_screen.dart',
    'lib/screens/log_graph_screen.dart',
    'lib/screens/logging_screen.dart',
    'lib/screens/events_screen.dart',
    'lib/screens/dtc_screen.dart',
    'lib/screens/analyzer_screen.dart',
    'lib/screens/ecu_read_screen.dart',
    'lib/screens/service_screen.dart',
    'lib/screens/performance_screen.dart',
    'lib/screens/export_screen.dart',
    'lib/screens/custom_pid_screen.dart',
    'lib/screens/profile_screen.dart',
    'lib/screens/rom_compare_screen.dart',
    'lib/screens/terminal_screen.dart',
]

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) if exists else 0
    status = '✅' if exists and size > 50 else '❌'
    if not exists or size < 50:
        all_ok = False
        print(f"  {status} {f} ({size} B)")

if all_ok:
    print(f"  ✅ Все {len(required_files)} файлов на месте")
else:
    print(f"\n⚠️ Некоторые файлы отсутствуют!")

# Считаем общий размер кода
total_lines = 0
total_bytes = 0
for root, dirs, files in os.walk('lib'):
    for fname in files:
        if fname.endswith('.dart'):
            fpath = os.path.join(root, fname)
            sz = os.path.getsize(fpath)
            total_bytes += sz
            with open(fpath, 'r') as f:
                total_lines += len(f.readlines())

print(f"\n  📊 Всего: {total_lines} строк Dart, {total_bytes/1024:.1f} KB")

# ── ЭТАП 4: СБОРКА ───────────────────────────────────────────
print()
print("=" * 60)
print("🔨 ЭТАП 4: СБОРКА APK V6 (~10-15 минут)")
print("=" * 60)

result = !JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64 \
  CMAKE_MAKE_PROGRAM=/usr/bin/ninja \
  flutter build apk \
    --release \
    --no-tree-shake-icons \
    --android-skip-build-dependency-validation 2>&1

# Показываем ключевые строки
important = []
for line in result:
    ll = line.lower()
    if any(k in ll for k in ['error:', 'failed', 'built', 'app-release.apk',
                               'exception', 'warning:', 'note:']):
        important.append(line)

print("\n📋 Ключевые события:")
for line in important[-50:]:
    print(line)

# ── РЕЗУЛЬТАТ ─────────────────────────────────────────────────
print()
print("=" * 60)
print("📱 РЕЗУЛЬТАТ")
print("=" * 60)

apk_path = '/content/nissan_logger_v6/build/app/outputs/flutter-apk/app-release.apk'

if os.path.exists(apk_path):
    size_mb = os.path.getsize(apk_path) / (1024 * 1024)

    print(f"\n🎉🎉🎉 APK V6 СОБРАН УСПЕШНО! 🎉🎉🎉")
    print(f"📁 Путь: {apk_path}")
    print(f"📏 Размер: {size_mb:.1f} MB")
    print(f"📊 Код: {total_lines} строк Dart")

    # Копируем и скачиваем
    output_name = 'NissanLoggerV6.apk'
    !cp {apk_path} /content/{output_name}

    print(f"\n📥 Скачиваю {output_name}...")
    from google.colab import files
    files.download(f'/content/{output_name}')

    print()
    print("=" * 60)
    print("✅ ЧТО В V6:")
    print("=" * 60)
    print()
    print("📱 16 вкладок:")
    print("  1.  Приборы     — настраиваемый дашборд (25 параметров, long-press)")
    print("  2.  Графики     — 4 профиля реального времени")
    print("  3.  ЛогГраф     — CSV + zoom + статистика + совмещённый")
    print("  4.  Лог         — start/stop/автолог/share")
    print("  5.  События     — алерты со snapshot + explanation")
    print("  6.  DTC         — A330 команда Nissan + русские описания")
    print("  7.  Анализатор  — Онлайн + Из лога + СПЛИТТЕР + паттерны")
    print("  8.  ЭБУ Карты   — АВТО + UNLOCK + чтение + ЗАПИСЬ В .BIN")
    print("  9.  Сервис      — тесты с repeater + обучения")
    print("  10. Замер       — 0-60, 0-100, 400м")
    print("  11. Экспорт     — WinOLS/ecuEdit/HEX")
    print("  12. PID         — CRUD (добавить/удалить кастомные)")
    print("  13. Профили     — ВСЕ настройки в профиле")
    print("  14. ROM Diff    — СРАВНЕНИЕ двух прошивок")
    print("  15. Терминал    — OBD терминал")
    print("  16. Настройки   — BT + инициализация + калибровки")
    print()
    print("🔧 Исправлено в V6 vs V5:")
    print("  ✅ MAF: вольтаж → g/s через таблицу Hitachi QR20DE")
    print("  ✅ Формула HP: BSFC метод вместо MAF*0.8")
    print("  ✅ Утечка памяти: StreamSubscription в HomeScreen")
    print("  ✅ NissanUnlock: 64-бит маски для Dart")
    print("  ✅ DTC база: P0120, P0121, U1001 добавлены")
    print("  ✅ PatternConfig: сравнение по type вместо ссылки")
    print("  ✅ IOSink: try/catch при записи логов")
    print("  ✅ Скролл: обе вкладки анализатора скроллятся")
    print("  ✅ Калибровка расхода: fuelCorrection в профиле")
    print()
    print("🆕 Новое в V6:")
    print("  ✅ Сплиттер логов (объединить 2+ CSV с усреднением)")
    print("  ✅ Запись правок в .bin → NLP_MOD1_filename.bin")
    print("  ✅ Сравнение двух ROM (оригинал vs мод)")
    print("  ✅ CRUD PID (добавить/изменить/удалить)")
    print("  ✅ Профиль = ВСЕ настройки (PID, калибровки, дашборд, алерты)")
    print("  ✅ 3 паттерна анализа (Мощность / Экономия / Стабильность)")
    print()
    print("🚗 УСТАНОВКА:")
    print("  1. Установи NissanLoggerV6.apk")
    print("  2. Разреши все разрешения")
    print("  3. Настройки → CONNECT ELM327")
    print("  4. ИНИЦИАЛИЗАЦИЯ ЭБУ")
    print("  5. Проверь MAF g/s на ХХ (~2.0-3.5)")
    print("  6. Если MAF не так — подкрути MAF множитель (×0.8-1.2)")
    print("  7. Проверь расход L/ч на ХХ (~0.8-1.2)")
    print("  8. Если расход не так — подкрути калибровку расхода")

else:
    print("\n❌ APK НЕ СОБРАН")
    print()
    print("Последние 100 строк лога:")
    print("=" * 60)
    for line in result[-100:]:
        print(line)
    print()
    print("=" * 60)
    print("Скинь ошибки — пофикшу.")

Java:
openjdk version "17.0.19" 2026-04-21

📦 ЭТАП 1: flutter clean + pub get
Deleting Generated.xcconfig...                                       0ms
Deleting flutter_export_environment.sh...                            0ms
Deleting ephemeral...                                                0ms
+ vibration_platform_interface 0.0.3 (0.1.2 available)
+ vm_service 15.3.0
+ web 1.1.1
+ win32 5.15.0 (6.4.0 available)
+ win32_registry 2.1.0 (3.0.3 available)
+ xdg_directories 1.1.0
+ yaml 3.1.3
Changed 92 dependencies!
23 packages have newer versions incompatible with dependency constraints.
Try `flutter pub outdated` for more information.

✅ pub get OK

🔧 ЭТАП 2: Патч flutter_bluetooth_serial
✅ Bluetooth плагин пропатчен: flutter_bluetooth_serial-0.4.0

📁 ЭТАП 3: Проверка структуры проекта
  ✅ Все 47 файлов на месте

  📊 Всего: 9598 строк Dart, 375.3 KB

🔨 ЭТАП 4: СБОРКА APK V6 (~10-15 минут)

📋 Ключевые события:
Note: /content/.pub-cache/hosted/pub.dev/flutter_bluetooth_serial-0.4.0/andro

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ ЧТО В V6:

📱 16 вкладок:
  1.  Приборы     — настраиваемый дашборд (25 параметров, long-press)
  2.  Графики     — 4 профиля реального времени
  3.  ЛогГраф     — CSV + zoom + статистика + совмещённый
  4.  Лог         — start/stop/автолог/share
  5.  События     — алерты со snapshot + explanation
  6.  DTC         — A330 команда Nissan + русские описания
  7.  Анализатор  — Онлайн + Из лога + СПЛИТТЕР + паттерны
  8.  ЭБУ Карты   — АВТО + UNLOCK + чтение + ЗАПИСЬ В .BIN
  9.  Сервис      — тесты с repeater + обучения
  10. Замер       — 0-60, 0-100, 400м
  11. Экспорт     — WinOLS/ecuEdit/HEX
  12. PID         — CRUD (добавить/удалить кастомные)
  13. Профили     — ВСЕ настройки в профиле
  14. ROM Diff    — СРАВНЕНИЕ двух прошивок
  15. Терминал    — OBD терминал
  16. Настройки   — BT + инициализация + калибровки

🔧 Исправлено в V6 vs V5:
  ✅ MAF: вольтаж → g/s через таблицу Hitachi QR20DE
  ✅ Формула HP: BSFC метод вместо MAF*0.8
  ✅ Утечка памяти: StreamSubscription в HomeScree